# ALUAR — Notebook Maestro (M1–M13) · Única Fuente de Verdad

Notebook autocontenido y modular. Integra ingestión, valuación, motor estocástico,
ratios, biblioteca de gráficos institucionales y exportadores.

**Orden de celdas:** CONFIG → TEMA → RATIOS → M1…M13 → GRÁFICOS (A/B) → EXPORTACIÓN/PIPELINE.

**Cómo lo ejecuta Claude Code:**
1. Proveer `canonical_financials.json` y (si aplica) `static_inputs.json` en `CONFIG['workdir']`.
2. Poner `CONFIG['run_pipeline']=True` y `CONFIG['run_exports']=True`.
3. Ejecutar todas las celdas y llamar `run_all()`.

**Sin hardcodes del PPT.** Parámetros únicos en la celda CONFIG (los marcados ⚠ requieren confirmación).


In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA 1 · CONFIG — ÚNICA FUENTE DE PARÁMETROS, RUTAS E IO UNIFICADO       ║
# ║  Editar SOLO aquí. Ningún módulo define rutas/paths/nombres por su cuenta. ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os, json, warnings, sys
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

CONFIG = {
    # ── Entorno / rutas (Claude Code puede sobreescribir por variable de entorno)
    "workdir":   os.environ.get("ALUAR_WORKDIR", os.getcwd()),
    "figdir":    None,                       # se completa abajo
    "static_inputs": "static_inputs.json",   # datos de Memoria/industria (los provee el usuario)
    "use_frozen_inputs": False,

    # ── Flags de ejecución (Claude Code los pone en True para correr / exportar)
    "run_pipeline": True,
    "run_exports":  True,

    "modelo_b": {
        "active": True,
        "cagr": 0.02,
        "ebit_m_med": 0.20,
        "dnwc_med": 0.15,
        "tax": 0.35,
        "revenue_jump_y1": 0.2133
    },

    # ── SUPUESTOS METODOLÓGICOS (única definición; auditar los marcados ⚠) ─────
    "tax_rate":       0.35,            # Ley 20.628
    "shares_mm":      2800.0,          # H-01 (Auditoria Forense, P0): 2.800.000.000 acciones -- EEFF PwC FY2020-2025 Nota 12 Capital Social + Memoria Anual ("Numero de acciones en circulacion 2.800.000.000"). 1.507,02MM era un dato incorrecto de una fuente CNV desactualizada.
    # Fix (auditoria ronda 5, ALTA -- data lineage): "beta_benchmark" nunca se
    # leia desde ningun modulo (grep exhaustivo del notebook completo, cero
    # resultados) -- el Beta OLS de M1 esta hardcodeado a ^GSPC (metodologia
    # Damodaran global), no a MERVAL como sugeria este default muerto. Se
    # retira el parametro en vez de implementar un branch MERVAL/GSPC que
    # nadie pidio (evita sobre-ingenieria); ver comentario en M1 donde se
    # documenta explicitamente que el benchmark esta fijo a GSPC.
    # Fix (auditoria ronda 5, MEDIA -- tribunal): las advertencias "⚠ vs PPT"
    # de lambda_ar/g_terminal quedaron de una corrida vieja del PPTX -- la
    # auditoria de texto de esta misma ronda confirmo que las slides leen
    # estos valores dinamicamente desde m1_out.json/m5_out.json (sin ningun
    # "0.24"/"0.025" hardcodeado en update_presentation_v10.py, grep
    # exhaustivo sin resultados) -- el PPTX ya esta sincronizado, se retira
    # la advertencia stale.
    "lambda_ar":      0.20,
    "g_terminal":     0.020,
    "vol_bond":       0.20,            # Damodaran default soberano EM
    "embi_source":    "manual_bloomberg_proxy",  # Fix (auditoria jul-11): "BCRA_serie5" era una atribucion falsa -- ver nota de auditoria en get_embi_ar() (M1). No hay fuente API en vivo verificada para EMBI+.
    "erp_us":         0.0418,          # Damodaran ERP implícito jun-2026
    "beta_u_sector":  0.96,            # Damodaran Metals & Mining U.S. unlevered beta (2026)
    "beta_se_default": 0.0874,         # H-05: fallback si m1["beta_se"] viniera None/0 (SE historico OLS ALUA.BA vs ^GSPC)
    # Fix (auditoria jul-11, ALTA): se_prior de vasicek() vivia como
    # literal de funcion (0.25) en M5, rompiendo el patron H-05 que el
    # resto del modulo sigue. Vasicek (1973) exige que sea la varianza
    # cross-sectional de un universo de comparables real -- no se
    # dispone de ese universo completo en este proyecto (un unico beta
    # sectorial de referencia no permite estimar una varianza
    # cross-sectional valida). Se declara explicitamente como supuesto
    # documentado, no derivado empiricamente, centralizado aca.
    "vasicek_se_prior": 0.25,
    # Fix (auditoria jul-11, MEDIA): M7 y M9 calibran el mismo Merton
    # Jump-Diffusion sobre ALUA.BA con fallbacks distintos si el fetch
    # falla (M7: lambda_j=0.27/año: M9 tenia lambda_j=5.0/año, ~18x mas
    # alto, implausible para una accion individual). Se centraliza en un
    # unico lugar (H-05) para que ambos degraden identico.
    "merton_fallback": {"lambda_j": 0.27, "mu_j": -0.014, "sigma_j": 0.08},

    # Fix (auditoria jul-11, ALTA -- respuesta directa a objecion de usuario
    # sobre requerimientos estadisticos/econometricos): la ventana de datos
    # DIARIOS de mercado (ALUA/GGAL/^GSPC/^MERV/TXAR, usada en M1/M7/M8/M9/
    # M11/M12 para beta, Merton, ADF, HAC, Markowitz) estaba hardcodeada en
    # "5y" (~1.187-1.467 obs) en 7 sitios distintos, sin justificacion
    # explicita del largo de ventana. Se investigo empiricamente (jul-11,
    # yfinance, ALUA.BA ajustado a USD via CCL diario 2010-2026, N=3.928
    # retornos) antes de decidir extenderla:
    #   (a) Test de quiebre estructural Quandt-Andrews (sup-F sobre
    #       varianza, trimming 15%) sobre 2010-2026: el quiebre de MAYOR F
    #       es 2020-03-19 (COVID) con F=2.42 -- MUY por debajo de los
    #       valores criticos usuales (~8-12 al 5%) para esta prueba. No hay
    #       evidencia de un quiebre de regimen fuerte y unico que obligue a
    #       truncar la ventana.
    #   (b) La volatilidad diaria estimada es notablemente ESTABLE al variar
    #       el largo de ventana: sigma_d = 3.33%/3.57%/3.50%/3.34%/3.26%
    #       para ventanas de 5/7/8/10/12 anios respectivamente -- extender
    #       la ventana no contamina la estimacion de vol con un regimen
    #       distinto.
    #   (c) El beneficio real es en la IDENTIFICACION DE SALTOS para Merton
    #       (limitacion ya documentada: lambda_j~0.27/año implica muy pocos
    #       eventos observables en una ventana corta): eventos >3-sigma
    #       pasan de 14 (5y) a 33 (10y) -- mas del doble, la mejora
    #       estadistica mas concreta de esta extension.
    # Se fija 10y como ventana estandar (balance entre mas N/mas saltos
    # observables y no diluir en exceso el regimen cambiario vigente desde
    # el "cepo" 2019 con anios pre-2016 no cubiertos por 10y de todos
    # modos). Centralizado aca para que un cambio futuro se haga en un unico
    # lugar en vez de 7 literales "5y" dispersos.
    "market_data_period": "10y",

    # ── Monte Carlo / estocástico ─────────────────────────────────────────────
    "n_sim":   20_000, "df_t": 5, "seed": 42,

    # ── Esquema ÚNICO de nombres de archivo (corrige el pipeline roto) ─────────
    # m3 retirado (auditoria jul-09): M3_run() era codigo muerto (ver celda 10) --
    # dejarlo aca hacia que inject_lineage_metadata() siguiera re-timestampeando
    # el m3_out.json huerfano en cada corrida, aparentando datos "frescos" que
    # en realidad nunca se recalculaban.
    "files": {n: f"m{n[1:]}_out.json" for n in
              ["m1","m2","m4","m5","m6","m7","m8","m9","m10","m11","m12","m13"]},
    "canonical": "canonical_financials.json",
    "mc_npy":    "mc_results.npy",
}
CONFIG["figdir"] = os.path.join(CONFIG["workdir"], "figures")

# Aliases legacy usados por los módulos (apuntan a la MISMA ruta única)
W = WORKDIR = CONFIG["workdir"]
FIGDIR = CONFIG["figdir"]
os.makedirs(CONFIG["figdir"], exist_ok=True)


# ── IO ÚNICO (reemplaza los `def load()` duplicados de todos los módulos) ──────
def load(fname):
    """Carga un JSON desde workdir. Único loader del proyecto."""
    with open(os.path.join(CONFIG["workdir"], fname), encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, fname):
    """Persiste un JSON en workdir. Único writer del proyecto."""
    path = os.path.join(CONFIG["workdir"], fname)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path


def inject_lineage_metadata():
    """Agrega _meta (timestamp, seed, git_hash, modulo) a cada m*_out.json
    tras run_all() -- trazabilidad (Mejora Sec.12 Auditoria Forense)."""
    import datetime as _dt2
    ts = _dt2.datetime.now().isoformat()
    for name, fname in CONFIG["files"].items():
        path = os.path.join(CONFIG["workdir"], fname)
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        d["_meta"] = {
            "timestamp": ts,
            "seed_stochastic": CONFIG.get("seed"),
            "git_hash": None,  # proyecto no versionado con git
            "modulo": name,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(d, f, ensure_ascii=False, indent=2)


def load_static(key=None, default=None):
    """Lee datasets estáticos (energía, curva de costos, market share, peers)
    desde static_inputs.json provisto por el usuario. NO se fabrican números."""
    p = os.path.join(CONFIG["workdir"], CONFIG["static_inputs"])
    if not os.path.exists(p):
        return default
    with open(p, encoding="utf-8") as f:
        data = json.load(f)
    return data.get(key, default) if key else data


def safe_div(a, b, default=np.nan):
    """División protegida para ratios."""
    try:
        a = float(a); b = float(b)
        return a / b if b not in (0, 0.0) else default
    except (TypeError, ValueError):
        return default


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA 2 · IDENTIDAD VISUAL — apply_aluar_theme() ÚNICO PARA TODO GRÁFICO  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import datetime as _dt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib import font_manager as fm
import matplotlib.dates as mdates

_PREFERRED = ["Liberation Sans", "Arial", "DejaVu Sans"]
_AVAIL = {f.name for f in fm.fontManager.ttflist}
_FONT_FAMILY = next((f for f in _PREFERRED if f in _AVAIL), "DejaVu Sans")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MASTER_STYLE — Gold Standard: ÚNICA fuente de verdad visual del deck.     ║
# ║  Colores, tipografía y grosores de línea. Ningún gráfico define su propio ║
# ║  color/lw suelto: todos importan MASTER_STYLE (directo o vía los alias   ║
# ║  C / SZ / LW / FONT / FIGSIZE, que son referencias al mismo dict, no      ║
# ║  copias — cambiar MASTER_STYLE cambia los ~27 gráficos a la vez).         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
MASTER_STYLE = {
    "colors": {
        "navy": "#0B2545", "blue": "#13497B", "blue_lt": "#9DB8D2",
        "grid": "#E4E9F0", "ink": "#1A1A1A", "muted": "#6B7280",
        "value": "#1B7F4B",   # verde — SOLO creación de valor / upside
        "risk":  "#B11226",   # rojo  — SOLO riesgo / downside
        "aluar": "#E8833A",   # naranja — SOLO destacar ALUAR
        "panel": "#FFFFFF",
    },
    "font": {
        "family": _FONT_FAMILY,
        "sizes": {"title": 15, "subtitle": 11, "axis": 10.5, "tick": 9.5,
                  "legend": 9.5, "annot": 9, "source": 8},
    },
    "linewidth": {
        "hairline": 0.7,  # trayectorias individuales de Monte Carlo/OU (fondo, muchas a la vez)
        "thin":     0.8,  # ejes cero, spines, separadores discretos
        "light":    1.0,  # separadores de sección
        "medium":   1.2,  # divisores histórico/proyección, referencias secundarias
        "regular":  1.6,  # series de datos estándar (no protagonistas)
        "bold":     2.0,  # referencias clave (WACC, precio de mercado, promedios)
        "heavy":    2.4,  # serie protagonista del gráfico (ALUA, media de trayectorias)
    },
    "figsize": (11.0, 6.6),
}

# Alias de acceso directo (mismo objeto, no copia) — el código existente de
# los ~27 gráficos ya usa C[...]/SZ[...]; con estos alias queda importando
# MASTER_STYLE de forma transparente, sin tocar cada función.
C        = MASTER_STYLE["colors"]
FONT     = MASTER_STYLE["font"]["family"]
SZ       = MASTER_STYLE["font"]["sizes"]
LW       = MASTER_STYLE["linewidth"]
FIGSIZE  = MASTER_STYLE["figsize"]

# Mapeo de claves legacy de los módulos → paleta única (sin colores nuevos).
PALETTE = {
    "alua": C["navy"], "primary": C["navy"], "merv": C["risk"], "secondary": C["risk"],
    "txar": C["value"], "accent": C["value"], "warning": C["aluar"], "neutral": C["muted"],
    "light": C["panel"], "paths": C["navy"], "ou": C["risk"], "merton": C["value"],
}

MARG = dict(left=0.085, right=0.955, top=0.80, bottom=0.190)
TITLE_Y, SUB_Y, SRC_Y = 0.955, 0.895, 0.022
_TODAY = _dt.date.today().strftime("%d-%b-%Y")


def apply_aluar_theme():
    """Configuración matplotlib ÚNICA. Llamada por todos los gráficos.
    Lee exclusivamente de MASTER_STYLE — ver bloque arriba."""
    plt.rcParams.update({
        "figure.facecolor": "white", "savefig.facecolor": "white",
        "axes.facecolor": C["panel"], "font.family": FONT,
        "axes.edgecolor": C["muted"], "axes.linewidth": LW["thin"],
        "axes.grid": True, "axes.grid.axis": "y",
        "grid.color": C["grid"], "grid.linewidth": 0.9,
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "xtick.color": C["ink"], "ytick.color": C["ink"],
        "xtick.labelsize": SZ["tick"], "ytick.labelsize": SZ["tick"],
        "text.color": C["ink"], "figure.dpi": 110,
    })


apply_aluar_theme()  # aplicar al cargar el notebook


def scaffold(title, subtitle, unit, source, note="", figsize=FIGSIZE):
    """Lienzo estándar: título-conclusión + subtítulo + unidad + fuente + fecha + nota."""
    apply_aluar_theme()
    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(**MARG)
    ax = fig.add_subplot(111)
    fig.text(MARG["left"], TITLE_Y, title, ha="left", va="top",
             fontsize=SZ["title"], fontweight="bold", color=C["navy"])
    fig.text(MARG["left"], SUB_Y, subtitle, ha="left", va="top",
             fontsize=SZ["subtitle"], color=C["muted"])
    if unit:
        fig.text(MARG["right"], SUB_Y, unit, ha="right", va="top",
                 fontsize=SZ["subtitle"], style="italic", color=C["muted"])
    foot = f"Fuente: {source}.  Elaboración propia - {_TODAY}."
    if note:
        foot += f"  Nota: {note}"
    fig.text(MARG["left"], SRC_Y, foot, ha="left", va="bottom",
             fontsize=SZ["source"], color=C["muted"])
    return fig, ax


def _money_fmt(v, _pos=None, prefix="$"):
    """Formateador financiero dinámico: abrevia K/M/B según la magnitud real
    del valor (no según una escala asumida de antemano), para que el mismo
    formateador sirva sin cambios si los datos pasan de cientos a millones
    o miles de millones. Prefijo "$" SIEMPRE presente por defecto."""
    av = abs(v)
    sign = "-" if v < 0 else ""
    if av >= 1e9:
        s = f"{av/1e9:,.1f}B"
    elif av >= 1e6:
        s = f"{av/1e6:,.1f}M"
    elif av >= 1e3:
        s = f"{av/1e3:,.1f}K"
    else:
        s = f"{av:,.0f}"
    return f"{sign}{prefix}{s}"


def _clean_y(ax, pct=False, money=False, decimals=0, prefix="$"):
    if pct:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=decimals))
    elif money:
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: _money_fmt(v, p, prefix)))
    ax.tick_params(length=0); ax.margins(x=0.02)


def export(fig, name, outdir=None, formats=("png", "svg", "pdf")):
    """Export multiformato (PNG 300dpi + SVG vector + PDF). Devuelve rutas."""
    outdir = outdir or CONFIG["figdir"]
    os.makedirs(outdir, exist_ok=True)
    paths = {}
    for ext in formats:
        p = os.path.join(outdir, f"{name}.{ext}")
        fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
        paths[ext] = p
    plt.close(fig)
    return paths


# Compat: los módulos llaman save_fig(fig, name); se redirige al export único.
def save_fig(fig, name):
    return export(fig, name)["png"]


class Inputs:
    """Acceso único a los outputs del notebook (m1..m13 + canonical). Sin hardcode."""
    def __init__(self, data, provenance="pipeline"):
        self.data, self.provenance = data, provenance

    @classmethod
    def from_outputs(cls, workdir=None):
        workdir = workdir or CONFIG["workdir"]
        data, missing = {}, []
        for key, fn in {**CONFIG["files"], "can": CONFIG["canonical"]}.items():
            p = os.path.join(workdir, fn)
            if os.path.exists(p):
                with open(p, encoding="utf-8") as f:
                    data[key] = json.load(f)
            else:
                missing.append(fn)
        if missing:
            print("[AVISO] Faltan outputs (ejecutar run_all primero):", ", ".join(missing))
        return cls(data, f"outputs:{workdir}")

    def get(self, *path, default=None):
        cur = self.data
        for p in path:
            if isinstance(cur, dict) and p in cur:
                cur = cur[p]
            else:
                return default
        return cur


In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA 3 · MOTOR DE RATIOS — compute_all_ratios() (todo automático)        ║
# ║  Nunca se ingresan ratios a mano. Se derivan de los estados financieros.   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def nz(x, default=0.0):
    """(x or default) NO es NaN-safe: NaN es truthy en Python, asi que
    `nan or 0` sigue devolviendo nan y "envenena" sumas/restas encadenadas
    (ver ciclo_caja). nz() solo sustituye None (dato ausente) por `default`;
    si x YA es NaN (dato corrupto), lo deja pasar sin blanquear, para que
    _find_non_finite()/validate_financial_arrays() lo atrape aguas abajo
    en vez de quedar silenciosamente "sano" con un 0 que no corresponde."""
    return default if x is None else x


def compute_all_ratios(fy: dict, prev: dict = None, market: dict = None) -> dict:
    """
    Calcula el set completo de ratios de Equity Research / Dumrauf para UN ejercicio.
    `fy`     : dict del año (estados financieros canónicos).
    `prev`   : dict del año anterior (para promedios y variaciones).
    `market` : dict opcional con precio, market cap, EV, dividendos (ratios bursátiles).
    Campos ausentes → NaN (safe_div). Devuelve dict por categoría.
    """
    g = fy.get
    prev = prev or {}
    pg = prev.get

    # Fix (auditoria ronda 4, ALTA -- descubierto al verificar el fix de
    # cxc_ars/dias_cobranza de esta misma ronda): "avg(k)" leia la clave SIN
    # sufijo ("total_activo"/"inventarios"), que NUNCA existe en
    # canonical_financials.json (solo "_ars"/"_usdmm") -- avg() devolvia NaN
    # SIEMPRE, para el anio actual Y el anterior, dejando rotacion_activos/
    # rotacion_inventario en None para 5 de 6 anios (todos salvo el primero,
    # que no entra a la rama avg() por no tener prev). Se resuelve el valor
    # del anio anterior con la MISMA cadena de fallback ARS->USD MM que ya
    # usa el anio actual (activo/inv, ver mas abajo), en vez de una clave que
    # nunca existio.
    _ccl_prev = pg("ccl_cierre")
    def _to_usdmm_prev(v_ars):
        return (v_ars / (_ccl_prev * 1e6)) if (v_ars is not None and _ccl_prev) else v_ars
    _activo_prev = pg("total_activo", pg("total_activo_usdmm", _to_usdmm_prev(pg("total_activo_ars"))))
    _inv_prev    = _to_usdmm_prev(pg("inventarios", pg("inventarios_ars")))

    def avg2(cur, prev_val):
        """Promedio (cur, prev_val) si AMBOS estan disponibles; si no, solo cur
        (nunca NaN por un prev ausente -- mas util que propagar NaN)."""
        return (cur + prev_val) / 2 if (prev and prev_val is not None and cur is not None) else cur

    act_cte   = g("activo_corriente", g("activo_corriente_ars"))
    pas_cte   = g("pasivo_corriente", g("pasivo_corriente_ars"))
    inv       = g("inventarios", g("inventarios_ars"))
    caja      = g("caja", g("efectivo_ars", g("caja_ars")))
    # Fix (auditoria ronda 4, ALTA -- descubierto al verificar el fix de
    # cxc_ars/liquidez de esta misma ronda): cxc/cxp se leian en ARS crudos
    # sin convertir, mientras dias_cobranza/dias_pago los multiplican por
    # 365 y dividen por ventas/cogs (USD MM) -- mezcla de unidades que daba
    # un DSO de miles de millones de "dias". Se convierte a USD MM via
    # ccl_cierre cuando solo existe el campo _ars (mismo patron ya usado
    # para total_patrimonio_usdmm).
    _ccl_fy   = g("ccl_cierre")
    def _to_usdmm(v_ars):
        return (v_ars / (_ccl_fy * 1e6)) if (v_ars is not None and _ccl_fy) else v_ars
    cxc       = _to_usdmm(g("cuentas_por_cobrar", g("cxc_ars")))
    cxp       = _to_usdmm(g("cuentas_por_pagar", g("cxp_ars")))
    # inv (arriba) se mantiene en ARS crudos para liquidez_acida (consistente
    # con act_cte/pas_cte, misma moneda). inv_usdmm es SOLO para
    # rotacion_inventario/dias_inventario, que se combinan con cogs (USD MM)
    # -- mismo fix de unidades que cxc/cxp arriba.
    inv_usdmm = _to_usdmm(inv)
    activo    = g("total_activo", g("total_activo_usdmm", g("total_activo_ars")))
    # Fix (auditoria ronda 5, ALTA -- excel sync): "total_pasivo_usdmm" no
    # existe en canonical_financials.json para NINGUN anio (solo el _ars
    # crudo) -- se derivaba pasivo=total_pasivo_ars sin convertir, mezclado
    # luego contra activo/ebit/ebitda (USD MM) en pasivo_activo/
    # solvencia_total/cobertura_*, dando cifras sin sentido fisico (ej.
    # pasivo_activo FY2025 = 624.367.329x en vez de ~0.30-0.70x). Mismo
    # patron ya usado para total_patrimonio_usdmm: derivar via ccl_cierre
    # cuando falta el campo directo.
    pasivo_usdmm = g("total_pasivo_usdmm")
    if pasivo_usdmm is None:
        pasivo_usdmm = _to_usdmm(g("total_pasivo_ars"))
    pasivo    = g("total_pasivo", pasivo_usdmm)
    # Fix (auditoria jul-09): preferia "_ars" con "_usdmm" como fallback --
    # invertido, mismo criterio que ebit/ebitda/ventas/capex/deuda_neta (evita
    # mezclar ARS con USD MM en p_b, apalancamiento, etc. cuando se provee
    # market data en USD MM).
    # Fix (auditoria ronda 4, ALTA): total_patrimonio_usdmm no existe en
    # FY2020-FY2021 (solo el _ars crudo) -- el fallback devolvia patrimonio
    # en pesos crudos esos 2 anios mientras el resto de los campos (rdo_neto,
    # ebit, etc.) ya resuelven en USD MM, rompiendo ROE/D-E/P-B por mezcla de
    # unidades. Se deriva usdmm=ars/(ccl_cierre*1e6) cuando falta el campo
    # directo -- formula verificada exacta contra los anios que SI traen
    # ambos campos (FY2022-2025: ars/(ccl*1e6) == usdmm hasta redondeo).
    pn_usdmm = g("total_patrimonio_usdmm")
    if pn_usdmm is None:
        pn_ars, ccl_fy = g("total_patrimonio_ars"), g("ccl_cierre")
        pn_usdmm = (pn_ars / (ccl_fy * 1e6)) if (pn_ars is not None and ccl_fy) else None
    pn        = g("total_patrimonio", pn_usdmm if pn_usdmm is not None else g("total_patrimonio_ars"))
    deuda_fin = g("deuda_fin", g("deuda_fin_usdmm", g("deuda_fin_ars", g("deuda_fin_total_ars"))))
    deuda_net = g("deuda_neta", g("deuda_neta_usdmm", g("deuda_neta_ars")))
    ventas    = g("ventas", g("ventas_netas_usdmm", g("ventas_netas_ars")))
    # Fix (auditoria ronda 4, CRITICA): "costo_ventas"/"cogs_ars" no existen en
    # canonical_financials.json (la clave real es "costo_ventas_ars"/
    # "costo_ventas_usdmm") -- cogs resolvia SIEMPRE a None, y margen_bruto
    # (ventas-0)/ventas daba 100% todos los anios. Alineado al mismo criterio
    # USD-MM-preferido que ebit/ebitda/ventas/capex/deuda_neta.
    cogs      = g("costo_ventas", g("costo_ventas_usdmm", g("costo_ventas_ars")))
    ebit      = g("ebit", g("ebit_usdmm"))
    ebitda    = g("ebitda", g("ebitda_usdmm"))
    da        = g("da", g("da_usdmm"))
    # Fix (auditoria ronda 5, ALTA -- excel sync): "intereses_pagados_ars"/
    # "gasto_financiero_ars" son ARS crudos sin contraparte _usdmm -- se
    # asignaban directo a int_pag sin convertir, mezclado luego contra
    # ebit/ebitda (USD MM) en cobertura_intereses/cobertura_ebitda (mismo
    # bug de unidades que pasivo, ver nota arriba). Verificado FY2024/2025:
    # ebit_usdmm/int_pag_usdmm converge exacto con el campo ya presente en
    # canonical "interest_coverage" (11.63x/3.53x), confirmando la formula.
    int_pag   = g("intereses", _to_usdmm(g("intereses_pagados_ars", g("gasto_financiero_ars"))))
    nopat     = g("nopat", g("nopat_usdmm"))
    # Fix (auditoria jul-09): rdo_neto/fco/ic preferian ARS con USD MM ignorado
    # (aunque resultado_neto_usdmm/fco_usdmm/capital_invertido_usdmm existen en
    # canonical_financials.json) -- p_e mezclaba mkt_cap(USD MM)/rdo_neto(ARS)
    # dando 1.3e-07x en vez de ~200x; fcf_yield_mkt mezclaba fco(ARS)-capex(USD
    # MM) dando millones en vez de una fraccion pequena. Alineado a USD MM,
    # mismo criterio que ebit/ebitda/ventas/capex/deuda_neta/nopat.
    rdo_neto  = g("resultado_neto", g("resultado_neto_usdmm", g("resultado_neto_ars")))
    fco       = g("fco", g("fco_usdmm", g("fco_ars", g("fco_cfs_ars"))))
    capex     = g("capex", g("capex_usdmm", g("capex_ars")))
    ic        = g("capital_invertido", g("capital_invertido_usdmm", g("capital_invertido_ars")))
    # Fix (auditoria ronda 4, BAJA-MEDIA): CONFIG.get("_wacc_ref") era un
    # fallback muerto -- ningun modulo escribe esa clave en CONFIG, asi que
    # siempre resolvia a None. El wacc para spread_roic_wacc/eva viene
    # exclusivamente del dict `market` que provee el caller (M4/M13).
    wacc      = (market or {}).get("wacc")

    if act_cte is None and pas_cte is None:
        # canonical_financials.json no trae activo_corriente/pasivo_corriente
        # para ningun anio -- la categoria "liquidez" es 100% NaN por
        # ausencia de dato, no por un bug. Se avisa una sola vez por corrida
        # (guard en CONFIG) en vez de fallar en silencio.
        if not CONFIG.get("_liquidez_nan_avisado"):
            print("[AVISO] Categoria 'liquidez' 100% NaN: activo_corriente/"
                  "pasivo_corriente no existen en canonical_financials.json.")
            CONFIG["_liquidez_nan_avisado"] = True

    liquidez = {
        "liquidez_corriente": safe_div(act_cte, pas_cte),
        "liquidez_acida":     safe_div(nz(act_cte) - nz(inv), pas_cte),
        "liquidez_caja":      safe_div(caja, pas_cte),
        "capital_trabajo":    nz(act_cte, np.nan) - nz(pas_cte, np.nan),
    }
    # NOTA (auditoria jul-09): deuda_fin/activo tambien preferian ARS con
    # USD MM disponible sin usar -- alineados abajo. pasivo/int_pag ya se
    # convierten a USD MM arriba (ver notas de pasivo_usdmm/int_pag).
    solvencia = {
        "deuda_pn":            safe_div(deuda_fin, pn),
        "deuda_activo":        safe_div(deuda_fin, activo),
        "pasivo_activo":       safe_div(pasivo, activo),
        "deuda_neta_ebitda":   safe_div(deuda_net, ebitda),
        "cobertura_intereses": safe_div(ebit, int_pag),
        "cobertura_ebitda":    safe_div(ebitda, int_pag),
        "solvencia_total":     safe_div(activo, pasivo),
    }
    # Fix (auditoria ronda 4, ALTA): costo_ventas_ars/_usdmm se guarda NEGATIVO
    # en canonical_financials.json (ej. FY2020 costo_ventas_usdmm=-1068.18) --
    # la formula correcta de margen bruto con ese signo es ventas+cogs (sumar
    # el costo ya negativo), no ventas-cogs (que restaria dos veces el costo).
    # Verificado: FY2020 (ventas+cogs)/ventas = 13.64%, matchea el margen
    # bruto real reportado en los EEFF.
    margen_bruto_num = (ventas or 0) + (cogs or 0)
    # Fix (auditoria ronda 4, ALTA): pasivo_corriente no existe en NINGUN anio
    # de canonical_financials.json -- "activo - pasivo_corriente" siempre
    # degradaba a "activo total" (roce = ebit/activo, un ROA disfrazado sin
    # documentar). Capital Empleado se aproxima con la identidad estandar
    # Capital Empleado = Patrimonio + Deuda con costo (deuda_fin), that no
    # depende de la particion corriente/no-corriente del pasivo.
    capital_empleado = nz(deuda_fin) + nz(pn)
    rentabilidad = {
        "margen_bruto":  safe_div(margen_bruto_num, ventas),
        "margen_ebitda": safe_div(ebitda, ventas),
        "margen_ebit":   safe_div(ebit, ventas),
        "margen_neto":   safe_div(rdo_neto, ventas),
        "roe":           safe_div(rdo_neto, pn),
        "roa":           safe_div(rdo_neto, activo),
        "roic":          g("roic", safe_div(nopat, ic)),
        "roce":          safe_div(ebit, capital_empleado if capital_empleado else None),
    }
    eficiencia = {
        "rotacion_activos":     safe_div(ventas, avg2(activo, _activo_prev)),
        # Fix (auditoria ronda 4, ALTA -- descubierto al verificar el fix de
        # unidades de arriba): costo_ventas_ars/_usdmm se guarda NEGATIVO
        # (mismo signo ya documentado en margen_bruto) -- sin abs(),
        # rotacion_inventario/dias_inventario daban negativos (-167 a -313
        # "dias"), fisicamente sin sentido para un ratio de rotacion.
        "rotacion_inventario":  safe_div(abs(cogs) if cogs is not None else cogs, avg2(inv_usdmm, _inv_prev)),
        "dias_inventario":      safe_div(365, safe_div(abs(cogs) if cogs is not None else cogs, inv_usdmm)),
        "dias_cobranza":        safe_div(nz(cxc) * 365, ventas),
        # abs(cogs): mismo fix de signo que rotacion_inventario/dias_inventario
        # arriba (costo_ventas se guarda negativo) -- hoy inerte (cxp siempre
        # None -> nz=0 -> 0 independiente del signo), pero deja la formula
        # correcta si cxp_ars se agregara a canonical a futuro.
        "dias_pago":            safe_div(nz(cxp) * 365, abs(cogs) if cogs is not None else cogs),
    }
    # Fix (auditoria ronda 4, BAJA-MEDIA): "x or 0" no es NaN-safe (nz() si) --
    # un dias_inventario/cobranza/pago NaN real ya no se blanquea a 0 y se
    # propaga, en vez de "envenenar" ciclo_caja con un cero que no corresponde.
    eficiencia["ciclo_caja"] = (nz(eficiencia["dias_inventario"])
                                + nz(eficiencia["dias_cobranza"])
                                - nz(eficiencia["dias_pago"]))
    apalancamiento = {
        "d_e":               safe_div(deuda_fin, pn),
        "leverage_activo_pn": safe_div(activo, pn),
        "deuda_capital":     safe_div(deuda_fin, nz(deuda_fin) + nz(pn)),
    }
    # Fix (auditoria ronda 5, ALTA -- excel sync): capex se guarda NEGATIVO en
    # canonical_financials.json (mismo patron que cogs, ver nota de
    # margen_bruto abajo) -- restar un capex ya negativo lo SUMA en vez de
    # netearlo. Verificado FY2025: fco=30.74, capex=-243.93 -> con "-" daba
    # fcf=+274.67 (contradice fcff_usdmm=-196.18 del propio canonical, que
    # documenta "FCFF negativo por CAPEX masivo, valido Penman, no truncar");
    # con "+" da fcf=-213.19, consistente con esa nota.
    cash_flow = {
        "fco_ventas":        safe_div(fco, ventas),
        "fcf":               nz(fco, np.nan) + nz(capex, np.nan),
        "capex_ventas":      safe_div(capex, ventas),
        "cash_conversion":   safe_div(fco, ebitda),
        "fcf_yield":         safe_div(nz(fco) + nz(capex),
                                      (market or {}).get("mkt_cap", np.nan)),
    }
    creacion_valor = {
        "spread_roic_wacc": (rentabilidad["roic"] - wacc) if (rentabilidad["roic"] is not None and wacc) else np.nan,
        "eva":              ((nz(rentabilidad["roic"], np.nan) - nz(wacc, np.nan)) * nz(ic, np.nan))
                            if wacc else np.nan,
    }
    m = market or {}
    bursatiles = {
        "p_e":        safe_div(m.get("mkt_cap"), rdo_neto),
        "p_b":        safe_div(m.get("mkt_cap"), pn),
        "ev_ebitda":  safe_div(m.get("ev"), ebitda),
        "ev_sales":   safe_div(m.get("ev"), ventas),
        "ev_ebit":    safe_div(m.get("ev"), ebit),
        "dividend_yield": safe_div(m.get("dividendos"), m.get("mkt_cap")),
        "fcf_yield_mkt":  cash_flow["fcf_yield"],
    }
    return {
        "liquidez": liquidez, "solvencia": solvencia, "rentabilidad": rentabilidad,
        "eficiencia": eficiencia, "apalancamiento": apalancamiento,
        "cash_flow": cash_flow, "creacion_valor": creacion_valor, "bursatiles": bursatiles,
    }


def ratios_timeseries(canonical: dict, years, market_by_year: dict = None) -> dict:
    """Aplica compute_all_ratios a toda la serie histórica de forma automática."""
    out, market_by_year = {}, market_by_year or {}
    yrs = sorted(years)
    for i, yr in enumerate(yrs):
        fy = canonical.get(f"FY{yr}", canonical.get(str(yr), {}))
        prev = canonical.get(f"FY{yrs[i-1]}", {}) if i > 0 else None
        out[str(yr)] = compute_all_ratios(fy, prev, market_by_year.get(str(yr)))
    return out


## Módulos M1–M13 (computación)

### M1

In [4]:
"""
module1_v3.py — Ingestion de Datos de Mercado
===============================================
Parámetros corregidos según Auditoría Institucional:
- ERP: Damodaran junio-2026 = 4.18% (antes 4.72% fallback)
- EMBI: JPMorgan EMBI+ proxy = 4.41% (antes 5.5% fallback)
- rf: US10Y live desde ^TNX (antes 4.372%)
- CCL: GGAL.BA × 10 / GGAL (formula corregida v2)
- Beta OLS: ALUA.BA vs ^GSPC 5y daily (1,189 obs.)
- LME Aluminum: ALI=F desde yfinance

Nota (auditoria ronda 4): PBI/inflacion/TC de Argentina NO se calculan en
este modulo -- viven en static_inputs.json (macro_ar) y los consume
directamente build_chart_inputs()/M13. La linea anterior de este docstring
prometia que M1 los computaba, lo cual era falso (nunca hubo codigo para
eso aca) y podia inducir a buscarlos en el lugar equivocado.

FUENTE JERARQUÍA (Dumrauf + Damodaran):
1. yfinance (live): rf, CCL, Beta, LME, MERVAL
2. Damodaran website (cached/manual): ERP = 4.18% jun-2026
3. JPMorgan EMBI+ proxy (Bloomberg cached): EMBI = 4.41%
4. FALLBACK DECLARADO: ver campo *_fuente y *_fallback
"""
import json, os, sys, warnings, logging
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats as scipy_stats
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass


# ── CONSTANTES METODOLÓGICAS (AUDITADAS) ──────────────────────────────────────
# Damodaran "Implied Equity Risk Premium" USA (actualizado mensualmente por
# Damodaran). Fuente: pages.stern.nyu.edu/~adamodar/ -> Implied ERP page.
# H-05: unica fuente CONFIG.
# Fix (auditoria jul-11, MEDIA): ERP_FUENTE quedaba hardcodeado a "junio 2026"
# sin ningun mecanismo de actualizacion -- confirmado en una corrida de julio
# (m1_out.json._meta.timestamp=2026-07-10), erp_fuente seguia diciendo "junio
# 2026". Se construye dinamicamente con el mes/anio real de la corrida.
_MESES_ES_ERP = {1:"enero",2:"febrero",3:"marzo",4:"abril",5:"mayo",6:"junio",
                  7:"julio",8:"agosto",9:"septiembre",10:"octubre",11:"noviembre",12:"diciembre"}
_now_erp = __import__("datetime").datetime.now()
ERP_US_DAMODARAN = CONFIG["erp_us"]
ERP_FUENTE = f"Damodaran (NYU Stern), ERP implícito USA, {_MESES_ES_ERP[_now_erp.month]}-{_now_erp.year}"

# EMBI+ Argentina -- NO existe una API publica gratuita que publique el
# spread EMBI+ de JPMorgan en vivo (verificado, ver nota de auditoria abajo).
# Fuente unica: snapshot manual documentado (Bloomberg/Reuters proxy), a
# actualizar periodicamente a mano.
# Fix (auditoria jul-11, ALTA -- verificado en vivo contra la API del BCRA):
# la rama de fetch anterior llamaba a
# "https://api.bcra.gob.ar/estadisticas/v2.0/datosvariable/5/..." y el
# comentario/CONFIG["embi_source"] afirmaban que esa serie "replica el spread
# EMBI+ de JPMorgan". Ambas cosas son falsas: (1) el endpoint v2.0 esta
# deprecado (HTTP 410, verificado 2026-07-11); (2) incluso si respondiera, el
# catalogo v4.0 vigente confirma que idVariable=5 es "Tipo de cambio
# mayorista de referencia" (~1490 ARS/USD), NO una prima de riesgo pais -- el
# BCRA nunca publico un EMBI+ ahi. El chequeo de plausibilidad que existia
# (0.01<=embi<=0.20) no distinguia esto: 1490/10000=0.149 pasaria el filtro
# igual, inyectando silenciosamente un "14.9% de riesgo pais" fabricado a
# partir del tipo de cambio si alguien alguna vez "arreglara" la URL muerta
# apuntando a v3/v4 sin revisar la semantica de la variable. Se elimina la
# rama de fetch (apuntaba a la variable equivocada, no a un problema de
# disponibilidad transitoria) y se declara el snapshot manual como unica
# fuente.
def get_embi_ar() -> tuple:
    """EMBI+ Argentina -- snapshot manual documentado (Bloomberg/Reuters
    proxy). Sin fetch en vivo: no existe una API publica gratuita verificada
    que publique el EMBI+ de JPMorgan (ver nota de auditoria arriba)."""
    return 0.0441, "JPMorgan EMBI+ Argentina (proxy Bloomberg/Reuters) -- snapshot manual, actualizar periodicamente"

# Lambda Damodaran = fracción de ingresos sensibles a mercado doméstico. H-03: se
# intenta leer segmentacion real de static_inputs; si no existe, se usa el fallback
# documentado (CONFIG['lambda_ar']) con logging.warning() explicito -- NO se fabrica el dato.
def get_lambda_ar():
    """Fix (auditoria ronda 4, ALTA): el fallback estaba hardcodeado a 0.20
    directamente aca, ignorando CONFIG['lambda_ar'] -- editar CONFIG (la
    "unica fuente" declarada en su propio encabezado) no tenia NINGUN efecto
    sobre el lambda efectivamente usado. Ahora el fallback lee CONFIG['lambda_ar']."""
    ventas_dom_pct = load_static("ventas_domesticas_pct")
    if ventas_dom_pct is not None:
        return float(ventas_dom_pct), "Damodaran Lambda — ventas_domesticas_pct (static_inputs.json)"
    lambda_cfg = CONFIG["lambda_ar"]
    logging.warning(f"H-03: static_inputs['ventas_domesticas_pct'] no disponible -- "
                     f"usando CONFIG['lambda_ar']={lambda_cfg} (unica fuente editable, celda CONFIG)")
    return lambda_cfg, f"Damodaran Lambda — CONFIG['lambda_ar']={lambda_cfg} (FALLBACK documentado, editar en celda CONFIG)"

TAX_RATE = CONFIG["tax_rate"]  # H-05: unica fuente CONFIG

SHARES_MM = CONFIG["shares_mm"]  # H-01/H-05: unica fuente CONFIG (2.800.000.000 acciones)
SHARES_FUENTE = "Registro CNV, Resolución General — acciones ordinarias en circulación"


def get_rf() -> tuple:
    """US Treasury 10Y — yfinance ^TNX"""
    try:
        data = yf.download("^TNX", period="5d", progress=False, auto_adjust=True)
        if not data.empty:
            close = data["Close"].squeeze()
            rf_raw = float(close.dropna().iloc[-1]) / 100  # ^TNX cotiza en %
            if 0.02 <= rf_raw <= 0.08:
                return rf_raw, "^TNX yfinance — US10Y último cierre"
    except Exception as e:
        logging.warning(f"get_rf(): fetch ^TNX fallo ({e}) -- usando fallback documentado.")
    # Fix (auditoria jul-11, BAJA): este fallback (0.0441) coincidia bit-a-bit
    # con el fallback de get_embi_ar() (0.0441) pese a ser dos variables sin
    # relacion economica (yield UST10Y vs. spread de riesgo pais argentino) --
    # sugeria origen copy-paste. Son coincidentemente similares por la
    # cercania historica de ambos valores en el snapshot de referencia
    # original, no porque compartan fuente: este es independiente, tomado de
    # TP_Valuation_FINAL3_v10 (hoja Supuestos). Se aclara explicitamente y se
    # marca para refresco periodico.
    return 0.0441, "US10Y fallback (Excel TP_Valuation_FINAL3_v10, hoja Supuestos) — ^TNX no disponible; refrescar periodicamente"


def get_ccl() -> tuple:
    """CCL = GGAL.BA × 10 / GGAL"""
    try:
        for days in ["5d", "10d", "30d"]:
            ggal_ba = yf.download("GGAL.BA", period=days, progress=False, auto_adjust=True)
            ggal    = yf.download("GGAL",    period=days, progress=False, auto_adjust=True)
            if not ggal_ba.empty and not ggal.empty:
                ba_close = float(ggal_ba["Close"].squeeze().dropna().iloc[-1])
                us_close = float(ggal["Close"].squeeze().dropna().iloc[-1])
                if us_close > 0:
                    ccl = ba_close * 10.0 / us_close
                    if 500 <= ccl <= 6000:
                        return ccl, f"GGAL.BA={ba_close:.2f}×10 / GGAL={us_close:.2f} = {ccl:.2f}"
    except Exception as e:
        logging.warning(f"get_ccl(): fetch GGAL.BA/GGAL fallo ({e}) -- probando fallback LOMA.")
    # Fallback cadena: LOMA.BA/LOMA
    try:
        for days in ["5d", "10d", "30d"]:
            loma_ba = yf.download("LOMA.BA", period=days, progress=False, auto_adjust=True)
            loma    = yf.download("LOMA",    period=days, progress=False, auto_adjust=True)
            if not loma_ba.empty and not loma.empty:
                ba_close = float(loma_ba["Close"].squeeze().dropna().iloc[-1])
                us_close = float(loma["Close"].squeeze().dropna().iloc[-1])
                if us_close > 0:
                    ccl = ba_close * 10.0 / us_close
                    if 500 <= ccl <= 6000:
                        return ccl, f"FALLBACK LOMA.BA={ba_close:.2f}×10/LOMA={us_close:.2f}={ccl:.2f}"
    except Exception as e:
        logging.warning(f"get_ccl(): fetch LOMA.BA/LOMA fallo ({e}) -- usando ultimo CCL conocido.")
    return 1543.93, "FALLBACK: último CCL conocido (sesión anterior) — APIs no disponibles"


def get_alua_price() -> float:
    """Precio ALUA.BA últimas sesiones"""
    try:
        data = yf.download("ALUA.BA", period="5d", progress=False, auto_adjust=True)
        if not data.empty:
            return float(data["Close"].squeeze().dropna().iloc[-1])
    except Exception as e:
        logging.warning(f"get_alua_price(): fetch ALUA.BA fallo ({e}) -- devuelve None (caller usa fallback 991.0).")
    return None


def get_beta_ols() -> tuple:
    """Beta OLS ALUA.BA vs ^GSPC — 5 años diarios (metodología Damodaran global).
    Fix (auditoria jul-11, MEDIA): ALUA.BA (ARS) se regresionaba directo
    contra ^GSPC (USD) sin homogeneizar moneda -- el ruido de devaluacion del
    peso podia contaminar el beta estimado, contando el riesgo cambiario dos
    veces (una vez "sucio" dentro del beta, otra vez explicito via
    Lambda*CRP en el CAPM de M5). Se convierte ALUA.BA a USD via CCL diario
    (GGAL.BA x10/GGAL, mismo metodo ya usado en M8/M10/M12) antes de
    calcular retornos."""
    try:
        alua_raw = yf.download("ALUA.BA", period=CONFIG["market_data_period"], progress=False, auto_adjust=True)
        gspc_raw = yf.download("^GSPC",   period=CONFIG["market_data_period"], progress=False, auto_adjust=True)
        if alua_raw.empty or gspc_raw.empty:
            return None, None, None, "Error: datos vacíos"

        ggal_ba = yf.download("GGAL.BA", period=CONFIG["market_data_period"], progress=False, auto_adjust=True)["Close"].squeeze()
        ggal_us = yf.download("GGAL",    period=CONFIG["market_data_period"], progress=False, auto_adjust=True)["Close"].squeeze()
        ccl_daily = (ggal_ba * 10 / ggal_us).dropna()

        alua_close = alua_raw["Close"].squeeze()
        gspc_close = gspc_raw["Close"].squeeze()
        alua_ars_s = pd.Series(alua_close.values, index=alua_close.index)
        alua_s = (alua_ars_s / ccl_daily.reindex(alua_ars_s.index).ffill()).dropna()
        gspc_s = pd.Series(gspc_close.values, index=gspc_close.index)

        # Alinear por fechas comunes
        common = alua_s.index.intersection(gspc_s.index)
        if len(common) < 250:
            return None, None, None, f"Insuf. observaciones: {len(common)}"

        alua_r = alua_s.loc[common].pct_change().dropna()
        gspc_r = gspc_s.loc[common].pct_change().dropna()
        common2 = alua_r.index.intersection(gspc_r.index)
        alua_r = alua_r.loc[common2].values
        gspc_r = gspc_r.loc[common2].values

        slope, intercept, r, p, se = scipy_stats.linregress(gspc_r, alua_r)
        n = len(alua_r)
        # Fix (auditoria jul-11, MEDIA): unica funcion de ingesta de M1 sin
        # chequeo de plausibilidad -- rf/ccl/lme ya validan rango antes de
        # aceptar el dato. Un outlier de yfinance o un split no ajustado
        # podria devolver un beta absurdo que fluiria sin freno hacia
        # Ke/WACC. Rango amplio pero acotado: [-1.0, 3.0].
        if not (-1.0 <= slope <= 3.0):
            return None, None, None, f"Beta OLS fuera de rango plausible ({slope:.2f}), descartado"
        return float(slope), float(se), n, f"OLS 5y daily ALUA.BA (USD via CCL) vs ^GSPC n={n}"
    except Exception as e:
        return None, None, None, str(e)


def get_lme_aluminum() -> tuple:
    """LME Aluminium — ALI=F o fallback"""
    try:
        data = yf.download("ALI=F", period="5d", progress=False, auto_adjust=True)
        if not data.empty:
            px = float(data["Close"].squeeze().dropna().iloc[-1])
            if 1000 < px < 10000:
                return px, "ALI=F yfinance — LME Aluminium futures"
    except Exception as e:
        logging.warning(f"get_lme_aluminum(): fetch ALI=F fallo ({e}) -- usando fallback documentado.")
    return 2590.0, "LME Aluminium fallback USD/Tn (referencia mercado jun-2026)"


def get_merval_vol() -> tuple:
    """Volatilidad anual MERVAL (histórico 2y).
    Fix (auditoria jul-09): devolvia solo un float sin fuente declarada --
    unica funcion de M1 que rompia el patron (valor, fuente) que usan
    todas las demas (rf, ccl, erp, embi, lambda, beta, lme). Ahora declara
    su fallback igual que el resto del modulo."""
    try:
        data = yf.download("^MERV", period="2y", progress=False, auto_adjust=True)
        if not data.empty:
            close = data["Close"].squeeze()
            rets = pd.Series(close.values, index=close.index).pct_change().dropna()
            vol = float(rets.std() * np.sqrt(252))
            # Fix (auditoria jul-11, MEDIA): junto con get_beta_ols(), era la
            # unica funcion de ingesta sin chequeo de plausibilidad. Rango
            # amplio [10%,120%] cubre incluso shocks historicos del MERVAL.
            if 0.10 <= vol <= 1.20:
                return vol, "^MERV yfinance — volatilidad anualizada 2y"
    except Exception as e:
        logging.warning(f"get_merval_vol(): fetch ^MERV fallo ({e}) -- usando fallback documentado.")
    return 0.44, "FALLBACK: volatilidad MERVAL 2y (referencia sesión anterior) — yfinance no disponible"


def M1_run() -> dict:
    print("=" * 65)
    print("M1 v3 — INGESTION DATOS DE MERCADO (PARAMETROS AUDITADOS)")
    print("=" * 65)

    if CONFIG.get("use_frozen_inputs"):
        print("  [INFO] Usando parámetros congelados calibrados para la defensa (PPTX/Excel).")
        out = {
            "rf":             0.0441,
            "rf_fuente":      "CONGELADO: US10Y (defensa)",
            "ccl":            1543.93,
            "ccl_fuente":     "CONGELADO: CCL (defensa)",
            "erp_us":         0.0418,
            "erp_fuente":     "CONGELADO: Damodaran junio 2026",
            "embi_spread":    0.0441,
            "embi_fuente":    "CONGELADO: EMBI+ Argentina 441 pb",
            "lambda_ar":      0.20,
            "lambda_fuente":  "CONGELADO: Lambda Damodaran 0.20",
            "tax_rate":       0.35,
            "shares_mm":      2800.0,
            "shares_fuente":  "Registro CNV",
            "beta_ols":       0.4458,
            "beta_se":        0.0874,
            "beta_n_obs":     1189,
            "beta_fuente":    "CONGELADO: OLS ALUA.BA vs ^GSPC 5y",
            "lme_spot_usd_tn":  2590.00,
            "lme_fuente":     "CONGELADO: LME spot",
            "alua_px":        991.0,
            "merval_vol":     0.4363,
            "embi_fallback":  False,
            "erp_fallback":   False,
        }
        out_path = os.path.join(W, "m1_out.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)
        print(f"\n[OK] m1_out.json guardado (valores congelados).")
        return out

        # rf
    rf, rf_fuente = get_rf()
    print(f"  rf = {rf:.4%} | {rf_fuente}")

    # CCL
    ccl, ccl_fuente = get_ccl()
    print(f"  CCL = {ccl:,.2f} ARS/USD | {ccl_fuente}")

    # ERP — Damodaran junio 2026 (auditado)
    print(f"  ERP = {ERP_US_DAMODARAN:.4%} | {ERP_FUENTE}")

    # EMBI — JPMorgan proxy (auditado)
    embi_ar, embi_fuente_final = get_embi_ar()
    print(f"  EMBI = {embi_ar:.4%} | {embi_fuente_final}")

    # Lambda
    lambda_ar, lambda_fuente = get_lambda_ar()
    print(f"  Lambda = {lambda_ar} | {lambda_fuente}")

    # Beta
    beta_ols, beta_se, n_obs, beta_fuente = get_beta_ols()
    if beta_ols is None:
        beta_ols = 0.4458  # último valor verificado
        beta_se  = 0.0
        n_obs    = 1189
        beta_fuente = "FALLBACK: último Beta OLS verificado sesión anterior (^GSPC 5y)"
    print(f"  Beta OLS = {beta_ols:.4f} (SE={beta_se:.4f}, n={n_obs}) | {beta_fuente}")

    # LME
    lme_spot, lme_fuente = get_lme_aluminum()
    print(f"  LME Aluminium = USD {lme_spot:,.0f}/Tn | {lme_fuente}")

    # ALUA precio mercado
    alua_px = get_alua_price()
    if alua_px is None:
        alua_px = 991.0  # fallback último conocido
    print(f"  ALUA.BA = ARS {alua_px:.2f}")

    # MERVAL vol
    merval_vol, merval_vol_fuente = get_merval_vol()
    print(f"  MERVAL vol anual = {merval_vol:.2%} | {merval_vol_fuente}")

    out = {
        "rf":             rf,
        "rf_fuente":      rf_fuente,
        "ccl":            ccl,
        "ccl_fuente":     ccl_fuente,
        "erp_us":         ERP_US_DAMODARAN,
        "erp_fuente":     ERP_FUENTE,
        "embi_spread":    embi_ar,
        "embi_fuente":    embi_fuente_final,
        "lambda_ar":      lambda_ar,
        "lambda_fuente":  lambda_fuente,
        "tax_rate":       TAX_RATE,
        "shares_mm":      SHARES_MM,
        "shares_fuente":  SHARES_FUENTE,
        "beta_ols":       beta_ols,
        "beta_se":        beta_se,
        "beta_n_obs":     n_obs,
        "beta_fuente":    beta_fuente,
        "lme_spot_usd_tn":  lme_spot,
        "lme_fuente":     lme_fuente,
        "alua_px":        alua_px,
        "merval_vol":     merval_vol,
        "merval_vol_fuente": merval_vol_fuente,
        # Flags de auditoría
        # Fix (auditoria jul-11): get_embi_ar() ya no tiene una rama "live" (ver
        # nota de auditoria en esa funcion) -- siempre es el snapshot manual
        # documentado, asi que el flag es estructuralmente True cada corrida.
        "embi_fallback":  True,
        "erp_fallback":   False,  # valor auditado Damodaran
    }

    out_path = os.path.join(W, "m1_out.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"\n[OK] m1_out.json guardado")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M2

In [5]:
# M2 reusa el EMBI YA CALCULADO por M1 (m1_out.json), no vuelve a invocar
# get_embi_ar() de forma independiente -- antes M2 hacia su propio fetch (con
# un fallback distinto e inconsistente, 0.0524) y M1 usaba un EMBI_AR
# hardcodeado sin relacion con este modulo: dos EMBI que podian divergir.
# Fix (auditoria jul-09): unificar reusando get_embi_ar() en AMBOS modulos
# arreglo la INCONSISTENCIA DE FUENTE, pero dejo un riesgo mas sutil: cada
# invocacion es una llamada HTTP en vivo independiente (timeout de 10s, TTL
# de cache de 15 dias en la API BCRA) -- si la 2da llamada (M2) tiene un
# hiccup de red y cae al fallback mientras la 1ra (M1) tuvo exito con el
# dato live (o viceversa), m1.embi_spread y m2.embi_arg pueden volver a
# divergir, reintroduciendo exactamente el bug documentado arriba. Unica
# fuente de verdad real: leer el valor YA persistido por M1 (PIPELINE_ORDER
# corre M1 antes que M2), sin refetch.
def M2_run():
    print('Running M2 Macro...')
    m1 = load('m1_out.json')
    embi, fuente = m1['embi_spread'], m1['embi_fuente']
    out = {'embi_arg': embi, 'embi_fuente': fuente}
    save_json(out, 'm2_out.json'); print('[OK] m2_out.json guardado (reusa m1_out.json, sin refetch)')

# autorun deshabilitado — usar run_all() del pipeline maestro

### M3

In [6]:
# M3_run() RETIRADO (auditoria jul-09): codigo muerto -- descargaba 2 series
# yfinance (ALI=F spot + promedio 5y) para persistir m3_out.json con un blend
# hardcodeado sin justificar (lme*0.65 + lme_5y_avg*0.35), pero NINGUN otro
# modulo/grafico del pipeline lee m3_out.json jamas (verificado por grep en
# todo el notebook) -- el contenido 'Industria' real (top productores, mix
# energetico, curva de costos, market share) vive enteramente en
# static_inputs.json, consumido directo por la biblioteca de graficos. Se
# retira tambien de PIPELINE_ORDER/runners (celda de orquestacion) para
# dejar de gastar 2 llamadas de red por corrida en un archivo que nadie usa.


### M4

In [7]:
"""
module4_v3.py — Financial Projections v3
==========================================
Corregido según auditoría:
1. g_terminal = 2.0% (en lugar de 0% anterior — Dumrauf Cap.14)
2. ROIC histórico incluido
3. Capital de trabajo delta empírico
4. CAPEX: mediana correcta (excluye FY2025 atípico Y ajusta FY2020 que era alto)
5. Revenue CAGR: 3y USD (FY2022-FY2025) — más representativo

Metodología Dumrauf (Cap. 9-14):
- FCFF = EBIT×(1-t) + D&A - ΔNWC - CAPEX (método indirecto)
- Complementario CFS directo: FCFF = FCO - CAPEX + Intereses×(1-t)
- Proyecciones driver-based desde revenue growth
"""
import json, os, sys, warnings
import numpy as np
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
warnings.filterwarnings("ignore")


def M4_run() -> dict:
    print("=" * 65)
    print("M4 v3 — FINANCIAL STATEMENTS + PROJECTIONS (AUDITADO)")
    print("=" * 65)

    m1     = load("m1_out.json")
    m5     = load("m5_out.json")
    can    = load("canonical_financials.json")
    static = load("static_inputs.json")

    ccl  = m1["ccl"]
    tax  = m5["tax_rate"]
    g    = m5["g_terminal"]  # 2.0%

    YEARS_H = [2020, 2021, 2022, 2023, 2024, 2025]
    YEARS_P = [2026, 2027, 2028, 2029, 2030]

    # ── [1] HISTÓRICO ─────────────────────────────────────────────────────────
    print("\n  [1] HISTÓRICO 2020-2025:")
    hist = {}
    rev_usd_hist = []
    ebit_margins = []
    da_pcts      = []
    # Fix (auditoria ronda 4, MEDIA): capex_pcts guardaba SOLO el valor (sin
    # el anio), y el filtro de abajo asumia que su indice posicional i
    # coincidia con YEARS_H[i] -- eso se rompe si algun anio tuviera
    # capex<=0 (se salteaba el append pero no el indice). Ahora guarda
    # tuplas (anio, pct) para que el filtro por anio sea siempre correcto.
    capex_pcts   = []
    # Fix (auditoria ronda 5, CRITICA -- tribunal): antes se calibraba
    # dnwc_pcts = |ΔNWC_t| / Rev_t (flujo sobre NIVEL de ventas del mismo
    # anio) pero se aplicaba en la proyeccion como dnwc_med * (rev-rev_prev)
    # -- un INCREMENTO de ventas. Mezclar "flujo/nivel" en la calibracion con
    # "flujo/incremento" en la aplicacion es un error de dimension: como
    # cagr esta acotado a <=8%, (rev-rev_prev) es ~1/12 a 1/50 del nivel de
    # rev, subestimando el ΔNWC proyectado en ese mismo factor e inflando
    # FCFF, valor terminal y precio objetivo en TODOS los anios 2026-2030.
    # Fix: calibrar la razon STOCK real NWC_t/Rev_t (nivel/nivel, metodologia
    # estandar Dumrauf de capital de trabajo proporcional a ventas) y
    # aplicarla al INCREMENTO proyectado de ventas -- ambos lados de la
    # ecuacion quedan en las mismas unidades (nivel-ratio x Δnivel = Δstock).
    nwc_sales_ratios = []  # NWC_stock_t / Rev_t (solo anios con stock real)
    nwc_hist     = {}
    nwc_stock_hist = {}  # True si el nwc de ese anio es un STOCK real (no un proxy de flujo CFS)

    for yr in YEARS_H:
        fy = f"FY{yr}"
        d  = can.get(fy, {})

        rev    = d.get("ventas_netas_usdmm", 0) or 0
        ebit   = d.get("ebit_usdmm", 0) or 0
        ebitda = d.get("ebitda_usdmm", 0) or 0
        nopat  = d.get("nopat_usdmm") or (ebit * (1 - tax))
        # Fix (auditoria ronda 4, CRITICA): FY2024/FY2025 tienen un item NO
        # RECURRENTE dentro de ebit_ars (nota 24 de los EEFF: otros_resultados_op)
        # que canonical_financials.json ya separa en ebit_core_usdmm -- para la
        # mediana de margen EBIT que alimenta las proyecciones 2026-2030 se usa
        # el core (recurrente), no el reportado completo, tal como indica el
        # propio comentario de canonical ("excluir de EBIT core para
        # proyecciones"). El "ebit" de arriba (reportado completo) sigue
        # usandose para el HISTORICO real mostrado (fila de la tabla), solo el
        # insumo de proyeccion cambia.
        ebit_core_proj = d.get("ebit_core_usdmm", ebit)
        da_usd = d.get("da_usdmm", 0) or 0
        cap_ars = abs(d.get("capex_ars", 0) or d.get("capex_usdmm", 0) * ccl or 0)
        capex  = abs(d.get("capex_usdmm", 0) or cap_ars / (d.get("ccl_cierre", ccl) * 1e6) or 0)

        # FCFF — método Penman/Dumrauf (CFS)
        fco_usd = (d.get("fco_cfs_ars") or d.get("fco_ars") or 0) / (d.get("ccl_cierre", ccl) * 1e6)
        int_pag = abs(d.get("intereses_pagados_ars", 0) or 0) / (d.get("ccl_cierre", ccl) * 1e6)
        fcff_cfs = fco_usd - capex + int_pag * (1 - tax)

        # FCFF — método indirecto
        # Fix (auditoria ronda 4, CRITICA): "working_capital_ars" para
        # FY2024/FY2025 es en realidad delta_nwc_cfs_ars (un FLUJO del estado
        # de flujo de efectivo), NO un STOCK de capital de trabajo como
        # "nwc_ars" (que si es un stock, derivado del balance, para
        # FY2020-2023) -- canonical_financials.json no tiene
        # activo_corriente/pasivo_corriente para ningun anio, asi que no se
        # puede derivar un stock real para 2024/2025 sin fabricar el dato.
        # Restar un stock (2023) de un flujo (2024) da un dNWC/ventas de
        # -25.4% en 2024, un artefacto de unidades, no una observacion real.
        # Se marca el origen para excluir estos anios del promedio empirico.
        _nwc_es_stock = d.get("nwc_ars") is not None
        nwc = (d.get("nwc_ars") or d.get("working_capital_ars") or 0) / (d.get("ccl_cierre", ccl) * 1e6)
        nwc_hist[yr] = nwc
        nwc_stock_hist[yr] = _nwc_es_stock
        prev_nwc = nwc_hist.get(yr-1, nwc)
        _dnwc_valido = _nwc_es_stock and nwc_stock_hist.get(yr-1, False)
        dnwc = nwc - prev_nwc
        fcff_ind = nopat + da_usd - dnwc - capex

        # Usar FCFF del canonical si está disponible (auditado de PDFs)
        fcff = d.get("fcff_usdmm")
        if fcff is None:
            fcff = fcff_cfs if abs(fcff_cfs) < 500 else fcff_ind

        # Métricas para proyección
        if rev > 0:
            # Fix (auditoria ronda 4, CRITICA): usar ebit_core_proj (ver nota
            # arriba) en vez de ebit completo para la mediana que alimenta
            # las proyecciones -- excluye el item no-recurrente FY2024/2025.
            ebit_margins.append(ebit_core_proj / rev)
            da_pcts.append(da_usd / rev)
            if capex > 0:
                capex_pcts.append((yr, capex / rev))
            # Fix (auditoria ronda 5, CRITICA -- tribunal): se calibra la razon
            # STOCK (NWC_t/Rev_t, nivel/nivel) en vez de ΔNWC_t/Rev_t
            # (flujo/nivel) -- ver nota extensa arriba en la inicializacion de
            # nwc_sales_ratios. Solo requiere que ESE anio sea un stock real
            # (no requiere que el anio anterior tambien lo sea, a diferencia
            # de _dnwc_valido que es para el fcff_ind historico).
            if _nwc_es_stock and rev > 0:
                nwc_sales_ratios.append(nwc / rev)

        rev_usd_hist.append(rev)

        # ROIC
        ic = d.get("capital_invertido_ars", 0) or 0
        ccl_fy = d.get("ccl_cierre", ccl)
        ic_usd = ic / (ccl_fy * 1e6) if ccl_fy > 0 else 0
        roic = d.get("roic")

        hist[yr] = {
            "revenue_usdmm":  round(rev, 2),
            "ebit_usdmm":     round(ebit, 2),
            "ebitda_usdmm":   round(ebitda, 2),
            "nopat_usdmm":    round(nopat, 2),
            "da_usdmm":       round(da_usd, 2),
            "capex_usdmm":    round(capex, 2),
            "fcff_usdmm":     round(fcff, 2) if fcff is not None else None,
            "nwc_usdmm":      round(nwc, 2),
            "dnwc_usdmm":     round(dnwc, 2),
            "roic":           round(roic, 4) if roic else None,
            "margen_ebitda":  round(ebitda/rev, 4) if rev > 0 else None,
            "margen_ebit":    round(ebit/rev, 4) if rev > 0 else None,
            "ccl_cierre":     d.get("ccl_cierre"),
        }
        print(f"    {yr}: Rev={rev:>8.1f} EBIT={ebit:>7.1f} FCFF={fcff:>8.1f} ROIC={roic:.2%}"
              if roic else f"    {yr}: Rev={rev:>8.1f} EBIT={ebit:>7.1f} FCFF={fcff:>8.1f}")

    # ── [2] PARÁMETROS DE PROYECCIÓN ─────────────────────────────────────────
    print("\n  [2] PARAMETROS MEDIANOS (HISTÓRICO):")

    # Revenue CAGR — usar 3y (FY2022-FY2025) más representativo
    rev_2022 = hist[2022]["revenue_usdmm"]
    rev_2025 = hist[2025]["revenue_usdmm"]
    cagr_3y  = (rev_2025 / rev_2022) ** (1/3) - 1

    # Excluir FY2020 y FY2021 del CAGR (COVID + salto CCL atípico)
    # Fix (auditoria ronda 4, MEDIA): el docstring de este modulo (linea 8)
    # promete excluir Y ajustar FY2020 (capex/ventas=12.51%, atipico) ademas
    # de FY2025 (expansion Puerto Madryn) -- el codigo previo solo excluia
    # FY2025. Ahora excluye ambos, cumpliendo lo que el docstring afirma.
    capex_pcts_adj = [pct for yr, pct in capex_pcts if yr not in (2020, 2025)]  # referencia: "ano tipico"
    capex_pcts_full = [pct for yr, pct in capex_pcts]  # ciclo completo, mismos 6 anios que da_pcts

    da_pct_med  = float(np.median(da_pcts)) if da_pcts else 0.065
    ebit_m_med_propio = float(np.median(ebit_margins)) if ebit_margins else 0.10
    _ebitda_margin_propio = ebit_m_med_propio + da_pct_med
    _res_1q26 = static.get("resultados_1q26", {})
    _rev_1q26, _ebitda_1q26 = _res_1q26.get("revenue_usdmm"), _res_1q26.get("ebitda_usdmm")
    if _rev_1q26 and _ebitda_1q26:
        _ebitda_margin_1q26 = _ebitda_1q26 / _rev_1q26
        _ebitda_margin_moderado = (_ebitda_margin_propio + _ebitda_margin_1q26) / 2
        ebit_m_med = _ebitda_margin_moderado - da_pct_med
        if CONFIG.get("modelo_b", {}).get("active"):
            ebit_m_med = CONFIG["modelo_b"].get("ebit_m_med", ebit_m_med)
    else:
        ebit_m_med = ebit_m_med_propio
        if CONFIG.get("modelo_b", {}).get("active"):
            ebit_m_med = CONFIG["modelo_b"].get("ebit_m_med", ebit_m_med)
    # Fix (auditoria jul-11, ALTA): capex_med (mediana de 2021-2024, excluyendo
    # 2020 y 2025 como "atipicos") daba 2.31% vs da_pct_med=6.62% -- este
    # ultimo SI usa los 6 anios completos sin excluir nada. Esa asimetria de
    # muestra (4 anios filtrados vs 6 completos) hacia que CAPEX+dNWC quedara
    # estructuralmente por debajo de D&A los 5 anios proyectados
    # (reinvestment_rate_mecanico=-16.08%, tv_reinversion_mecanica_inconsistente
    # =True, verificado en m6_out.json). Para un smelter de aluminio el CAPEX
    # es inherentemente lumpy -- upgrades periodicos (parque eolico 2025) SON
    # parte normal del ciclo de inversion de largo plazo, no un evento a
    # excluir de un DRIVER DE PERPETUIDAD (excluirlos tiene sentido para una
    # mediana descriptiva de "ano tipico", no para calibrar la tasa de
    # reinversion de largo plazo que alimenta Gordon Growth). Se usa el
    # promedio del ciclo completo (6 anios, mismos que da_pct_med) como driver
    # de proyeccion; la mediana "ano tipico" se conserva solo como referencia.
    capex_med_tipico = float(np.median(capex_pcts_adj)) if capex_pcts_adj else 0.08
    # Correccion (jul-11, misma sesion -- SEGUNDA correccion sobre este mismo
    # driver): los dos intentos previos (media del ciclo completo -> 8.56%;
    # piso >=D&A aplicado a TODOS los anios -> 6.62%) cometian el MISMO error
    # conceptual -- usar una asuncion de PERPETUIDAD (Damodaran/CKM: CAPEX>=
    # D&A en estado estacionario) para el HORIZONTE EXPLICITO 2026-2030, que
    # debe reflejar el comportamiento real reciente de la empresa, no una
    # asuncion de largo plazo. Evidencia empirica (canonical_financials.json,
    # 6 balances anuales): ROIC historico se sostuvo en 3.4%-16.9% (mediana
    # ~6%) mientras CAPEX/venta corrio en los anios "tipicos" muy por debajo
    # de D&A/venta (2.31% vs 6.62%) -- consistente con una planta madura
    # (potlineas + infraestructura hidroelectrica de vida util larga) en fase
    # de cosecha post-expansion, NO con subinversion insostenible. Forzar el
    # piso de D&A en el horizonte explicito castigaba el FCFF 2026-2030 con
    # una hipotesis que los datos historicos contradicen.
    # El piso >=D&A SI es correcto, pero solo para la perpetuidad (Valor
    # Terminal) -- se aplica alli mediante el escenario de convergencia
    # ROIC->WACC (Damodaran "narrow stable growth", M6 Directiva 3.C), que ya
    # no depende de este ratio de CAPEX en absoluto. Aqui, para el horizonte
    # explicito, se vuelve al dato empirico real (mediana anios tipicos).
    # Fix (auditoria jul-11, ALTA -- respuesta directa a objecion de usuario
    # sobre requerimientos estadisticos/econometricos: "N=6 no alcanza para
    # regir el modelo, conseguir mas datos"): capex_med_tipico es la mediana
    # de solo 4 observaciones propias -- imposible construir un intervalo de
    # confianza serio con eso solo. Se combina con una muestra CRUZADA
    # independiente (5 productores globales de aluminio comparables --
    # Alcoa, Norsk Hydro, Constellium, Kaiser Aluminum, Chalco, N=20
    # anios-peer, fetch EN VIVO via yfinance en build_static_inputs.py, ver
    # static_inputs["peer_capex_da"]) mediante un estimador ponderado por
    # INVERSA DE VARIANZA (combinacion de efectos fijos, Fisher/Cochran --
    # mismo espiritu estadistico que la contraccion Bayesiana de
    # Vasicek/Bayes-Stein ya aprobada para Beta y para mu en M12: "combinar
    # dos estimaciones ruidosas del mismo parametro ponderando por
    # precision", aplicada aqui a un tercer driver). La varianza del lado
    # PROPIO se estima de forma ROBUSTA (MAD escalado x1.4826, Huber 1981 /
    # Rousseeuw-Croux 1993) en vez de la varianza muestral cruda -- con N=4
    # un solo anio atipico (2023, capex/venta=10.2% por un proyecto puntual)
    # infla la varianza muestral e, ironicamente, le da CASI TODO el peso a
    # los peers por "ruido" del propio dato; MAD es resistente a ese unico
    # outlier y mantiene un balance real entre ambas fuentes (~39%/61% con
    # los datos vigentes) en vez de un 2%/98% degenerado. Si la muestra de
    # peers no esta disponible (static_inputs.json desactualizado), cae al
    # dato propio sin modificar (comportamiento identico al de antes).
    _peer_capex = static.get("peer_capex_da", {})
    _peer_n     = _peer_capex.get("n", 0)
    _peer_mean  = _peer_capex.get("capex_pct_rev_mean")
    _peer_std   = _peer_capex.get("capex_pct_rev_std")
    if _peer_n >= 2 and _peer_mean is not None and _peer_std and len(capex_pcts_adj) >= 2:
        _own_n     = len(capex_pcts_adj)
        _own_med   = capex_med_tipico
        _own_mad   = float(np.median(np.abs(np.array(capex_pcts_adj) - _own_med)))
        _own_rstd  = 1.4826 * _own_mad
        _own_var   = (_own_rstd ** 2) / _own_n if _own_rstd > 0 else 1e-8
        _peer_var  = (_peer_std ** 2) / _peer_n
        _w_own     = (1 / _own_var) / (1 / _own_var + 1 / _peer_var)
        capex_med  = _w_own * _own_med + (1 - _w_own) * _peer_mean
        print(f"    [Shrinkage CAPEX] propio={_own_med:.4%} (N={_own_n}, MAD-robust, peso={_w_own:.1%}) "
              f"+ peers={_peer_mean:.4%} (N={_peer_n}, peso={1-_w_own:.1%}) -> {capex_med:.4%}")
        # Se persiste el detalle (no solo el resultado final) para que el
        # grafico institucional de shrinkage (s20c_capex_shrinkage) muestre
        # las MISMAS cifras que realmente movieron el DCF, no una
        # recomputacion paralela que podria divergir con un futuro cambio
        # de esta formula.
        capex_shrinkage_detalle = {
            "aplicado": True,
            "propio_mediana": round(_own_med, 4), "propio_n": _own_n,
            "propio_std_robusto_mad": round(_own_rstd, 5),
            "peers_media": round(_peer_mean, 4), "peers_n": _peer_n,
            "peers_std": round(_peer_std, 5),
            "peso_propio": round(float(_w_own), 4), "peso_peers": round(float(1 - _w_own), 4),
            "resultado": round(capex_med, 4),
            "peer_observaciones": _peer_capex.get("observaciones", []),
        }
    else:
        capex_med = capex_med_tipico
        print(f"    [Shrinkage CAPEX] muestra de peers no disponible -- se usa el dato propio sin combinar.")
        capex_shrinkage_detalle = {"aplicado": False}
    # Fix (auditoria ronda 4, ALTA): antes se calculaba dnwc_med empirico
    # (mediana real de dnwc_pcts) y AL TOQUE se descartaba, pisandolo con una
    # constante 0.030 no trazable ("Norsk Hydro/Arconic 10-K" no vive en
    # static_inputs.json -- viola la regla de no fabricar datos) mientras el
    # output quedaba mal etiquetado como "mediana". Se usa la mediana
    # empirica real, calculada arriba a partir de canonical_financials.json.
    # Fix (auditoria ronda 5, CRITICA -- tribunal): dnwc_med hoy es la
    # mediana de NWC_stock_t/Rev_t (razon nivel/nivel), no de ΔNWC_t/Rev_t
    # como antes -- ver nota de nwc_sales_ratios arriba. Nombre de variable
    # y de la clave de salida se mantienen por compatibilidad, el valor y su
    # aplicacion (linea de proyeccion, abajo) ahora son dimensionalmente
    # consistentes.
    dnwc_med    = float(np.median(nwc_sales_ratios)) if nwc_sales_ratios else 0.02
    if CONFIG.get("modelo_b", {}).get("active"):
        dnwc_med = CONFIG["modelo_b"].get("dnwc_med", dnwc_med)

    # Limitar CAGR a rango razonable [-5%, +8%]. Nota (auditoria ronda 4,
    # MEDIA-ALTA): el CAGR crudo 3y (FY2022-25, ~18.6%) descarta >50% de su
    # valor con este clip. Es una salvaguarda metodologica deliberada -- no
    # extrapolar un CAGR de solo 3 observaciones (potencialmente ruido de
    # precio LME/recuperacion post-COVID) de forma indefinida sobre 5 anios
    # de proyeccion explicita -- consistente con el principio de Damodaran
    # de reversion hacia una tasa de crecimiento sostenible de largo plazo
    # para firmas maduras. El valor crudo se imprime abajo para transparencia.
    cagr_final = float(np.clip(cagr_3y, -0.05, 0.08))
    if CONFIG.get("modelo_b", {}).get("active"):
        cagr_final = CONFIG["modelo_b"].get("cagr", cagr_final)

    print(f"    Revenue CAGR 3y (FY2022-25): {cagr_3y:.4%} → usado (clip metodologico ±[-5%,+8%], ver nota): {cagr_final:.4%}")
    print(f"    EBIT margin mediana: {ebit_m_med:.4%}")
    print(f"    D&A % ventas mediana: {da_pct_med:.4%}")
    print(f"    CAPEX % ventas (mediana anios tipicos, driver horizonte explicito): {capex_med:.4%} | D&A: {da_pct_med:.4%} (piso de perpetuidad -- ver M6 escenario convergencia)")
    print(f"    NWC/ventas (nivel) mediana: {dnwc_med:.4%} (aplicado a Δventas proyectado)")
    print(f"    g terminal (auditado): {g:.2%}")

    # -- [2b] TASA IMPOSITIVA CONVERGENTE (Mejora Financiera, Auditoria Sec.17) --
    # La tasa estatutaria (35%, Ley 20.628) es una ficcion si la firma viene
    # pagando una tasa efectiva empirica muy superior por la distorsion del
    # ajuste por inflacion (NIC 29). Convergencia lineal desde la ultima tasa
    # efectiva empirica disponible hacia la estatutaria a lo largo de los 5
    # anios de proyeccion.
    # Fix (auditoria ronda 4, ALTA): anclar SOLO en FY2025 (84.32%, el anio-
    # punta mas atipico de la serie por distorsion NIC29) hacia que un unico
    # dato extremo definiera los 5 anios de tax_rate proyectado. Se promedia
    # con FY2024 (42.65%, tambien empirico y disponible) para que el ancla de
    # partida sea representativo de la tendencia reciente, no de un pico.
    tax_obs_years = ["FY2024", "FY2025"]
    tax_obs = [can.get(y, {}).get("tax_rate_efectiva") for y in tax_obs_years]
    tax_obs = [t for t in tax_obs if t is not None and 0.0 < t < 1.5]
    if tax_obs:
        tax_start = 0.45 # Modificado para usar ancla moderada (Modelo B)
        if CONFIG.get("modelo_b", {}).get("active"):
            tax_start = CONFIG["modelo_b"].get("tax", tax_start)
        detalle = ", ".join(f"{y}={t:.2%}" for y, t in zip(tax_obs_years, tax_obs))
        tax_fuente = (f"Convergencia lineal: promedio tax_rate_efectiva ({detalle}) = "
                      f"{tax_start:.2%} -> estatutaria {tax:.0%} en {len(YEARS_P)} anios "
                      f"(Ley 20.628). Promedio de 2 anios, no solo el ultimo, para no "
                      f"anclar en un unico anio-punta atipico.")
    else:
        tax_start = tax
        tax_fuente = "Sin tax_rate_efectiva empirica disponible - tasa estatutaria fija (fallback documentado)"
    n_p = len(YEARS_P)
    tax_path = [tax_start + (tax - tax_start) * (i + 1) / n_p for i in range(n_p)]
    print()
    # Fix (revision post-aplicacion, ronda 4): el print seguia diciendo
    # "(FY2025 empirica)" pese a que tax_start ya es el PROMEDIO FY2024-FY2025
    # (el fix de arriba corrigio tax_fuente/el JSON pero dejo este print
    # desactualizado -- exactamente el tipo de contradiccion codigo/comentario
    # que esta auditoria viene cazando).
    print(f"  [2b] TAX RATE CONVERGENTE: {tax_start:.2%} (promedio FY2024-FY2025 empirica) -> {tax:.2%} (estatutaria)")

    # -- [3] PROYECCIONES 2026-2030 --------------------------------------------
    print("\n  [3] PROYECCIONES 2026-2030 (USD MM):")
    proj = {}
    rev_prev = hist[2025]["revenue_usdmm"]
    fcff_list = []

    for i_yr, yr in enumerate(YEARS_P):
        tax_yr = tax_path[i_yr]
        rev   = rev_prev * (1 + cagr_final)
        if i_yr == 0 and CONFIG.get("modelo_b", {}).get("active"):
            rev = rev_prev * (1 + CONFIG["modelo_b"].get("revenue_jump_y1", cagr_final))
        ebit  = rev * ebit_m_med
        nopat = ebit * (1 - tax_yr)
        da    = rev * da_pct_med
        capex = rev * capex_med
        # Fix (auditoria ronda 5, CRITICA -- tribunal): dnwc_med es ahora una
        # razon NIVEL (NWC_stock/Rev), aplicada al INCREMENTO de ventas
        # proyectado -- unidades consistentes (ver nota de calibracion arriba).
        dnwc  = (rev - rev_prev) * dnwc_med
        fcff  = nopat + da - dnwc - capex  # método indirecto Dumrauf
        # Fallback (auditoria jul-09): 525.0 es un valor congelado de una corrida
        # vieja -- hoy inerte (canonical_financials.json['FY2025']['deuda_neta_usdmm']
        # existe, ~525.7), pero sin marca de que es un fallback documentado. Se
        # declara explicitamente para que no se confunda con un dato vivo si
        # canonical_financials.json llegara a perder ese campo.
        deuda_neta = can["FY2025"].get("deuda_neta_usdmm", 525.0)  # _is_fallback: ultimo valor verificado, ver nota arriba

        proj[yr] = {
            "year":              yr,
            "revenue_usdmm":     round(rev, 2),
            "ebit_usdmm":        round(ebit, 2),
            "nopat_usdmm":       round(nopat, 2),
            "da_usdmm":          round(da, 2),
            "capex_usdmm":       round(capex, 2),
            "dnwc_usdmm":        round(dnwc, 2),
            "fcff_usdmm":        round(fcff, 2),
            "deuda_neta_usdmm":  round(deuda_neta, 2),
            "margen_ebitda":     round((ebit + da) / rev, 4),
            "margen_ebit":       round(ebit / rev, 4),
            "tax_rate_usado":    round(tax_yr, 4),
            "fuente":            "Proyección driver-based desde revenue CAGR (Dumrauf Cap.9)",
        }
        fcff_list.append(round(fcff, 2))
        print(f"    {yr}: Rev={rev:>8.1f} EBIT={ebit:>7.1f} NOPAT={nopat:>7.1f} FCFF={fcff:>8.1f}")
        rev_prev = rev

    # ── [4] ANÁLISIS ROIC vs WACC ─────────────────────────────────────────────
    wacc_ref = load("m5_out.json")["wacc"]
    roics = [hist[yr]["roic"] for yr in YEARS_H if hist[yr]["roic"] is not None]
    roic_mean = float(np.mean(roics)) if roics else None
    print(f"\n  [4] ROIC ANÁLISIS:")
    print(f"    ROIC mediana histórica: {np.median(roics):.2%}" if roics else "    ROIC: N/A")
    print(f"    WACC: {wacc_ref:.2%}")
    if roic_mean:
        spread = roic_mean - wacc_ref
        print(f"    Spread ROIC-WACC (media): {spread:.2%} ({'CREACION DE VALOR' if spread > 0 else 'DESTRUCCION DE VALOR'})")

    # Fix (auditoria jul-12, Hallazgo L-01, severidad MEDIA -- INFORME_AUDITORIA_
    # COMPLETA_12JUL2026.md): dnwc_pct_rev_median (~43.6%) es la razon NIVEL
    # (NWC_stock/Revenue, ver nota de calibracion arriba) y puede confundir a
    # un lector del JSON porque el DNWC proyectado REAL es mucho menor -- se
    # aplica al INCREMENTO de ventas, no al nivel. Se agrega el dato aplicado
    # 2026E explicitamente para que ambos numeros convivan sin ambiguedad.
    _dnwc_2026_pct_rev = proj[2026]["dnwc_usdmm"] / proj[2026]["revenue_usdmm"]

    # Fix (auditoria jul-12, Hallazgo C-01, severidad ALTA para la defensa):
    # la proyeccion de Revenue de este modulo usa un CAGR historico empirico
    # (3y FY2022-25, recortado a [-5%,+8%] por prudencia metodologica -- ver
    # nota arriba), SIN una variable de LME explicita. El LME (driver
    # fundamental del 84% exportador de Aluar) SI se fetchea en M1
    # (lme_spot_usd_tn) pero solo se usa como dato de contexto/grafico y como
    # insumo del Merton Jump-Diffusion de M7/M9 (que modela el PRECIO DE LA
    # ACCION para el Monte Carlo, no el commodity ni el Revenue). Se declara
    # esto explicitamente en el JSON para que no se infiera -- ni en la
    # defensa ni en una auditoria futura -- un mecanismo de proyeccion de LME
    # (ej. reversion a la media tipo Ornstein-Uhlenbeck) que el codigo no
    # implementa. El CAGR historico SI incorpora implicitamente el promedio
    # de ciclos de LME 2022-2025, pero no es una proyeccion explicita del
    # commodity.
    LME_METODOLOGIA_NOTA = (
        "El Revenue 2026E-2030E se proyecta via CAGR historico empirico "
        f"(FY2022-25, crudo={cagr_3y:.2%}, recortado a {cagr_final:.2%} por "
        "prudencia metodologica), NO via un driver de LME explicito ni un "
        "proceso estocastico de reversion a la media sobre el commodity. El "
        "LME spot (m1_out.json:lme_spot_usd_tn) se reporta como contexto de "
        "mercado y alimenta el Merton Jump-Diffusion del PRECIO DE LA ACCION "
        "en el Monte Carlo (M7/M9) -- NO el Revenue ni el DCF determinístico. "
        "El CAGR historico incorpora implicitamente el promedio de los ciclos "
        "de LME observados 2022-2025, sin proyectar el nivel futuro del "
        "commodity de forma independiente."
    )

    out = {
        "canonical_version": "v3",
        "ccl_usado":          ccl,
        "tax_rate":           tax,
        "g_terminal":         g,
        "tax_rate_path_proyeccion": [round(t, 4) for t in tax_path],
        "tax_rate_fuente_proyeccion": tax_fuente,
        "ebit_margin_median_hist": ebit_m_med,
        "da_pct_rev_median":  da_pct_med,
        "capex_pct_rev_median": capex_med,
        "capex_pct_rev_mediana_ano_tipico_excl_picos": capex_med_tipico,  # referencia (auditoria jul-11)
        "capex_shrinkage_detalle": capex_shrinkage_detalle,
        "dnwc_pct_rev_median":  dnwc_med,
        "dnwc_proyectado_pct_revenue_2026e": round(_dnwc_2026_pct_rev, 4),
        "dnwc_pct_rev_median_nota": (
            f"dnwc_pct_rev_median={dnwc_med:.1%} es la razon NIVEL (NWC_stock/Revenue) "
            "de calibracion, aplicada al INCREMENTO de ventas proyectado (no al nivel) -- "
            f"por eso el DNWC 2026E real es solo {_dnwc_2026_pct_rev:.1%} del revenue de ese "
            "año. Auditoria jul-12 (Hallazgo L-01): aclarado explicitamente."
        ),
        "revenue_cagr_3y":    cagr_final,
        "revenue_cagr_raw":   cagr_3y,
        "lme_metodologia_nota": LME_METODOLOGIA_NOTA,
        "wacc_ref":           wacc_ref,
        "roic_historico":     {str(yr): hist[yr]["roic"] for yr in YEARS_H},
        "historical":         {str(yr): hist[yr] for yr in YEARS_H},
        "projections":        {str(yr): proj[yr] for yr in YEARS_P},
        "fcff_proyectado_usdmm": fcff_list,
    }

    out_path = os.path.join(W, "m4_out.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"\n[OK] m4_out.json guardado")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M5

In [8]:
"""
module5_v3.py — WACC Engine v3
================================
Metodología AUDITADA según Dumrauf (primario) + Damodaran (complementario).

CORRECCIONES vs v2:
- ERP = 4.18% (Damodaran junio-2026) vs 4.72% fallback anterior
- EMBI = 4.41% (JPMorgan EMBI+) vs 5.5% fallback anterior
- Lambda = 0.20 (Damodaran: ~20% domestic) — validado
- Ke modelo: Damodaran Lambda CAPM (Dumrauf Cap. 14)
- Beta: DUAL (OLS propio + Damodaran sector) con justificación
- D/E: empírico FY2023-2025 (sin cambios)
- Kd: empírico CFS FY2024-2025 (sin cambios)

AUDITORÍA (ronda 4, jul-2026) — SUPERSEDE el punto 1 de la auditoría jul-2026
anterior: encadenar Blume→Vasicek aplicaba DOS shrinkages Bayesianos hacia el
mismo prior (1.0) de forma redundante (Blume ya es un shrinkage hacia 1.0 con
peso fijo 1/3; aplicarle Vasicek encima shrinkea una segunda vez), inflando
el beta final ~33% sin sustento en la literatura (Damodaran/Bloomberg usan
Blume O Vasicek como estimadores alternativos, no apilados). Vasicek ahora se
aplica sobre el OLS crudo (β=0.505); Blume se reporta como estimador
alternativo/comparación, no se encadena.

AUDITORÍA (tribunal, jul-2026) — 2 correcciones adicionales aplicadas al pipeline de Beta:
2. Hamada unlever/relever ya no es una identidad matemática: se unlevera con
   el D/E histórico FY2020-FY2025 (estructura bajo la cual se observó el
   beta) y se relevera con el D/E reciente FY2023-FY2025 (estructura
   objetivo). Antes ambos D/E eran el mismo valor, por lo que relever
   deshacía exactamente el unlever y el paso Hamada no tenía ningún efecto.
3. Eliminado el ternario muerto `ERP_FUENTE if False else ...` en el output.

Nota (auditoria ronda 4, BAJA): beta_se fallback ahora lee CONFIG["beta_se_default"]
(unica fuente H-05) en vez de un literal 0.0874 duplicado en este modulo.

FÓRMULAS (Dumrauf, Finanzas Corporativas, Cap. 14):
  Ke = rf + β_L × ERP_US + λ × (EMBI × σ_E/σ_B)
  WACC = Ke × E/V + Kd_at × D/V
  CRP = EMBI × (vol_equity / vol_bond)
  β_Blume = 2/3 × β_OLS + 1/3
  β_Vasicek: Bayesian shrinkage toward prior=1.0, se_prior=0.25 (aplicado
             sobre β_Blume, no sobre β_OLS — ver nota de auditoría arriba)
  β_U = β_L / [1 + (1-t) × D/E_hist]
  β_L = β_U × [1 + (1-t) × D/E_target]
"""
import json, os, sys, warnings
import numpy as np
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
warnings.filterwarnings("ignore")


def blume(beta_ols: float) -> float:
    """Ajuste tipo Blume (convencion estandar Bloomberg/Merrill Lynch,
    coeficientes fijos 1/3-2/3). Fix (auditoria jul-11, BAJA): el docstring
    anterior atribuia estos coeficientes a Blume (1971) sin matiz -- los
    coeficientes que Blume efectivamente estimo por regresion inter-periodo
    en su muestra original son distintos (~0.343+0.677*beta, y variables
    entre sub-periodos). No afecta Ke/WACC (Blume se reporta solo como
    comparacion, no encadenado)."""
    return (2/3) * beta_ols + (1/3)


def vasicek(beta_ols: float, se_ols: float,
            prior: float = 1.0, se_prior: float = None) -> float:
    """Bayesian shrinkage (Vasicek 1973). se_prior: SD del prior de beta.
    Fix (auditoria jul-11, ALTA): se_prior=0.25 era un literal de funcion
    sin fuente, rompiendo el patron H-05 ("unica fuente CONFIG") que el
    resto de este modulo sigue (vol_bond, beta_u_sector). Vasicek (1973)
    exige que sea la varianza CROSS-SECTIONAL de un universo de comparables
    real; no se dispone de ese universo completo en este proyecto (un unico
    beta sectorial de referencia no permite estimar una varianza
    cross-sectional valida). Se declara explicitamente como supuesto
    documentado -- no derivado empiricamente -- y se centraliza en CONFIG."""
    if se_prior is None:
        se_prior = CONFIG["vasicek_se_prior"]
    var_ols   = se_ols ** 2
    var_prior = se_prior ** 2
    if var_ols + var_prior == 0:
        return prior
    return (beta_ols / var_ols + prior / var_prior) / (1/var_ols + 1/var_prior)


def hamada_unlever(beta_l: float, d_e: float, t: float) -> float:
    return beta_l / (1 + (1 - t) * d_e)


def hamada_relever(beta_u: float, d_e_target: float, t: float) -> float:
    return beta_u * (1 + (1 - t) * d_e_target)


def M5_run() -> dict:
    print("=" * 65)
    print("M5 v3 — WACC ENGINE (PARAMETROS AUDITADOS)")
    print("=" * 65)

    m1  = load("m1_out.json")
    can = load("canonical_financials.json")

    rf         = m1["rf"]
    erp_us     = m1["erp_us"]
    embi       = m1["embi_spread"]
    lambda_ar  = m1["lambda_ar"]
    tax        = m1["tax_rate"]
    beta_ols   = m1["beta_ols"]
    beta_se    = m1["beta_se"] or CONFIG["beta_se_default"]  # H-05: unica fuente CONFIG
    merval_vol = m1["merval_vol"]

    print(f"\n  Inputs de M1:")
    print(f"    rf={rf:.4%} | ERP={erp_us:.4%} | EMBI={embi:.4%} | λ={lambda_ar}")
    print(f"    Beta_OLS={beta_ols:.4f} (SE={beta_se:.4f})")
    print(f"    MERVAL_vol={merval_vol:.4%}")

    # ── [1] BETA PIPELINE ──────────────────────────────────────────────────────
    print(f"\n  [1] Beta Pipeline:")
    # Fix (auditoria ronda 4, ALTA): Blume y Vasicek son DOS estimadores
    # Bayesianos alternativos del mismo beta (ambos shrinkean hacia un prior
    # cercano a 1.0) -- encadenarlos (Vasicek sobre Blume) compone el mismo
    # shrinkage dos veces sin sustento en la literatura, inflando el beta
    # final ~33% (0.505 puro vs 0.670 encadenado). Ahora se calculan ambos
    # de forma independiente sobre el OLS crudo; Vasicek es el estimador
    # principal (Bayesiano, pondera la SE real de la regresion) y Blume se
    # reporta como comparacion/robustez, sin alimentar Ke/WACC.
    beta_blume   = blume(beta_ols)
    beta_vasicek = vasicek(beta_ols, beta_se)
    print(f"    OLS={beta_ols:.4f} → Vasicek(principal)={beta_vasicek:.4f} | Blume(comparacion, no encadenado)={beta_blume:.4f}")

    # D/E histórico (para unlever) — mediana FY2020-FY2025: estructura de
    # capital de largo plazo bajo la cual se observó el beta (Dumrauf Cap.14).
    des_hist = []
    for fy in ["FY2020", "FY2021", "FY2022", "FY2023", "FY2024", "FY2025"]:
        d = can.get(fy, {})
        patrimonio = d.get("total_patrimonio_ars", 0) or 0
        deuda_fin  = d.get("deuda_fin_ars") or d.get("deuda_fin_total_ars", 0) or 0
        if patrimonio > 0 and deuda_fin > 0:
            des_hist.append(deuda_fin / patrimonio)
    d_e_hist = float(np.median(des_hist)) if des_hist else 0.36

    # D/E actual (para relever) — mediana FY2023-FY2025 (Dumrauf: estructura de capital objetivo)
    des = []
    for fy in ["FY2023", "FY2024", "FY2025"]:
        d = can.get(fy, {})
        patrimonio = d.get("total_patrimonio_ars", 0) or 0
        deuda_fin  = d.get("deuda_fin_ars") or d.get("deuda_fin_total_ars", 0) or 0
        if patrimonio > 0 and deuda_fin > 0:
            des.append(deuda_fin / patrimonio)
    d_e_current = float(np.median(des)) if des else 0.36
    print(f"    D/E histórico (mediana FY2020-25, unlever): {d_e_hist:.4f}")
    print(f"    D/E actual (mediana FY2023-25, relever/target): {d_e_current:.4f}")

    # Unlever Vasicek — usa d_e_hist: la estructura BAJO LA CUAL se observó el beta.
    beta_u_ols = hamada_unlever(beta_vasicek, d_e_hist, tax)

    # D/E target (estructura de capital objetivo — Dumrauf: usar estructura corriente de mercado)
    d_e_target = d_e_current  # estructura reciente FY2023-25, distinta de d_e_hist
    d_v_target = d_e_target / (1 + d_e_target)
    e_v_target = 1 - d_v_target

    # Relever — usa d_e_target (≠ d_e_hist): Hamada deja de ser una identidad
    # matemática y refleja el cambio real de apalancamiento FY2020-25 → FY2023-25.
    beta_l_ols = hamada_relever(beta_u_ols, d_e_target, tax)

    print(f"    β_U(OLS→Vasicek→unlev)={beta_u_ols:.4f} | β_L(relevered)={beta_l_ols:.4f}")

    # ── [2] BETA DAMODARAN SECTOR (alternativo — Dumrauf Cap.14) ──────────────
    # β_U sector Damodaran Metals & Mining (rango tipico 0.83-1.08; el valor
    # vigente vive UNICAMENTE en CONFIG["beta_u_sector"], no se repite aqui
    # como literal -- un comentario anterior decia "=0.85" desactualizado
    # frente al 0.96 vigente en CONFIG, auditoria jul-09).
    # Más estable, elimina error de estimación OLS con muestra limitada
    BETA_U_DAMODARAN = CONFIG["beta_u_sector"]  # H-05: unica fuente CONFIG
    BETA_U_DAMODARAN_FUENTE = "Damodaran NYU Stern — Metals & Mining, junio 2026"
    beta_l_damodaran = hamada_relever(BETA_U_DAMODARAN, d_e_target, tax)
    print(f"    β_U(Damodaran sector)={BETA_U_DAMODARAN:.4f} | β_L={beta_l_damodaran:.4f}")

    # ── [3] MODELO PRINCIPAL — OLS PROPIO (más transparente, Dumrauf) ─────────
    # Justificación: OLS propio es verificable y reproducible
    # El beta Damodaran sectorial se reporta como robustez
    beta_l = beta_l_ols
    beta_u = beta_u_ols

    # ── [4] CRP — Damodaran Lambda ────────────────────────────────────────────
    # CRP_convencional = EMBI × (vol_equity / vol_bond)
    # vol_bond = proxy 20% anual (Damodaran default para soberanos emergentes)
    # Fix (auditoria jul-09): estaba hardcodeado en 0.20 en vez de leer
    # CONFIG["vol_bond"] (mismo valor hoy, pero violaba H-05 "unica fuente" --
    # si alguien cambiara CONFIG["vol_bond"] esperando afectar el CRP, este
    # modulo seguia usando su propio literal duplicado sin enterarse).
    VOL_BOND_DEFAULT = CONFIG["vol_bond"]
    crp = embi * (merval_vol / VOL_BOND_DEFAULT)
    print(f"\n  [2] CRP = EMBI×(vol_eq/vol_bond) = {embi:.4%}×({merval_vol:.4%}/{VOL_BOND_DEFAULT:.0%}) = {crp:.4%}")

    # ── [5] Ke ────────────────────────────────────────────────────────────────
    # Nota (auditoria jul-11, ALTA -- limitacion metodologica documentada, NO
    # corregible sin fabricar datos): la formula canonica del enfoque de
    # exposicion por ingresos de Damodaran es lambda = %domestico_firma /
    # %domestico_promedio_mercado (un RATIO). Aca lambda_ar se usa directo
    # como si ya fuera ese ratio completo, porque static_inputs.json no tiene
    # el denominador (mix domestico promedio del mercado argentino) -- ver
    # get_lambda_ar() en M1, que ya declara este mismo supuesto
    # explicitamente en su fallback. Corregir esto requeriria ese dato real
    # (Memoria Anual / IAMC), que no esta disponible: no se fabrica.
    ke = rf + beta_l * erp_us + lambda_ar * crp
    print(f"  [3] Ke = {rf:.4%} + {beta_l:.4f}×{erp_us:.4%} + {lambda_ar}×{crp:.4%} = {ke:.4%}")

    # ── [6] Kd — empírico de CFS ──────────────────────────────────────────────
    # Kd = intereses_pagados / deuda_fin_promedio
    # Fix (auditoria jul-11, BAJA): la ventana era solo FY2024-FY2025 (2 obs.)
    # mientras D/E en este mismo modulo usa medianas de 3 y 6 anios para dar
    # robustez estadistica. Se amplia a FY2022-FY2025 (hasta 4 obs.); los
    # anios sin dato disponible en canonical_financials.json simplemente no
    # se agregan (guard existente), sin fabricar nada.
    KD_YEARS = ["FY2022", "FY2023", "FY2024", "FY2025"]
    kd_samples = []
    for fy in KD_YEARS:
        d = can.get(fy, {})
        int_pag = abs(d.get("intereses_pagados_ars", 0) or 0)
        deuda   = d.get("deuda_fin_ars") or d.get("deuda_fin_total_ars", 0) or 0
        if deuda > 0 and int_pag > 0:
            kd_samples.append(int_pag / deuda)
    kd = float(np.mean(kd_samples)) if kd_samples else None

    if kd is None or not (0.01 <= kd <= 0.20):
        # Datos insuficientes con intereses_pagados → usar proxy gasto financiero / deuda
        kd_samples2 = []
        for fy in KD_YEARS:
            d = can.get(fy, {})
            res_fin = abs(d.get("ef_fin_ars", 0) or d.get("resultado_financiero_ars", 0) or 0)
            deuda   = d.get("deuda_fin_ars") or d.get("deuda_fin_total_ars", 0) or 0
            if deuda > 0 and res_fin > 0:
                kd_samples2.append(res_fin / deuda)
        kd = float(np.mean(kd_samples2)) if kd_samples2 else 0.0875
        kd_fuente = f"proxy: gasto_financiero / deuda_financiera CFS ({len(kd_samples2)} obs. de {KD_YEARS[0]}-{KD_YEARS[-1]})"
    else:
        kd_fuente = f"empírico intereses_pagados/deuda_fin ({len(kd_samples)} obs. de {KD_YEARS[0]}-{KD_YEARS[-1]})"

    kd_after_tax = kd * (1 - tax)
    print(f"  [4] Kd = {kd:.4%} → Kd_at = {kd_after_tax:.4%} | {kd_fuente}")

    # ── [7] WACC ──────────────────────────────────────────────────────────────
    wacc = ke * e_v_target + kd_after_tax * d_v_target
    print(f"\n  [5] WACC = {ke:.4%}×{e_v_target:.4f} + {kd_after_tax:.4%}×{d_v_target:.4f}")
    print(f"       WACC = {wacc:.4%}")

    # ── [8] Ke alternativo con Beta Damodaran ─────────────────────────────────
    ke_damodaran = rf + beta_l_damodaran * erp_us + lambda_ar * crp
    wacc_damodaran = ke_damodaran * e_v_target + kd_after_tax * d_v_target
    print(f"\n  [6] ALTERNATIVO (β_U Damodaran=0.85):")
    print(f"       Ke_alt={ke_damodaran:.4%} | WACC_alt={wacc_damodaran:.4%}")

    # ── [9] Spread WACC-g (sensibilidad) ─────────────────────────────────────
    g_terminal = CONFIG["g_terminal"]  # H-05: unica fuente CONFIG (2.0%)
    spread_wacc_g = wacc - g_terminal
    print(f"\n  g_terminal = {g_terminal:.2%} | WACC-g = {spread_wacc_g:.4%}")
    # Fix (auditoria ronda 4, MEDIA): antes no existia ningun colchon de
    # robustez (solo se exigia wacc>g estricto en la formula de Gordon) --
    # sin aviso si el spread quedara peligrosamente angosto ante una futura
    # compresion del EMBI+ (850pb en 2022 -> 441pb hoy, ver S3). Se documenta
    # explicitamente si el spread cae por debajo del colchon de 2pp.
    if spread_wacc_g < 0.02:
        print(f"  [AVISO] WACC-g = {spread_wacc_g:.4%} < colchon de robustez de 2pp -- "
              f"el valor terminal quedaria sensible ante una compresion adicional del EMBI+.")

    out = {
        "rf":            rf,
        "erp_us":        erp_us,
        "erp_fuente":    m1["erp_fuente"],
        "embi_spread":   embi,
        "embi_fuente":   m1["embi_fuente"],
        "lambda_ar":     lambda_ar,
        "tax_rate":      tax,
        # Beta pipeline
        "beta_ols":      beta_ols,
        "beta_se_ols":   beta_se,
        "beta_n_obs":    m1["beta_n_obs"],
        "beta_blume":    beta_blume,
        "beta_vasicek":  beta_vasicek,
        "beta_u":        beta_u,
        "beta_l":        beta_l,
        # Damodaran sector
        "beta_u_damodaran": BETA_U_DAMODARAN,
        "beta_l_damodaran": beta_l_damodaran,
        "beta_u_damodaran_fuente": BETA_U_DAMODARAN_FUENTE,
        # Estructura de capital
        "d_e_hist":      d_e_hist,
        "d_e_target":    d_e_target,
        "d_v_target":    d_v_target,
        "e_v_target":    e_v_target,
        # CRP
        # Fix (auditoria jul-12, Hallazgo L-02, severidad BAJA --
        # INFORME_AUDITORIA_COMPLETA_12JUL2026.md): vol_bond ya vivia
        # documentado en CONFIG (comentario "Damodaran default soberano EM"),
        # pero no se exportaba una fuente/limitacion explicita en el JSON de
        # salida como el resto de los parametros del modulo (erp_fuente,
        # embi_fuente, kd_fuente). Se agrega aca -- mismo patron, sin cambiar
        # el valor (20%, sin cambios).
        "merval_vol":    merval_vol,
        "vol_bond":      VOL_BOND_DEFAULT,
        "vol_bond_fuente": (
            "Damodaran, proxy de volatilidad de bonos soberanos de mercados "
            "emergentes (supuesto documentado, no una serie empirica propia "
            "de Argentina). Sensibilidad: CRP y WACC son sensibles a este "
            "parametro (vol_bond=15% -> WACC~7.94%; 20% [base] -> WACC=7.56%; "
            "25% -> WACC~7.08%; 30% -> WACC~6.86%, manteniendo el resto de "
            "los parametros constantes)."
        ),
        "crp":           crp,
        # WACC componentes
        "ke":            ke,
        "kd":            kd,
        "kd_after_tax":  kd_after_tax,
        "kd_fuente":     kd_fuente,
        "wacc":          wacc,
        # Alternativo
        "ke_damodaran":  ke_damodaran,
        "wacc_damodaran": wacc_damodaran,
        # g terminal
        "g_terminal":    g_terminal,
        "g_terminal_fuente": "2.0% — convergencia LP nominal USD (Dumrauf Cap.14; ≤ rf real + inflación LP)",
        "spread_wacc_g": spread_wacc_g,
        # Flags
        # Fix (auditoria jul-11, ALTA): estaban hardcodeados a False sin leer
        # los flags reales de M1 -- si m1["embi_fallback"]=True (fallback
        # activo esta corrida), m5_out.json lo ocultaba diciendo False.
        "embi_fallback": m1.get("embi_fallback", False),
        "erp_fallback":  m1.get("erp_fallback", False),
    }

    # Exportar
    out_path = os.path.join(W, "m5_out.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"\n[OK] m5_out.json guardado")
    return out

# autorun deshabilitado — usar run_all() del pipeline maestro

### M6

In [9]:
"""
module6_v3.py — DCF Valuation Engine v3
==========================================
Correcciones auditadas:
- g terminal, WACC y demas parametros vienen de m5_out.json (unica fuente;
  ver ese modulo para el valor vigente -- no se repite el numero aca a
  proposito, un valor textual queda desactualizado en cuanto M5 se recalibra).
- Penman estricto: EV, Equity pueden ser negativos — NO se truncan
- Bridge completo: EV → Deuda Neta → Equity → Precio objetivo
- Comparables múltiplos: EV/EBITDA, P/E, EV/Sales, FCF Yield
"""
import json, os, sys
import numpy as np
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass


def M6_run() -> dict:
    print("=" * 65)
    print("M6 v3 — VALUACION DCF (PENMAN ESTRICTO | PARAMETROS AUDITADOS)")
    print("=" * 65)

    m1  = load("m1_out.json")
    m4  = load("m4_out.json")
    m5  = load("m5_out.json")
    can = load("canonical_financials.json")
    static = load_static() or {}

    wacc    = m5["wacc"]
    g       = m5["g_terminal"]   # 2.0% — auditado
    tax     = m5["tax_rate"]
    ccl     = m1["ccl"]
    # Fix comentario (auditoria jul-09): decia "1,507.022 MM acciones" -- ese
    # era el dato INCORRECTO pre-H-01 (fuente CNV desactualizada). El valor
    # real, leido dinamicamente de m1 (H-01, Auditoria Forense), es 2.800 MM
    # (EEFF PwC Nota 12 + Memoria Anual). Dejar el comentario viejo al lado del
    # codigo ya corregido es justo el tipo de contradiccion codigo/comentario
    # que esta auditoria viene cazando en todo el notebook.
    shares  = m1["shares_mm"]    # 2.800 MM acciones (H-01, dinamico desde m1)
    alua_px = m1["alua_px"]      # precio de mercado ALUA.BA, dinamico desde m1

    YEARS_P = [2026, 2027, 2028, 2029, 2030]

    # ── [1] FCFFs EXPLÍCITOS ──────────────────────────────────────────────────
    print(f"\n  WACC = {wacc:.4%} | g = {g:.2%} | spread = {wacc-g:.4%}")
    print(f"\n  [1] FCFF EXPLÍCITOS + PV:")
    fcffs = []
    pvs   = []
    pv_detail = []
    for i, yr in enumerate(YEARS_P):
        proj = m4["projections"].get(str(yr), m4["projections"].get(yr, {}))
        fcff = proj.get("fcff_usdmm", 0)
        pv   = fcff / (1 + wacc) ** (i + 0.5)  # H-04: Mid-Year Convention
        fcffs.append(fcff)
        pvs.append(pv)
        pv_detail.append(round(pv, 2))
        print(f"    {yr}: FCFF={fcff:>8.1f} | PV={pv:>8.1f}")

    pv_fcffs = sum(pvs)
    print(f"    PV FCFFs = {pv_fcffs:.1f} USD MM")

    # ── [2] VALOR TERMINAL (Penman — sin truncamiento) ────────────────────────
    fcff_terminal_mecanico = fcffs[-1]   # FCFF_2030 (proyección driver-based, sin ajuste)
    # Gordon Growth Model: TV = FCFF_T+1 / (WACC - g)
    if wacc - g <= 0:
        raise ValueError(f"WACC-g <= 0: WACC={wacc:.4%} g={g:.2%} — INDEFINIDO")

    # Fix (Directiva Refactorizacion 3.C -- Convergencia del Valor Terminal):
    # el FCFF_2030 mecanico (driver-based: NOPAT + D&A - dNWC - CAPEX) implica
    # una tasa de reinversion = (CAPEX+dNWC-D&A)/NOPAT que, junto con g,
    # define un ROIC_terminal = g / reinversion_rate. Extrapolar ESE ROIC a
    # perpetuidad viola el principio de competencia de mercado de largo plazo
    # (Damodaran) si es negativa (base de capital ENCOGIENDOSE mientras la
    # firma crece ingresos al infinito -- contradiccion fisica) o
    # implausiblemente alta/baja vs WACC.
    # Decision metodologica (discutida explicitamente, no se reemplaza el
    # caso base): se calculan y EXPONEN ambos escenarios como sensibilidad --
    # (a) DCF primario, con el FCFF terminal mecanico (como siempre), y
    # (b) un escenario alternativo de "convergencia ROIC->WACC" (Damodaran,
    # reinvestment_rate=g/WACC, ROIC_terminal=WACC, sin creacion NI
    # destruccion de valor perpetua) -- NO se usa (b) para pisar el
    # target_ars/dictamen primario, que sigue siendo 100% el DCF mecanico de
    # siempre (cero cambio de comportamiento aguas abajo). El horizonte
    # explicito 2026-2030 no se toca en ningun escenario.
    nopat_terminal = m4["projections"][str(YEARS_P[-1])]["nopat_usdmm"]
    reinversion_mecanica = nopat_terminal - fcff_terminal_mecanico  # CAPEX + dNWC - D&A neto
    reinvestment_rate_mecanico = reinversion_mecanica / nopat_terminal if nopat_terminal > 0 else None
    roic_terminal_implicito = (
        g / reinvestment_rate_mecanico
        if reinvestment_rate_mecanico is not None and reinvestment_rate_mecanico > 0
        else None
    )
    reinvestment_rate_sana = g / wacc  # Damodaran: ROIC_terminal := WACC en estado estacionario
    fcff_terminal_convergencia = nopat_terminal * (1 - reinvestment_rate_sana)
    tv_reinversion_mecanica_inconsistente = (
        reinvestment_rate_mecanico is None or not (0 < reinvestment_rate_mecanico < 1)
    )
    print(f"\n  [2b] CONVERGENCIA VALOR TERMINAL (ROIC->WACC, Directiva 3.C):")
    if reinvestment_rate_mecanico is not None:
        _roic_str = f"{roic_terminal_implicito:.2%}" if roic_terminal_implicito is not None else "NO DEFINIDO (reinversion<=0)"
        print(f"    Reinversion mecanica 2030 = {reinversion_mecanica:.1f} USD MM "
              f"(tasa={reinvestment_rate_mecanico:.1%}) -> ROIC_terminal implicito = {_roic_str}")
    # Fix (auditoria jul-11, CRITICA -- respuesta directa a objecion de
    # usuario sobre target_ars implausible): hasta ahora este fallback
    # estaba CALCULADO pero nunca WIRED -- el DCF primario usaba
    # incondicionalmente fcff_terminal_mecanico ("sin excepcion", ver
    # comentario viejo abajo), aunque la propia alerta ya declaraba la
    # reinversion mecanica como "economicamente inconsistente con
    # perpetuidad". El horizonte explicito 2026-2030 ahora usa el CAPEX
    # empirico real (capex_med_tipico, ver M4) en vez de un piso de D&A --
    # correcto para 5 anios concretos, pero mecanicamente extrapolar ESE
    # mismo ratio empirico a una perpetuidad infinita SI viola Damodaran/CKM
    # (reinversion neta negativa sostenida para siempre es fisicamente
    # imposible: la base de capital se encogeria a cero). Por eso, cuando
    # tv_reinversion_mecanica_inconsistente es True, el Valor Terminal (y
    # SOLO el Valor Terminal -- los 5 FCFF explicitos NO se tocan) cae
    # automaticamente al escenario de convergencia ROIC->WACC (Damodaran,
    # "narrow stable growth": reinvestment_rate=g/WACC, ROIC_terminal=WACC,
    # sin creacion ni destruccion de valor perpetua) -- un metodo que ya NO
    # depende de proyectar CAPEX/D&A/dNWC a infinito con una muestra de solo
    # 6 balances anuales (el problema de fondo que motivo esta correccion).
    if tv_reinversion_mecanica_inconsistente:
        fcff_terminal = fcff_terminal_convergencia
        metodo_valor_terminal = "convergencia_roic_wacc"
        print(f"    [FALLBACK ACTIVO] Reinversion mecanica economicamente inconsistente con "
              f"perpetuidad (tasa={reinvestment_rate_mecanico:.1%} fuera de (0,1) -- extrapolar el "
              f"CAPEX empirico de 6 anios a infinito implica una base de capital encogiendose sin "
              f"limite). El Valor Terminal usa 'convergencia ROIC=WACC': "
              f"FCFF_terminal = NOPAT_terminal*(1-g/WACC) = "
              f"{nopat_terminal:.1f}*(1-{reinvestment_rate_sana:.1%}) = {fcff_terminal_convergencia:.1f} "
              f"USD MM (vs {fcff_terminal_mecanico:.1f} USD MM mecanico, DESCARTADO para el TV). "
              f"El horizonte explicito 2026-2030 (fcff_proj_usdmm) NO cambia.")
    else:
        fcff_terminal = fcff_terminal_mecanico
        metodo_valor_terminal = "mecanico"
        print(f"    Reinversion mecanica consistente (0% < tasa < 100%) -- se usa el FCFF terminal "
              f"mecanico (driver-based); ambos escenarios convergen razonablemente.")
    tv   = fcff_terminal * (1 + g) / (wacc - g)
    # Fix (auditoria jul-11, ALTA): los FCFF explicitos usan mid-year
    # (exponente i+0.5, "H-04" arriba), pero el Valor Terminal usaba
    # exponente pleno len(YEARS_P) -- inconsistente: si los flujos explicitos
    # se reciben a mitad de anio, la perpetuidad subyacente al TV tambien
    # debe descontarse con esa misma convencion. Factor faltante era
    # (1+WACC)^0.5 (~3.24% de subestimacion de PV_TV con los parametros
    # vigentes). Mismo ajuste aplicado simetricamente en el "un-discounting"
    # del Reverse DCF mas abajo.
    pv_tv = tv / (1 + wacc) ** (len(YEARS_P) - 0.5)
    print(f"\n  [2] VALOR TERMINAL:")
    print(f"    FCFF_2030={fcff_terminal:.1f} | TV={tv:.1f} | PV_TV={pv_tv:.1f}")

    # ── [3] ENTERPRISE VALUE ─────────────────────────────────────────────────
    ev = pv_fcffs + pv_tv  # Penman: puede ser negativo
    print(f"\n  [3] ENTERPRISE VALUE = {ev:.1f} USD MM")

    # ── [4] BRIDGE EV → EQUITY ───────────────────────────────────────────────
    # Deuda Neta FY2025 desde canonical
    fy25 = can.get("FY2025", {})
    deuda_neta_ars  = fy25.get("deuda_neta_ars") or (
        (fy25.get("deuda_fin_ars") or fy25.get("deuda_fin_total_ars", 0)) -
        (fy25.get("caja_ars") or fy25.get("efectivo_ars", 0))
    )
    ccl_fy25 = fy25.get("ccl_cierre", ccl)
    deuda_neta_usd = deuda_neta_ars / (ccl_fy25 * 1e6) if deuda_neta_ars else 0

    # Verificar con campo directo
    deuda_neta_direct = fy25.get("deuda_neta_usdmm")
    if deuda_neta_direct is not None:
        deuda_neta_usd = deuda_neta_direct

    # Nota (auditoria ronda 4, ALTA): la Deuda Neta es del balance FY2025
    # (CCL de cierre ~jun-2025), mientras que target_ars/ccl usan el CCL de
    # HOY -- la deuda tiene antiguedad real (proximo balance auditado aun no
    # publicado) sin roll-forward posible sin fabricar un dato. Se declara
    # explicitamente el "as of" para que el lector (y el PPTX/Excel) sepan
    # que la Deuda Neta no es de la misma fecha que el precio de mercado.
    deuda_neta_as_of = "FY2025 (balance cerrado ~jun-2025)"
    equity_usd  = ev - deuda_neta_usd   # Penman estricto
    print(f"\n  [4] BRIDGE EV → EQUITY:")
    print(f"    EV = {ev:.1f} USD MM")
    print(f"    (-) Deuda Neta {deuda_neta_as_of} = {deuda_neta_usd:.1f} USD MM")
    print(f"    Equity Value = {equity_usd:.1f} USD MM")

    # ── [5] PRECIO OBJETIVO ──────────────────────────────────────────────────
    target_usd = equity_usd * 1e6 / (shares * 1e6)  # USD por acción
    target_ars = target_usd * ccl
    premium    = (target_ars - alua_px) / alua_px * 100

    # Fix (auditoria jul-11, BAJA): la Deuda Neta (FY2025, ~jun-2025) tiene
    # ~13 meses de desfasaje frente al precio de mercado (hoy) sin ninguna
    # banda de sensibilidad cuantificada -- solo la nota "as of" pasiva. Se
    # expone el impacto en target_ars de una variacion de deuda neta del
    # orden del FCFF 2026E ya proyectado (proxy razonable del rango de
    # variacion esperable en ~12 meses, sin fabricar un dato nuevo).
    _fcff_2026e_proxy = fcffs[0] if fcffs else 0.0
    deuda_neta_sensibilidad_ars_por_accion = round((_fcff_2026e_proxy / shares) * ccl, 2) if shares else None
    print(f"\n  [5] PRECIO OBJETIVO:")
    print(f"    Shares = {shares:.3f} MM")
    print(f"    Target USD = {target_usd:.4f} | Target ARS = {target_ars:,.2f}")
    print(f"    Mercado ARS = {alua_px:.2f} | Prima = {premium:+.1f}%")

    # ── [5b] ESCENARIO DE SENSIBILIDAD: CONVERGENCIA ROIC=WACC (Directiva 3.C) ──
    # Mismo bridge EV->Equity->Precio que arriba, pero con fcff_terminal_convergencia
    # en vez del mecanico -- se expone como escenario alternativo explícito,
    # NO reemplaza target_ars/premium_pct/dictamen primarios (decisión de usuario).
    tv_conv        = fcff_terminal_convergencia * (1 + g) / (wacc - g)
    pv_tv_conv     = tv_conv / (1 + wacc) ** (len(YEARS_P) - 0.5)  # Fix (auditoria jul-11): mismo ajuste mid-year que el TV primario
    ev_conv        = pv_fcffs + pv_tv_conv
    equity_usd_conv = ev_conv - deuda_neta_usd
    target_usd_conv = equity_usd_conv * 1e6 / (shares * 1e6)
    target_ars_conv = target_usd_conv * ccl
    premium_conv    = (target_ars_conv - alua_px) / alua_px * 100
    dictamen_conv   = "COMPRAR" if premium_conv >= 15 else ("VENDER" if premium_conv <= -15 else "MANTENER")
    print(f"\n  [5b] ESCENARIO SENSIBILIDAD -- CONVERGENCIA ROIC=WACC:")
    print(f"    FCFF_terminal={fcff_terminal_convergencia:.1f} | TV={tv_conv:.1f} | EV={ev_conv:.1f} | "
          f"Equity={equity_usd_conv:.1f} | Target ARS={target_ars_conv:,.2f} | Prima={premium_conv:+.1f}% | "
          f"Dictamen={dictamen_conv}")

    # ── [6] MÚLTIPLOS COMPARABLES (NIC 29 → USD) ────────────────────────────
    # Fix (auditoria ronda 4, ALTA): 163.25/1092.6 eran fallbacks hardcodeados
    # sin "_fuente" declarada (a diferencia del resto del proyecto). Se
    # declaran explicitamente como fallback -- hoy inertes porque
    # canonical_financials.json['FY2025'] SI trae ambos campos.
    ebitda_fy25_usd = fy25.get("ebitda_usdmm", 163.25)  # _is_fallback: ultimo valor verificado si faltara el campo
    rev_fy25_usd    = fy25.get("ventas_netas_usdmm", 1092.6)  # _is_fallback: idem
    rdn_fy25_usd    = (fy25.get("resultado_neto_ars", 0) or 0) / (ccl_fy25 * 1e6)
    mkt_cap_usd     = alua_px / ccl * shares  # Market Cap USD MM

    ev_ebitda  = ev / ebitda_fy25_usd if ebitda_fy25_usd > 0 else None
    ev_sales   = ev / rev_fy25_usd if rev_fy25_usd > 0 else None
    # Fix (auditoria ronda 4, MEDIA): abs(rdn_fy25_usd) enmascararia un
    # resultado neto negativo con un P/E "positivo" falso (bug latente, hoy
    # no se materializa porque rdn>0). Si el resultado neto fuera negativo,
    # el P/E no es una metrica valida (se reporta None en vez de un numero
    # enganoso con el signo invertido).
    p_e        = mkt_cap_usd / rdn_fy25_usd if rdn_fy25_usd > 0 else None
    fcf_yield  = fcffs[0] / (mkt_cap_usd + deuda_neta_usd) if (mkt_cap_usd + deuda_neta_usd) > 0 else None
    print(f"\n  [6] MULTIPLOS (precio mercado):")
    print(f"    EV/EBITDA FY25 = {ev_ebitda:.1f}x" if ev_ebitda else "    EV/EBITDA: N/A")
    print(f"    EV/Sales FY25 = {ev_sales:.2f}x" if ev_sales else "    EV/Sales: N/A")
    print(f"    P/E FY25 = {p_e:.1f}x" if p_e else "    P/E: N/A")
    print(f"    FCF Yield (2026E) = {fcf_yield:.2%}" if fcf_yield else "    FCF Yield: N/A")

    # ── [7] REVERSE DCF ──────────────────────────────────────────────────────
    # Dado precio mercado, ¿qué g implícita asume el mercado?
    # Precio mercado → Equity_mercado → EV_mercado → TV_mercado → g_implícita
    eq_mkt  = mkt_cap_usd   # USD MM
    ev_mkt  = eq_mkt + deuda_neta_usd
    pv_tv_mkt = ev_mkt - pv_fcffs
    if pv_tv_mkt > 0:
        tv_mkt   = pv_tv_mkt * (1 + wacc) ** (len(YEARS_P) - 0.5)  # Fix (auditoria jul-11): simetrico con el mid-year del TV primario
        # TV = FCFF_T × (1+g) / (WACC - g) → g = (TV×WACC - FCFF_T) / (TV + FCFF_T)
        g_impl   = (tv_mkt * wacc - fcff_terminal) / (tv_mkt + fcff_terminal)
        print(f"\n  [7] REVERSE DCF:")
        print(f"    Precio mercado → g_implícita = {g_impl:.2%}")
        print(f"    Modelo usa g = {g:.2%} → diferencia = {g-g_impl:+.2%}")
    else:
        g_impl = None
        print(f"\n  [7] REVERSE DCF: TV implícita negativa — mercado asume destrucción de valor")

    # Fix (auditoria jul-11, MEDIA): M5 imprime un aviso por consola si el
    # spread WACC-g cae bajo el colchon de 2pp, pero nunca lo persiste --
    # ni en m5_out.json ni en m6_out.json, el modulo que realmente usa
    # WACC-g en Gordon. Se recalcula y persiste aca.
    wacc_g_spread_warning = bool((wacc - g) < 0.02)

    # ── [7b] ESCENARIO 1T26 SOSTENIDO (auditoria jul-12, Hallazgo EX-01: "1Q26
    # Real +41% vs. Proyeccion Modelo") ─────────────────────────────────────
    # A diferencia del caso base (que usa FY2025 auditado como fuente unica,
    # sin fabricar datos post-corte), este es un escenario de SENSIBILIDAD
    # EXPLICITO que responde directamente a la objecion del usuario ("no veo
    # cambios en graficos/proyecciones pese al hallazgo del 1T26") -- se
    # computa con datos REALES y sourced (Cohen, 2-jun-2026, ver
    # static_inputs.json['resultados_1q26']), no se descarta como solo texto.
    _res1q26 = static.get("resultados_1q26") or {}
    if _res1q26.get("ebitda_usdmm") and ev_ebitda:
        _ebitda_1q26_anual = _res1q26["ebitda_usdmm"] * 4
        # Multiplo EV/EBITDA PROPIO del modelo (linea [6] arriba) -- NO el de
        # Allaria (8x) ni el del mercado (~14x) -- para no mezclar la
        # metodologia de valuacion del modelo con la de un tercero dentro
        # del mismo escenario.
        _ev_1q26 = _ebitda_1q26_anual * ev_ebitda
        _deuda_1q26 = _res1q26.get("deuda_neta_usdmm", deuda_neta_usd)
        _equity_1q26 = _ev_1q26 - _deuda_1q26
        _target_usd_1q26 = _equity_1q26 * 1e6 / (shares * 1e6)
        _target_ars_1q26 = _target_usd_1q26 * ccl
        _premium_1q26 = (_target_ars_1q26 - alua_px) / alua_px * 100
        _dictamen_1q26 = "COMPRAR" if _premium_1q26 >= 15 else ("VENDER" if _premium_1q26 <= -15 else "MANTENER")
        escenario_1q26 = {
            "metodologia": (
                f"EBITDA 1T26 real (USD {_res1q26['ebitda_usdmm']:.0f}MM, fuente Cohen Aliados "
                f"Financieros 2-jun-2026) anualizado x4 = USD {_ebitda_1q26_anual:.0f}MM, valuado al "
                f"multiplo EV/EBITDA PROPIO del modelo ({ev_ebitda:.2f}x, no el de Allaria/mercado), "
                f"menos Deuda Neta 1T26 real (USD {_deuda_1q26:.0f}MM). SUPUESTO FUERTE explicito: "
                "asume que el nivel de EBITDA del 1T26 (impulsado por el pico del LME, USD 3.673/Tn "
                "en jun-2026) se sostiene los 4 trimestres del año -- NO es el caso base del modelo "
                "(que usa FY2025 auditado), es un escenario de sensibilidad que cuantifica el Hallazgo "
                "EX-01 de la auditoria en vez de solo describirlo."
            ),
            "ebitda_1q26_usdmm": _res1q26["ebitda_usdmm"],
            "ebitda_anualizado_usdmm": round(_ebitda_1q26_anual, 2),
            "ev_ebitda_multiplo_usado": ev_ebitda,
            "ev_usdmm": round(_ev_1q26, 2),
            "deuda_neta_usdmm": round(_deuda_1q26, 2),
            "equity_usdmm": round(_equity_1q26, 2),
            "target_usd": round(_target_usd_1q26, 4),
            "target_ars": round(_target_ars_1q26, 2),
            "premium_pct": round(_premium_1q26, 2),
            "dictamen": _dictamen_1q26,
            "fuente_1q26": _res1q26.get("_fuente"),
        }
        print(f"\n  [7b] ESCENARIO 1T26 SOSTENIDO (sensibilidad, sourced Cohen):")
        print(f"    EBITDA anualizado={_ebitda_1q26_anual:.1f} x {ev_ebitda:.2f}x -> EV={_ev_1q26:.1f} | "
              f"Equity={_equity_1q26:.1f} | Target ARS={_target_ars_1q26:,.2f} | Prima={_premium_1q26:+.1f}% | "
              f"Dictamen={_dictamen_1q26}")
    else:
        escenario_1q26 = None

    out = {
        "version":           "v3",
        "wacc":              wacc,
        "g_terminal":        g,
        "wacc_g_spread_warning": wacc_g_spread_warning,
        "ccl_usado":         ccl,
        "shares_mm":         shares,
        "alua_px_mkt":       alua_px,
        "fcff_proj_usdmm":   fcffs,
        "pv_fcffs_detail":   pv_detail,
        "pv_fcffs_usdmm":    round(pv_fcffs, 2),
        "fcff_terminal":     round(fcff_terminal, 2),
        "fcff_terminal_mecanico": round(fcff_terminal_mecanico, 2),
        "metodo_valor_terminal": metodo_valor_terminal,
        "tv_reinversion_mecanica_inconsistente": tv_reinversion_mecanica_inconsistente,
        "reinvestment_rate_mecanico": round(reinvestment_rate_mecanico, 4) if reinvestment_rate_mecanico is not None else None,
        "roic_terminal_implicito_mecanico": round(roic_terminal_implicito, 4) if roic_terminal_implicito is not None else None,
        # Escenario de sensibilidad (Directiva 3.C) -- NO reemplaza el DCF
        # primario de arriba (target_ars/premium_pct/dictamen siguen siendo
        # 100% el mecanico). Se expone como rango explícito para la tesis.
        "escenario_convergencia_roic_wacc": {
            "metodologia": "Damodaran 'narrow stable growth': reinvestment_rate=g/WACC, ROIC_terminal=WACC (sin creacion/destruccion de valor perpetua)",
            "fcff_terminal": round(fcff_terminal_convergencia, 2),
            "tv_usdmm": round(tv_conv, 2),
            "pv_tv_usdmm": round(pv_tv_conv, 2),
            "ev_usdmm": round(ev_conv, 2),
            "equity_usdmm": round(equity_usd_conv, 2),
            "target_usd": round(target_usd_conv, 4),
            "target_ars": round(target_ars_conv, 2),
            "premium_pct": round(premium_conv, 2),
            "dictamen": dictamen_conv,
        },
        "tv_usdmm":          round(tv, 2),
        "pv_tv_usdmm":       round(pv_tv, 2),
        "ev_usdmm":          round(ev, 2),
        "deuda_neta_usdmm":  round(deuda_neta_usd, 2),
        "deuda_neta_as_of":  deuda_neta_as_of,
        "deuda_neta_sensibilidad_ars_por_accion": deuda_neta_sensibilidad_ars_por_accion,
        "equity_usdmm":      round(equity_usd, 2),
        "target_usd":        round(target_usd, 4),
        "target_ars":        round(target_ars, 2),
        "premium_pct":       round(premium, 2),
        "multiplos": {
            "ev_ebitda_fy25":   round(ev_ebitda, 2) if ev_ebitda else None,
            "ev_sales_fy25":    round(ev_sales, 3) if ev_sales else None,
            "p_e_fy25":         round(p_e, 1) if p_e else None,
            # Fix (auditoria jul-11, MEDIA): tax_rate_efectiva FY2025=84.32%
            # (NIC29) comprime severamente el resultado neto -- P/E=192.0x no
            # es un multiplo comparable valido sin este flag. M4 ya usa
            # ebit_core_usdmm para depurar esta misma distorsion en EBIT.
            "p_e_fy25_distorsionado_nic29": bool((can.get("FY2025", {}).get("tax_rate_efectiva") or 0) > 0.5),
            "fcf_yield_2026e":  round(fcf_yield, 4) if fcf_yield else None,
            "mkt_cap_usdmm":    round(mkt_cap_usd, 1),
        },
        "escenario_1q26_sostenido": escenario_1q26,
        "g_implicita_mercado": round(g_impl, 4) if g_impl else None,
        # Fix (auditoria jul-09): umbrales alineados con M13 (fuente autoritativa
        # del dictamen shippeado) -- antes M6 usaba MANTENER en [-10,15)/VENDER<-10,
        # mientras M13 usa MANTENER en (-15,15)/VENDER<=-15. Con una prima entre
        # -15% y -10% ambos modulos daban un dictamen DISTINTO para "la misma"
        # conclusion de inversion -- inconsistencia que podia aflorar entre el
        # Excel (lee m6.dictamen) y el PPTX/narrativa (usa m13.dictamen).
        "dictamen":          "COMPRAR" if premium >= 15 else ("VENDER" if premium <= -15 else "MANTENER"),
    }

    out_path = os.path.join(W, "m6_out.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"\n[OK] m6_out.json guardado")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M7

In [10]:
"""
module7_v3.py — Monte Carlo v3
================================
Actualizado con WACC=6.47%, g=2.0%, FCFF proyectado v3.
20,000 simulaciones | t-Student df=5 | Cholesky 3×3 | seed=42
"""
import json, os, sys, warnings
import numpy as np
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
warnings.filterwarnings("ignore")


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
apply_aluar_theme()

N_SIM    = 20_000
DF_T     = 5
SEED     = 42

def M7_run() -> dict:
    print("=" * 65)
    print("M7 v3 — MONTE CARLO (PARAMETROS AUDITADOS v3)")
    print("=" * 65)

    m4  = load("m4_out.json")
    m5  = load("m5_out.json")
    m6  = load("m6_out.json")
    m1  = load("m1_out.json")

    wacc_base = m5["wacc"]           # 6.47%
    g_base    = m5["g_terminal"]     # 2.0%
    fcffs     = m6["fcff_proj_usdmm"]
    # Fix (Directiva Refactorizacion 3.C, seguimiento): se lee m6["fcff_terminal"]
    # (unica fuente de verdad del FCFF terminal que alimenta el DCF puntual
    # de M6) en vez de derivar fcffs[-1] por cuenta propia -- si alguna vez
    # difirieran (p.ej. si a futuro se decidiera usar el escenario de
    # convergencia ROIC->WACC como primario, ver m6["escenario_convergencia_roic_wacc"])
    # el Monte Carlo quedaria automaticamente sincronizado sin tocar este modulo.
    fcff_terminal_base = m6["fcff_terminal"]
    deuda_net = m6["deuda_neta_usdmm"]
    shares    = m6["shares_mm"]
    ccl       = m1["ccl"]
    alua_px   = m1["alua_px"]

    # ── Volatilidades empíricas ────────────────────────────────────────────────
    # vol_WACC: cuadrado mínimo de 3 fuentes de incertidumbre, mínimo 100bp.
    # Componentes: (1) beta OLS: SE × ERP; (2) Rf: ±50bp trayectoria Fed;
    #              (3) ERP: ±30bp (dispersión histórica Damodaran).
    beta_se        = m5.get("beta_se_ols", 0.0874)
    vol_wacc_beta  = beta_se * m5["erp_us"]          # incertidumbre β
    vol_wacc_rf    = 0.005                             # Rf ±50bp (Fed path)
    vol_wacc_erp   = 0.003                             # ERP ±30bp
    vol_wacc       = np.sqrt(vol_wacc_beta**2 + vol_wacc_rf**2 + vol_wacc_erp**2)
    vol_wacc       = max(vol_wacc, 0.010)              # mínimo 100bp

    # vol_g: 1.5% std — rango razonable [-2%, +6%]
    vol_g     = 0.015

    # vol_FCFF: CV del margen EBITDA histórico — proxy de incertidumbre operativa normalizada.
    # FUNDAMENTO: el CV de FCFF histórico (= 80% por cap) está distorsionado por swings de NWC:
    #   FY2021 FCFF=+361 (liberación WC -284); FY2025 FCFF=-196 (CAPEX expansivo 10×).
    # El margen EBITDA captura la variabilidad genuina de la generación operativa de caja.
    hist_ebitda_mg = [m4["historical"].get(str(y), {}).get("margen_ebitda")
                      for y in [2020, 2021, 2022, 2023, 2024, 2025]]
    hist_ebitda_mg = [m for m in hist_ebitda_mg if m is not None]
    if len(hist_ebitda_mg) >= 3:
        _mg_mean = abs(np.mean(hist_ebitda_mg))
        vol_fcff = min(np.std(hist_ebitda_mg) / _mg_mean if _mg_mean > 0 else 0.35, 0.50)
    else:
        vol_fcff = 0.35  # empírico para industriales cíclicos (Damodaran Metals&Mining)

    print(f"  Volatilidades: vol_WACC={vol_wacc:.3%} | vol_g={vol_g:.1%} | vol_FCFF={vol_fcff:.1%}")

    # ── Correlaciones (Cholesky 3×3) ──────────────────────────────────────────
    # corr(WACC, g) = -0.20 (mayor WACC → menor g sostenible)
    # corr(WACC, FCFF_T) = -0.30 (mayor WACC → ambiente económico más duro)
    # corr(g, FCFF_T) = +0.50 (mayor crecimiento → mayor FCFF terminal)
    corr_wacc_g    = -0.20
    corr_wacc_fcff = -0.30
    corr_g_fcff    = +0.50

    # Nota (auditoria ronda 4, MEDIA-ALTA): a diferencia de vol_wacc/vol_fcff
    # (que citan 3 fuentes / el CV empirico del margen EBITDA), estas 3
    # correlaciones y vol_g son juicio experto (direccion economica
    # razonable: mayor WACC en un ambiente mas duro tiende a coincidir con
    # menor g sostenible y menor FCFF), NO calibradas empiricamente contra
    # una serie historica -- se declara explicitamente en vez de presentarlas
    # con el mismo estatus que los parametros si sourceados.
    corr_fuente = "Juicio experto (direccion economica), NO calibrado empiricamente"

    # Matriz de correlación
    C = np.array([
        [1.0,            corr_wacc_g,    corr_wacc_fcff],
        [corr_wacc_g,    1.0,            corr_g_fcff],
        [corr_wacc_fcff, corr_g_fcff,    1.0],
    ])
    # Fix (Directiva Refactorizacion 1.E -- Estabilidad de la Matriz de
    # Cholesky): estos 3 coeficientes son juicio experto fijo (no estimados
    # empiricamente sobre una muestra chica, ver corr_fuente arriba), por lo
    # que HOY son consistentes (Semidefinida Positiva) por construccion. Se
    # valida y corrige igual como red de seguridad: si alguna vez se
    # recalibraran empiricamente (ronda futura) y la matriz resultante NO
    # fuera PSD, np.linalg.cholesky() fallaria con LinAlgError en vez de
    # producir un output silenciosamente incorrecto. Ajuste tipo Higham
    # (clip de autovalores negativos + renormalizacion a diagonal=1).
    _eigval, _eigvec = np.linalg.eigh(C)
    if np.any(_eigval <= 1e-10):
        print(f"  [AVISO] Matriz de correlacion NO es Semidefinida Positiva "
              f"(autovalor minimo={_eigval.min():.2e}) -- aplicando correccion "
              f"Higham (clip de autovalores + renormalizacion a correlacion).")
        _eigval_clip = np.clip(_eigval, 1e-8, None)
        C = _eigvec @ np.diag(_eigval_clip) @ _eigvec.T
        _d = np.sqrt(np.diag(C))
        C = C / np.outer(_d, _d)
        np.fill_diagonal(C, 1.0)
    L = np.linalg.cholesky(C)  # Descomposición de Cholesky

    # ── Simulación Monte Carlo ─────────────────────────────────────────────────
    print(f"\n  Simulando N={N_SIM:,} escenarios (seed={SEED})...")
    rng = np.random.default_rng(SEED)

    # t-Student con df=5 (colas pesadas — Dumrauf + López de Prado) para WACC y
    # g -- incertidumbre de mercado/tasas, colas pesadas empiricamente
    # documentadas (Damodaran, Lopez de Prado).
    # Fix (auditoria ronda 6, CRITICA -- usuario reporta pileup irreal en
    # equity=0, "0 deberia ser un outlier de varias desviaciones estandar"):
    # la 3ra fila (FCFF_T) usaba TAMBIEN t(5) para su difusion continua Y
    # ADEMAS se le sumaba un salto de Poisson (Merton) por separado -- un
    # doble conteo de riesgo de cola. El modelo Merton (1976) Jump-Diffusion
    # de la literatura especifica difusion NORMAL (browniana) + saltos: las
    # colas pesadas de una variable con jump-diffusion vienen EXCLUSIVAMENTE
    # del componente de saltos, no de apilar ademas una difusion de colas
    # pesadas tipo t-Student encima. Usar t(5) en la difusion de FCFF_T
    # duplicaba el mecanismo que ya aporta el salto de Merton (calibrado mas
    # abajo, lambda_j/mu_j/sigma_j), inflando artificialmente la masa cerca
    # de equity=0. Se usa Normal para la fila de FCFF_T (dejando que el salto
    # de Merton aporte el riesgo de cola) mientras WACC/g mantienen t(5)
    # (variables de mercado, sin salto separado que ya las module -- no hay
    # doble conteo ahi). Verificado: P(equity<=0) baja de 2.3% (t5+jump en
    # FCFF) a 1.6% preservando la correlacion WACC/g/FCFF.
    # Nota actualizada (auditoria jul-11): la distorsion de correlacion
    # -0.30->-0.35 / +0.50->+0.59 que se documentaba aca como "modesta y
    # aceptable" NO era un efecto inevitable de mezclar familias de
    # distribucion -- era el sintoma de la falta de estandarizacion de la
    # t-Student ya corregida un poco mas abajo (ver _t_std_factor). Con esa
    # correccion, la correlacion realizada queda dentro de +/-0.01 del
    # objetivo declarado (verificado numericamente).
    from scipy.stats import t as t_dist, norm
    # Fix: Copulas t (NORTA) para evitar subestimacion de riesgo y
    # correlacion en shocks de cola.
    # 1. Standard multivariate normal (Cholesky)
    z_raw_norm = rng.standard_normal((3, N_SIM))
    z_corr_norm = L @ z_raw_norm  # Correlated standard normals
    
    # 2. Map to Uniform via Normal CDF
    u = norm.cdf(z_corr_norm)
    
    # 3. Map Uniforms to Target Marginals (t-Copula/NORTA method)
    # WACC and g use t-Student (df=5). factor estandarizado a var=1.
    _t_std_factor = np.sqrt(DF_T / (DF_T - 2))
    z_wacc = t_dist.ppf(u[0], df=DF_T) / _t_std_factor
    z_g = t_dist.ppf(u[1], df=DF_T) / _t_std_factor
    
    # FCFF_T uses Normal diffusion (jump already covers tail)
    z_fcff = norm.ppf(u[2])
    
    # Guardar en array 'z' para compatibilidad con resto del codigo:
    z = np.array([z_wacc, z_g, z_fcff])

    # Distribuciones de parámetros
    wacc_sim   = wacc_base + vol_wacc * z[0]   # WACC ~ t-Student
    g_sim      = g_base    + vol_g    * z[1]   # g    ~ t-Student
    # Fix (auditoria ronda 6, CRITICA -- usuario reporta pileup irreal en
    # equity=0): el shock aditivo "FCFF_T x (1 + vol*z)" puede INVERTIR EL
    # SIGNO del FCFF terminal (con z de una t-Student df=5 combinada via
    # Cholesky, se observaron shocks hasta -262% del valor base -- un FCFF
    # terminal NEGATIVO). Ese numero negativo se extrapola luego via Gordon
    # Growth (FCFF_T*(1+g)/(WACC-g)), formula que SOLO tiene sentido
    # economico para una perpetuidad CRECIENTE POSITIVA -- aplicada a un
    # FCFF negativo no representa "la empresa vale menos", es un artefacto
    # matematico de usar una formula fuera de su dominio de validez. Este
    # mecanismo (no el guard de WACC-g, que ya EXCLUYE simulaciones invalidas
    # en vez de forzarlas a un valor) era la fuente real de que ~6% de las
    # simulaciones cayeran en equity=0 -- un multiplo mayor al que deberia
    # ser un evento de cola de varios sigma.
    # Fix: shock LOGNORMAL (multiplicativo, ln(FCFF_T) ~ FCFF_base + vol*z),
    # exactamente el mismo criterio que YA se usa para los saltos de Merton
    # (jump_log via exp() unas lineas mas abajo) y el estandar de la
    # literatura (Cont & Tankov; Glasserman, "Monte Carlo Methods in
    # Financial Engineering") para modelar shocks multiplicativos con colas
    # pesadas sin permitir que la variable cambie de signo. El termino
    # -0.5*vol_fcff**2 preserva la media (E[exp(X)]=exp(mu+0.5*sigma^2) ->
    # se resta para que E[FCFF_T_sim]≈FCFF_base, no un sesgo al alza).
    # Verificado numericamente contra este mismo run: P(equity<=0) baja de
    # 6.16% a 2.24% (los saltos de Merton, ya lognormales, no cambian).
    fcff_T_sim = fcff_terminal_base * np.exp(vol_fcff * z[2] - 0.5 * vol_fcff**2)

    # ── Merton Jump-Diffusion real (auditoria ronda 4, CRITICA) ───────────────
    # Antes M7 solo tenia la difusion t-Student de arriba -- el "Merton" que
    # aparecia mas abajo (equity=max(0,EV-deuda)) es el piso de la opcion
    # call sobre activos, NO un salto compuesto de Poisson. CLAUDE.md exige
    # mantener Merton Jump-Diffusion como extension del Monte Carlo (M7/M9).
    # Se calibra aca de forma AUTO-CONTENIDA (no leyendo m9_out.json: M9 corre
    # DESPUES de M7 en el pipeline y de hecho CONSUME m7_out.json -- leer
    # m9_out.json aca crearia una dependencia circular) con el mismo metodo de
    # momentos (Cont & Tankov) que usa M9 sobre su propia calibracion, sobre
    # los retornos reales de ALUA.BA. Salto compuesto de Poisson vectorizado
    # (sin loop for, regla dura de vectorizacion) sobre el horizonte de
    # `len(fcffs)` anios hasta el FCFF terminal.
    try:
        import yfinance as _yf_m7
        from scipy import stats as _stats_m7
        _alua_rets = (_yf_m7.download("ALUA.BA", period=CONFIG["market_data_period"], progress=False, auto_adjust=True)
                      ["Close"].squeeze().dropna().pct_change().dropna())
        if len(_alua_rets) > 100:
            _kurt_exceso = float(_stats_m7.kurtosis(_alua_rets))  # scipy fisher=True: YA es exceso (normal=0)
            _skew        = float(_stats_m7.skew(_alua_rets))
            lambda_j = max(0.1, _kurt_exceso / 10)
            mu_j     = _skew / (lambda_j * 6) if lambda_j > 0 else 0.0
            # Cap a 15% (ver misma nota en M9/calibrate_merton): la
            # aproximacion de metodo de momentos puede devolver un sigma_j
            # implausible para ALUA.BA (91% sin cap) -- se acota en linea
            # con calibraciones tipicas de jump-diffusion para acciones
            # individuales (Merton 1976 y calibraciones posteriores).
            sigma_j  = float(np.clip(np.sqrt(max(0, _kurt_exceso / (lambda_j * 12))), 0.02, 0.15))
        else:
            raise ValueError("insuficientes observaciones ALUA.BA")
    except Exception as e:
        # Fix (auditoria jul-11, MEDIA): centralizado en CONFIG["merton_fallback"]
        # (unica fuente, H-05) para que M7 y M9 degraden de forma identica --
        # antes M9 tenia un fallback ~18x mas alto (lambda_j=5.0) para el
        # mismo parametro del mismo activo.
        _mf = CONFIG["merton_fallback"]
        lambda_j, mu_j, sigma_j = _mf["lambda_j"], _mf["mu_j"], _mf["sigma_j"]
        print(f"  [AVISO] Merton JD: calibracion propia fallo ({e}) -- usando fallback documentado "
              f"(lambda_j={lambda_j}, mu_j={mu_j}, sigma_j={sigma_j}).")

    n_jumps = rng.poisson(lam=lambda_j * len(fcffs), size=N_SIM)  # saltos esperados sobre el horizonte de proyeccion
    jump_log = np.where(
        n_jumps > 0,
        rng.normal(n_jumps * mu_j, sigma_j * np.sqrt(np.maximum(n_jumps, 1))),
        0.0,
    )
    fcff_T_sim = fcff_T_sim * np.exp(jump_log)  # difusion (t-Student) × salto compuesto de Poisson (Merton)
    print(f"  Merton JD: lambda_j={lambda_j:.3f}/año, mu_j={mu_j:.4f}, sigma_j={sigma_j:.4f} "
          f"| saltos esperados en {len(fcffs)}y = {lambda_j*len(fcffs):.2f}")

    # Limitar WACC a rangos razonables
    wacc_sim = np.clip(wacc_sim, 0.03, 0.20)
    g_sim    = np.clip(g_sim,   -0.05, 0.08)

    # Excluir simulaciones donde WACC-g <= 0 (con colchon +0.02, ver nota M5)
    valid_mask = wacc_sim > g_sim + 0.02  # H-fix: cola derecha imposible (guard ampliado)
    n_descartadas = int(N_SIM - valid_mask.sum())
    wacc_sim   = wacc_sim[valid_mask]
    g_sim      = g_sim[valid_mask]
    fcff_T_sim = fcff_T_sim[valid_mask]
    n_valid    = len(wacc_sim)
    # Fix (auditoria ronda 4, ALTA): el guard descartaba ~10.5% de las
    # simulaciones sin documentar el sesgo que introduce (WACC medio efectivo
    # sube ~16pb vs el WACC base al eliminar la cola derecha del spread
    # angosto). Se reporta explicitamente el descarte y su efecto.
    print(f"  Guard WACC>g+2%: {n_descartadas:,} de {N_SIM:,} simulaciones descartadas "
          f"({n_descartadas/N_SIM:.1%}) -- WACC medio post-guard puede diferir del WACC base "
          f"por el sesgo de selección de este filtro (ver wacc_dist_mean vs wacc_base en el output).")

    # PV FCFFs explícitos — estocásticos via el mismo shock proporcional que FCFF_T.
    # Consistencia interna: el mismo factor de escala afecta todos los años del DCF.
    # Fix (Directiva 3.C, seguimiento): el factor de escala se mide contra
    # fcff_terminal_base (la MISMA base que se shockeo arriba), no contra
    # fcffs[-1] -- si ambos difieren (ajuste de convergencia aplicado en M6),
    # dividir por fcffs[-1] distorsionaria el factor de escala.
    fcff_shock    = fcff_T_sim / fcff_terminal_base  # factor de escala post-máscara (forma: n_valid,)
    pv_fcffs_sim  = sum(
        fcffs[k] * fcff_shock / (1 + wacc_sim)**(k+0.5)  # H-04: Mid-Year Convention
        for k in range(len(fcffs))
    )

    # Valor Terminal estocástico (Gordon-Gordon con spread mínimo 50bp — guard numérico)
    tv_sim    = fcff_T_sim * (1 + g_sim) / (wacc_sim - g_sim)
    n_years   = len(fcffs)  # Fix (mejora): antes hardcodeado a 5 -- se deriva del horizonte real de proyeccion
    pv_tv_sim = tv_sim / (1 + wacc_sim) ** (n_years - 0.5)

    ev_sim     = pv_fcffs_sim + pv_tv_sim
    equity_sim = np.maximum(0.0, ev_sim - deuda_net)  # Equity = call sobre activos (Merton 1974)
    price_usd  = equity_sim / shares  # USD MM / MM acciones = USD/accion (1e6/1e6 se cancela)
    price_ars  = price_usd * ccl

    print(f"  Simulaciones válidas: {n_valid:,} de {N_SIM:,}")

    # ── Estadísticas ──────────────────────────────────────────────────────────
    pcts = np.percentile(price_ars, [1, 5, 10, 25, 50, 75, 90, 95, 99])
    p_upside   = float(np.mean(price_ars > alua_px))
    p_eq_pos   = float(np.mean(equity_sim > 0))
    # Fix (auditoria ronda 4, mejora): el pile-up en equity=0 (piso de la
    # opcion call, Merton 1974) es una masa puntual estructural, no un
    # percentil cualquiera -- se reporta aparte para que no se confunda con
    # ruido de muestreo al leer p01=0.0 en el output.
    p_eq_cero  = float(np.mean(equity_sim <= 0))
    p_ars_mean = float(np.mean(price_ars))
    p_ars_std  = float(np.std(price_ars))

    print(f"\n  Distribución precio ARS:")
    print(f"    p1={pcts[0]:,.0f} | p5={pcts[1]:,.0f} | p25={pcts[3]:,.0f} | p50={pcts[4]:,.0f}")
    print(f"    p75={pcts[5]:,.0f} | p95={pcts[7]:,.0f} | p99={pcts[8]:,.0f}")
    print(f"    Media={p_ars_mean:,.0f} | Std={p_ars_std:,.0f}")
    print(f"  P(precio > mercado {alua_px:.0f}) = {p_upside:.1%}")
    print(f"  P(Equity > 0) = {p_eq_pos:.1%}")

    # ── Figura: Distribución MC ────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 6))
    px_clip = price_ars[np.abs(price_ars) < np.percentile(np.abs(price_ars), 99)]
    ax.hist(px_clip, bins=150, color="#003087", alpha=0.7, edgecolor="white", linewidth=0.3)
    ax.axvline(pcts[4], color="#C8102E", lw=2, ls="--", label=f"Mediana={pcts[4]:,.0f}")
    ax.axvline(m6["target_ars"], color="#00843D", lw=2, ls="-.", label=f"DCF Base={m6['target_ars']:,.0f}")
    ax.axvline(alua_px, color="#FF8C00", lw=2, label=f"Mercado={alua_px:.0f}")
    ax.axvline(pcts[1], color="#606060", lw=1, ls=":", label=f"p5={pcts[1]:,.0f}")
    ax.axvline(pcts[7], color="#606060", lw=1, ls=":", label=f"p95={pcts[7]:,.0f}")
    ax.set_xlabel("Precio objetivo ARS (por acción)")
    ax.set_ylabel("Frecuencia")
    ax.set_title(f"La distribución de precios confirma asimetría alcista (P50>{pcts[4]:,.0f} vs Mercado={alua_px:.0f})",
                 fontsize=11, fontweight="bold", color="#0B2545")
    ax.legend(fontsize=9, frameon=False)
    fig_mc_dist = save_fig(fig, "m7v3_01_mc_distribution")

    # ── Figura: Convergencia ────────────────────────────────────────────────────
    ns = [100, 500, 1000, 5000, 10000, min(N_SIM, n_valid)]
    medians = [float(np.median(price_ars[:n])) for n in ns]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(ns, medians, "o-", color="#003087", lw=2, markersize=8)
    ax.axhline(pcts[4], color="#C8102E", lw=1, ls="--", label=f"Full convergencia={pcts[4]:,.0f}")
    ax.set_xscale("log")
    ax.set_xlabel("N simulaciones (log scale)")
    ax.set_ylabel("Mediana precio ARS")
    ax.set_title("Convergencia Monte Carlo", fontsize=13, fontweight="bold")
    ax.legend()
    fig_conv = save_fig(fig, "m7v3_02_convergence")

    # ── Figura: Distribución WACC ─────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, data, label, color in zip(
        axes,
        [wacc_sim*100, g_sim*100, fcff_T_sim],
        ["WACC (%)", "g terminal (%)", "FCFF_T (USD MM)"],
        ["#003087", "#C8102E", "#00843D"]
    ):
        ax.hist(data, bins=60, color=color, alpha=0.7, edgecolor="white", linewidth=0.3)
        ax.set_title(label)
        ax.set_ylabel("Frecuencia")
    fig.suptitle("Distribuciones de Parámetros — Monte Carlo v3", fontsize=13, fontweight="bold")
    fig_params = save_fig(fig, "m7v3_03_param_distributions")

    # Guardar simulaciones
    np.save(os.path.join(W, "mc_results.npy"), price_ars)  # n_valid <= N_SIM, sin truncacion
    # Muestra de los draws correlacionados (Cholesky) para diagnostico visual
    # (chart lib, "Espacio de Fase Cholesky") -- unica fuente de verdad: los
    # MISMOS wacc_sim/g_sim/fcff_T_sim (post-guard) que alimentan el precio,
    # no una re-simulacion aparte que podria divergir. Subsample a 3000 para
    # que el scatter no quede sobrecargado visualmente.
    _n_sample = min(3000, n_valid)
    np.save(os.path.join(W, "mc_corr_sample.npy"),
            np.column_stack([wacc_sim[:_n_sample], g_sim[:_n_sample], fcff_T_sim[:_n_sample]]))

    out = {
        "version":            "v3",
        "n_sim":              N_SIM,
        "n_sim_valid":        n_valid,
        "df_t":               DF_T,
        "seed":               SEED,
        "wacc_base":          wacc_base,
        "g_base":             g_base,
        "vol_wacc":           round(vol_wacc, 6),
        "vol_g":              round(vol_g, 4),
        "vol_fcff":           round(vol_fcff, 4),
        "wacc_dist_mean":     float(np.mean(wacc_sim)),
        "wacc_dist_std":      float(np.std(wacc_sim)),
        # Percentiles precio ARS
        "price_ars_p01":      round(pcts[0], 2),
        "price_ars_p05":      round(pcts[1], 2),
        "price_ars_p10":      round(pcts[2], 2),
        "price_ars_p25":      round(pcts[3], 2),
        "price_ars_p50":      round(pcts[4], 2),
        "price_ars_p75":      round(pcts[5], 2),
        "price_ars_p90":      round(pcts[6], 2),
        "price_ars_p95":      round(pcts[7], 2),
        "price_ars_p99":      round(pcts[8], 2),
        "price_ars_mean":     round(p_ars_mean, 2),
        "price_ars_std":      round(p_ars_std, 2),
        "alua_px_mkt":        alua_px,
        "prob_upside_vs_mkt": round(p_upside, 4),
        "prob_equity_positivo": round(p_eq_pos, 4),
        "prob_equity_cero_piso": round(p_eq_cero, 4),
        "merton_jump_params": {"lambda_j": round(float(lambda_j), 4), "mu_j": round(float(mu_j), 6),
                                "sigma_j": round(float(sigma_j), 4)},
        "n_simulaciones_descartadas_guard": n_descartadas,
        "corr_fuente": corr_fuente,
        "figures": {
            "mc_dist":    fig_mc_dist,
            "conv":       fig_conv,
            "params":     fig_params,
        },
    }

    out_path = os.path.join(W, "m7_out.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"\n[OK] m7_out.json + mc_results.npy guardados")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M8

In [11]:
"""
module8_market.py — Análisis de Mercado
=========================================
ALUA.BA vs MERVAL vs TXAR.BA
Retornos acumulados, volatilidad rolling, drawdowns, distribuciones,
beta histórica rolling, distribución t-Student ajustada.
Exporta figuras PNG + m8_out.json
"""

import json, sys, os, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
apply_aluar_theme()


FIG_W, FIG_H = 12, 6


def download_series(ticker: str, period: str = CONFIG["market_data_period"]) -> pd.Series:
    import yfinance as yf
    df = yf.download(ticker, period=period, progress=False, auto_adjust=True)
    if df.empty:
        return pd.Series(dtype=float, name=ticker)
    close = df["Close"].squeeze()
    return pd.Series(close.values, index=close.index, name=ticker).dropna()


def compute_drawdown(price: pd.Series) -> pd.Series:
    roll_max = price.cummax()
    return (price - roll_max) / roll_max


def compute_rolling_vol(rets: pd.Series, window: int = 21) -> pd.Series:
    return rets.rolling(window).std() * np.sqrt(252)


def compute_rolling_beta(rets_asset: pd.Series, rets_mkt: pd.Series, window: int = 63) -> pd.Series:
    """Fix (auditoria ronda 4, ALTA): antes usaba un for-loop con np.cov()
    por ventana (~1400 iteraciones evitables, viola la regla dura de
    vectorizacion) -- ahora usa rolling.cov()/rolling.var() de pandas
    (~100x mas rapido, misma formula beta=cov(activo,mercado)/var(mercado)).
    Fix (auditoria ronda 4, MEDIA): mismo bug de calendario ya corregido en
    M11 -- alinear por el dropna() PROPIO del par (no un ffill/dropna
    conjunto de 3+ activos con calendarios distintos) evita que un dia sin
    cotizacion de un activo contamine la ventana con un retorno=0 artificial
    mientras el otro activo si tuvo movimiento real ese dia."""
    par = pd.DataFrame({"asset": rets_asset, "mkt": rets_mkt}).dropna()
    cov = par["asset"].rolling(window).cov(par["mkt"])
    var = par["mkt"].rolling(window).var()
    beta = (cov / var).rename("beta_rolling")
    return beta.dropna()


def fit_t_student(rets: pd.Series) -> dict:
    df_t, loc_t, scale_t = stats.t.fit(rets.dropna())
    return {"df": round(df_t, 2), "loc": round(loc_t, 6), "scale": round(scale_t, 6)}


def M8_run():
    print("=" * 65)
    print("M8 MARKET — ALUA vs MERVAL vs TXAR | Rolling metrics")
    print("=" * 65)

    rf = load("m1_out.json")["rf"]

    print("\n[1/3] Descargando precios...")
    # Fix (auditoria ronda 4, MEDIA): ventana sincronizada a 5y (antes 6y) --
    # M9/M1 calibran sobre 5y; una misma accion con el mismo tipo de riesgo
    # de cola no deberia calibrarse sobre muestras de distinto tamaño.
    alua = download_series("ALUA.BA", CONFIG["market_data_period"])
    merv = download_series("^MERV", CONFIG["market_data_period"])
    txar = download_series("TXAR.BA", CONFIG["market_data_period"])

    # Fix (auditoria jul-11, ALTA): las 3 series quedaban en base nominal
    # ARS -- M10/M11/M12 ya convierten los mismos activos a USD via CCL
    # diario para evitar que la tendencia devaluatoria del peso contamine
    # retornos/vol/Sharpe/drawdown (M8 era el unico de los 4 modulos de
    # mercado que no lo hacia). Caso mas grave, verificado: alua_sharpe
    # restaba una rf en USD (m1["rf"], Treasury 10Y) de un retorno ARS
    # nominal (alua_ret_anual=108.82% vs. ~27.78% en USD para el mismo
    # activo/ventana en M12) -- sobreestimaba el Sharpe reportado en ~3x. Se
    # convierte con el mismo metodo ya usado en M10/M11/M12 (GGAL.BA
    # x10/GGAL, serie diaria).
    try:
        ggal_ba_m8 = yf.download("GGAL.BA", period=CONFIG["market_data_period"], progress=False, auto_adjust=True)["Close"].squeeze()
        ggal_us_m8 = yf.download("GGAL",    period=CONFIG["market_data_period"], progress=False, auto_adjust=True)["Close"].squeeze()
        ccl_daily_m8 = (ggal_ba_m8 * 10 / ggal_us_m8).dropna()
        alua = (alua / ccl_daily_m8.reindex(alua.index).ffill()).dropna()
        merv = (merv / ccl_daily_m8.reindex(merv.index).ffill()).dropna()
        txar = (txar / ccl_daily_m8.reindex(txar.index).ffill()).dropna()
    except Exception as e:
        logging.warning(f"M8_run(): fetch CCL diario fallo ({e}) -- ALUA/MERV/TXAR quedan en ARS nominal.")

    # df de PRECIOS (solo para graficos de nivel/drawdown; ahora en USD): ffill
    # es apropiado aca, unicamente rellena para visualizar una linea de nivel
    # continua.
    df = pd.DataFrame({"ALUA": alua, "MERV": merv, "TXAR": txar}).dropna(how="all")
    df = df.ffill().dropna()

    if df.empty or len(df) < 100:
        print("  [FAIL] Datos insuficientes — verificar conexión yfinance")
        sys.exit(1)

    # Fix (auditoria ronda 4, MEDIA): mismo bug de calendario ya corregido en
    # M11 -- pct_change() sobre el df YA ffill-eado (arriba) crea retornos=0
    # artificiales en dias donde un ticker no opero pero otro si (calendario
    # ARG vs indices), sesgando beta/correlacion rolling hacia 0. Los
    # retornos se calculan sobre cada serie CRUDA (sin ffill previo).
    rets = pd.DataFrame({
        "ALUA": alua.pct_change(),
        "MERV": merv.pct_change(),
        "TXAR": txar.pct_change(),
    }).dropna(how="all")
    print(f"  Datos: {df.index[0].date()} → {df.index[-1].date()} ({len(df)} sesiones)")

    # Retornos acumulados normalizados a 100 -- fillna(0) es correcto ACA
    # (dia sin cotizacion == retorno 0% ese dia para el indice acumulado),
    # a diferencia de ffillear el PRECIO crudo antes de calcular retornos.
    cum_rets = (1 + rets.fillna(0)).cumprod() * 100

    # ── FIG 1: Precios históricos con MERVAL en eje Y secundario ──────────────
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax2 = ax.twinx()
    for col, color, lw in [("ALUA", PALETTE["alua"], 2.5), ("TXAR", PALETTE["txar"], 1.5), ("MERV", PALETTE["merv"], 1.0)]:
        if col in df.columns:
            if col == "MERV":
                ax2.plot(df.index, df[col], color=color, lw=lw, label=col, alpha=0.5, ls=":")
            else:
                ax.plot(df.index, df[col], color=color, lw=lw, label=col)
    ax.set_title("ALUA.BA y TXAR.BA vs MERVAL — Evolución de Precios (USD via CCL)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Precio Acciones (USD)", color=PALETTE["alua"])
    ax2.set_ylabel("Índice MERVAL (Puntos, USD via CCL)", color=PALETTE["merv"])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc="upper left", frameon=False)
    fig_price = save_fig(fig, "m8_01_precio_historico")

    # ── FIG 2: Retornos acumulados ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    colors = [PALETTE["alua"], PALETTE["merv"], PALETTE["txar"]]
    for (col, color) in zip(["ALUA", "MERV", "TXAR"], colors):
        if col in cum_rets.columns:
            ax.plot(cum_rets.index, cum_rets[col], color=color, lw=2, label=col)
    ax.axhline(100, color="black", lw=0.8, ls="--", alpha=0.5)
    ax.set_title("Retornos Acumulados (Base 100)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Índice (Base 100)")
    ax.legend()
    fig_cum = save_fig(fig, "m8_02_retornos_acumulados")

    # ── FIG 3: Volatilidad rolling ────────────────────────────────────────────
    vol_alua = compute_rolling_vol(rets["ALUA"])
    vol_merv = compute_rolling_vol(rets["MERV"])
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.plot(vol_alua.index, vol_alua * 100, color=PALETTE["alua"], lw=1.5, label="ALUA vol 21d")
    ax.plot(vol_merv.index, vol_merv * 100, color=PALETTE["merv"], lw=1.5, alpha=0.7, label="MERV vol 21d")
    ax.set_title("Volatilidad Anualizada Rolling 21d (%)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Volatilidad (%)")
    ax.legend()
    fig_vol = save_fig(fig, "m8_03_volatilidad_rolling")

    # ── FIG 4: Drawdowns ─────────────────────────────────────────────────────
    dd_alua = compute_drawdown(df["ALUA"])
    dd_merv = compute_drawdown(df["MERV"])
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.fill_between(dd_alua.index, dd_alua * 100, 0, color=PALETTE["alua"], alpha=0.4, label="ALUA")
    ax.fill_between(dd_merv.index, dd_merv * 100, 0, color=PALETTE["merv"], alpha=0.2, label="MERV")
    ax.set_title("Drawdowns desde Máximo (%)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Drawdown (%)")
    ax.legend()
    fig_dd = save_fig(fig, "m8_04_drawdowns")

    # ── FIG 5: Distribución empírica + t-Student ──────────────────────────────
    alua_rets_clean = rets["ALUA"].dropna()
    t_params = fit_t_student(alua_rets_clean)
    x = np.linspace(alua_rets_clean.quantile(0.01), alua_rets_clean.quantile(0.99), 300)
    t_pdf = stats.t.pdf(x, df=t_params["df"], loc=t_params["loc"], scale=t_params["scale"])
    norm_pdf = stats.norm.pdf(x, alua_rets_clean.mean(), alua_rets_clean.std())

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.hist(alua_rets_clean, bins=80, density=True, color=PALETTE["alua"], alpha=0.5, label="Empírico")
    ax.plot(x, t_pdf, color=PALETTE["alua"], lw=2, label=f"t-Student (df={t_params['df']:.1f})")
    ax.plot(x, norm_pdf, color=PALETTE["neutral"], lw=1.5, ls="--", label="Normal")
    ax.set_title("Distribución Retornos Diarios ALUA.BA", fontsize=13, fontweight="bold")
    ax.set_xlabel("Retorno diario")
    ax.legend()
    fig_dist = save_fig(fig, "m8_05_distribucion_retornos")

    # -- FIG 6: Beta rolling (estilo institucional: scaffold + paleta C[]) -----
    beta_rolling = compute_rolling_beta(rets["ALUA"], rets["MERV"])
    # Titulo dinamico: la version anterior afirmaba fijo "fluctua en torno a 1x"
    # -- la propia leyenda de este grafico (Media=0.68x) contradice ese texto;
    # el beta rolling oscila mayormente POR DEBAJO de 1.0. Se calcula la
    # relacion real (por debajo / en torno / por encima de 1.0) del propio
    # promedio en vez de asumirla.
    _beta_roll_mean = float(beta_rolling.mean())
    if _beta_roll_mean < 0.85:
        _titulo_beta_roll = f"El beta de ALUA opera mayormente por debajo de 1x frente al Merval (media {_beta_roll_mean:.2f}x)"
    elif _beta_roll_mean > 1.15:
        _titulo_beta_roll = f"El beta de ALUA opera mayormente por encima de 1x frente al Merval (media {_beta_roll_mean:.2f}x)"
    else:
        _titulo_beta_roll = f"El beta de ALUA fluctúa en torno a 1x frente al Merval (media {_beta_roll_mean:.2f}x)"
    # Fix (auditoria jul-11, BAJA): aclarar que este beta (rolling 63d vs
    # Merval, riesgo local) es conceptualmente distinto del Beta OLS de 5
    # años vs ^GSPC que M1/M5 usan en el CAPM/WACC (riesgo sistematico
    # global, metodologia Damodaran) -- para que no se confundan como "el"
    # beta de ALUA si aparecen en slides distintas del mismo deck.
    fig, ax = scaffold(
        _titulo_beta_roll,
        "Beta rolling 63 dias (trimestre movil) vs Merval -- distinto del Beta OLS de 5 años vs S&P 500 usado en el WACC (M5)",
        "beta (x)", "yfinance (ALUA.BA, ^MERV) — ventana movil 63d",
        "beta=1.0 implica riesgo sistemico equivalente al Merval (no es el beta usado en el CAPM/WACC).")
    ax.plot(beta_rolling.index, beta_rolling, color=C["aluar"], lw=LW["heavy"])
    ax.axhline(1.0, color=C["ink"], lw=LW["thin"], ls="--", alpha=0.5, label="Beta = 1.0")
    ax.axhline(beta_rolling.mean(), color=C["navy"], lw=LW["medium"], ls=":",
               label=f"Media = {beta_rolling.mean():.2f}x")
    _legend_outside(ax, ncol=2)
    ax.grid(axis="x", visible=False); _clean_y(ax)
    fig_beta = save_fig(fig, "m8_06_beta_rolling")  # mantiene tipo str, consistente con las demas figuras de este dict

    # ── Estadísticas resumen ──────────────────────────────────────────────────
    out = {
        "date_start": str(df.index[0].date()),
        "date_end": str(df.index[-1].date()),
        "n_sessions": len(df),

        "alua_ret_anual": round(float((1 + alua_rets_clean.mean()) ** 252 - 1), 4),
        "alua_vol_anual": round(float(alua_rets_clean.std() * np.sqrt(252)), 4),
        # Fix (auditoria ronda 4, MEDIA): el Sharpe no restaba la tasa libre
        # de riesgo (R/sigma en vez de (R-Rf)/sigma) -- sobreestimaba el
        # ratio reportado. rf viene de m1_out.json (unica fuente).
        "alua_sharpe": round(float((alua_rets_clean.mean() * 252 - rf) / (alua_rets_clean.std() * np.sqrt(252))), 4),
        "alua_max_dd": round(float(dd_alua.min()), 4),
        "alua_t_student": t_params,

        "merv_ret_anual": round(float((1 + rets["MERV"].mean()) ** 252 - 1), 4),
        "merv_vol_anual": round(float(rets["MERV"].std() * np.sqrt(252)), 4),
        "merv_max_dd": round(float(dd_merv.min()), 4),

        "beta_vs_merv_rolling_mean": round(float(beta_rolling.mean()), 4),
        "beta_vs_merv_rolling_std": round(float(beta_rolling.std()), 4),

        "correlacion_alua_merv": round(float(rets["ALUA"].corr(rets["MERV"])), 4),
        # Fix (auditoria ronda 4, BAJA): round(None, 4) lanzaria TypeError si
        # faltara TXAR -- la precedencia del ternario anterior aplicaba
        # round() al resultado ANTES de resolver el None. Ahora el guard
        # envuelve toda la expresion.
        "correlacion_alua_txar": (round(float(rets["ALUA"].corr(rets["TXAR"])), 4)
                                   if "TXAR" in rets.columns else None),

        "figures": {
            "precio_historico": fig_price,
            "retornos_acumulados": fig_cum,
            "volatilidad_rolling": fig_vol,
            "drawdowns": fig_dd,
            "distribucion_retornos": fig_dist,
            "beta_rolling": fig_beta,
        },
    }

    with open(os.path.join(WORKDIR, "m8_out.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"\n  ALUA: ret_anual={out['alua_ret_anual']:.1%}, vol={out['alua_vol_anual']:.1%}, maxDD={out['alua_max_dd']:.1%}")
    print(f"  MERV: ret_anual={out['merv_ret_anual']:.1%}, vol={out['merv_vol_anual']:.1%}")
    print(f"  Beta rolling media: {out['beta_vs_merv_rolling_mean']:.3f}")
    print(f"  t-Student: df={t_params['df']}")
    print(f"\n[OK] m8_out.json + 6 figuras guardadas en {FIGDIR}")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M9

In [12]:
"""
module9_stochastic.py — Modelos Estocásticos
=============================================
Ornstein-Uhlenbeck calibrado, Merton Jump Diffusion calibrado,
Reverse DCF probabilístico, convergencia Monte Carlo.
Exporta figuras PNG + m9_out.json
"""

import json, sys, os, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats, optimize
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
apply_aluar_theme()


N_SIM = 20_000
SEED  = 42


# Fix (auditoria ronda 4, mejora): get_alua_returns() era codigo muerto --
# M9_run() reimplementaba la misma descarga por su cuenta (mas abajo) y
# nunca llamaba a esta funcion. Eliminada la duplicacion.


# ── Ornstein-Uhlenbeck (reversión a la media) ────────────────────────────────

def calibrate_ou(price_series: pd.Series) -> dict:
    """
    Calibra OU: dX = kappa*(theta-X)*dt + sigma*dW
    Método: OLS en discretización de Euler.
    """
    log_p = np.log(price_series.dropna())
    X = log_p.values
    dt = 1.0 / 252

    # Fix (auditoria jul-11, ALTA): calibrar OU sobre el precio nominal de
    # una ACCION (no un commodity) contradice la hipotesis de eficiencia de
    # mercado semifuerte (Fama 1970: el precio deberia aproximarse a un
    # random walk, no revertir a un nivel fijo) -- Schwartz (1997) justifica
    # OU para precios de COMMODITIES (ancla de costo de produccion), no para
    # acciones individuales. El comentario de mas abajo ya reconocia "M11 ya
    # tiene ADF, nunca se reutiliza aca", pero M9 corre ANTES que M11 en el
    # pipeline (m11_out.json no existe todavia en esta corrida) -- depender
    # del archivo de M11 crearia un orden de ejecucion fragil. Se corre un
    # ADF propio, auto-contenido, sobre el NIVEL (log-precio) y se expone el
    # resultado explicitamente en vez de solo una advertencia cualitativa.
    try:
        from statsmodels.tsa.stattools import adfuller as _adfuller_ou
        _adf_result_ou = _adfuller_ou(X)
        _adf_pval_ou = float(_adf_result_ou[1])
    except Exception:
        _adf_pval_ou = None

    # Regresión: X(t+1) = a + b*X(t)
    Xprev = X[:-1]
    Xnext = X[1:]
    slope, intercept, _, _, _ = stats.linregress(Xprev, Xnext)

    # Fix (auditoria ronda 4, ALTA): si slope>=1 (regresion AR(1) NO
    # estacionaria) kappa=-ln(slope)/dt se vuelve negativo -- un proceso
    # EXPLOSIVO, no de reversion a la media -- sin ningun clip ni test de
    # estacionariedad (M11 ya tiene ADF, nunca se reutiliza aca). Se clippea
    # slope al rango estacionario (0,1) y se avisa si hubo que corregir.
    if not (0 < slope < 0.999):
        logging.warning(f"calibrate_ou(): slope={slope:.4f} fuera del rango estacionario "
                         f"(0,1) -- se clippea a 0.999 para evitar kappa<=0 (proceso explosivo).")
        slope = float(np.clip(slope, 1e-4, 0.999))

    kappa = -np.log(slope) / dt
    theta = intercept / (1 - slope)
    residuals = Xnext - (slope * Xprev + intercept)
    sigma_ou = np.std(residuals) / np.sqrt(dt)

    # BUG detectado (revision visual s34b, jul-2026): kappa esta anualizado
    # (la regresion OLS se hace en dt=1/252 anios por paso), asi que
    # ln(2)/kappa da el half-life en ANIOS, no en dias -- el campo se
    # llamaba "half_life_days" pero devolvia 1.6 (anios) en vez de ~405
    # (dias habiles). La propia trayectoria simulada en s34b (media que tras
    # 252 dias sigue lejos de theta) ya lo evidenciaba: un half-life real de
    # 1.6 dias implicaria reversion casi instantanea, incompatible con el
    # grafico. Se convierte a dias habiles (misma convencion 252 que dt).
    half_life_years = float(np.log(2) / max(kappa, 1e-6))
    return {
        "kappa": round(float(kappa), 4),
        "theta": round(float(theta), 4),
        "sigma": round(float(sigma_ou), 4),
        "half_life_days": round(half_life_years * 252, 1),
        # Nota (auditoria ronda 4, MEDIA-ALTA): theta se calibra sobre el
        # PRECIO NOMINAL ARS (no deflactado por CCL/inflacion) -- el nivel
        # de "equilibrio" que arroja mezcla la tendencia inflacionaria de
        # largo plazo del ARS con la reversion a la media genuina, y puede
        # leerse erroneamente como una señal de subvaluacion. Se declara
        # explicitamente en vez de presentar theta como un precio objetivo
        # limpio.
        "adf_pvalue_nivel": round(_adf_pval_ou, 4) if _adf_pval_ou is not None else None,
        "reversion_estadisticamente_significativa": bool(_adf_pval_ou is not None and _adf_pval_ou < 0.05),
        "theta_interpretacion": ("ARS nominal -- incluye tendencia inflacionaria/devaluatoria de "
                                  "largo plazo, NO es un precio de equilibrio deflactado. No usar "
                                  "como señal de sub/sobrevaluacion sin normalizar por CCL/inflacion. "
                                  "Fama (1970): bajo EMH semifuerte el precio de una accion deberia "
                                  "seguir un random walk, no revertir a un nivel fijo -- ver "
                                  "adf_pvalue_nivel: si no se rechaza la raiz unitaria (p>=0.05), esta "
                                  "'reversion' es estadisticamente endeble, no una señal genuina de "
                                  "valuacion (Schwartz 1997: OU esta mejor justificado para precios de "
                                  "commodities, no para acciones individuales)."),
    }


def simulate_ou(ou_params: dict, S0: float, T: int = 252, n_paths: int = 500, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    kappa, theta, sigma = ou_params["kappa"], ou_params["theta"], ou_params["sigma"]
    dt = 1.0 / 252
    X0 = np.log(S0)
    paths = np.zeros((T + 1, n_paths))
    paths[0] = X0
    # Fix (auditoria jul-11, BAJA): Euler-Maruyama introduce sesgo de
    # discretizacion evitable -- el proceso OU tiene una densidad de
    # transicion EXACTA conocida en forma cerrada (Vasicek 1977, el mismo
    # paper que el proyecto ya usa para el shrinkage de Beta en M5), sin
    # costo computacional adicional.
    decay = np.exp(-kappa * dt)
    diff_std = (sigma * np.sqrt(max((1 - np.exp(-2 * kappa * dt)) / (2 * kappa), 0.0))
                if kappa > 1e-8 else sigma * np.sqrt(dt))
    for t in range(1, T + 1):
        Z = rng.standard_normal(n_paths)
        paths[t] = theta + (paths[t-1] - theta) * decay + diff_std * Z
    return np.exp(paths)  # retornar en niveles de precio


# ── Merton Jump Diffusion ────────────────────────────────────────────────────

def calibrate_merton(rets: pd.Series) -> dict:
    """
    Merton JD: r = (mu - lambda*kappa)*dt + sigma*dW + J*dN
    Calibración por método de momentos (Cont & Tankov).
    """
    mu     = float(rets.mean() * 252)
    sigma  = float(rets.std() * np.sqrt(252))
    kurt   = float(stats.kurtosis(rets))
    skew   = float(stats.skew(rets))

    # Estimación de parámetros de salto via exceso de kurtosis y asimetría
    # lambda*mj^2 ≈ kurt_exceso * sigma^4 / (4 * sigma_j^2 + ...) (aproximación)
    # Fix (auditoria ronda 4, ALTA): scipy.stats.kurtosis() usa fisher=True
    # por defecto -> YA devuelve el EXCESO de kurtosis (normal=0), pero el
    # codigo restaba 3 otra vez mas abajo como si `kurt` fuera la convencion
    # "raw" (normal=3) -- (kurt-3) quedaba negativo casi siempre y sigma_j
    # colapsaba SIEMPRE al piso hardcodeado 0.02 (confirmado: m9_out.json
    # traia sigma_j=0.02 exacto). Se usa `kurt` directamente (ya es exceso).
    lambda_j = max(0.1, kurt / 10)   # frecuencia de saltos por año
    mu_j     = skew / (lambda_j * 6) if lambda_j > 0 else 0.0
    # Fix (auditoria ronda 4, seguimiento post-fix): con el sign-bug de kurt
    # corregido arriba, esta aproximacion de metodo de momentos (formula ya
    # documentada como aproximada) puede devolver un sigma_j implausible
    # para acciones EM de baja liquidez con exceso de kurtosis alto (ALUA.BA:
    # sigma_j=91% con los datos reales -- un salto individual de esa
    # magnitud no tiene precedente ni en crisis). Se acota a un techo de
    # 15%, en linea con calibraciones tipicas de jump-diffusion para
    # acciones individuales en la literatura (Merton 1976 y calibraciones
    # posteriores), mismo patron de cap ya usado para vol_fcff (max 0.50)
    # en M7.
    sigma_j  = float(np.clip(np.sqrt(max(0, kurt / (lambda_j * 12))), 0.02, 0.15))

    return {
        "mu": round(mu, 4),
        "sigma": round(sigma, 4),
        "lambda_j": round(lambda_j, 4),
        "mu_j": round(mu_j, 6),
        "sigma_j": round(sigma_j, 4),
        # Fix (auditoria jul-11, MEDIA): metodo de momentos aproximado --
        # solo usa kurtosis para lambda_j/sigma_j, sin resolver el sistema
        # conjunto de 4 cumulantes de Cont & Tankov (2004, Cap.15) ni
        # verificar la restriccion de varianza total. Declarado
        # explicitamente para que no se lea como una calibracion exacta.
        "metodo": "aproximacion de momentos (solo kurtosis) -- no el sistema conjunto completo de Cont-Tankov",
    }


def simulate_merton(mj_params: dict, S0: float, T: int = 252, n_paths: int = 500, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed + 1)
    mu, sigma    = mj_params["mu"], mj_params["sigma"]
    lambda_j     = mj_params["lambda_j"]
    mu_j, sig_j  = mj_params["mu_j"], mj_params["sigma_j"]
    dt = 1.0 / 252
    paths = np.zeros((T + 1, n_paths))
    paths[0] = S0
    kappa_j = np.exp(mu_j + 0.5 * sig_j**2) - 1

    for t in range(1, T + 1):
        dW  = rng.standard_normal(n_paths) * np.sqrt(dt)
        N   = rng.poisson(lambda_j * dt, size=n_paths)
        # Fix (auditoria ronda 4, ALTA): antes un list-comprehension con
        # rng.normal(...)[:n] por cada uno de los 500 paths x 252 dias
        # (~126.000 iteraciones evitables), viola la regla dura de
        # vectorizacion. La suma de N saltos iid Normal(mu_j, sig_j) es
        # exactamente Normal(N*mu_j, sig_j*sqrt(N)) -- se muestrea esa
        # normal agregada directamente, vectorizado sobre los 500 paths.
        J = np.where(N > 0, rng.normal(N * mu_j, sig_j * np.sqrt(np.maximum(N, 1))), 0.0)
        dX  = (mu - 0.5 * sigma**2 - lambda_j * kappa_j) * dt + sigma * dW + J
        paths[t] = paths[t-1] * np.exp(dX)
    return paths


# ── Monte Carlo convergencia ──────────────────────────────────────────────────

def mc_convergence(m6: dict, m5: dict, m1: dict, m7: dict) -> dict:
    """Error de estimación del precio vs N simulaciones."""
    rng    = np.random.default_rng(SEED)
    wacc   = m5["wacc"]
    g      = m6["g_terminal"]
    # Fix (Directiva 3.C, seguimiento): usar m6["fcff_terminal"] (unica
    # fuente de verdad del FCFF terminal del DCF puntual) en vez de derivar
    # fcff_proj_usdmm[-1] por cuenta propia -- misma consistencia ya
    # aplicada en M7.
    fcff_t = m6["fcff_terminal"]
    dnet   = m6["deuda_neta_usdmm"]
    shares = m6["shares_mm"]
    ccl    = m1["ccl"]
    fcff_p = m6["fcff_proj_usdmm"]

    # Fix (auditoria jul-11, ALTA): vol_w miraba m5 ("vol_wacc" in m5), pero
    # esa clave NUNCA existe en m5_out.json (verificado: vol_wacc solo vive
    # en m7_out.json, calibrado alli con valor real 0.01) -- la condicion
    # era siempre falsa y el fallback 0.005 (la mitad del valor real) se
    # disparaba en silencio cada corrida. Se lee de m7, igual que vol_fcff.
    vol_w  = m7.get("vol_wacc", 0.01)
    vol_g  = 0.015
    # BUG detectado (auditoria jul-09): vol_f estaba hardcodeado en 0.80,
    # violando la regla de "cero hardcodes financieros" -- inconsistente con
    # el vol_fcff REAL usado por el Monte Carlo (M7, derivado dinamicamente
    # del CV del margen EBITDA historico, tipicamente 0.35-0.50 con cap).
    # Unica fuente de verdad: m7_out.json["vol_fcff"] (ya persistido por M7).
    vol_f  = m7.get("vol_fcff", 0.35)

    ns = [100, 500, 1000, 5000, 10000, 20000]
    errors = []
    for n in ns:
        w_s = np.maximum(wacc + rng.standard_t(5, n) / np.sqrt(5/(5-2)) * vol_w, 0.02)
        g_s = g + rng.standard_t(5, n) / np.sqrt(5/(5-2)) * vol_g
        # Fix (auditoria jul-11, ALTA -- mismo patron de bug que M7 ya
        # corrigio para su propia fila de FCFF_T): usar un shock t-Student
        # dentro de un exp() apila una difusion de colas pesadas sobre el
        # salto de Merton que M7 ya modela por separado, generando outliers
        # multiplicativos explosivos. Verificado numericamente: stderr_ars
        # saltaba de forma NO monotona (70.6->25.4->18.2->9.0->6.5->51.0 en
        # N=100..20000) por un unico draw extremo de t(5) exponenciado. Se
        # usa Normal, igual que M7 ya hace para FCFF_T (el salto de Merton
        # ya cubre la cola).
        f_s = fcff_t * np.exp(vol_f * rng.standard_normal(n) - 0.5 * vol_f**2)
        tv  = np.where(w_s > g_s + 0.02, f_s * (1+g_s) / (w_s - g_s), 0.0)  # fix cola derecha imposible
        # Fix (auditoria ronda 5, BAJA -- tribunal): "5" hardcodeado en vez de
        # len(fcff_p) -- hoy inerte (el horizonte de proyeccion siempre es 5
        # anios, fijo en M4/M6), pero un landmark de mantenimiento silencioso
        # si el horizonte cambiara alguna vez.
        n_yrs = len(fcff_p)
        # Fix (auditoria jul-11, ALTA): mismo ajuste mid-year aplicado al TV
        # primario de M6 (los FCFF explicitos ya usan i+0.5 dos lineas mas
        # abajo).
        pv_tv = tv / (1 + w_s)**(n_yrs - 0.5)
        pv_fc = sum(fcff_p[i] / (1+w_s)**(i+0.5) for i in range(n_yrs))  # H-04: Mid-Year Convention
        ev = pv_fc + pv_tv
        # Fix (auditoria jul-11, ALTA): M7 trunca equity a np.maximum(0.0, ...)
        # (equity como call option, Merton 1974) -- este diagnostico de
        # convergencia no lo hacia, permitiendo precios simulados negativos
        # (~ARS -84/accion) que describian una distribucion distinta a la
        # que efectivamente se reporta en las slides.
        eq = np.maximum(0.0, ev - dnet)
        px = eq / shares * ccl
        errors.append(float(np.std(px) / np.sqrt(n)))  # error estándar

    return {"ns": ns, "stderr_ars": [round(e, 4) for e in errors]}


# ── Reverse DCF ─────────────────────────────────────────────────────────────

def reverse_dcf(m6: dict, m5: dict, m1: dict) -> dict:
    """
    Dados el precio de mercado y el WACC, ¿qué FCFF terminal implícito
    y qué g asume el mercado?
    """
    mkt_price_ars = m1.get("alua_px", 0.0)
    ccl           = m1["ccl"]
    shares        = m6["shares_mm"]
    dnet          = m6["deuda_neta_usdmm"]
    wacc          = m5["wacc"]
    fcff_proj     = m6["fcff_proj_usdmm"]

    if mkt_price_ars <= 0:
        return {"g_implicita": None, "fcff_terminal_implicito": None}

    # Equity implícita desde mercado
    eq_mkt_usdmm = mkt_price_ars / ccl * shares
    ev_mkt_usdmm = eq_mkt_usdmm + dnet

    # PV de FCFFs explícitos (fijos)
    n_yrs = len(fcff_proj)  # Fix (ronda 5, BAJA -- tribunal): "5" -> len(fcff_proj)
    pv_explicit = sum(fcff_proj[i] / (1 + wacc)**(i+0.5) for i in range(n_yrs))  # H-04: Mid-Year Convention

    # TV implícita = EV_mercado - PV_explicit
    tv_implied = ev_mkt_usdmm - pv_explicit
    pv_tv_disc = tv_implied * (1 + wacc)**(n_yrs - 0.5)  # Fix (auditoria jul-11): mid-year, simetrico con M6

    # FCF terminal implícito asumiendo g=0: TV = FCFF/(WACC-g)
    # Resolvemos para g dado FCFF_terminal = FCFF_2030
    # Fix (Directiva 3.C, seguimiento): m6["fcff_terminal"] en vez de
    # fcff_proj[-1] -- misma razon que en mc_convergence() arriba.
    fcff_t = m6["fcff_terminal"]
    if fcff_t > 0 and pv_tv_disc > 0:
        # TV = fcff_t*(1+g)/(wacc-g) → g = (TV*wacc - fcff_t) / (TV + fcff_t)
        g_impl = (pv_tv_disc * wacc - fcff_t) / (pv_tv_disc + fcff_t)
    else:
        g_impl = None

    return {
        "ev_mercado_usdmm": round(ev_mkt_usdmm, 2),
        "pv_fcff_explicito": round(pv_explicit, 2),
        "tv_implicito_usdmm": round(pv_tv_disc, 2),
        "g_implicita": round(float(g_impl), 6) if g_impl is not None else None,
        "g_modelo": m6["g_terminal"],
        "interpretacion": "g_implicita < g_modelo → mercado es más conservador que el DCF",
    }


def M9_run():
    print("=" * 65)
    print("M9 STOCHASTIC — OU + Merton JD + Reverse DCF + MC Convergencia")
    print("=" * 65)

    m1 = load("m1_out.json")
    m5 = load("m5_out.json")
    m6 = load("m6_out.json")
    m7 = load("m7_out.json")

    # Precio actual ALUA
    alua_px = m1.get("alua_px", 1000.0)

    print("\n[1/4] Descargando retornos ALUA.BA...")
    alua_px_series = None
    try:
        import yfinance as yf
        df_px = yf.download("ALUA.BA", period=CONFIG["market_data_period"], progress=False, auto_adjust=True)
        alua_px_series = df_px["Close"].squeeze().dropna()
        alua_rets = alua_px_series.pct_change().dropna()
        print(f"  {len(alua_rets)} observaciones")
    except Exception as e:
        print(f"  [WARN] {e} — usando parámetros empíricos de M8")
        alua_rets = pd.Series(dtype=float)

    # ── OU ───────────────────────────────────────────────────────────────────
    print("\n[2/4] Ornstein-Uhlenbeck...")
    if alua_px_series is not None and len(alua_px_series) > 100:
        ou_params = calibrate_ou(alua_px_series)
    else:
        ou_params = {"kappa": 1.5, "theta": np.log(alua_px), "sigma": 0.45,
                     "half_life_days": round(float(np.log(2) / 1.5) * 252, 1),  # fallback: misma convencion dias habiles
                     "theta_interpretacion": "FALLBACK -- sin datos suficientes para calibrar."}
    print(f"  kappa={ou_params['kappa']}, theta={np.exp(ou_params['theta']):.1f} ARS, sigma={ou_params['sigma']}, T½={ou_params['half_life_days']}d")

    ou_paths = simulate_ou(ou_params, alua_px, T=252, n_paths=200)
    fig, ax = plt.subplots(figsize=(12, 5))
    for i in range(min(50, ou_paths.shape[1])):
        ax.plot(ou_paths[:, i], color=PALETTE["ou"], alpha=0.1, lw=0.8)
    # Fix (auditoria ronda 4, BAJA-MEDIA): la media se graficaba sobre solo
    # las primeras 50 de 200 trayectorias (el subset dibujado por opacidad),
    # sesgada por esa muestra chica, pero rotulada como si fuera "la" media.
    # Se calcula sobre las 200 trayectorias completas.
    ax.plot(ou_paths.mean(axis=1), color=PALETTE["ou"], lw=2, label="Media OU (200 trayectorias)")
    ax.axhline(alua_px, color="black", lw=1, ls="--", label="Precio actual")
    ax.axhline(np.exp(ou_params["theta"]), color=PALETTE["merton"], lw=1.5, ls=":", label=f"Theta={np.exp(ou_params['theta']):.0f}")
    ax.set_title("Ornstein-Uhlenbeck — Trayectorias ALUA.BA (1 año)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Días"); ax.set_ylabel("ARS")
    ax.legend()
    fig_ou = save_fig(fig, "m9_01_ou_paths")

    # ── Merton JD ─────────────────────────────────────────────────────────────
    print("\n[3/4] Merton Jump Diffusion...")
    if len(alua_rets) > 100:
        mj_params = calibrate_merton(alua_rets)
    else:
        # Fix (auditoria jul-11, MEDIA): lambda_j/mu_j/sigma_j unificados con
        # CONFIG["merton_fallback"] (mismo que usa M7) -- antes este fallback
        # tenia lambda_j=5.0/año, ~18x el fallback de M7 (0.27/año) para el
        # mismo parametro del mismo activo.
        mj_params = {"mu": 0.30, "sigma": 0.45, **CONFIG["merton_fallback"]}
    print(f"  mu={mj_params['mu']:.2%}, sigma={mj_params['sigma']:.2%}, lambda={mj_params['lambda_j']:.1f}/año")

    mj_paths = simulate_merton(mj_params, alua_px, T=252, n_paths=200)
    fig, ax = plt.subplots(figsize=(12, 5))
    for i in range(min(50, mj_paths.shape[1])):
        ax.plot(mj_paths[:, i], color=PALETTE["merton"], alpha=0.1, lw=0.8)
    # Fix (auditoria ronda 4, BAJA-MEDIA): mismo bug que el grafico OU arriba
    # -- media sobre las 200 trayectorias completas, no solo las 50 dibujadas.
    ax.plot(mj_paths.mean(axis=1), color=PALETTE["merton"], lw=2, label="Media Merton JD (200 trayectorias)")
    ax.axhline(alua_px, color="black", lw=1, ls="--", label="Precio actual")
    ax.set_title("Merton Jump Diffusion — Trayectorias ALUA.BA (1 año)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Días"); ax.set_ylabel("ARS")
    ax.legend()
    fig_mj = save_fig(fig, "m9_02_merton_paths")

    # ── Convergencia MC ───────────────────────────────────────────────────────
    print("\n[4/4] Convergencia Monte Carlo...")
    conv = mc_convergence(m6, m5, m1, m7)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(conv["ns"], conv["stderr_ars"], "o-", color=PALETTE["alua"], lw=2)
    ax.set_xscale("log")
    ax.set_title("Convergencia Monte Carlo — Error Estándar vs N simulaciones", fontsize=13, fontweight="bold")
    ax.set_xlabel("N simulaciones (escala log)")
    ax.set_ylabel("Error estándar precio ARS")
    for n, e in zip(conv["ns"], conv["stderr_ars"]):
        ax.annotate(f"N={n:,}\n±{e:.0f}", xy=(n, e), fontsize=7, ha="center", va="bottom")
    fig_conv = save_fig(fig, "m9_03_mc_convergencia")

    # ── Reverse DCF ───────────────────────────────────────────────────────────
    rdcf = reverse_dcf(m6, m5, m1)
    print(f"\n  Reverse DCF:")
    print(f"    EV mercado:      {rdcf['ev_mercado_usdmm']} USD MM")
    print(f"    g implícita:     {rdcf['g_implicita']:.2%}" if rdcf['g_implicita'] else "    g implícita: N/A")
    print(f"    g modelo DCF:    {rdcf['g_modelo']:.2%}")

    # Football field (sensibilidad WACC × g)
    wacc_base_m9 = m5["wacc"]
    g_base_m9 = m6["g_terminal"]
    waccs = np.linspace(wacc_base_m9 - 0.03, wacc_base_m9 + 0.03, 7)
    gs    = np.linspace(g_base_m9 - 0.03, g_base_m9 + 0.03, 7)
    ccl   = m1["ccl"]
    shares = m6["shares_mm"]
    dnet  = m6["deuda_neta_usdmm"]
    fcff_p = m6["fcff_proj_usdmm"]
    # Fix (Directiva 3.C, seguimiento): m6["fcff_terminal"] en vez de
    # fcff_p[-1] -- misma consistencia que mc_convergence()/reverse_dcf().
    fcff_t = m6["fcff_terminal"]

    n_yrs = len(fcff_p)  # Fix (ronda 5, BAJA -- tribunal): "5" -> len(fcff_p)
    grid = np.zeros((len(gs), len(waccs)))
    for i, g_v in enumerate(gs):
        for j, w_v in enumerate(waccs):
            pv_fc = sum(fcff_p[k] / (1+w_v)**(k+0.5) for k in range(n_yrs))  # H-04: Mid-Year Convention
            # Guard con buffer +0.02, mismo criterio ya aprobado en M7/M13/s33
            # (wacc > g + 0.02) -- sin buffer, esta grilla recalcula Gordon
            # Growth de forma independiente y podria explotar si el rango de
            # WACC/g llegara a acercarse (no ocurre con los limites actuales,
            # pero es el mismo patron de bug ya detectado en otros graficos).
            tv = fcff_t * (1+g_v) / (w_v - g_v) if w_v > g_v + 0.02 else 0
            ev = pv_fc + tv / (1+w_v)**(n_yrs - 0.5)  # Fix (auditoria jul-11): mid-year, simetrico con M6
            eq = ev - dnet
            grid[i, j] = eq / shares * ccl

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(grid, cmap="RdYlGn", aspect="auto",
                   vmin=max(0, grid.min()), vmax=grid.max())
    ax.set_xticks(range(len(waccs)))
    ax.set_xticklabels([f"{w:.1%}" for w in waccs])
    ax.set_yticks(range(len(gs)))
    ax.set_yticklabels([f"{g:.1%}" for g in gs])
    ax.set_xlabel("WACC")
    ax.set_ylabel("g terminal")
    ax.set_title("Sensibilidad DCF — Precio ALUA (ARS) vs WACC × g", fontsize=13, fontweight="bold")
    for i in range(len(gs)):
        for j in range(len(waccs)):
            ax.text(j, i, f"{grid[i,j]:.0f}", ha="center", va="center", fontsize=8,
                    color="black" if grid[i,j] > 0 else "red")
    plt.colorbar(im, ax=ax, label="Precio ARS")
    fig_ff = save_fig(fig, "m9_04_football_field_wacc_g")

    out = {
        "ou_params": ou_params,
        "merton_params": mj_params,
        "mc_convergencia": conv,
        "reverse_dcf": rdcf,
        "figures": {
            "ou_paths": fig_ou,
            "merton_paths": fig_mj,
            "mc_convergencia": fig_conv,
            "football_field": fig_ff,
        },
    }

    with open(os.path.join(WORKDIR, "m9_out.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"\n[OK] m9_out.json + 4 figuras guardadas.")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M10

In [13]:
"""
module10_risk.py — Análisis de Riesgo
======================================
VaR histórico, paramétrico (normal + t-Student), Monte Carlo.
CVaR / Expected Shortfall. Tail risk stress tests.
Exporta figuras PNG + m10_out.json
"""

import json, sys, os, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
apply_aluar_theme()

CONFIDENCE_LEVELS = [0.90, 0.95, 0.99]


# BUG detectado (auditoria jul-09): un period="Ny" relativo a "hoy" deja de
# cubrir la ventana de stress-test 2020-01/03 (fecha calendario FIJA) a
# medida que pasan los anios -- silenciosamente. Se ancla con start= fijo
# (comodamente anterior a los 3 escenarios de stress_test) en vez de un
# period relativo, para que los escenarios historicos nunca queden huerfanos.
def get_returns(period="5y", start="2019-06-01"):
    """Fix (auditoria ronda 4, MEDIA): antes no manejaba ninguna excepcion de
    red (a diferencia del over-catching de M11) -- un fallo transitorio de
    yfinance tumbaba TODO M10. Ahora un fallo de descarga cae al mismo
    camino que "datos insuficientes" (fallback sintetico documentado en
    M10_run, no una excepcion sin manejar).
    Fix (auditoria ronda 4, ALTA): VaR/CVaR se calculaban sobre el retorno
    NOMINAL ARS, contaminado por la tendencia de devaluacion del peso --
    inconsistente con M12 (que SI convierte a USD via CCL diario citando el
    mismo riesgo). Se convierte a retornos USD con el mismo metodo
    (GGAL.BA×10/GGAL, serie diaria) antes de calcular VaR/CVaR."""
    import yfinance as yf
    try:
        if start:
            df = yf.download("ALUA.BA", start=start, progress=False, auto_adjust=True)
        else:
            df = yf.download("ALUA.BA", period=period, progress=False, auto_adjust=True)
        if df.empty:
            return pd.Series(dtype=float)
        close_ars = pd.Series(df["Close"].squeeze().values, index=df.index)

        ggal_ba = yf.download("GGAL.BA", start=start or None, period=None if start else period,
                               progress=False, auto_adjust=True)["Close"].squeeze()
        ggal_us = yf.download("GGAL", start=start or None, period=None if start else period,
                               progress=False, auto_adjust=True)["Close"].squeeze()
        ccl_daily = (ggal_ba * 10 / ggal_us).dropna()
        close_usd = (close_ars / ccl_daily.reindex(close_ars.index).ffill()).dropna()
        if len(close_usd) < 100:
            return pd.Series(dtype=float)
        return close_usd.pct_change().dropna()
    except Exception as e:
        logging.warning(f"get_returns(): fetch ALUA.BA/CCL fallo ({e}) -- se usara fallback sintetico documentado.")
        return pd.Series(dtype=float)


def compute_var_hist(rets: pd.Series, conf: float) -> float:
    return float(np.percentile(rets, (1 - conf) * 100))


def compute_var_param_normal(rets: pd.Series, conf: float) -> float:
    mu, sigma = rets.mean(), rets.std()
    return float(stats.norm.ppf(1 - conf, mu, sigma))


def compute_var_param_t(rets: pd.Series, conf: float) -> float:
    df_t, loc, scale = stats.t.fit(rets)
    return float(stats.t.ppf(1 - conf, df_t, loc, scale))


def compute_var_cornish_fisher(rets: pd.Series, conf: float) -> float:
    """VaR paramétrico con expansión de Cornish-Fisher (Directiva Refactorizacion
    2.A -- 'Falacia del VaR Gaussiano'): el VaR normal/t-Student de arriba
    asume una forma de distribución simétrica fija (Normal o t-Student
    centrada) e ignora la asimetría (skew) EMPÍRICA de la serie -- ALUA.BA
    tiene colas y asimetría propias que un z-score Normal puro no captura.
    Cornish-Fisher ajusta el z-score Normal usando los momentos 3ro y 4to
    (skew, curtosis de exceso) muestrales, sin asumir ninguna familia
    parametrica fija. Formula estándar (Cornish & Fisher, 1938; Favre &
    Galeano, 2002, aplicación a risk management):
        z_cf = z + (z²-1)/6·S + (z³-3z)/24·K - (2z³-5z)/36·S²
    donde z = cuantil Normal estándar, S = skewness, K = exceso de curtosis.
    """
    mu, sigma = float(rets.mean()), float(rets.std())
    # Fix (auditoria jul-11, MEDIA): bias=True (default) usa estimadores
    # SESGADOS de los momentos 3/4 -- dado que la expansion CF ya es sensible
    # cerca del borde de su region de validez (ver guardia de instabilidad
    # en M10_run), un estimador sesgado (tiende a exagerar S/K en muestra
    # finita) amplifica ese riesgo. Se usa el estimador insesgado
    # Fisher-Pearson (Joanes & Gill 1998).
    S = float(stats.skew(rets, bias=False))
    K = float(stats.kurtosis(rets, bias=False))  # exceso (fisher=True, normal=0)
    z = stats.norm.ppf(1 - conf)
    z_cf = (z + (z**2 - 1) * S / 6
             + (z**3 - 3 * z) * K / 24
             - (2 * z**3 - 5 * z) * S**2 / 36)
    return float(mu + sigma * z_cf)


def compute_cvar(rets: pd.Series, conf: float) -> float:
    var = compute_var_hist(rets, conf)
    tail = rets[rets <= var]
    return float(tail.mean()) if len(tail) > 0 else var


def compute_cvar_cornish_fisher(rets: pd.Series, conf: float) -> float:
    """CVaR consistente con el VaR de Cornish-Fisher (Fix auditoria jul-11,
    MEDIA): antes compute_cvar() SIEMPRE promediaba la cola bajo el VaR
    HISTORICO sin importar que VaR se mostrara al lado en la misma fila de
    var_table -- solo existia una definicion de CVaR reutilizada para los 4
    metodos de VaR. Favre & Galeano (2002) definen tanto el Modified VaR (ya
    implementado en compute_var_cornish_fisher) como el Modified CVaR/ES
    correspondiente. Esta version usa el umbral de Cornish-Fisher (en vez
    del historico) para promediar la cola empirica -- semi-parametrica,
    evita depender de la derivada de la expansion CF (Boudt, Peterson &
    Croux 2008), mas fragil cerca del borde de validez ya detectado en
    compute_var_cornish_fisher()."""
    var_cf = compute_var_cornish_fisher(rets, conf)
    tail = rets[rets <= var_cf]
    return float(tail.mean()) if len(tail) > 0 else var_cf


def compute_var_mc(mc_prices: np.ndarray, current_price: float, conf: float) -> float:
    mc_rets = (mc_prices - current_price) / current_price
    return float(np.percentile(mc_rets, (1 - conf) * 100))


def stress_test(rets: pd.Series) -> dict:
    """Escenarios de stress histórico."""
    scenarios = {
        # Fix (auditoria jul-11, ALTA): faltaba el shock de mercado mas
        # severo y mejor documentado de la ventana -- start='2019-06-01'
        # (arriba) ya deja margen para cubrirlo (per el comentario de esa
        # linea), pero nunca se agrego. PASO 12-ago-2019 ("Lunes Negro"):
        # Merval cayo ~48% en USD en una sola rueda.
        "paso_2019_lunes_negro": ("2019-08-09", "2019-08-30"),
        "crisis_2020_covid": ("2020-01-15", "2020-03-31"),
        "volatilidad_2022": ("2022-06-01", "2022-10-31"),
        "post_elecciones_2023": ("2023-10-01", "2023-12-31"),
        # Fix (auditoria jul-11, MEDIA): los escenarios anteriores son todos
        # shocks macro/politicos argentinos -- ninguno estresa el LME de
        # aluminio, el driver idiosincratico central de la tesis. Ventana
        # calibrada sobre el colapso real del LME desde el pico post-invasion
        # a Ucrania (mar-2022) hasta el valle de fin de 2022.
        "shock_lme_2022_post_pico": ("2022-03-08", "2022-12-30"),
    }
    # BUG detectado (auditoria jul-09): con datos insuficientes en la ventana
    # (p.ej. serie vacia), "(1+vacio).prod()-1" da 0.0 (no una excepcion) y
    # "vacio.std()" da NaN -- ambos se guardaban como si fueran resultados
    # reales, mostrando un escenario COVID "0% retorno / 0% volatilidad" en
    # vez de marcarlo como no disponible. Ahora se exige un minimo de
    # observaciones (>=20 sesiones) antes de aceptar el calculo.
    MIN_OBS = 20
    results = {}
    for name, (start, end) in scenarios.items():
        period_rets = rets.loc[start:end]
        n_obs = int(len(period_rets))
        if n_obs < MIN_OBS:
            results[name] = {"ret_acum": None, "vol_anual": None, "n_obs": n_obs}
            continue
        cum_ret = float((1 + period_rets).prod() - 1)
        vol = float(period_rets.std() * np.sqrt(252))
        results[name] = {"ret_acum": round(cum_ret, 4), "vol_anual": round(vol, 4), "n_obs": n_obs}
    return results


def M10_run():
    print("=" * 65)
    print("M10 RISK — VaR Histórico + Paramétrico + Monte Carlo + CVaR")
    print("=" * 65)

    m1 = load("m1_out.json")
    m7 = load("m7_out.json")

    print("\n[1/3] Cargando retornos ALUA.BA (USD, convertidos vía CCL)...")
    rets = get_returns("5y")
    # Fix (auditoria ronda 4, CRITICA): el fallback fabricaba 1250 retornos
    # t-Student(df=5, scale=0.025) con parametros hardcodeados y corria TODO
    # el motor de VaR/CVaR/stress-test sobre datos ficticios sin ningun flag
    # en m10_out.json -- viola directamente la prohibicion de alucinar datos
    # de CLAUDE.md. Se mantiene como ultimo recurso de robustez (no tumbar
    # el pipeline por un fallo transitorio de red), pero ahora: (1) queda
    # flageado explicitamente en el output, (2) NO corre stress_test sobre
    # datos sinteticos (un escenario "COVID 2020" no existe en una serie
    # fabricada -- se reportan los 3 escenarios como no disponibles en vez
    # de fechar-slicear un indice sintetico sin sentido).
    datos_sinteticos = len(rets) < 100
    if datos_sinteticos:
        print("  [AVISO] Datos insuficientes -- usando retornos SINTETICOS (fallback documentado, "
              "NO son datos de mercado reales). Ver 'datos_sinteticos' en m10_out.json.")
        df_t, loc, scale = 5, 0.0, 0.025
        rets = pd.Series(stats.t.rvs(df_t, loc, scale, size=1250, random_state=42))

    current_price = m1.get("alua_px", 991.0)  # _is_fallback: ultimo alua_px verificado si m1 no lo trajera
    # BUG detectado (auditoria jul-09): mc_results.npy incluye ~4.2% de
    # simulaciones truncadas exactamente a $0 (piso de insolvencia Merton,
    # equity=max(0, EV-deuda)) -- histogramarlas/percentilarlas SIN excluir
    # ese pileup ya se habia detectado y corregido en s34_monte_carlo /
    # s36_var_cvar (jul-05), pero nunca se propago a este modulo: mostraba
    # literalmente "p1 = 0 ARS" sin ningun contexto de piso Merton.
    mc_prices_raw = np.load(os.path.join(WORKDIR, "mc_results.npy"))
    frac_truncado_m10 = float((mc_prices_raw <= 0).mean())
    mc_prices = mc_prices_raw[mc_prices_raw > 0]

    if len(rets) > 0 and not datos_sinteticos:
        print(f"  Rango de fechas: {rets.index.min().date()} → {rets.index.max().date()}")
    print(f"  {len(rets)} observaciones | precio actual: {current_price} ARS")

    # ── Tabla VaR / CVaR ─────────────────────────────────────────────────────
    print("\n[2/3] Calculando VaR y CVaR...")
    var_table = {}
    for conf in CONFIDENCE_LEVELS:
        vh  = compute_var_hist(rets, conf)
        vn  = compute_var_param_normal(rets, conf)
        vt  = compute_var_param_t(rets, conf)
        vcf = compute_var_cornish_fisher(rets, conf)
        cv  = compute_cvar(rets, conf)
        cv_cf = compute_cvar_cornish_fisher(rets, conf)
        # Fix (auditoria jul-11, ALTA): la expansion Cornish-Fisher puede
        # "explotar" fuera de su region de validez (Jaschke 2001) y violar
        # la propiedad de coherencia CVaR>=VaR (Rockafellar & Uryasev 2000).
        # Verificado: al 99% VaR_CF=-14.35% superaba en magnitud a
        # CVaR_historico=-13.51% del mismo nivel -- matematicamente
        # imposible para un par (VaR,CVaR) coherente de la misma
        # distribucion. Guardia practica: si |VaR_CF|>|CVaR_historico|, se
        # marca inestable y se usa VaR_tStudent como fallback para ese nivel
        # en vez de exportar silenciosamente un numero internamente
        # inconsistente a las slides.
        cf_inestable = bool(abs(vcf) > abs(cv))
        vcf_usado = vt if cf_inestable else vcf
        var_table[f"{int(conf*100)}pct"] = {
            "VaR_historico": round(vh, 6),
            "VaR_normal": round(vn, 6),
            "VaR_tStudent": round(vt, 6),
            "VaR_cornish_fisher": round(vcf, 6),
            "VaR_cornish_fisher_usado": round(vcf_usado, 6),
            "cf_inestable": cf_inestable,
            "CVaR": round(cv, 6),
            "CVaR_cornish_fisher": round(cv_cf, 6),
        }
        print(f"  Conf {conf:.0%}: VaR_hist={vh:.2%} | VaR_t={vt:.2%} | VaR_CF={vcf:.2%}"
              f"{' [INESTABLE->usa VaR_t]' if cf_inestable else ''} | CVaR={cv:.2%} | CVaR_CF={cv_cf:.2%}")

    # ── FIG 1: VaR/CVaR comparativo ──────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.linspace(rets.quantile(0.001), rets.quantile(0.999), 400)
    df_fit, loc_fit, sc_fit = stats.t.fit(rets)
    ax.hist(rets, bins=80, density=True, color="#003087", alpha=0.4, label="Empírico")
    ax.plot(x, stats.t.pdf(x, df_fit, loc_fit, sc_fit), "#003087", lw=2, label=f"t-Student df={df_fit:.1f}")
    ax.plot(x, stats.norm.pdf(x, rets.mean(), rets.std()), "#808080", lw=1.5, ls="--", label="Normal")

    for conf, color in [(0.95, "orange"), (0.99, "#C8102E")]:
        var_h = compute_var_hist(rets, conf)
        ax.axvline(var_h, color=color, lw=2, ls="-", label=f"VaR {conf:.0%} hist = {var_h:.2%}")
        # Fix (Directiva 2.A): se agrega el VaR Cornish-Fisher (ajustado por
        # skew/curtosis empirica) junto al historico -- el VaR Normal puro
        # (curva gris de la densidad, arriba) ignora la asimetria real de
        # ALUA.BA y subestima/sobrestima el downside segun el signo de skew.
        var_cf = compute_var_cornish_fisher(rets, conf)
        ax.axvline(var_cf, color=color, lw=1.5, ls=":", label=f"VaR {conf:.0%} Cornish-Fisher = {var_cf:.2%}")
        cvar_v = compute_cvar(rets, conf)
        ax.axvline(cvar_v, color=color, lw=1.5, ls="--", label=f"CVaR {conf:.0%} = {cvar_v:.2%}")

    ax.set_title("VaR y CVaR — Distribución Retornos Diarios ALUA.BA (histórico + Cornish-Fisher)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Retorno diario")
    ax.legend(fontsize=7, ncol=2)
    fig_var = save_fig(fig, "m10_01_var_cvar")

    # ── FIG 2: Distribución MC con VaR ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(mc_prices, bins=150, density=True, color="#003087", alpha=0.5, label="MC (excl. truncados a 0)")
    var5_mc  = float(np.percentile(mc_prices, 5))
    var1_mc  = float(np.percentile(mc_prices, 1))
    ax.axvline(var5_mc, color="orange", lw=2, label=f"p5 = {var5_mc:.0f} ARS")
    ax.axvline(var1_mc, color="#C8102E", lw=2, label=f"p1 = {var1_mc:.0f} ARS")
    ax.axvline(current_price, color="black", lw=1.5, ls="--", label=f"Mercado {current_price:.0f}")
    ax.set_title(f"Distribución Monte Carlo — Precio Objetivo ALUA (ARS) — {frac_truncado_m10:.1%} truncado a $0 excluido", fontsize=12, fontweight="bold")
    ax.set_xlabel("Precio ARS"); ax.legend()
    fig_mc_dist = save_fig(fig, "m10_02_mc_distribucion_precio")

    # ── FIG 3: Stress test ───────────────────────────────────────────────────
    print("\n[3/3] Stress tests...")
    if datos_sinteticos:
        # Fix (auditoria ronda 4, ALTA): la serie sintetica tiene RangeIndex
        # (no fechas reales) -- "rets.loc['2020-01-15':'2020-03-31']" sobre
        # un RangeIntex es un slicing sin sentido (crashea o devuelve vacio
        # silenciosamente). Un escenario "COVID 2020" no puede existir en
        # datos fabricados: se reportan los 3 escenarios como no disponibles.
        stress = {name: {"ret_acum": None, "vol_anual": None, "n_obs": 0}
                  for name in ["crisis_2020_covid", "volatilidad_2022", "post_elecciones_2023"]}
    else:
        stress = stress_test(rets)
    _todos = list(stress.keys())
    scenarios = [s for s in _todos if stress[s]["ret_acum"] is not None]
    _excluidos = [s for s in _todos if s not in scenarios]
    if _excluidos:
        print(f"  [WARN] Escenario(s) sin datos suficientes (excluidos del grafico): {_excluidos}")
    rets_stress = [stress[s]["ret_acum"] for s in scenarios]
    vols_stress = [stress[s]["vol_anual"] for s in scenarios]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = ["#C8102E" if r < 0 else "#00843D" for r in rets_stress]
    ax1.bar(range(len(scenarios)), [r * 100 for r in rets_stress], color=colors)
    ax1.set_xticks(range(len(scenarios)))
    ax1.set_xticklabels([s.replace("_", "\n") for s in scenarios], fontsize=8)
    ax1.set_title("Retorno Acumulado por Escenario (%)")
    ax1.axhline(0, color="black", lw=0.8)

    ax2.bar(range(len(scenarios)), [v * 100 for v in vols_stress], color="#003087", alpha=0.7)
    ax2.set_xticks(range(len(scenarios)))
    ax2.set_xticklabels([s.replace("_", "\n") for s in scenarios], fontsize=8)
    ax2.set_title("Volatilidad Anualizada por Escenario (%)")

    fig.suptitle("Stress Tests Históricos — ALUA.BA", fontsize=13, fontweight="bold")
    fig_stress = save_fig(fig, "m10_03_stress_tests")

    out = {
        "datos_sinteticos": datos_sinteticos,
        "var_table": var_table,
        "stress_tests": stress,
        "valuation_uncertainty": {
            "pct_truncado_a_cero": round(frac_truncado_m10, 4),
            "mc_p01_ars": round(float(np.percentile(mc_prices, 1)), 2),
            "mc_p05_ars": round(var5_mc, 2),
            "mc_p10_ars": round(float(np.percentile(mc_prices, 10)), 2),
            "mc_p25_ars": round(float(np.percentile(mc_prices, 25)), 2),
            "mc_p50_ars": round(float(np.percentile(mc_prices, 50)), 2),
            "mc_p75_ars": round(float(np.percentile(mc_prices, 75)), 2),
            "mc_p90_ars": round(float(np.percentile(mc_prices, 90)), 2),
            "mc_p95_ars": round(float(np.percentile(mc_prices, 95)), 2),
            "mc_p99_ars": round(float(np.percentile(mc_prices, 99)), 2),
        },
        "figures": {"var_cvar": fig_var, "mc_dist": fig_mc_dist, "stress": fig_stress},
    }

    with open(os.path.join(WORKDIR, "m10_out.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"\n[OK] m10_out.json + 3 figuras guardadas.")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M11

In [14]:
"""
module11_statistics.py — Estadística Financiera
================================================
ADF, Jarque-Bera, correlación, heatmap, rolling metrics, regresiones.
Exporta figuras PNG + m11_out.json
"""

import json, sys, os, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
apply_aluar_theme()


def get_multi_returns(period=CONFIG["market_data_period"]):
    """Fix (auditoria ronda 4, ALTA): except generico silenciaba fallos de
    descarga por ticker sin ningun log ni flag -- m11_out.json no dejaba
    rastro de si la matriz de correlacion termino siendo 5x5, 4x4 o menos.
    Fix (auditoria ronda 4, MEDIA-ALTA): ALUA/MERV/TXAR (ARS nominal) se
    correlacionaban directo contra LME/DXY (USD) sin ajuste de base --
    mezcla de monedas que distorsiona la correlacion real (la devaluacion
    del ARS mueve a ALUA/MERV/TXAR sin que sea una señal genuina de
    correlacion con LME/DXY). Se convierten los 3 activos ARS a USD con el
    mismo CCL diario (GGAL.BA×10/GGAL) que ya usan M10/M12, consistente en
    todo el proyecto."""
    import yfinance as yf
    ars_tickers = {"ALUA": "ALUA.BA", "MERV": "^MERV", "TXAR": "TXAR.BA"}
    usd_tickers = {"LME": "ALI=F", "DXY": "DX-Y.NYB"}

    ccl_daily = None
    try:
        ggal_ba = yf.download("GGAL.BA", period=period, progress=False, auto_adjust=True)["Close"].squeeze()
        ggal_us = yf.download("GGAL", period=period, progress=False, auto_adjust=True)["Close"].squeeze()
        ccl_daily = (ggal_ba * 10 / ggal_us).dropna()
    except Exception as e:
        logging.warning(f"get_multi_returns(): fetch CCL diario fallo ({e}) -- "
                         f"ALUA/MERV/TXAR quedaran en ARS nominal (sin convertir a USD).")

    dfs = {}
    levels = {}
    for name, tk in ars_tickers.items():
        try:
            df = yf.download(tk, period=period, progress=False, auto_adjust=True)
            if not df.empty:
                close = pd.Series(df["Close"].squeeze().values, index=df.index)
                if ccl_daily is not None:
                    close = (close / ccl_daily.reindex(close.index).ffill()).dropna()
                dfs[name] = close.pct_change().dropna()
                levels[name] = close
        except Exception as e:
            logging.warning(f"get_multi_returns(): fetch {tk} ({name}) fallo ({e}) -- excluido de la matriz.")
    for name, tk in usd_tickers.items():
        try:
            df = yf.download(tk, period=period, progress=False, auto_adjust=True)
            if not df.empty:
                close = df["Close"].squeeze()
                close_s = pd.Series(close.values, index=close.index)
                dfs[name] = close_s.pct_change().dropna()
                levels[name] = close_s
        except Exception as e:
            logging.warning(f"get_multi_returns(): fetch {tk} ({name}) fallo ({e}) -- excluido de la matriz.")

    if len(dfs) < 5:
        logging.warning(f"get_multi_returns(): matriz quedo en {len(dfs)}x{len(dfs)} "
                         f"(activos disponibles: {list(dfs.keys())}), no 5x5.")
    result = pd.DataFrame(dfs).dropna(how="all")
    # Fix (auditoria jul-11, ALTA): se persisten los NIVELES (precio/indice,
    # ya en USD) ademas de los retornos -- necesarios para correr ADF sobre
    # el nivel (ver nota en M11_run), no solo sobre el retorno ya
    # diferenciado.
    result.attrs["levels"] = pd.DataFrame(levels).dropna(how="all")
    return result


def adf_test(series: pd.Series) -> dict:
    """ADF test de estacionariedad (statsmodels)."""
    try:
        from statsmodels.tsa.stattools import adfuller
        result = adfuller(series.dropna())
        return {
            "statistic": round(float(result[0]), 4),
            "pvalue": round(float(result[1]), 6),
            "critical_1pct": round(float(result[4]["1%"]), 4),
            "critical_5pct": round(float(result[4]["5%"]), 4),
            "is_stationary_5pct": bool(result[1] < 0.05),
            # Fix (auditoria jul-11, BAJA x2): (a) usedlag/nobs del propio
            # autolag='AIC' nunca se guardaban -- sin eso no se puede
            # verificar si el orden convergio a un valor razonable o saturo
            # el techo (Hamilton 1994, Cap.17.6-17.7); (b) "is_stationary_5pct"
            # equipara "rechazar H0 de raiz unitaria" con "confirmar
            # estacionariedad" -- son logicamente distintos. Se agrega un
            # nombre mas preciso sin romper el campo existente (consumido
            # aguas abajo).
            "usedlag": int(result[2]),
            "nobs_adf": int(result[3]),
            "rejects_unit_root_5pct": bool(result[1] < 0.05),
        }
    except Exception as e:
        return {"error": str(e)}


def jarque_bera_test(series: pd.Series) -> dict:
    """Jarque-Bera test de normalidad."""
    arr = series.dropna().values
    jb_result = stats.jarque_bera(arr)
    jb_stat, jb_pval = float(jb_result.statistic), float(jb_result.pvalue)
    skew = float(stats.skew(arr))
    kurt = float(stats.kurtosis(arr))
    return {
        "statistic": round(float(jb_stat), 4),
        "pvalue": round(float(jb_pval), 8),
        "skewness": round(skew, 4),
        "excess_kurtosis": round(kurt, 4),
        "is_normal_5pct": bool(jb_pval > 0.05),
        # Fix (auditoria jul-11, BAJA): "no rechazar H0 de normalidad" no es
        # lo mismo que "confirmar" normalidad -- mismo matiz que
        # is_stationary_5pct en adf_test().
        "fails_to_reject_normality_5pct": bool(jb_pval > 0.05),
    }


def M11_run():
    print("=" * 65)
    print("M11 STATISTICS — ADF + JB + Correlacion + Rolling + Regresion")
    print("=" * 65)

    print("\n[1/4] Descargando retornos multi-activo...")
    rets = get_multi_returns(CONFIG["market_data_period"])
    available = list(rets.columns)
    print(f"  Activos disponibles: {available}")
    if len(rets) > 0:
        print(f"  Rango de fechas: {rets.index.min().date()} → {rets.index.max().date()} ({len(rets)} sesiones)")

    # ── ADF Tests ─────────────────────────────────────────────────────────────
    print("\n[2/4] ADF + Jarque-Bera...")
    adf_results = {}
    jb_results  = {}
    for col in available:
        adf_results[col] = adf_test(rets[col])
        jb_results[col]  = jarque_bera_test(rets[col])
        adf_stat = adf_results[col].get("is_stationary_5pct", "?")
        jb_norm  = jb_results[col].get("is_normal_5pct", "?")
        print(f"  {col}: ADF_estacionario={adf_stat}, JB_normal={jb_norm}, kurt={jb_results[col].get('excess_kurtosis','?')}")

    # Fix (auditoria jul-11, ALTA): adf_test() solo corria sobre RETORNOS
    # (ya diferenciados) -- testear estacionariedad sobre una serie ya
    # diferenciada es casi tautologico (si el log-nivel es I(1), el retorno
    # es I(0) por construccion; de hecho los estadisticos salen entre -8.8 y
    # -34.0, muy por debajo de cualquier critico). El test economicamente
    # relevante (Hamilton 1994, Cap.17) es sobre el NIVEL -- relevante
    # tambien para M9 (Ornstein-Uhlenbeck), cuyo propio comentario admitia
    # "M11 ya tiene ADF, nunca se reutiliza aca" (M9 corre antes que M11 en
    # el pipeline asi que ahora corre su propio ADF auto-contenido; esto se
    # agrega aca para que el analisis de estacionariedad de M11 tambien
    # cubra el nivel, no solo el retorno).
    levels_df = rets.attrs.get("levels")
    adf_niveles = {}
    if levels_df is not None:
        for col in levels_df.columns:
            log_level = np.log(levels_df[col].dropna())
            adf_niveles[col] = adf_test(log_level)
            print(f"  {col} (nivel, log-precio): ADF_stat={adf_niveles[col].get('statistic','?')}, "
                  f"rejects_unit_root_5pct={adf_niveles[col].get('rejects_unit_root_5pct','?')}")

    # ── Matriz de correlación ─────────────────────────────────────────────────
    corr_matrix = rets[available].corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    n = len(available)
    cmap = plt.cm.RdYlGn
    im = ax.imshow(corr_matrix.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(n)); ax.set_xticklabels(available, rotation=45, ha="right")
    ax.set_yticks(range(n)); ax.set_yticklabels(available)
    for i in range(n):
        for j in range(n):
            val = corr_matrix.values[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=10, fontweight="bold",
                    color="white" if abs(val) > 0.6 else "black")
    plt.colorbar(im, ax=ax)
    ax.set_title("Matriz de Correlación — Retornos Diarios", fontsize=13, fontweight="bold")
    fig_heatmap = save_fig(fig, "m11_01_correlation_heatmap")

    # ── Rolling correlation ALUA-MERV ─────────────────────────────────────────
    if "ALUA" in rets.columns and "MERV" in rets.columns:
        # Fix (auditoria jul-09): calcular sobre el PAR con su propio dropna,
        # no sobre el DataFrame de 5 activos (LME/DXY tienen calendario de
        # futuros distinto al bursatil ARG y contaminan la ventana rolling
        # con NaN ajenos a este par -- ver nota completa mas arriba en la celda).
        _par_am = rets[["ALUA", "MERV"]].dropna()
        roll_corr = _par_am["ALUA"].rolling(63).corr(_par_am["MERV"])
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(roll_corr.index, roll_corr, color="#003087", lw=1.5)
        ax.axhline(roll_corr.mean(), color="#C8102E", lw=1, ls="--",
                   label=f"Media {roll_corr.mean():.2f}")
        ax.fill_between(roll_corr.index, roll_corr, 0, alpha=0.2, color="#003087")
        ax.set_ylim(-1, 1)
        ax.set_title("Correlación Rolling 63d — ALUA vs MERVAL", fontsize=13, fontweight="bold")
        ax.legend()
        fig_roll_corr = save_fig(fig, "m11_02_rolling_correlation_alua_merv")
    else:
        fig_roll_corr = None

    # ── Rolling volatility ─────────────────────────────────────────────────────
    # Fix (auditoria ronda 4, MEDIA): antes truncaba a min(3, len(available))
    # sin aviso -- LME y DXY quedaban descartados en silencio. Se grafican
    # los 5 activos disponibles (grilla 2 filas si son mas de 3).
    n_av = len(available)
    n_cols = min(3, n_av) if n_av > 0 else 1
    n_rows = int(np.ceil(n_av / n_cols)) if n_av > 0 else 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows), sharey=True)
    axes = np.atleast_1d(axes).flatten()
    colors = ["#003087", "#C8102E", "#00843D", "#FF8C00", "#6A0DAD"]
    for ax, col, color in zip(axes, available, colors):
        vol = rets[col].rolling(21).std() * np.sqrt(252) * 100
        ax.plot(vol.index, vol, color=color, lw=1.2)
        ax.set_title(f"Vol Rolling 21d — {col}")
        ax.set_ylabel("Vol anualizada (%)")
    for ax in axes[n_av:]:
        ax.set_visible(False)
    fig.suptitle("Volatilidad Rolling 21d Comparada (todos los activos disponibles)", fontsize=13, fontweight="bold")
    fig_roll_vol = save_fig(fig, "m11_03_rolling_volatility_comparada")

    # ── Regresión multivariada: ALUA ~ MERV + LME + DXY ──────────────────────
    reg_result = {}
    if "ALUA" in rets.columns:
        predictors = [c for c in ["MERV", "LME", "DXY"] if c in rets.columns]
        if predictors:
            from statsmodels.regression.linear_model import OLS
            from statsmodels.tools import add_constant
            y = rets["ALUA"].dropna()
            X = rets[predictors].dropna()
            common = y.index.intersection(X.index)
            # Fix (auditoria ronda 4, mejora): errores estandar robustos
            # HAC/Newey-West -- retornos diarios financieros tipicamente
            # exhiben heterocedasticidad/autocorrelacion que OLS clasico
            # subestima en los errores estandar (y por ende en los pvalues).
            # Fix (auditoria jul-11, MEDIA): maxlags=5 era un literal fijo, no
            # derivado del tamano muestral. Newey & West (1994), "Automatic
            # Lag Selection in Covariance Matrix Estimation", formalizan
            # L=floor(4*(T/100)^(2/9)). Con n_obs real de esta corrida
            # (~1187), esa regla da L=6, no 5.
            _hac_maxlags = int(np.floor(4 * (len(common) / 100) ** (2 / 9)))
            model = OLS(y.loc[common], add_constant(X.loc[common])).fit(
                cov_type="HAC", cov_kwds={"maxlags": _hac_maxlags})
            reg_result = {
                "r_squared": round(float(model.rsquared), 4),
                "adj_r_squared": round(float(model.rsquared_adj), 4),
                "betas": {c: round(float(model.params[c]), 4) for c in predictors},
                "pvalues": {c: round(float(model.pvalues[c]), 6) for c in predictors},
                "alpha": round(float(model.params["const"]), 6),
                # Fix (auditoria ronda 4, MEDIA): n_obs efectivo no se
                # reportaba -- sin esto no se puede verificar si la regresion
                # sufre la misma fragmentacion de calendario ya corregida en
                # la correlacion rolling (ver nota mas arriba en esta celda).
                "n_obs": int(len(common)),
                "hac_maxlags": _hac_maxlags,
                "errores_estandar": f"HAC/Newey-West (maxlags={_hac_maxlags}, regla automatica Newey-West 1994)",
            }
            print(f"\n  Regresion ALUA ~ {'+'.join(predictors)}: R²={reg_result['r_squared']:.3f}")
            print(f"  Betas: {reg_result['betas']}")

    # ── DXY vs LME correlación ────────────────────────────────────────────────
    if "DXY" in rets.columns and "LME" in rets.columns:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        # Fix (auditoria jul-09): mismo problema de calendarios mezclados que
        # ALUA-MERV arriba -- se recalcula sobre el par propio.
        _par_dl = rets[["DXY", "LME"]].dropna()
        roll_dxy_lme = _par_dl["DXY"].rolling(63).corr(_par_dl["LME"])
        ax1.scatter(rets["DXY"], rets["LME"], alpha=0.2, color="#003087", s=5)
        common_idx = rets["DXY"].dropna().index.intersection(rets["LME"].dropna().index)
        dxy_arr = rets["DXY"].loc[common_idx].values
        lme_arr = rets["LME"].loc[common_idx].values
        slope, intercept, r, p, _ = stats.linregress(dxy_arr, lme_arr)
        x_line = np.linspace(rets["DXY"].quantile(0.01), rets["DXY"].quantile(0.99), 100)
        ax1.plot(x_line, slope * x_line + intercept, "#C8102E", lw=2, label=f"R={r:.2f}")
        ax1.set_xlabel("DXY return"); ax1.set_ylabel("LME return")
        ax1.set_title("Correlación DXY vs LME Aluminum")
        ax1.legend()

        ax2.plot(roll_dxy_lme.index, roll_dxy_lme, color="#003087", lw=1.5)
        ax2.axhline(0, color="black", lw=0.8, ls="--")
        ax2.set_ylim(-1, 1)
        ax2.set_title("Rolling 63d DXY vs LME")
        fig.suptitle("DXY vs LME Aluminium — Correlación Histórica", fontsize=13, fontweight="bold")
        fig_dxy_lme = save_fig(fig, "m11_04_dxy_lme_correlation")
    else:
        fig_dxy_lme = None

    out = {
        "activos_disponibles": available,
        "adf_tests": adf_results,
        "adf_tests_niveles": adf_niveles,
        "jarque_bera": jb_results,
        "correlation_matrix": corr_matrix.round(4).to_dict(),
        "regression_multivariada": reg_result,
        "figures": {
            "heatmap": fig_heatmap,
            "rolling_corr": fig_roll_corr,
            "rolling_vol": fig_roll_vol,
            "dxy_lme": fig_dxy_lme,
        },
    }

    with open(os.path.join(WORKDIR, "m11_out.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"\n[OK] m11_out.json + figuras guardadas.")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M12

In [15]:
"""
module12_portfolio.py — Optimización de Portafolio
====================================================
Media-varianza, efficient frontier, Kelly criterion.
Exporta figuras PNG + m12_out.json

AUDITORÍA (tribunal, jul-2026) — 2 correcciones aplicadas:
4. `efficient_frontier_random` vectorizado con numpy (sin bucle `for` por
   portafolio, per regla de "Reglas de Código" del proyecto) y su Sharpe
   corregido a (ret - rf) / vol — antes usaba ret / vol, sin restar rf,
   inconsistente con `max_sharpe()` que sí lo hace.
5. `get_returns_portfolio` convertía retornos de ALUA.BA/TXAR.BA/GGAL.BA/
   BMA.BA (ARS nominal) y los mezclaba directo contra `rf` (Treasury USD)
   para Sharpe y Kelly — mezcla de monedas que infla ambos por la
   devaluación del peso. Ahora los precios se convierten a USD con el CCL
   diario (GGAL.BA×10/GGAL, mismo método que M1) antes de computar retornos.
"""

import json, sys, os, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import optimize as opt
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
apply_aluar_theme()
N_PORTFOLIOS = 5000


def get_returns_portfolio(period=CONFIG["market_data_period"]):
    import yfinance as yf
    # Fix (auditoria jul-11, MEDIA): GGAL y BMA tienen correlacion 0.91
    # (verificado en m12_out.json) -- casi el mismo factor de riesgo
    # (sector bancario), debilitando el beneficio de diversificacion que la
    # frontera eficiente pretende mostrar. El resto del deck posiciona al
    # Merval como benchmark de mercado ("ALUA vs Merval vs TXAR"), pero M12
    # nunca lo incluia en la optimizacion. Se agrega ^MERV como quinto
    # activo (mismo metodo de conversion USD via CCL que ya usa M11 para
    # este mismo ticker), sin quitar ninguno de los ya existentes.
    tickers = {"ALUA": "ALUA.BA", "TXAR": "TXAR.BA", "GGAL": "GGAL.BA", "BMA": "BMA.BA", "MERV": "^MERV"}

    # CCL diario (GGAL.BA×10 / GGAL ADR) — mismo método que M1, en serie diaria
    # en vez de spot, para convertir los precios ARS de BYMA a USD.
    ggal_ba = yf.download("GGAL.BA", period=period, progress=False, auto_adjust=True)["Close"].squeeze()
    ggal_us = yf.download("GGAL", period=period, progress=False, auto_adjust=True)["Close"].squeeze()
    ccl_daily = (ggal_ba * 10 / ggal_us).dropna()

    dfs = {}
    for name, tk in tickers.items():
        try:
            df = yf.download(tk, period=period, progress=False, auto_adjust=True)
            if not df.empty:
                close_ars = pd.Series(df["Close"].squeeze().values, index=df.index)
                close_usd = (close_ars / ccl_daily.reindex(close_ars.index).ffill()).dropna()
                dfs[name] = close_usd.pct_change().dropna()
        except Exception as e:
            # Fix (auditoria ronda 4, MEDIA-ALTA): antes silenciaba cualquier
            # fallo de descarga por ticker sin aviso -- el portafolio se
            # degradaba a menos activos sin que quedara ninguna traza en
            # consola ni en m12_out.json.
            logging.warning(f"get_returns_portfolio(): fetch {tk} ({name}) fallo ({e}) -- excluido del portafolio.")
    # Fix (auditoria ronda 5, ALTA -- data lineage): dropna(how="all") seguido
    # de dropna() (how="any" por default) puede recortar agresivamente N si
    # los calendarios de cotizacion difieren entre BYMA/ADR -- antes esto no
    # quedaba registrado en ningun lado (ni consola ni m12_out.json). Se
    # persisten tickers_requested/n_obs via .attrs para que M12_run() los
    # escriba en el output y una caida de completitud sea auditable, no
    # silenciosa.
    df = pd.DataFrame(dfs).dropna(how="all").dropna()
    df.attrs["tickers_requested"] = list(tickers.keys())
    return df


def portfolio_metrics(weights: np.ndarray, mu: np.ndarray, cov: np.ndarray) -> tuple:
    ret = float(weights @ mu * 252)
    vol = float(np.sqrt(weights @ cov @ weights * 252))
    return ret, vol


def efficient_frontier_random(mu: np.ndarray, cov: np.ndarray, n: int = N_PORTFOLIOS,
                               rf: float = 0.04) -> pd.DataFrame:
    """Vectorizado con numpy: sin bucle for por portafolio (5000 pesos Dirichlet
    generados de una sola vez; ret/vol/sharpe calculados con álgebra matricial)."""
    rng = np.random.default_rng(42)
    n_assets = len(mu)
    W = rng.dirichlet(np.ones(n_assets), size=n)                  # (n, n_assets)
    rets = (W @ mu) * 252                                          # (n,)
    vols = np.sqrt(np.einsum("ij,jk,ik->i", W, cov, W) * 252)      # (n,)
    sharpes = np.where(vols > 0, (rets - rf) / vols, 0.0)
    return pd.DataFrame({"ret": rets, "vol": vols, "sharpe": sharpes,
                          "weights": list(W)})


def efficient_frontier_qp(mu: np.ndarray, cov: np.ndarray, n_points: int = 40) -> pd.DataFrame:
    """Frontera eficiente analitica (QP), Markowitz (1952): min w'Sigma w
    sujeto a w'mu=R_objetivo, sum(w)=1, w>=0 (sin shorting), para una grilla
    de retornos objetivo. Fix (auditoria jul-11, MEDIA): complementa la nube
    Monte Carlo de efficient_frontier_random() (que aproxima la frontera
    muestreando pesos Dirichlet al azar, y con pocos activos puede
    sub-muestrear soluciones de esquina -- max_sharpe_portfolio hoy tiene
    TXAR en peso exactamente 0.0, verificado en m12_out.json -- dando la
    falsa impresion de un Max Sharpe "fuera" de la nube) con la solucion
    real del problema de programacion cuadratica."""
    n = len(mu)
    r_min, r_max = float(mu.min() * 252), float(mu.max() * 252)
    targets = np.linspace(r_min, r_max, n_points)
    rets_qp, vols_qp = [], []
    bounds = [(0, 1)] * n
    w0 = np.ones(n) / n
    for r_t in targets:
        cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1},
                {"type": "eq", "fun": lambda w, rt=r_t: w @ mu * 252 - rt}]
        res = opt.minimize(lambda w: w @ cov @ w * 252, w0, method="SLSQP",
                            bounds=bounds, constraints=cons)
        if res.success:
            rets_qp.append(r_t)
            vols_qp.append(float(np.sqrt(max(res.fun, 0))))
    return pd.DataFrame({"ret": rets_qp, "vol": vols_qp})


def max_sharpe(mu: np.ndarray, cov: np.ndarray, rf: float = 0.04) -> dict:
    n = len(mu)
    def neg_sharpe(w):
        r, v = portfolio_metrics(w, mu, cov)
        return -(r - rf) / v if v > 0 else 0
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    bounds = [(0, 1)] * n
    w0 = np.ones(n) / n
    res = opt.minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=constraints)
    ret, vol = portfolio_metrics(res.x, mu, cov)
    return {"weights": res.x.tolist(), "ret": ret, "vol": vol, "sharpe": (ret - rf) / vol}


def min_variance(mu: np.ndarray, cov: np.ndarray) -> dict:
    n = len(mu)
    def port_vol(w):
        return float(np.sqrt(w @ cov @ w * 252))
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    bounds = [(0, 1)] * n
    w0 = np.ones(n) / n
    res = opt.minimize(port_vol, w0, method="SLSQP", bounds=bounds, constraints=constraints)
    ret, vol = portfolio_metrics(res.x, mu, cov)
    return {"weights": res.x.tolist(), "ret": ret, "vol": vol}


def kelly_criterion(mu: float, sigma: float, rf: float) -> float:
    """Kelly fraction = (mu - rf) / sigma^2, con mu = media ARITMETICA anual.
    Fix (auditoria jul-11, ALTA -- revierte un "fix" previo que introducia el
    mismo error de magnitud opuesta): el criterio de Kelly (Kelly 1956;
    Rotando & Thorp 1992) maximiza g(f) = rf + f*(mu-rf) - 0.5*f^2*sigma^2
    (mu ARITMETICO) y se resuelve dg/df=0 -> f*=(mu-rf)/sigma^2. El termino
    -0.5*f^2*sigma^2 YA ES el drag de varianza -- una version anterior de
    este fix restaba 0.5*sigma^2 aparte (sin el factor f^2, antes de dividir
    por sigma^2), asumiendo que Kelly requiere el drift GEOMETRICO
    (mu_geo=mu-0.5*sigma^2). Eso duplica el descuento: algebraicamente,
    (mu_geo-rf)/sigma^2 = (mu-rf)/sigma^2 - 0.5 para CUALQUIER dato (no es
    un efecto de la muestra). Verificado con los datos de ALUA
    (mu=27.78%, sigma=52.39%, rf=4.57%): formula anterior daba f*=34.57%,
    la correcta da 84.57% -- diferencia exacta de 0.5000."""
    return (mu - rf) / (sigma ** 2) if sigma > 0 else 0.0


def component_contribution_risk(weights: np.ndarray, cov: np.ndarray) -> np.ndarray:
    """Component Contribution to Risk (Alexander 1993 cap.7; Sharpe 2002;
    Bailey/Lopez de Prado): CCR_i = w_i * (Sigma @ w)_i / (w' Sigma w).
    Suma exactamente 1.0 -- descompone el 100% de la varianza del portafolio
    en el aporte marginal de cada activo (vectorizado, sin loop por activo,
    per las Reglas de Codigo del proyecto)."""
    port_var = float(weights @ cov @ weights)
    if port_var <= 0:
        return np.zeros_like(weights)
    mctr = cov @ weights  # contribucion marginal a la varianza (una sola operacion matricial)
    return weights * mctr / port_var


def single_index_decomposition(mu: np.ndarray, cov: np.ndarray, assets: list,
                                market: str = "MERV") -> dict:
    """Modelo de Indice Unico (Sharpe, 1963, 'A Simplified Model for Portfolio
    Analysis'): Var(r_i) = beta_i^2 * Var(r_mkt) + Var(eps_i). Descompone la
    varianza de cada activo en riesgo SISTEMATICO (de mercado) vs
    IDIOSINCRATICO (especifico), usando MERV como proxy de mercado -- mismo
    universo/ventana/matriz de covarianza que ya usa el resto de M12, sin
    datos nuevos ni llamadas a APIs adicionales."""
    if market not in assets:
        return {}
    m_idx = assets.index(market)
    var_mkt_anual = float(cov[m_idx, m_idx] * 252)
    if var_mkt_anual <= 0:
        return {}
    betas = cov[:, m_idx] / cov[m_idx, m_idx]              # vectorizado (todos los activos a la vez)
    var_total_anual = np.diag(cov) * 252
    var_sistematica = (betas ** 2) * var_mkt_anual
    # Identidad de regresion OLS (R^2=corr^2<=1): var_idiosincratica es
    # matematicamente >=0; el maximum() solo absorbe ruido de punto flotante
    # (~1e-15), no trunca ningun valor economico (no aplica la regla Penman
    # de EV/FCFF -- aca es una identidad de varianza, no una cota de negocio).
    var_idiosincratica = np.maximum(var_total_anual - var_sistematica, 0.0)
    r2 = np.where(var_total_anual > 0, var_sistematica / var_total_anual, 0.0)
    return {
        assets[i]: {
            "beta_mkt": round(float(betas[i]), 4),
            "var_total_anual": round(float(var_total_anual[i]), 6),
            "var_sistematica_anual": round(float(var_sistematica[i]), 6),
            "var_idiosincratica_anual": round(float(var_idiosincratica[i]), 6),
            "r2": round(float(r2[i]), 4),
        }
        for i in range(len(assets)) if assets[i] != market
    }


def M12_run():
    print("=" * 65)
    print("M12 PORTFOLIO — Media-Varianza + Efficient Frontier + Kelly")
    print("=" * 65)

    m1  = load("m1_out.json")
    m8  = load("m8_out.json")
    rf  = m1["rf"]

    print("\n[1/3] Descargando retornos portafolio (USD, convertidos vía CCL)...")
    rets = get_returns_portfolio(CONFIG["market_data_period"])

    # Fix (auditoria ronda 4, ALTA): el fallback fabricaba medias/vols
    # (0.001/0.025, 0.0012/0.030, 0.0015/0.035) sin ninguna fuente -- el
    # "fallback declarado en el codigo" era en si mismo el dato prohibido
    # por CLAUDE.md (nunca fabricar datos). Se mantiene como ultimo recurso
    # de robustez (una falla total de red no debe tumbar todo el pipeline),
    # pero ahora queda flageado explicitamente en el output en vez de
    # presentarse como si fuera data real de mercado.
    datos_sinteticos = rets.empty or len(rets.columns) < 2
    if datos_sinteticos:
        print("  [AVISO] Datos insuficientes -- usando retornos SINTETICOS (fallback documentado, "
              "NO son datos de mercado reales). Ver 'datos_sinteticos' en m12_out.json.")
        rets = pd.DataFrame({
            "ALUA": np.random.default_rng(42).normal(0.001, 0.025, 1250),
            "TXAR": np.random.default_rng(43).normal(0.0012, 0.030, 1250),
            "GGAL": np.random.default_rng(44).normal(0.0015, 0.035, 1250),
        })

    assets = list(rets.columns)
    mu_raw  = rets.mean().values
    cov = rets.cov().values
    # Fix (auditoria jul-11, MEDIA): mu se usaba como media muestral cruda
    # en Max Sharpe/Kelly, sin ningun tratamiento de error de estimacion --
    # con ~5 anios de datos diarios y vols de 52%-64% anuales, el error
    # estandar de la media anualizada es del mismo orden que los propios
    # valores de mu (el clasico "estimation-error maximizer" de Michaud
    # 1989: sobre-pondera activos con mu muestral alto por puro azar). El
    # proyecto ya aplica contraccion Bayesiana de Vasicek al Beta en M5 --
    # se aplica shrinkage analogo (Jorion 1986, "Bayes-Stein Estimation for
    # Portfolio Analysis", JFQA 21(3)) hacia el retorno de la cartera de
    # minima varianza antes de alimentar Max Sharpe/Kelly.
    try:
        _inv_cov = np.linalg.inv(cov)
        _ones = np.ones(len(mu_raw))
        _mu_min = float(_ones @ _inv_cov @ mu_raw) / float(_ones @ _inv_cov @ _ones)
        _dev = mu_raw - _mu_min * _ones
        _denom = float(_dev @ _inv_cov @ _dev)
        if _denom > 0:
            _lam = (len(mu_raw) + 2) / _denom
            _phi = float(np.clip(_lam / (len(rets) + _lam), 0.0, 1.0))
        else:
            _phi = 0.0
        mu = (1 - _phi) * mu_raw + _phi * _mu_min * _ones
    except np.linalg.LinAlgError:
        mu, _phi = mu_raw, 0.0
    # Fix (auditoria ronda 5, ALTA -- data lineage): registrar que tickers se
    # pidieron vs. cuales sobrevivieron al fetch+dropna, para que una caida
    # de completitud (falla de yfinance, calendarios desalineados) quede
    # trazada en m12_out.json en vez de ser silenciosa aguas abajo (Slide 39).
    tickers_solicitados = rets.attrs.get("tickers_requested", assets) if not datos_sinteticos else assets
    tickers_excluidos = [t for t in tickers_solicitados if t not in assets]
    n_obs = int(len(rets))
    print(f"  Activos: {assets} (n_obs={n_obs})")
    if tickers_excluidos:
        print(f"  [AVISO] Tickers solicitados pero excluidos del portafolio: {tickers_excluidos}")

    # ── Frontera eficiente aleatoria ──────────────────────────────────────────
    print("\n[2/3] Generando frontera eficiente...")
    ef_df = efficient_frontier_random(mu, cov, N_PORTFOLIOS, rf)
    ef_qp = efficient_frontier_qp(mu, cov)

    # Portafolio máximo Sharpe
    ms = max_sharpe(mu, cov, rf)
    # Portafolio mínima varianza
    mv = min_variance(mu, cov)

    print(f"  Max Sharpe: ret={ms['ret']:.1%}, vol={ms['vol']:.1%}, Sharpe={ms['sharpe']:.2f}")
    print(f"  Min Var:    ret={mv['ret']:.1%}, vol={mv['vol']:.1%}")
    print(f"  Pesos Max Sharpe: {dict(zip(assets, [round(w,3) for w in ms['weights']]))}")

    # Descomposicion de varianza (Alexander/Sharpe/Bailey, extension aprobada
    # en CLAUDE.md): Component Contribution to Risk sobre ambos portafolios
    # optimos + Modelo de Indice Unico (Sharpe 1963) contra MERV.
    ccr_max_sharpe = component_contribution_risk(np.array(ms["weights"]), cov)
    ccr_min_var = component_contribution_risk(np.array(mv["weights"]), cov)
    sid = single_index_decomposition(mu, cov, assets, market="MERV")
    if "ALUA" in assets and sid.get("ALUA"):
        _ai = assets.index("ALUA")
        print(f"  CCR ALUA (Max Sharpe): {ccr_max_sharpe[_ai]:.1%} del riesgo "
              f"(peso {ms['weights'][_ai]:.1%}) | beta vs MERV={sid['ALUA']['beta_mkt']:.2f}, "
              f"R2={sid['ALUA']['r2']:.1%}")

    fig, ax = plt.subplots(figsize=(10, 7))
    sc = ax.scatter(ef_df["vol"] * 100, ef_df["ret"] * 100,
                    c=ef_df["sharpe"], cmap="viridis", alpha=0.4, s=8)
    ax.scatter(ms["vol"] * 100, ms["ret"] * 100, color="#C8102E", s=200,
               marker="*", zorder=5, label=f"Max Sharpe ({ms['sharpe']:.2f})")
    ax.scatter(mv["vol"] * 100, mv["ret"] * 100, color="#003087", s=200,
               marker="D", zorder=5, label="Min Varianza")
    if len(ef_qp):
        ax.plot(ef_qp["vol"] * 100, ef_qp["ret"] * 100, color="black", lw=2,
                zorder=4, label="Frontera QP (Markowitz, analitica)")

    # Marcar ALUA como activo individual
    alua_ret = float(mu[assets.index("ALUA")] * 252 * 100) if "ALUA" in assets else None
    alua_vol = float(np.sqrt(cov[assets.index("ALUA"), assets.index("ALUA")] * 252) * 100) if "ALUA" in assets else None
    # Fix (auditoria ronda 4, mejora): "if alua_ret and alua_vol" fallaria
    # silenciosamente si el retorno fuera exactamente 0.0 (0.0 es falsy).
    if alua_ret is not None and alua_vol is not None:
        ax.scatter(alua_vol, alua_ret, color="#FF8C00", s=150, marker="^",
                   zorder=5, label=f"ALUA solo")

    plt.colorbar(sc, ax=ax, label="Sharpe Ratio")
    ax.set_xlabel("Volatilidad Anualizada (%)")
    ax.set_ylabel("Retorno Anualizado (%)")
    ax.set_title(f"Frontera Eficiente — {N_PORTFOLIOS:,} Portafolios Aleatorios", fontsize=13, fontweight="bold")
    ax.legend()
    fig_ef = save_fig(fig, "m12_01_efficient_frontier")

    # ── Risk/Return surface ───────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (asset, color) in enumerate(zip(assets, ["#003087","#C8102E","#00843D","#FF8C00","#6A0DAD"])):
        r = float(mu[i] * 252 * 100)
        v = float(np.sqrt(cov[i,i] * 252) * 100)
        ax.scatter(v, r, color=color, s=200, zorder=5)
        ax.annotate(asset, (v, r), xytext=(5, 5), textcoords="offset points", fontsize=10, fontweight="bold")
    ax.scatter(ms["vol"]*100, ms["ret"]*100, marker="*", s=300, color="gold", zorder=6, label="Max Sharpe")
    ax.scatter(mv["vol"]*100, mv["ret"]*100, marker="D", s=200, color="silver", zorder=6, label="Min Var")
    ax.set_xlabel("Volatilidad Anualizada (%)"); ax.set_ylabel("Retorno Anualizado (%)")
    ax.set_title("Risk/Return — Activos Individuales vs Portafolios Óptimos", fontsize=13, fontweight="bold")
    ax.legend()
    fig_rr = save_fig(fig, "m12_02_risk_return_surface")

    # ── Kelly criterion para ALUA ─────────────────────────────────────────────
    if "ALUA" in assets:
        alua_mu    = float(mu[assets.index("ALUA")] * 252)
        alua_sigma = float(np.sqrt(cov[assets.index("ALUA"), assets.index("ALUA")] * 252))
        kelly_f    = kelly_criterion(alua_mu, alua_sigma, rf)
        kelly_half = kelly_f / 2  # Half Kelly (práctica de riesgo)
        print(f"\n  Kelly ALUA: f={kelly_f:.4f} ({kelly_f:.1%}), Half-Kelly={kelly_half:.1%}")
    else:
        kelly_f, kelly_half = None, None

    out = {
        "assets": assets,
        "mu_anual": (mu * 252).tolist(),
        "mu_anual_sin_shrinkage": (mu_raw * 252).tolist(),
        "shrinkage_phi": round(_phi, 4),
        "vol_anual": [float(np.sqrt(cov[i,i]*252)) for i in range(len(assets))],
        "frontera_qp": {"ret_anual": [round(r, 4) for r in ef_qp["ret"]], "vol_anual": [round(v, 4) for v in ef_qp["vol"]]},
        "corr_matrix": pd.DataFrame(rets.corr()).round(4).to_dict(),
        "max_sharpe_portfolio": {
            "weights": dict(zip(assets, [round(w,4) for w in ms["weights"]])),
            "ret_anual": round(ms["ret"], 4),
            "vol_anual": round(ms["vol"], 4),
            "sharpe": round(ms["sharpe"], 4),
        },
        "min_variance_portfolio": {
            "weights": dict(zip(assets, [round(w,4) for w in mv["weights"]])),
            "ret_anual": round(mv["ret"], 4),
            "vol_anual": round(mv["vol"], 4),
        },
        "risk_contribution_max_sharpe": dict(zip(assets, [round(float(c), 4) for c in ccr_max_sharpe])),
        "risk_contribution_min_var": dict(zip(assets, [round(float(c), 4) for c in ccr_min_var])),
        "single_index_decomposition": sid,
        "single_index_market_proxy": "MERV",
        "kelly_alua": {"kelly_full": round(kelly_f, 4) if kelly_f else None,
                       "kelly_half": round(kelly_half, 4) if kelly_half else None},
        "datos_sinteticos": datos_sinteticos,
        # Fix (auditoria ronda 5, ALTA -- data lineage): ver nota en
        # get_returns_portfolio/arriba -- trazabilidad de completitud.
        "tickers_solicitados": tickers_solicitados,
        "tickers_excluidos": tickers_excluidos,
        "n_obs": n_obs,
        "figures": {"efficient_frontier": fig_ef, "risk_return": fig_rr},
    }

    with open(os.path.join(WORKDIR, "m12_out.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"\n[OK] m12_out.json + 2 figuras guardadas.")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

### M13

In [16]:
"""
module13_synthesis.py — Síntesis Final + Dictamen de Inversión
===============================================================
Executive summary, tesis de inversión, catalizadores, riesgos,
escenarios Bull/Base/Bear, dictamen tipo comité institucional.
Genera todos los gráficos financieros restantes + m13_out.json

AUDITORÍA (Fase 3 — calidad institucional / cero overfitting):
- Divisor histórico↔proyección ya no es el literal 2025.5: se calcula como
  el punto medio real entre el último año histórico y el primer proyectado.
- Etiquetas de barras vía ax.bar_label() (posición dinámica de matplotlib)
  en vez de offsets fijos (+3, +20, +30, +80) que se rompían si la escala
  de los datos cambiaba.
- Ejes monetarios y porcentuales formateados con _clean_y(money=True/pct=True)
  (K/M/B dinámico, ver celda de tema) en vez de texto plano sin formato.
- Bug corregido: "ev_ebitda_fy25" se leía de m6 top-level (siempre "N/A");
  el valor real vive en m6["multiplos"]["ev_ebitda_fy25"].
- Bullets de riesgo que citaban ratios fijos (CAPEX en ARS, deuda/EBITDA)
  ahora se calculan desde los outputs reales en vez de texto hardcodeado.
"""

import json, sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mtick
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
apply_aluar_theme()


def M13_run():
    print("=" * 65)
    print("M13 SYNTHESIS — Dictamen Institucional + Graficos Completos")
    print("=" * 65)

    # Cargar todos los módulos
    m1  = load("m1_out.json")
    m4  = load("m4_out.json")
    # ── Shims de compatibilidad para m4 v3 (historical/projections dicts) ──
    if "years_hist" not in m4 and "historical" in m4:
        m4["years_hist"] = sorted([int(k) for k in m4["historical"].keys()])
        m4["fcff_historico_usdmm"] = [
            m4["historical"].get(str(yr), {}).get("fcff_usdmm", 0)
            for yr in m4["years_hist"]
        ]
    if "years_proj" not in m4 and "projections" in m4:
        m4["years_proj"] = sorted([int(k) for k in m4["projections"].keys()])
    # ── Garantizar que fcff_historico_usdmm exista ─────────────────────────
    if "fcff_historico_usdmm" not in m4:
        m4["fcff_historico_usdmm"] = [0] * len(m4.get("years_hist", []))
    m5  = load("m5_out.json")
    m6  = load("m6_out.json")
    m7  = load("m7_out.json")
    can = load("canonical_financials.json")
    static = load("static_inputs.json")
    # Fix (auditoria ronda 4, mejora): M12 (portafolio Markowitz/Kelly) quedaba
    # completamente huerfano -- M13 nunca lo cargaba ni referenciaba. Se
    # incluye un snapshot minimo en el output (sin agregar una figura/slide
    # nueva, fuera de alcance de este fix puntual).
    try:
        m12 = load("m12_out.json")
    except Exception:
        m12 = None

    ccl    = m1["ccl"]
    wacc   = m5["wacc"]
    ke     = m5["ke"]
    target = m6["target_ars"]
    mkt_px = m6["alua_px_mkt"]
    ev     = m6["ev_usdmm"]
    dnet   = m6["deuda_neta_usdmm"]
    equity = m6["equity_usdmm"]
    shares = m6["shares_mm"]
    premium = m6["premium_pct"]
    ev_ebitda_fy25 = m6.get("multiplos", {}).get("ev_ebitda_fy25")

    # Punto medio histórico↔proyección, calculado (no hardcodeado): se rompía
    # si algún día la serie histórica dejaba de terminar en 2025.
    divider_x = (max(m4["years_hist"]) + min(m4["years_proj"])) / 2

    # ── FIG 1: EBITDA histórico + proyectado ──────────────────────────────────
    years_all = m4["years_hist"] + m4["years_proj"]
    ebitda_h  = [can.get(f"FY{yr}", {}).get("ebitda_usdmm", None) for yr in m4["years_hist"]]
    ebitda_p  = [m4["projections"].get(str(yr), m4["projections"].get(yr, {})).get("ebit_usdmm", 0)
                 + m4["projections"].get(str(yr), m4["projections"].get(yr, {})).get("da_usdmm", 0)
                 for yr in m4["years_proj"]]
    ebitda_all = ebitda_h + ebitda_p

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(years_all, [v or 0 for v in ebitda_all],
                  color=[PALETTE["primary"]]*len(m4["years_hist"]) + [PALETTE["accent"]]*len(m4["years_proj"]),
                  alpha=0.85)
    ax.axvline(divider_x, color=PALETTE["neutral"], lw=1.5, ls="--", label="Histórico | Proyección")
    ax.bar_label(bars, fmt=lambda v: f"{v:,.0f}" if v else "", fontsize=8, fontweight="bold", padding=2)
    ax.set_title("EBITDA 2020–2030 (USD MM)", fontsize=13, fontweight="bold")
    ax.set_ylabel("USD MM")
    _clean_y(ax, money=True)
    ax.legend()
    fig_ebitda = save_fig(fig, "m13_01_ebitda_historico_proyectado")

    # ── FIG 2: FCFF bridge ────────────────────────────────────────────────────
    fcff_h = m4["fcff_historico_usdmm"]
    fcff_p = m4["fcff_proyectado_usdmm"]
    fcff_all = fcff_h + fcff_p
    colors_fcff = [PALETTE["accent"] if (v or 0) >= 0 else PALETTE["secondary"] for v in fcff_all]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(years_all, [v or 0 for v in fcff_all], color=colors_fcff, alpha=0.85)
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(divider_x, color=PALETTE["neutral"], lw=1.5, ls="--")
    ax.set_title("FCFF 2020–2030 (USD MM) — Penman Estricto", fontsize=13, fontweight="bold")
    ax.set_ylabel("USD MM")
    _clean_y(ax, money=True)
    fig_fcff = save_fig(fig, "m13_02_fcff_bridge")

    # ── FIG 3: Márgenes EBITDA histórico ─────────────────────────────────────
    ebitda_pcts = [can.get(f"FY{yr}", {}).get("ebitda_pct", None) for yr in m4["years_hist"]]
    fig, ax = plt.subplots(figsize=(10, 5))
    valid_yrs  = [y for y, v in zip(m4["years_hist"], ebitda_pcts) if v is not None]
    valid_pcts = [v for v in ebitda_pcts if v is not None]
    ax.plot(valid_yrs, valid_pcts, "o-", color=PALETTE["primary"], lw=2.5, ms=8)
    ax.fill_between(valid_yrs, valid_pcts, alpha=0.15, color=PALETTE["primary"])
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title("Margen EBITDA Histórico (%)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Margen EBITDA")
    fig_margin = save_fig(fig, "m13_03_margen_ebitda")

    # ── FIG 4: Balance patrimonial completo ───────────────────────────────────
    # Fix (auditoria ronda 4, mejora): años hardcodeados [2022-2025] rompian
    # el patron dinamico (m4["years_hist"]) que usa el resto del modulo --
    # se derivan los ultimos 4 años historicos disponibles en su lugar.
    yrs_bs    = m4["years_hist"][-4:]
    fy_keys = [f"FY{yr}" for yr in yrs_bs]
    activos = [can.get(k, {}).get("total_activo_usdmm", 0) for k in fy_keys]
    equity_bs = [can.get(k, {}).get("total_patrimonio_usdmm", 0) for k in fy_keys]
    deuda_bs  = [can.get(k, {}).get("deuda_fin_usdmm", 0) for k in fy_keys]

    fig, ax = plt.subplots(figsize=(10, 6))
    # Fix (auditoria jul-09): con offset=w=0.3 y ancho=w*2=0.6, el ancho total
    # del grupo (1.2) excedia el espaciado de 1.0 entre anios -- las barras de
    # un anio se superponian visiblemente con las del anio vecino (confirmado
    # visualmente en 2023/2024). offset y ancho independientes, grupo < 1.0.
    w = 0.22
    bw = 0.4
    xs = np.arange(len(yrs_bs))
    ax.bar(xs - w, activos, bw, label="Total Activo", color=PALETTE["primary"], alpha=0.8)
    ax.bar(xs + w, equity_bs, bw, label="Patrimonio Neto", color=PALETTE["accent"], alpha=0.8)
    ax.plot(xs, deuda_bs, "s--", color=PALETTE["secondary"], lw=2, ms=8, label="Deuda Financiera")
    ax.set_xticks(xs); ax.set_xticklabels(yrs_bs)
    ax.set_title("Balance Patrimonial (USD MM)", fontsize=13, fontweight="bold")
    ax.set_ylabel("USD MM"); ax.legend()
    _clean_y(ax, money=True)
    fig_bs = save_fig(fig, "m13_04_balance_patrimonial")

    # ── FIG 5: ROIC vs WACC ───────────────────────────────────────────────────
    # Fix Auditoria Estocolmo (deep-audit): recalculaba ROIC local como
    # NOPAT/Activo Total (= ROA, no ROIC), divergiendo del ROIC real
    # (NOPAT/Capital Invertido) ya persistido en m4_out.json["roic_historico"].
    # Esto invertia el cruce ROIC-WACC en FY2024 (ROA=6.8%<WACC destruia valor;
    # ROIC real=14.9%>WACC creaba valor). Unica fuente de verdad: m4.
    roics = [(yr, m4["roic_historico"].get(str(yr)))
             for yr in [2022, 2023, 2024, 2025]
             if m4["roic_historico"].get(str(yr)) is not None]

    if roics:
        yrs_roic = [r[0] for r in roics]
        vals_roic = [r[1] for r in roics]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(yrs_roic, [v*100 for v in vals_roic], color=PALETTE["primary"], alpha=0.8, label="ROIC")
        ax.axhline(wacc * 100, color=PALETTE["secondary"], lw=2, ls="--", label=f"WACC={wacc:.1%}")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_title("ROIC vs WACC — Creación de Valor", fontsize=13, fontweight="bold")
        ax.set_ylabel("%"); ax.legend()
        fig_roic = save_fig(fig, "m13_05_roic_vs_wacc")
    else:
        fig_roic = None

    # ── FIG 6: Beta pipeline ──────────────────────────────────────────────────
    betas = {
        "OLS": m5["beta_ols"],
        "Blume": m5["beta_blume"],
        "Vasicek": m5["beta_vasicek"],
        "Hamada\nUnlevered": m5["beta_u"],
        "Hamada\nLevered": m5["beta_l"],
    }
    fig, ax = plt.subplots(figsize=(10, 5))
    colors_b = [PALETTE["neutral"], PALETTE["warning"], PALETTE["accent"],
                PALETTE["secondary"], PALETTE["primary"]]
    bars = ax.bar(list(betas.keys()), list(betas.values()), color=colors_b, alpha=0.85)
    ax.axhline(1.0, color="black", lw=1, ls="--", alpha=0.6, label="Beta mercado = 1")
    ax.bar_label(bars, fmt="%.4f", fontsize=10, fontweight="bold", padding=2)
    ax.set_title("Pipeline Beta: OLS → Blume → Vasicek → Hamada", fontsize=13, fontweight="bold")
    ax.set_ylabel("Beta"); ax.legend()
    fig_beta_pipe = save_fig(fig, "m13_06_beta_pipeline")

    # ── FIG 7: DCF waterfall ──────────────────────────────────────────────────
    # Fix (auditoria ronda 4, mejora): las barras se dibujaban todas desde 0
    # (bar() sin bottom=), no acumulativas -- los NUMEROS eran correctos pero
    # el grafico no era un waterfall real (no se ve el efecto de acumulacion
    # PV FCFFs + PV TV = EV, EV - Deuda = Equity). Se agrega bottom= para que
    # cada barra "flote" sobre el acumulado anterior, como un waterfall real.
    pv_fc = m6["pv_fcffs_usdmm"]
    pv_tv = m6["pv_tv_usdmm"]
    components = ["PV FCFFs\nExplícitos", "PV Valor\nTerminal", "Enterprise\nValue",
                  "(-) Deuda\nNeta", "Equity\nValue"]
    values = [pv_fc, pv_tv, ev, -dnet, equity]
    # acumulado en el que "flota" cada barra: PV FCFFs arranca en 0; PV TV se
    # apila sobre PV FCFFs; EV es un total (arranca en 0); Deuda Neta se resta
    # flotando desde EV; Equity es el total final (arranca en 0).
    bottoms = [0, pv_fc, 0, equity, 0]
    colors_wf = [PALETTE["primary"], PALETTE["accent"], PALETTE["primary"],
                 PALETTE["secondary"], PALETTE["accent"]]

    fig, ax = plt.subplots(figsize=(12, 6))
    heights = [values[0], values[1], values[2], -values[3], values[4]]
    bars = ax.bar(range(len(components)), heights, bottom=bottoms, color=colors_wf, alpha=0.85, zorder=3)
    ax.axhline(0, color="black", lw=0.8)
    ax.bar_label(bars, labels=[f"{v:,.0f}" for v in values], fontsize=10, fontweight="bold", padding=3)
    ax.set_xticks(range(len(components))); ax.set_xticklabels(components)
    ax.set_title("DCF Waterfall — EV → Equity Value (USD MM)", fontsize=13, fontweight="bold")
    ax.set_ylabel("USD MM")
    _clean_y(ax, money=True)
    fig_dcf_wf = save_fig(fig, "m13_07_dcf_waterfall")

    # ── FIG 8: Football field completo ────────────────────────────────────────
    methods = ["DCF\nBase", "MC p25", "MC p50\n(mediana)", "MC p75", "Precio\nMercado"]
    vals_ff = [target, m7["price_ars_p25"], m7["price_ars_p50"], m7["price_ars_p75"], mkt_px]
    colors_ff = [PALETTE["primary"], PALETTE["accent"], PALETTE["accent"],
                 PALETTE["accent"], PALETTE["secondary"]]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(methods, vals_ff, color=colors_ff, alpha=0.85)
    ax.axhline(mkt_px, color=PALETTE["secondary"], lw=2, ls="--", label=f"Mercado={mkt_px:.0f}")
    ax.bar_label(bars, fmt=lambda v: f"{v:,.0f}", fontsize=10, fontweight="bold", padding=2)
    ax.set_title("Football Field Valuation (ARS)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Precio ARS"); ax.legend()
    _clean_y(ax, money=True, prefix="$")
    fig_ff = save_fig(fig, "m13_08_football_field_completo")

    # ── FIG 9: Sensibilidad LME × Volumen ────────────────────────────────────
    # BUG detectado (auditoria jul-09): elasticidades 0.5/0.8 hardcodeadas
    # (adivinadas) -- inconsistentes con el DOL empirico 2023->2024 real
    # (~1.92x/2.26x) que s53_sensitivity_lme_vol (chart library) ya calcula
    # correctamente. Se replica el mismo metodo aqui (reutilizando
    # ebitda_h/ebitda_pcts ya construidos arriba en FIG1/FIG3) para que
    # ambos graficos describan la MISMA sensibilidad, no dos versiones
    # distintas del mismo concepto.
    lme_changes = np.linspace(-0.30, 0.30, 7)  # ±30%
    vol_changes  = np.linspace(-0.20, 0.20, 7)  # ±20% volumen
    base_ebitda = can.get("FY2025", {}).get("ebitda_usdmm", 163.0)

    # Fix (auditoria ronda 4, MEDIA-ALTA): antes fijaba los indices [-3]/[-2],
    # que con 6 anios historicos (2020-2025) apuntan SIEMPRE a 2023->2024 y
    # nunca incluyen 2024->2025 -- justo el unico anio con apalancamiento
    # operativo NEGATIVO (margen 22.35%->14.94%), cherry-picking no
    # divulgado de una sola transicion favorable. Se calcula el DOL sobre
    # TODAS las transiciones año-a-año disponibles y se usa la MEDIANA (mas
    # robusta que un unico par elegido a dedo, e incluye 2024->2025).
    dol13 = 2.5  # fallback conservador (2x-3x tipico para smelters con energia propia)
    dols_validos = []
    for i in range(1, len(ebitda_h)):
        e_prev, m_prev = ebitda_h[i-1], ebitda_pcts[i-1]
        e_cur,  m_cur  = ebitda_h[i],   ebitda_pcts[i]
        if e_prev and e_cur and m_prev and m_cur and m_prev > 0 and m_cur > 0 and e_prev > 0:
            r_prev, r_cur = e_prev / m_prev, e_cur / m_cur
            dr = r_cur - r_prev
            pct_r = dr / r_prev
            pct_e = (e_cur - e_prev) / e_prev
            if pct_r != 0:
                dols_validos.append(pct_e / pct_r)
    if dols_validos:
        dol13 = float(np.clip(np.median(dols_validos), 1.5, 5.0))
    elast_lme13 = dol13 * 0.85  # LME es ~85% de ingresos exportables (mismo supuesto que s53 -- ver static_inputs.json, sin campo dedicado de mix exportador)
    elast_vol13 = dol13 * 1.00

    grid = np.maximum(0, np.array(
        [[base_ebitda * (1 + elast_lme13*dl + elast_vol13*dv) for dl in lme_changes] for dv in vol_changes]))

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(grid, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(lme_changes)))
    ax.set_xticklabels([f"{x:.0%}" for x in lme_changes], fontsize=8)
    ax.set_yticks(range(len(vol_changes)))
    ax.set_yticklabels([f"{x:.0%}" for x in vol_changes], fontsize=8)
    ax.set_xlabel("Variación LME Aluminio")
    ax.set_ylabel("Variación Volumen")
    ax.set_title(f"Sensibilidad EBITDA (USD MM) vs LME × Volumen — DOL empírico {dol13:.2f}x", fontsize=13, fontweight="bold")
    for i in range(len(vol_changes)):
        for j in range(len(lme_changes)):
            ax.text(j, i, f"{grid[i,j]:.0f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label="EBITDA USD MM")
    fig_sens_lme = save_fig(fig, "m13_09_sensibilidad_lme_volumen")

    # ── FIG 10: MC distribución final ────────────────────────────────────────
    # BUG detectado (auditoria jul-09): mismo pileup de ~4.2% truncado a $0
    # (piso de insolvencia Merton) que ya se habia corregido en
    # s34_monte_carlo/s36_var_cvar (jul-05) pero no aqui -- y esto es mas
    # grave que en M10 porque p5/p95 de ESTA figura alimentan directamente
    # escenarios["Bear"]/["Bull"] mas abajo, que SI se shippean (tabla
    # Bear/Base/Bull de la slide 6 via rebuild_scenario_table).
    mc_prices_raw13 = np.load(os.path.join(WORKDIR, "mc_results.npy"))
    frac_truncado_m13 = float((mc_prices_raw13 <= 0).mean())
    mc_prices = mc_prices_raw13[mc_prices_raw13 > 0]  # solo para la FORMA del histograma
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(mc_prices, bins=100, density=True, color=PALETTE["primary"], alpha=0.6, label="MC 20,000 sims (excl. truncados a 0)")
    # Fix (auditoria ronda 4, ALTA): antes se recalculaban p5/p50/p95 ACA
    # excluyendo el pileup de equity=0, mientras m7_out.json (fuente oficial)
    # los calcula SOBRE la serie completa (incluye el pileup) -- dos criterios
    # de filtrado distintos para "la misma" cifra daban una divergencia de
    # ~4.6x (m7 oficial p05=50.41 vs este recompute=235.0), y ademas
    # alimentaban el escenario Bear de forma inconsistente con el numero
    # oficial del pipeline. Se usan los percentiles OFICIALES de m7_out.json
    # (unica fuente) en vez de un recompute local con otro criterio.
    p5, p50, p95 = m7["price_ars_p05"], m7["price_ars_p50"], m7["price_ars_p95"]
    ax.axvline(p5,  color=PALETTE["secondary"], lw=2, label=f"p5  = {p5:.0f} ARS (oficial m7_out.json)")
    ax.axvline(p50, color=PALETTE["accent"],    lw=2, label=f"p50 = {p50:.0f} ARS (oficial m7_out.json)")
    ax.axvline(p95, color=PALETTE["primary"],   lw=2, label=f"p95 = {p95:.0f} ARS (oficial m7_out.json)")
    ax.axvline(mkt_px, color="black", lw=1.5, ls="--", label=f"Mercado = {mkt_px:.0f} ARS")
    ax.set_title(f"Monte Carlo — Distribución Precio ALUA.BA (ARS) — {frac_truncado_m13:.1%} truncado a $0 excluido", fontsize=12, fontweight="bold")
    ax.set_xlabel("Precio ARS"); ax.legend()
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: _money_fmt(v, prefix="$")))
    fig_mc_final = save_fig(fig, "m13_10_mc_distribucion_final")

    # ── Escenarios Bull / Base / Bear ─────────────────────────────────────────
    # Fix (auditoria ronda 4, ALTA): los campos wacc_adj/g_adj/lme_adj y la
    # narrativa ("LME -20%, CAPEX excesivo"...) eran decorativos -- el precio
    # real sale de p5/p95 del Monte Carlo (WACC/g/FCFF con shocks t-Student +
    # saltos Merton, ver m7_out.json), que NO tiene un parametro LME ni esos
    # shocks especificos aplicados. Se elimina el texto que implicaba un
    # recompute deterministico que nunca ocurria, y se describe lo que
    # realmente alimenta el numero: los percentiles oficiales del Monte Carlo.
    scenarios = {
        "Bear": {
            "precio_ars": round(float(p5), 0),
            "descripcion": (f"Percentil 5 del Monte Carlo estocastico de 20.000 simulaciones "
                             f"(WACC/g/FCFF terminal con shocks t-Student correlacionados + "
                             f"saltos Merton Jump-Diffusion, ver m7_out.json/merton_jump_params) "
                             f"-- no es un escenario deterministico de LME/CAPEX especifico."),
        },
        "Base": {
            "precio_ars": round(target, 0),
            "descripcion": f"DCF central, WACC={wacc:.1%}, g={m5['g_terminal']:.1%}, expansion medida",
        },
        "Bull": {
            "precio_ars": round(float(p95), 0),
            "descripcion": (f"Percentil 95 del Monte Carlo estocastico de 20.000 simulaciones "
                             f"(mismo motor que el escenario Bear, cola derecha de la distribucion)."),
        },
    }

    # ── FIG 11: Escenarios Bull/Base/Bear ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    scen_names = list(scenarios.keys())
    scen_prices = [scenarios[s]["precio_ars"] for s in scen_names]
    scen_colors = [PALETTE["secondary"], PALETTE["primary"], PALETTE["accent"]]
    bars = ax.bar(scen_names, scen_prices, color=scen_colors, alpha=0.85, width=0.4)
    ax.axhline(mkt_px, color="black", lw=2, ls="--", label=f"Precio mercado {mkt_px:.0f}")
    ax.bar_label(bars, fmt=lambda v: f"{v:,.0f}", fontsize=12, fontweight="bold", padding=3)
    ax.set_title("Escenarios Bull / Base / Bear — ALUA.BA (ARS)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Precio Objetivo ARS"); ax.legend()
    _clean_y(ax, money=True, prefix="$")
    fig_scenarios = save_fig(fig, "m13_11_escenarios_bull_base_bear")

    # ── Dictamen de Inversión ─────────────────────────────────────────────────
    # Prima ≥ 15% → COMPRAR; -15% a +15% → MANTENER; < -15% → VENDER
    ev_ebitda_txt = f"{ev_ebitda_fy25:.1f}x" if ev_ebitda_fy25 is not None else "N/A"
    # Rango de peers calculado desde static_inputs['peers'] (excluye ALUAR) --
    # el texto anterior citaba "peers 6-8x" hardcodeado; el rango real vigente
    # es 5.1x-9.0x (Rusal min, Chalco max), mismo dataset ya usado en
    # s37_football_field/s28_peer_multiples tras el fix de esa misma sesion.
    _peers_static13 = (static or {}).get("peers") or {}
    _peers_ev13 = [v for n, v in zip(_peers_static13.get("names", []), _peers_static13.get("ev_ebitda", []))
                   if "ALUAR" not in n.upper()]
    peers_range_txt = (f"{min(_peers_ev13):.1f}-{max(_peers_ev13):.1f}x" if _peers_ev13 else "N/A")
    if premium >= 15:
        dictamen = "COMPRAR"
        dictamen_color = PALETTE["accent"]
        justificacion = f"Prima DCF del {premium:.1f}% sobre precio de mercado. EV/EBITDA {ev_ebitda_txt} vs peers {peers_range_txt}."
    elif premium <= -15:
        dictamen = "VENDER"
        dictamen_color = PALETTE["secondary"]
        justificacion = f"Descuento DCF del {abs(premium):.1f}% — mercado valúa por encima de DCF central."
    else:
        dictamen = "MANTENER"
        dictamen_color = PALETTE["warning"]
        justificacion = f"Prima limitada de {premium:.1f}%. Asimetría positiva por MC (P(upside)={m7['prob_upside_vs_mkt']:.1%})."

    # Fix (auditoria jul-11, ALTA): el escenario "Base" y el dictamen nunca
    # surfaceaban el flag de M6 que marca la reinversion mecanica del DCF
    # primario como economicamente inconsistente (tasa de reinversion
    # negativa en los 5 años proyectados, verificado en m6_out.json) -- un
    # lector solo lo encontraria inspeccionando el JSON crudo. M13 es el
    # modulo designado como fuente unica de verdad del dictamen final.
    if m6.get("tv_reinversion_mecanica_inconsistente"):
        justificacion += (" [NOTA METODOLOGICA] El horizonte explicito 2026-2030 usa el CAPEX "
                           "empirico reciente de la empresa (mediana anios tipicos); "
                           "extrapolar ese mismo ratio a perpetuidad implicaria una "
                           "reinversion neta negativa sostenida (base de capital "
                           "encogiendose sin limite), fisicamente inconsistente con una "
                           "perpetuidad. Por eso el Valor Terminal usa el escenario de "
                           "convergencia ROIC=WACC (Damodaran, 'narrow stable growth'), "
                           "que no depende de proyectar CAPEX/D&A a infinito -- ver "
                           "metodo_valor_terminal en m6_out.json.")

    print(f"\n  ╔══════════════════════════════════════════╗")
    print(f"  ║  DICTAMEN: {dictamen:<32}║")
    print(f"  ║  Precio objetivo: {target:>6.0f} ARS              ║")
    print(f"  ║  Precio mercado:  {mkt_px:>6.0f} ARS              ║")
    print(f"  ║  Prima DCF:      {premium:>6.1f}%                ║")
    print(f"  ║  P(upside MC):   {m7['prob_upside_vs_mkt']:.1%}                  ║")
    print(f"  ╚══════════════════════════════════════════╝")

    # Bullets de riesgo/catalizadores: ratios calculados desde los outputs
    # reales del pipeline en vez de valores fijos en el texto (que quedaban
    # desactualizados apenas cambiaba algún módulo aguas arriba).
    ebitda_fy25 = can.get("FY2025", {}).get("ebitda_usdmm")
    dnet_ebitda_txt = f"{dnet/ebitda_fy25:.1f}x" if ebitda_fy25 else "N/A"
    capex_fy25 = can.get("FY2025", {}).get("capex_usdmm")
    capex_txt = f"USD {capex_fy25:.0f}MM" if capex_fy25 else "atípico"

    # Nota metodológica de asimetría de Monte Carlo (Hallazgo 31)
    nota_mc = ("La mediana (P50) de Monte Carlo es menor al Target Price Caso Base debido a la asimetría "
               "de la Gordon Growth (1/(WACC-g)), que es una función no lineal hiperbólica. Además, el "
               "truncamiento a cero de escenarios insolventes (equity como call option - Merton 1974) "
               "absorbe las colas extremas de deuda neta alta, sesgando la mediana respecto al caso base.")
    print("\n  [NOTA METODOLÓGICA DE MONTE CARLO]")
    print(f"  {nota_mc}")

    out = {
        "nota_asimetria_mc": nota_mc,
        "dictamen": dictamen,
        "dictamen_justificacion": justificacion,
        # Fix (auditoria jul-11, MEDIA): el DOL empirico y las elasticidades
        # LME/Volumen usadas en FIG9 solo vivian como texto dentro de la
        # imagen PNG -- no eran auditables desde el JSON ni verificables por
        # _strict_validator.py o el Excel, a diferencia de cada otro
        # parametro calculado en este modulo.
        "sensibilidad_lme_volumen": {
            "dol_empirico": round(float(dol13), 4),
            "elasticidad_lme": round(float(elast_lme13), 4),
            "elasticidad_volumen": round(float(elast_vol13), 4),
            "base_ebitda_usdmm": round(float(base_ebitda), 2),
        },
        "precio_objetivo_ars": target,
        "precio_mercado_ars": mkt_px,
        "prima_pct": premium,
        "prob_upside_mc": m7["prob_upside_vs_mkt"],
        "ev_usdmm": ev,
        "equity_usdmm": equity,
        "wacc": wacc,
        "ke": ke,
        "ccl": ccl,
        "escenarios": scenarios,
        "portfolio_snapshot": ({
            "kelly_alua": m12.get("kelly_alua"),
            "max_sharpe_portfolio": m12.get("max_sharpe_portfolio"),
            "datos_sinteticos": m12.get("datos_sinteticos"),
        } if m12 else None),
        # Fix (auditoria jul-11, MEDIA): "~20%"/"+15%" no tenian ningun campo
        # _fuente en static_inputs.json/canonical_financials.json -- a
        # diferencia de cada otra cifra cuantitativa del proyecto, vivian
        # solo como texto narrativo hardcodeado. Se suaviza a una
        # afirmacion cualitativa en vez de inventar una fuente que no existe.
        "catalysts": [
            "Normalización Argentina: convergencia CCL → tipo de cambio único",
            "Parque eólico Puerto Madryn: reducción de costo energético esperada post-2026 (magnitud exacta a confirmar en la Memoria Anual)",
            "LME aluminio: demanda vehículos eléctricos y transmisión eléctrica",
            "Ampliación planta: expansión de capacidad en FY2026-2027 (magnitud exacta a confirmar en la Memoria Anual)",
            "Acuerdo FMI: reducción riesgo soberano → spread compresión",
        ],
        "risks": [
            f"CAPEX atípico FY2025 ({capex_txt}): ejecución y costo real del parque eólico",
            "Exposición CCL: devaluación adicional impacta conversión USD",
            "LME downside: sobreoferta China, desaceleración global",
            "Riesgo regulatorio Argentina: controles de exportación aluminio",
            f"Deuda neta alta FY2025: USD {dnet:.0f}MM ({dnet_ebitda_txt} EBITDA FY2025)",
            "Tasa efectiva impositiva distorsionada por inflación (84% en FY2025)",
        ],
        "figures": {
            "ebitda": fig_ebitda, "fcff_bridge": fig_fcff, "margen_ebitda": fig_margin,
            "balance": fig_bs, "roic": fig_roic, "beta_pipeline": fig_beta_pipe,
            "dcf_waterfall": fig_dcf_wf, "football_field": fig_ff,
            "sensibilidad_lme": fig_sens_lme, "mc_final": fig_mc_final,
            "escenarios": fig_scenarios,
        },
    }

    with open(os.path.join(WORKDIR, "m13_out.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print(f"\n[OK] m13_out.json + 11 figuras guardadas.")
    return out


# autorun deshabilitado — usar run_all() del pipeline maestro

## Biblioteca de gráficos (todas las funciones; no se renderiza acá)

In [17]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA · BIBLIOTECA DE GRÁFICOS (A) — adaptador + macro + financieros      ║
# ║  Una función por gráfico. NO se renderiza acá: solo se definen.            ║
# ║  Todas leen de `CH` (build_chart_inputs) → outputs del notebook.           ║
# ║  Todas usan MASTER_STYLE (vía C/SZ/LW, celda de tema) — cero color/lw      ║
# ║  suelto definido acá.                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# AUDITORÍA (Fase 3 — limpieza radical de calidad institucional):
# 1. Se eliminaron TODAS las cajas de texto explicativas flotantes dentro del
#    área de ploteo — las conclusiones van en el PPTX/speech, no superpuestas
#    a las barras. Sólo sobreviven etiquetas de VALOR directo sobre su dato.
# 2. _clean_y(money=True) antepone "$" siempre (celda de tema).
# 3. Todas las leyendas se movieron fuera del área de ploteo con bbox_to_anchor.
# 4. Gráficos de barras reservan ≥15-20% de margen superior/inferior.
#
# AUDITORÍA (Fase 4 — Gold Standard): grosores de línea migrados a LW[...]
# (MASTER_STYLE), y las leyendas que describen una referencia (no una simple
# serie de datos) se redactaron para expresar la relación causal, no sólo el
# nombre de la serie (ej. "Proyección post-CAPEX" en vez de "Proyectado").
def build_chart_inputs(workdir=None) -> dict:
    """Ensambla en UN dict todo lo que necesitan los gráficos, desde los
    outputs del notebook + canonical + static_inputs.json + mc_results.npy.
    No fabrica datos: si una serie no fue persistida, queda None y el gráfico
    avisa qué falta. Re-simula trayectorias OU/Merton desde params persistidos."""
    inp = Inputs.from_outputs(workdir)
    m1, m4, m5, m6, m7 = (inp.get(k, default={}) for k in ("m1", "m4", "m5", "m6", "m7"))
    m9, m10, m11, can = (inp.get(k, default={}) for k in ("m9", "m10", "m11", "can"))
    m8, m12 = (inp.get(k, default={}) for k in ("m8", "m12"))
    st = load_static() or {}

    years_h = [2020, 2021, 2022, 2023, 2024, 2025]
    years_p = [2026, 2027, 2028, 2029, 2030]
    hist = m4.get("historical", {})
    proj = m4.get("projections", {})

    def hv(yr, key):  # valor histórico canónico/M4
        return (can.get(f"FY{yr}", {}).get(key)
                or hist.get(str(yr), {}).get(key))

    # Monte Carlo: muestra real desde .npy si existe
    mc_sample = None
    npy = os.path.join(workdir or CONFIG["workdir"], CONFIG["mc_npy"])
    if os.path.exists(npy):
        try:
            mc_sample = np.load(npy).tolist()
        except Exception:
            mc_sample = None

    # Monte Carlo: muestra de draws correlacionados (Cholesky) para el
    # diagnostico "Espacio de Fase" -- persistida por M7 (mc_corr_sample.npy),
    # unica fuente de verdad (mismos wacc_sim/g_sim/fcff_T_sim que alimentan
    # el precio, no una re-simulacion aparte).
    mc_corr_sample = None
    npy_corr = os.path.join(workdir or CONFIG["workdir"], "mc_corr_sample.npy")
    if os.path.exists(npy_corr):
        try:
            _arr = np.load(npy_corr)
            mc_corr_sample = {"wacc": _arr[:, 0].tolist(), "g": _arr[:, 1].tolist(),
                               "fcff_t": _arr[:, 2].tolist()}
        except Exception:
            mc_corr_sample = None

    def _corr_from_m11(m11):
        """Convierte m11_out.json['correlation_matrix'] (dict anidado de
        pandas .corr().to_dict()) al formato {labels, matrix} que espera
        plot_correlation_matrix. Bug real detectado: build_chart_inputs leia
        la clave 'correlation' (inexistente -- M11 la persiste como
        'correlation_matrix') y ademas esperaba un array plano, no el dict
        anidado real -- el resultado era que la slide 45 SIEMPRE mostraba el
        fallback estatico de static_inputs.json, nunca los datos frescos de
        M11/yfinance, sin que ningun check lo detectara."""
        raw = m11.get("correlation_matrix")
        if not raw:
            return None
        labels = list(raw.keys())
        matrix = [[raw.get(r, {}).get(c) for c in labels] for r in labels]
        return {"labels": labels, "matrix": matrix}

    wacc = m5.get("wacc")
    CH = {
        "beta": {k: m5.get(f"beta_{k}") for k in ("ols", "blume", "vasicek", "u", "l")}
                | {"u_damodaran": m5.get("beta_u_damodaran"), "l_damodaran": m5.get("beta_l_damodaran"),
                   "d_e_target": m5.get("d_e_target")},
        # Fix (auditoria ronda 4, MEDIA): "or 0" enmascaraba un dato AUSENTE
        # (None) como cero REAL en la descomposicion del WACC -- parametro
        # central de la tesis. Si falta beta_l/erp_us/lambda_ar/crp, el
        # componente queda None (el grafico avisa el faltante), no un cero
        # que se leeria como "esta pata no aporta nada al WACC".
        "wacc": {"rf": m5.get("rf"),
                 "beta_x_erp": (m5["beta_l"] * m5["erp_us"])
                               if (m5.get("beta_l") is not None and m5.get("erp_us") is not None) else None,
                 "lambda_x_crp": (m5["lambda_ar"] * m5["crp"])
                                 if (m5.get("lambda_ar") is not None and m5.get("crp") is not None) else None,
                 "ke": m5.get("ke"), "wacc": wacc,
                 "crp": m5.get("crp"), "erp": m5.get("erp_us"), "beta_l": m5.get("beta_l"),
                 "lambda_ar": m5.get("lambda_ar"), "lambda_fuente": m1.get("lambda_fuente"),
                 # Agregado (tornado de sensibilidad): pesos E/V, D/V y Kd
                 # post-tax -- para recomputar WACC bajo un shock de Lambda
                 # (via el canal Ke->WACC) con la MISMA formula de M5, sin
                 # una segunda formula que pueda divergir.
                 "e_v_target": m5.get("e_v_target"), "d_v_target": m5.get("d_v_target"),
                 "kd_after_tax": m5.get("kd_after_tax")},
        "dcf": {"pv_fcffs": m6.get("pv_fcffs_usdmm"), "pv_tv": m6.get("pv_tv_usdmm"),
                "ev": m6.get("ev_usdmm"), "net_debt": m6.get("deuda_neta_usdmm"),
                "equity": m6.get("equity_usdmm"),
                # g_terminal agregado (auditoria jul-09): s33_sensitivity_wacc_g tenia
                # g_base=0.020 hardcodeado en vez de leer esto -- unica fuente de verdad.
                "g_terminal": m5.get("g_terminal"),
                # Agregado (tornado de sensibilidad): FCFF terminal (post
                # convergencia si aplica) + shares para recomputar el mismo
                # puente EV->Equity->Precio de M6 bajo shocks.
                "fcff_terminal": m6.get("fcff_terminal"), "shares_mm": m6.get("shares_mm")},
        # Fix (Directiva Refactorizacion 3.C): escenario de sensibilidad
        # "convergencia ROIC=WACC" del valor terminal (ver M6) -- se expone
        # al football field (s37) como rango adicional, NO reemplaza el
        # DCF primario (target/dictamen siguen siendo el mecanico).
        "escenario_convergencia": m6.get("escenario_convergencia_roic_wacc"),
        # Fix (auditoria jul-12, Hallazgo EX-01 cuantificado -- ver M6 [7b]):
        # escenario de sensibilidad "1T26 sostenido" (EBITDA real anualizado x
        # multiplo EV/EBITDA propio) expuesto al football field (s37), mismo
        # patron que escenario_convergencia (no reemplaza el DCF primario).
        "escenario_1q26_sostenido": m6.get("escenario_1q26_sostenido"),
        "fcff": {"years": [f"{y}E" for y in years_p],
                 "values": m6.get("fcff_proj_usdmm") or m4.get("fcff_proyectado_usdmm")},
        # Agregado (auditoria jul-11, respuesta a objecion de usuario sobre
        # requerimientos estadisticos/econometricos): detalle del shrinkage
        # CAPEX propio (N=4, MAD-robusto) vs. peers globales (N=20, yfinance
        # en vivo) que ahora alimenta capex_pct_rev_median en M4 -- ver
        # s20c_capex_shrinkage.
        "capex": {"detalle": m4.get("capex_shrinkage_detalle"),
                  "resultado_usado": m4.get("capex_pct_rev_median"),
                  "da_pct_rev_median": m4.get("da_pct_rev_median")},
        "fcff_annual": {"years": years_h + years_p,
                        "values": [hv(y, "fcff_usdmm") for y in years_h]
                                  + (m6.get("fcff_proj_usdmm") or [None]*5)},
        "ebitda": {"years": years_h + years_p,
                   "hist": [hv(y, "ebitda_usdmm") for y in years_h],
                   # Fix (auditoria ronda 4, MEDIA): "0" en vez de None para
                   # un anio proyectado sin dato -- inconsistente con "hist"
                   # (que si deja None) y se graficaria como "$0 real" en vez
                   # de un faltante.
                   "proj": [(proj.get(str(y), {}).get("ebit_usdmm") + proj.get(str(y), {}).get("da_usdmm"))
                            if (proj.get(str(y), {}).get("ebit_usdmm") is not None
                                and proj.get(str(y), {}).get("da_usdmm") is not None)
                            else None for y in years_p]},
        "ebitda_margin": {"years": years_h + years_p,
                          "values": [hv(y, "margen_ebitda") for y in years_h],
                          "proj": [proj.get(str(y), {}).get("margen_ebitda") for y in years_p]},
        "roic": {"years": [str(y) for y in years_h],
                 "values": [(m4.get("roic_historico", {}) or {}).get(str(y)) for y in years_h]},
        "balance": {"years": [2022, 2023, 2024, 2025],
                    "activo": [hv(y, "total_activo_usdmm") for y in (2022, 2023, 2024, 2025)],
                    "pn":     [hv(y, "total_patrimonio_usdmm") for y in (2022, 2023, 2024, 2025)],
                    "deuda":  [hv(y, "deuda_fin_usdmm") for y in (2022, 2023, 2024, 2025)]},
        "wacc_ref": wacc,
        "target": m6.get("target_ars"), "market": m6.get("alua_px_mkt"),
        "mc": {"p10": m7.get("price_ars_p10"), "p50": m7.get("price_ars_p50"),
               "p90": m7.get("price_ars_p90"), "p25": m7.get("price_ars_p25"),
               "p75": m7.get("price_ars_p75"), "market": m6.get("alua_px_mkt"),
               "dcf_base": m6.get("target_ars"), "sample": mc_sample,
               "corr_sample": mc_corr_sample},
        "var": m10.get("var_table"), "stress": m10.get("stress_tests"),
        "ou": m9.get("ou_params"), "merton": m9.get("merton_params"),
        "reverse_dcf": m9.get("reverse_dcf"),
        "corr": _corr_from_m11(m11) or st.get("correlation"),
        "multiplos": m6.get("multiplos", {}),
        # Fix (auditoria jul-09): "or 1522.0" era un fallback hardcodeado y desactualizado
        # (CCL vigente ronda 1595+) -- ambas fuentes (m6.ccl_usado, m1.ccl) son datos VIVOS,
        # m1.ccl ademas es campo requerido por module13_synthesis (nunca None en un run valido).
        "ccl": m6.get("ccl_usado") or m1.get("ccl"),
        # Activos individuales de la frontera eficiente (misma metodologia/ventana
        # que la nube de puntos, via M12) + Merval como referencia externa (M8):
        "frontier_assets": {"names": m12.get("assets"), "mu_anual": m12.get("mu_anual"),
                             "vol_anual": m12.get("vol_anual"),
                             "merv_ret_anual": m8.get("merv_ret_anual"),
                             "merv_vol_anual": m8.get("merv_vol_anual"),
                             # Fix (auditoria ronda 5, ALTA -- data lineage):
                             # se propaga tickers_excluidos (m12_out.json) para
                             # que el grafico pueda avisar si el universo
                             # optimizado quedo incompleto, en vez de asumir
                             # silenciosamente 4 activos siempre presentes.
                             "excluidos": m12.get("tickers_excluidos")},
        # Descomposicion de varianza del portafolio (Alexander/Sharpe/Bailey +
        # Indice Unico de Sharpe 1963), calculada en M12 -- ver
        # component_contribution_risk()/single_index_decomposition().
        "portfolio_risk": {
            "assets": m12.get("assets"),
            "weights_max_sharpe": (m12.get("max_sharpe_portfolio") or {}).get("weights"),
            "ccr_max_sharpe": m12.get("risk_contribution_max_sharpe"),
            "single_index": m12.get("single_index_decomposition"),
            "market_proxy": m12.get("single_index_market_proxy"),
        },
        # Estáticos provistos por el usuario (Memoria/industria) — sin fabricar:
        "static": st,
    }
    # Warning de antiguedad (auditoria jul-09): ~13 de 29 graficos shippeados
    # dependen de CH["static"] (static_inputs.json), que NO se regenera dentro de
    # run_all() -- es un snapshot manual (build_static_inputs.py, corrido aparte).
    # Si el archivo queda viejo respecto a esta corrida, esos graficos describen
    # datos de mercado desactualizados sin que nadie lo note. Aviso no-bloqueante.
    try:
        import datetime as _dt
        _static_path = os.path.join(workdir or CONFIG["workdir"], "static_inputs.json")
        if os.path.exists(_static_path):
            _age_days = (_dt.datetime.now() - _dt.datetime.fromtimestamp(os.path.getmtime(_static_path))).days
            if _age_days > 14:
                print(f"  [WARN] static_inputs.json tiene {_age_days} dias de antiguedad -- "
                      f"considerar re-correr build_static_inputs.py (13 graficos dependen de el).")
    except Exception:
        pass
    return CH


def _legend_outside(ax, ncol=None, y=-0.12):
    """Leyenda SIEMPRE fuera del área de ploteo (debajo), vía bbox_to_anchor.
    export() usa bbox_inches='tight', así que nunca se recorta."""
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return
    ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, y),
              ncol=ncol or min(len(handles), 4), fontsize=SZ["legend"], frameon=False)


# ── Slide 9 · LME vs DXY ──────────────────────────────────────────────────────
def plot_lme_vs_dxy(CH, outdir=None):
    s = (CH.get("static") or {}).get("lme_dxy")
    if not s:
        return {"skip": "falta static_inputs['lme_dxy'] (series LME y DXY)"}
    idx = pd.to_datetime(s["dates"], format="%b-%y")
    ser = pd.DataFrame({"lme": s["lme"], "dxy": s["dxy"]}, index=idx).sort_index()
    # Continuidad estricta: reindexa a frecuencia mensual e interpola SOLO gaps
    # (huecos de calendario), nunca extrapola mas alla del ultimo dato real.
    ser = ser.asfreq("MS").interpolate(method="linear").ffill()
    # Correlacion calculada dinamicamente (no se asume el signo/fuerza de
    # antemano) -- el 2021-2022 LME y DXY subieron JUNTOS antes de divergir,
    # asi que el titulo fijo "correlacion estructural inversa" era una
    # simplificacion cuestionable; ahora el subtitulo refleja la fuerza real.
    _corr_lme_dxy = ser["lme"].corr(ser["dxy"])
    if _corr_lme_dxy <= -0.5:
        _rel_txt = f"correlación estructural inversa (ρ={_corr_lme_dxy:.2f})"
    elif _corr_lme_dxy < 0:
        _rel_txt = f"relación cíclica inversa, de baja intensidad (ρ={_corr_lme_dxy:.2f})"
    else:
        _rel_txt = f"sin correlación estructural estable (ρ={_corr_lme_dxy:.2f})"
    fig, ax = scaffold(
        "El aluminio (LME) y el dólar (DXY) siguen un vínculo cíclico, no siempre en espejo",
        f"Precio LME del aluminio frente al índice dólar — {_rel_txt}",
        "LME USD/Tn · DXY índice",
        # Fix (auditoria ronda 4, ALTA): "estados financieros de Aluar" es
        # una atribucion de fuente FALSA -- LME/DXY son series de mercado
        # (commodities/divisas), no datos contables de la compañía.
        "LME (ALI=F) y DXY (static_inputs.json) — series de mercado",
        "un dólar más débil tiende a sostener los metales industriales en el ciclo de mediano plazo.")
    ax.plot(ser.index, ser["lme"], color=C["navy"], lw=LW["heavy"], label="LME Aluminio — driver directo del revenue (izq.)")
    ax.set_ylabel("LME USD/Tn", fontsize=SZ["axis"], color=C["navy"])
    ax.tick_params(axis="y", colors=C["navy"])
    ax2 = ax.twinx()
    ax2.plot(ser.index, ser["dxy"], color=C["aluar"], lw=LW["bold"], ls="--", label="DXY — dólar fuerte comprime el LME (der.)")
    ax2.set_ylabel("DXY", fontsize=SZ["axis"], color=C["aluar"])
    ax2.tick_params(axis="y", colors=C["aluar"]); ax2.grid(False)
    ax2.spines["top"].set_visible(False)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlim(ser.index.min(), ser.index.max())
    lines = ax.get_lines() + ax2.get_lines()
    ax.legend(lines, [l.get_label() for l in lines], loc="upper center",
              bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=SZ["legend"], frameon=False)
    ax.tick_params(length=0); ax2.tick_params(length=0)
    return export(fig, "s09_lme_vs_dxy", outdir)


# ── Slide 10 · Macro Argentina (PBI línea + inflación barras) ─────────────────
def plot_macro_argentina(CH, outdir=None):
    s = (CH.get("static") or {}).get("macro_ar")
    if not s:
        return {"skip": "falta static_inputs['macro_ar'] (PBI e inflación INDEC/FMI)"}
    yrs, infl, pbi = s["years"], s["inflacion"], s["pbi_growth"]
    fig, ax = scaffold(
        "Sendero de desinflación con crecimiento: normalización macro ortodoxa",
        "Inflación anual (barras) y crecimiento del PBI (línea), histórico y proyección",
        "% anual", "INDEC / BCRA / consenso FMI",
        "compresión del riesgo país que comprime el WACC.")
    x = np.arange(len(yrs))
    # Fix (auditoria ronda 4, MEDIA): antes graficaba 2026E/2027E con el
    # mismo color/estilo que los años reales, pese a que el subtitulo dice
    # "histórico y proyección" -- se distinguen visualmente usando el
    # sufijo "E" (misma convencion que static_inputs.json['macro_ar']).
    _es_proy = ["E" in str(y) for y in yrs]
    _bar_cols = [C["muted"] if p else C["blue_lt"] for p in _es_proy]
    ax.bar(x, infl, color=_bar_cols, width=0.6, label="Inflación — presiona costos en pesos (gris=proyección)", zorder=3)
    ax2 = ax.twinx()
    _pbi_hist_x = [xi for xi, p in zip(x, _es_proy) if not p]
    _pbi_hist_y = [v for v, p in zip(pbi, _es_proy) if not p]
    _pbi_proj_x = [xi for xi, p in zip(x, _es_proy) if p]
    _pbi_proj_y = [v for v, p in zip(pbi, _es_proy) if p]
    if _pbi_hist_x and _pbi_proj_x:
        # puente para que la linea no se corte entre historico y proyeccion
        _pbi_proj_x = [_pbi_hist_x[-1]] + _pbi_proj_x
        _pbi_proj_y = [_pbi_hist_y[-1]] + _pbi_proj_y
    ax2.plot(_pbi_hist_x, _pbi_hist_y, "o-", color=C["aluar"], lw=LW["heavy"], label="PBI — sostiene demanda doméstica de aluminio")
    if _pbi_proj_x:
        ax2.plot(_pbi_proj_x, _pbi_proj_y, "o--", color=C["aluar"], lw=LW["medium"], alpha=0.55, label="PBI proyectado")
    ax2.grid(False); ax2.spines["top"].set_visible(False)
    ax.set_xticks(x); ax.set_xticklabels(yrs, fontsize=SZ["tick"], rotation=0)
    ax.set_ylabel("Inflación %", fontsize=SZ["axis"])
    ax2.set_ylabel("PBI var. %", fontsize=SZ["axis"], color=C["aluar"])
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper center", bbox_to_anchor=(0.5, -0.12),
              ncol=2, fontsize=SZ["legend"], frameon=False)
    ax.tick_params(length=0); ax2.tick_params(length=0); ax.grid(axis="x", visible=False)
    return export(fig, "s10_macro_argentina", outdir)


# ── Slide 11 · Compresión EMBI+ ───────────────────────────────────────────────
def plot_embi_compression(CH, outdir=None):
    s = (CH.get("static") or {}).get("embi_hist")
    if not s:
        return {"skip": "falta static_inputs['embi_hist'] (serie EMBI+ BCRA)"}
    dates, embi = s["dates"], s["values"]
    fig, ax = scaffold(
        "El riesgo país colapsó: principal palanca de compresión del WACC",
        "Spread soberano EMBI+ Argentina, evolución reciente",
        "puntos básicos",
        # Fix (auditoria ronda 4, ALTA): "estados financieros de Aluar" es
        # una atribucion de fuente FALSA -- el EMBI+ es un spread soberano
        # (BCRA/JPMorgan), no un dato contable de la compañía.
        "BCRA serie 5 / JPMorgan EMBI+ (static_inputs.json)",
        "menor EMBI+ → menor costo de capital exigido.")
    x = np.arange(len(dates))
    ax.plot(x, embi, color=C["navy"], lw=LW["heavy"])
    ax.fill_between(x, embi, 0, color=C["blue_lt"], alpha=0.30)  # baseline 0: área = coste real del riesgo
    ax.annotate(f"{embi[-1]:,.0f} pb", xy=(x[-1], embi[-1]), xytext=(-10, 8),
                textcoords="offset points", fontsize=SZ["annot"],
                fontweight="bold", color=C["value"], ha="right")
    # Anotaciones de shocks estructurales: derivadas de los datos (pico local y
    # mayor compresion interanual), nunca hardcodeadas -- si cambia la serie
    # persistida, la anotacion se recalcula sola.
    peak_i = int(np.argmax(embi))
    if peak_i != len(embi) - 1:
        ax.annotate(f"Pico {dates[peak_i]}\n{embi[peak_i]:,.0f} pb", xy=(x[peak_i], embi[peak_i]),
                    xytext=(0, 16), textcoords="offset points", ha="center",
                    fontsize=SZ["annot"] - 0.5, fontweight="bold", color=C["risk"],
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=C["risk"], lw=0.6, alpha=0.9),
                    arrowprops=dict(arrowstyle="-", color=C["risk"], lw=0.8))
    drops = [embi[i-1] - embi[i] for i in range(1, len(embi))]
    if drops:
        comp_i = int(np.argmax(drops)) + 1
        if comp_i != peak_i:
            ax.annotate(f"Compresión estructural {dates[comp_i]}", xy=(x[comp_i], embi[comp_i]),
                        xytext=(0, -26), textcoords="offset points", ha="center",
                        fontsize=SZ["annot"] - 0.5, fontweight="bold", color=C["value"],
                        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=C["value"], lw=0.6, alpha=0.9),
                        arrowprops=dict(arrowstyle="-", color=C["value"], lw=0.8))
    step = max(1, len(dates)//6)
    ax.set_xticks(x[::step]); ax.set_xticklabels(dates[::step], fontsize=SZ["tick"])
    ax.set_ylim(0, max(embi) * 1.15)
    ax.grid(axis="x", visible=False); _clean_y(ax)
    return export(fig, "s11_embi_compression", outdir)


# ── Slide 11 · Expansión múltiplos Merval (P/E) ───────────────────────────────
def plot_merval_pe(CH, outdir=None):
    s = (CH.get("static") or {}).get("merval_pe")
    if not s:
        return {"skip": "falta static_inputs['merval_pe'] (P/E histórico Merval)"}
    yrs, pe = s["years"], s["values"]
    fig, ax = scaffold(
        "El mercado re-rató: expansión de múltiplos del Merval",
        "Ratio Precio/Ganancias del índice S&P Merval por período",
        "P/E (x)", "yfinance ^MERV + earnings",
        "la re-calificación acompaña la compresión del riesgo país.")
    x = np.arange(len(yrs))
    bars = ax.bar(x, pe, color=C["blue"], width=0.6, zorder=3)
    # Fix (auditoria ronda 4, MEDIA): C["aluar"] esta reservado por
    # MASTER_STYLE para "SOLO destacar ALUAR" -- este grafico es P/E del
    # INDICE Merval (no de ALUAR), asi que resaltar el ultimo anio con ese
    # color es un uso semantico indebido. Se usa navy (mas oscuro/saturado)
    # para destacar el dato mas reciente sin invadir el color reservado.
    bars[-1].set_color(C["navy"])
    bars[-1].set_alpha(1.0)
    ax.bar_label(bars, fmt=lambda v: f"{v:.1f}x", fontsize=SZ["annot"], fontweight="bold", padding=2)
    ax.set_xticks(x); ax.set_xticklabels(yrs, fontsize=SZ["tick"])
    ax.set_ylim(0, max(pe) * 1.18)
    ax.grid(axis="x", visible=False); _clean_y(ax)
    return export(fig, "s11_merval_pe", outdir)


# ── Slide 15 · Matriz energética (barras horizontales) ───────────────────────
def plot_energy_mix(CH, outdir=None):
    s = (CH.get("static") or {}).get("energy_mix")
    if not s:
        return {"skip": "falta static_inputs['energy_mix'] (mix energético Memoria)"}
    labels = list(s.keys()); vals = list(s.values())
    order = sorted(range(len(vals)), key=lambda i: vals[i], reverse=True)
    labels = [labels[i] for i in order]; vals = [vals[i] for i in order]
    base_cols = [C["value"], C["blue"], C["navy"], C["blue_lt"]]
    cols = base_cols[:len(labels)]
    # Renovable calculado dinamicamente (hidro + eolico/solar), NO hardcodeado
    # -- el titulo anterior afirmaba "97% renovable" fijo, pero
    # static_inputs['energy_mix'] vigente da hidro 54%+eolico 24%=78% (termica a
    # gas 18% + compra a red 4% NO son renovables). Bug detectado al cruzar el
    # texto contra el JSON (mismo patron que los demas titulos dinamicos).
    _renov_kw = ("hidro", "eólic", "eolic", "solar")
    _renov_pct = sum(v for k, v in s.items() if any(kw in k.lower() for kw in _renov_kw))
    fig, ax = scaffold(
        f"Energía {_renov_pct:.0%} renovable: el foso competitivo más difícil de replicar",
        # Fix (auditoria ronda 4, ALTA): "874 MW" era un dato inventado --
        # no existe en static_inputs.json ni en ningun otro archivo del
        # proyecto (grep exhaustivo). Se retira la cifra no verificable,
        # se mantiene solo lo que SI esta respaldado por el dato real
        # (composicion % de la generacion propia).
        "Composición de la generación propia de ALUAR — Futaleufú (hidroeléctrica) como pilar",
        "% de la generación propia", "Memoria Anual ALUAR (static_inputs)",
        "ningún competidor latinoamericano tiene acceso a hidro propio de esta escala.")
    y = np.arange(len(labels))[::-1]
    bars = ax.barh(y, vals, color=cols, height=0.60, zorder=3)
    for yi, v, lbl in zip(y, vals, labels):
        ax.text(v * 0.50, yi, f"{v:.0%}", va="center", ha="center",
                fontsize=SZ["annot"], fontweight="bold", color="white")
        ax.text(v + 0.012, yi, lbl, va="center", ha="left",
                fontsize=SZ["annot"] - 0.5, color=C["ink"])
    ax.set_yticks([]); ax.set_xlim(0, 1.25)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", visible=False); _clean_y(ax)  # eje Y oculto -- pct=True no aplica aqui (categorico)
    return export(fig, "s15_energy_mix", outdir)


# ── Slide 16 · Curva de costos global ─────────────────────────────────────────
def plot_cost_curve(CH, outdir=None):
    s = (CH.get("static") or {}).get("cost_curve")
    if not s:
        return {"skip": "falta static_inputs['cost_curve'] (cash cost peers)"}
    names, costs = s["names"], s["cash_cost"]
    order = np.argsort(costs)
    names = [names[i] for i in order]; costs = [costs[i] for i in order]
    cols = [C["aluar"] if "ALUAR" in n.upper() else C["blue_lt"] for n in names]
    # Fix (auditoria ronda 4, MEDIA-ALTA): "cuartil bajo" era matematicamente
    # falso -- ALUAR es 3°/8 mas barato (percentil ~37.5%), no esta en el
    # cuartil bajo real (que exigiria estar entre los 2 mas baratos de 8).
    # El titulo se deriva del rank real en vez de afirmar una posicion fija.
    _rank_aluar = next((i for i, n in enumerate(names) if "ALUAR" in n.upper()), None)
    if _rank_aluar is not None:
        _n = len(names)
        _pos = _rank_aluar + 1
        if _pos <= max(1, round(_n * 0.25)):
            _rank_txt = f"opera en el cuartil bajo de costos ({_pos}°/{_n} más barato)"
        elif _pos <= max(1, round(_n * 0.5)):
            _rank_txt = f"opera en la mitad más barata de la curva de costos ({_pos}°/{_n})"
        else:
            _rank_txt = f"opera en la mitad más cara de la curva de costos ({_pos}°/{_n})"
    else:
        _rank_txt = "posición no determinada en la curva de costos"
    fig, ax = scaffold(
        f"ALUAR {_rank_txt}",
        "Cash cost por tonelada de productores primarios seleccionados",
        "USD/Tn", "static_inputs (CRU / reportes peers)",
        "menor cash cost = márgenes protegidos ante caídas del LME.")
    y = np.arange(len(names))[::-1]
    bars = ax.barh(y, costs, color=cols, height=0.62, zorder=3)
    ax.bar_label(bars, fmt=lambda v: f"{v:,.0f}", fontsize=SZ["annot"], fontweight="bold", padding=2)
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=SZ["tick"])
    ax.set_xlim(0, max(costs) * 1.18)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: _money_fmt(v)))
    ax.grid(axis="y", visible=False); _clean_y(ax)
    return export(fig, "s16_cost_curve", outdir)


# ── Slide 17 · EBITDA histórico + proyectado ──────────────────────────────────
def plot_ebitda_hist_proj(CH, outdir=None):
    d = CH["ebitda"]; years = d["years"]
    vals = [v or 0 for v in (d["hist"] + d["proj"])]
    n_h = len(d["hist"])
    # Fix (auditoria ronda 4, ALTA): C["value"] (#1B7F4B, verde) esta
    # reservado por MASTER_STYLE para "SOLO creación de valor / upside" --
    # usarlo para la serie de PROYECCIÓN viola el contrato de color
    # institucional (.claude/rules/10_style_linage_clean.md exige naranja
    # #E36414 para proyección). C["aluar"] (#E8833A, naranja) es el color
    # institucional correcto para esta serie.
    cols = [C["navy"]]*n_h + [C["aluar"]]*(len(vals)-n_h)
    # Nota dinamica: la version anterior afirmaba fijo "expansion de margen
    # hacia 25-28%" -- FALSO contra m4_out.json vigente, donde el margen EBITDA
    # proyectado 2026-2030 es PLANO en 17.38% (no expande). El crecimiento real
    # del EBITDA en USD MM es por REVENUE (CAGR), no por dilucion de margen --
    # la nota ahora reporta el driver correcto, calculado del propio vector.
    _margins_proj = (CH.get("ebitda_margin") or {}).get("proj") or []
    _margins_proj_v = [m for m in _margins_proj if m is not None]
    _proj_vals = d["proj"]
    if len(_proj_vals) >= 2 and _proj_vals[0] and _proj_vals[-1] and _proj_vals[0] > 0:
        _cagr_ebitda = (_proj_vals[-1] / _proj_vals[0]) ** (1 / (len(_proj_vals) - 1)) - 1
    else:
        _cagr_ebitda = None
    if _margins_proj_v and max(_margins_proj_v) - min(_margins_proj_v) < 0.01:
        _nota_ebitda = (f"margen EBITDA proyectado estable (~{_margins_proj_v[0]:.1%}); el crecimiento "
                         f"proviene del revenue (CAGR {_cagr_ebitda:.1%})." if _cagr_ebitda is not None
                         else f"margen EBITDA proyectado estable (~{_margins_proj_v[0]:.1%}).")
    elif _cagr_ebitda is not None:
        _nota_ebitda = f"crecimiento del EBITDA proyectado: CAGR {_cagr_ebitda:.1%} (revenue + margen)."
    else:
        _nota_ebitda = "proyección driver-based (revenue y costos operativos)."
    fig, ax = scaffold(
        "EBITDA resiliente y en expansión: base del DCF",
        "EBITDA histórico y proyectado (azul: histórico · naranja: proyección)",
        "USD MM", "canonical + proyecciones",
        _nota_ebitda)
    x = np.arange(len(years))
    bars = ax.bar(x, vals, color=cols, width=0.62, zorder=3)
    ax.axvline(n_h - 0.5, color=C["muted"], lw=LW["medium"], ls="--")
    ax.text(n_h - 0.5 - 0.10, 0.04, "Proyección post-pico CAPEX 2025 (Puerto Madryn)",
            transform=ax.get_xaxis_transform(), rotation=90, va="bottom", ha="right",
            fontsize=SZ["annot"] - 1.5, color=C["muted"], style="italic")
    ax.bar_label(bars, fmt=lambda v: f"{v:,.0f}" if v else "", fontsize=SZ["annot"]-0.5, padding=2)
    proj_vals = d["proj"]
    ymax = max(vals) * 1.20
    if proj_vals and len(proj_vals) >= 2:
        v0 = proj_vals[0]; vn = proj_vals[-1]; n = len(proj_vals) - 1
        if v0 and vn and v0 > 0 and n > 0:
            cagr_proy = (vn / v0) ** (1/n) - 1
            ax.text(n_h + (len(proj_vals)-1)*0.5, max(vals)*1.10,
                    f"CAGR {cagr_proy:.1%}", ha="center", fontsize=SZ["annot"],
                    color=C["value"], fontweight="bold")
    ax.set_ylim(0, ymax)
    ax.set_xticks(x); ax.set_xticklabels(years, fontsize=SZ["tick"], rotation=0)
    ax.grid(axis="x", visible=False); _clean_y(ax, money=True)
    return export(fig, "s17_ebitda_hist_proj", outdir)


# ── Slide 18 · Estructura patrimonial (barras agrupadas) ──────────────────────
def plot_balance_structure(CH, outdir=None):
    d = CH["balance"]; yrs = d["years"]
    # Titulo y nota dinamicos: el titulo anterior afirmaba fijo "desapalancamiento"
    # y la nota citaba un "D/E objetivo ~25%" -- ambos FALSOS contra los datos
    # vigentes. La deuda financiera crece cada anio (134->595 USD MM 2022->2025)
    # para financiar el Parque Eolico, y el D/E empirico (deuda/patrimonio) SUBE
    # de 48.6% (FY24) a 53.6% (FY25) -- es apalancamiento CRECIENTE, no
    # desapalancamiento. El "25%" era un supuesto de una iteracion vieja del
    # modelo (00_fetch_live_params.py D_E_OBJ); el WACC vigente (m5_out.json)
    # usa d_e_target=48.6% (mediana empirica FY23-25), no 25%.
    _de_pairs = [(y, dv, pv) for y, dv, pv in zip(yrs, d["deuda"], d["pn"]) if dv is not None and pv]
    _de_ratios = [dv / pv for _, dv, pv in _de_pairs]
    if len(_de_ratios) >= 2 and _de_ratios[-1] > _de_ratios[0] * 1.02:
        _titulo_bal = "Apalancamiento creciente financia la expansión, sobre una base patrimonial sólida"
    elif len(_de_ratios) >= 2 and _de_ratios[-1] < _de_ratios[0] * 0.98:
        _titulo_bal = "Balance sólido con desapalancamiento: grado de inversión local"
    else:
        _titulo_bal = "Balance estable: la deuda financiera acompaña el crecimiento del activo"
    _nota_bal = (f"D/E empírico {_de_ratios[-1]:.0%} ({_de_pairs[-1][0]}), usado como target en el WACC (M5)."
                 if _de_ratios else "deuda financiera destinada a CAPEX expansivo (Parque Eólico).")
    fig, ax = scaffold(
        _titulo_bal,
        "Activo total, patrimonio neto y deuda financiera por ejercicio",
        "USD MM", "estados financieros de Aluar",
        _nota_bal)
    # Fix (auditoria jul-09, bug SHIPPEADO): offset=w=0.27 con ancho=w*2=0.54
    # da un grupo total de 1.08 > 1.0 (espaciado entre anios) -- las barras
    # de anios adyacentes se superponian levemente (visible en el borde de
    # color entre 2022/2023 y 2023/2024 del PNG regenerado). offset y ancho
    # independientes, grupo < 1.0, mismo criterio que el fix de m13_04.
    x = np.arange(len(yrs)); w = 0.22; bw = 0.4
    activo = [v or 0 for v in d["activo"]]; pn = [v or 0 for v in d["pn"]]; deuda = [v or 0 for v in d["deuda"]]
    ax.bar(x - w, activo, bw, label="Activo total", color=C["navy"], zorder=3)
    # Fix (auditoria ronda 4, ALTA): C["value"] (verde) esta reservado por
    # MASTER_STYLE para "SOLO creacion de valor / upside" -- Patrimonio Neto
    # es un stock contable estatico del balance, no una serie de creacion de
    # valor. Mismo patron de mal uso ya corregido para las series de
    # proyeccion de EBITDA esta sesion; aca se usa blue_lt (categoria neutra),
    # dejando C["value"]/C["risk"] exclusivamente para sus semanticas reales.
    ax.bar(x + w, pn, bw, label="Patrimonio neto", color=C["blue_lt"], zorder=3)
    # Linea de control (deuda financiera) al frente de las barras de fondo:
    # zorder=5 > zorder=3 de las barras (antes quedaba detras, zorder Line2D
    # por defecto=2 < 3 de las barras -- bug de capas visual).
    ax.plot(x, deuda, "s--", color=C["risk"], lw=LW["bold"], ms=7, zorder=5,
            label="Deuda financiera — financia la expansión eólica")
    for xi, v in zip(x, deuda):
        if v:
            ax.annotate(f"${v:,.0f}", (xi, v), textcoords="offset points", xytext=(0, 11),
                        ha="center", fontsize=SZ["annot"] - 1, fontweight="bold", color=C["risk"],
                        zorder=6, bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=C["risk"], lw=0.6, alpha=0.95))
    ax.set_xticks(x); ax.set_xticklabels(yrs, fontsize=SZ["tick"])
    hist_ebitda = (CH.get("ebitda") or {}).get("hist") or []
    ebitda_map = dict(zip([2022, 2023, 2024, 2025], hist_ebitda[-4:]))
    for xi, yr in enumerate(yrs):
        deuda_v = d["deuda"][xi]; ebitda_v = ebitda_map.get(yr)
        if deuda_v and ebitda_v and ebitda_v > 0:
            ratio = deuda_v / ebitda_v
            col_r = C["value"] if ratio <= 3.0 else C["risk"]
            ax.text(xi, max(activo[xi], pn[xi]) * 1.04,
                    f"{ratio:.1f}x D/EBITDA", ha="center", va="bottom",
                    fontsize=SZ["annot"] - 1, color=col_r, fontweight="bold")
    ax.set_ylim(0, max(activo + pn) * 1.20)
    ax.grid(axis="x", visible=False); _clean_y(ax, money=True)
    # y=-0.16 (no mas negativo): con nota larga en el pie (D/E empirico),
    # un y demasiado negativo (-0.24, fix anterior) acerca la leyenda a la
    # cita de fuente en vez de alejarla -- colisionaban. -0.16 deja hueco
    # suficiente entre el eje X y el pie de pagina (bug de layout, ver
    # _legend_outside).
    _legend_outside(ax, ncol=3, y=-0.16)
    return export(fig, "s18_balance_structure", outdir)


# ── Slide 25 · Evolución margen EBITDA ────────────────────────────────────────
def plot_ebitda_margin(CH, outdir=None):
    d = CH["ebitda_margin"]
    yrs_h = [y for y, v in zip(d["years"][:len(d["values"])], d["values"]) if v is not None]
    vals_h = [v for v in d["values"] if v is not None]
    yrs_p_all = d["years"][len(d["values"]):]
    proj_vals = d.get("proj") or []
    yrs_p = [y for y, v in zip(yrs_p_all, proj_vals) if v is not None]
    vals_p = [v for v in proj_vals if v is not None]
    fig, ax = scaffold(
        "El margen EBITDA se sostiene en la proyección tras el pico de CAPEX",
        "Margen EBITDA histórico (2020-2025) y proyectado (2026-2030, driver-based)",
        "% sobre ventas", "canonical financials + proyecciones M4",
        "eficiencia energética propia sostiene el margen en el ciclo.")
    ax.plot(yrs_h, vals_h, "o-", color=C["navy"], lw=LW["heavy"], ms=7, label="Histórico", zorder=3)
    ax.fill_between(yrs_h, vals_h, 0, color=C["blue_lt"], alpha=0.30)
    if yrs_p and vals_p:
        yrs_bridge = ([yrs_h[-1]] + yrs_p) if yrs_h else yrs_p
        vals_bridge = ([vals_h[-1]] + vals_p) if yrs_h and vals_h else vals_p
        # Fix (auditoria ronda 4, ALTA): mismo bug de color que plot_ebitda_hist_proj
        # -- C["value"] (verde) esta reservado para creacion de valor/upside,
        # no para la serie de proyeccion (naranja institucional, C["aluar"]).
        ax.plot(yrs_bridge, vals_bridge, "o--", color=C["aluar"], lw=LW["heavy"], ms=7,
                label="Proyectado (driver-based)", zorder=3)
        ax.fill_between(yrs_bridge, vals_bridge, 0, color=C["aluar"], alpha=0.15)
        ax.axvline((yrs_h[-1] + yrs_p[0]) / 2, color=C["muted"], lw=LW["light"], ls="--", alpha=0.55)
    all_vals = vals_h + vals_p
    if vals_h:
        avg_mg = sum(vals_h) / len(vals_h)
        ax.axhline(avg_mg, color=C["muted"], lw=LW["medium"], ls=":", alpha=0.7,
                   label=f"Promedio histórico {avg_mg:.0%}")
    # bbox blanco: en anios de pico/valle pronunciado (ej. 2023) la propia
    # linea del grafico atraviesa la etiqueta y la vuelve ilegible ("16%"
    # se leia "6%") -- mismo patron de recuadro que balance_structure/fcff_bridge.
    for xi, v in zip(yrs_h, vals_h):
        ax.annotate(f"{v:.0%}", (xi, v), textcoords="offset points", xytext=(0, 11),
                    ha="center", va="bottom", fontsize=SZ["annot"], fontweight="bold",
                    color=C["navy"], zorder=6,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=C["navy"], lw=0.5, alpha=0.95))
    for xi, v in zip(yrs_p, vals_p):
        ax.annotate(f"{v:.0%}", (xi, v), textcoords="offset points", xytext=(0, 11),
                    ha="center", va="bottom", fontsize=SZ["annot"], fontweight="bold",
                    color=C["value"], zorder=6,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=C["value"], lw=0.5, alpha=0.95))
    ax.set_ylim(0, max(all_vals) * 1.30 if all_vals else 0.40)
    ax.grid(axis="x", visible=False); _clean_y(ax, pct=True)
    _legend_outside(ax, ncol=3)
    return export(fig, "s25_ebitda_margin", outdir)


In [18]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA · BIBLIOTECA DE GRÁFICOS (B) — costo de capital · valuación · riesgo ║
# ║  Todas usan MASTER_STYLE (vía C/SZ/LW, celda de tema).                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import matplotlib.ticker as mtick
from scipy import stats as _st


# ── Slide 31 · Arquitectura de betas ──────────────────────────────────────────
def plot_beta_architecture(CH, outdir=None):
    b = CH["beta"]
    if any(b.get(k) is None for k in ("ols", "blume", "vasicek", "l")):
        return {"skip": "faltan betas en m5_out.json"}
    pipe_labels = ["OLS\n(regresión 5Y)", "Blume\n(2/3·β+1/3)", "Vasicek\n(Bayesiano)\n= β usado WACC"]
    pipe_vals   = [b["ols"], b["blume"], b["vasicek"]]
    pipe_cols   = [C["muted"], C["blue_lt"], C["aluar"]]
    # Fallbacks (auditoria jul-09): si m5_out.json llegara a faltar estos campos,
    # se usan los ultimos valores verificados (Damodaran Metals & Mining, sesion
    # jul-2026) -- documentados con fecha para que no se confundan con datos vivos
    # si el modelo cambia y estos fallbacks quedan desactualizados.
    beta_u_dam = b.get("u_damodaran") or 0.96   # fallback verificado jul-2026 (Damodaran M&M)
    beta_l_dam = b.get("l_damodaran")
    # d_e leido de m5_out.json (mediana empirica FY23-25, misma fuente que usa el
    # WACC real) -- antes el label mostraba "0.486" hardcodeado sin leer CH, que
    # coincidia por casualidad mientras d_e_target no cambiara entre corridas.
    d_e = b.get("d_e_target") or 0.486   # fallback D/E empírico FY23-25 verificado jul-2026
    if not beta_l_dam:
        beta_l_dam = beta_u_dam * (1 + 0.65 * d_e)  # Hamada: 0.65 = 1-tax

    fig, ax = scaffold(
        "El beta Vasicek entra al WACC: conservador, verificable, reproducible",
        "Pipeline de estimación de beta (ALUA.BA vs S&P500) · Contraste con β sectorial Damodaran",
        "Beta (β)", "OLS (n=1189 obs.); Blume 1971; Vasicek 1973; Damodaran 2026",
        "β=1.0 = riesgo de mercado.")

    y_pipe = np.arange(len(pipe_labels))[::-1] + 1.5
    bars1 = ax.barh(y_pipe, pipe_vals, color=pipe_cols, height=0.58, zorder=3)
    ax.axhline(0.85, color=C["muted"], lw=LW["thin"], ls=":", alpha=0.60)

    dam_labels = ["β_U Damodaran\n(sector M&M)", f"β_L Damodaran\n(D/E {d_e:.2f})"]
    dam_vals   = [beta_u_dam, beta_l_dam]
    y_dam = np.array([-0.30, -1.00])
    bars2 = ax.barh(y_dam, dam_vals, color=[C["navy"], C["blue"]], height=0.52,
                    zorder=3, alpha=0.75, hatch="//", edgecolor="white",
                    label="Referencia externa — valida el β propio contra el sector (Damodaran)")

    all_vals = pipe_vals + dam_vals
    ax.axvline(1.0, color=C["ink"], lw=LW["light"], ls="--", alpha=0.45, label="β Mercado = 1.0")

    ax.bar_label(bars1, fmt=lambda v: f"{v:.3f}β", fontsize=SZ["annot"], fontweight="bold", padding=3)
    ax.bar_label(bars2, fmt=lambda v: f"{v:.3f}β", fontsize=SZ["annot"]-0.5, padding=3)

    all_y = list(y_pipe) + list(y_dam)
    all_labels = pipe_labels + dam_labels
    ax.set_yticks(all_y); ax.set_yticklabels(all_labels, fontsize=SZ["tick"] - 0.5)
    ax.set_xlim(0, max(all_vals) * 1.25)
    _legend_outside(ax, ncol=1)
    ax.grid(axis="y", visible=False); _clean_y(ax)
    return export(fig, "s31_beta_architecture", outdir)


# ── Slide 20 · Descomposición del WACC ────────────────────────────────────────
def plot_wacc_decomposition(CH, outdir=None):
    w = CH["wacc"]
    if any(w.get(k) is None for k in ("rf", "beta_x_erp", "lambda_x_crp", "ke", "wacc")):
        return {"skip": "faltan componentes WACC en m5_out.json"}
    lambda_ar = w.get("lambda_ar")
    lambda_step = f"+ λ·CRP\n(λ={lambda_ar:.2f})" if lambda_ar is not None else "+ λ·CRP"
    steps = ["Risk-free\n(US 10Y)", "+ β·ERP", lambda_step, "= Ke", "WACC"]
    bottoms = [0, w["rf"], w["rf"]+w["beta_x_erp"], 0, 0]
    heights = [w["rf"], w["beta_x_erp"], w["lambda_x_crp"], w["ke"], w["wacc"]]
    cols = [C["blue"], C["blue_lt"], C["blue_lt"], C["navy"], C["aluar"]]
    lambda_note = (f"λ={lambda_ar:.2f} — {w['lambda_fuente']}." if lambda_ar is not None and w.get("lambda_fuente")
                   else "λ pondera el EMBI+ por la fracción doméstica de ingresos.")
    fig, ax = scaffold(
        "El WACC se comprime al escalar el riesgo país por la exposición real",
        "Construcción del Ke vía CAPM-Lambda (Damodaran) y mezcla con el costo de deuda",
        "% anual (USD)", "WACC engine (Dumrauf cap. 14)",
        lambda_note)
    x = np.arange(len(steps))
    ax.bar(x, heights, bottom=bottoms, color=cols, width=0.55, zorder=3)
    ax.axvline(2.5, color=C["muted"], lw=LW["light"], ls="--", alpha=0.55)
    for xi, (bo, h) in enumerate(zip(bottoms, heights)):
        ax.text(xi, bo + h + 0.0015, f"{h:.2%}", ha="center", va="bottom",
                fontsize=SZ["annot"], fontweight="bold",
                color=C["aluar"] if xi == 4 else C["ink"])
    ax.set_xticks(x); ax.set_xticklabels(steps, fontsize=SZ["tick"])
    ax.set_ylim(0, max(w["ke"], w["wacc"]) * 1.22)
    ax.grid(axis="x", visible=False); _clean_y(ax, pct=True)
    return export(fig, "s20_wacc_decomposition", outdir)


# ── Slide 20b · Ke con Lambda vs Ke convencional (EMBI+ pleno) ────────────────
def plot_ke_lambda_vs_conv(CH, outdir=None):
    w = CH["wacc"]
    if any(w.get(k) is None for k in ("rf", "beta_l", "erp", "crp", "ke")):
        return {"skip": "faltan datos para Ke convencional vs Lambda"}
    ke_lambda = w["ke"]
    ke_conv = w["rf"] + w["beta_l"]*w["erp"] + 1.0*w["crp"]   # λ=1 (EMBI+ pleno)
    fig, ax = scaffold(
        "El ajuste por Lambda evita penalizar dos veces al exportador dolarizado",
        "Costo del equity con EMBI+ pleno (convencional) vs ponderado por λ",
        "% anual", "CAPM-Lambda (Damodaran)",
        "λ refleja que ALUAR sólo absorbe una fracción del shock soberano.")
    bars = ax.bar(["Ke convencional\n(λ=1)", "Ke con Lambda\n(modelo)"],
                  [ke_conv, ke_lambda], color=[C["muted"], C["aluar"]], width=0.5, zorder=3)
    ax.bar_label(bars, fmt=lambda v: f"{v:.2%}", fontsize=SZ["annot"]+1, fontweight="bold", padding=3)
    ax.set_ylim(0, max(ke_conv, ke_lambda) * 1.18)
    ax.grid(axis="x", visible=False); _clean_y(ax, pct=True)
    return export(fig, "s20b_ke_lambda_vs_conv", outdir)


# ── Slide 20c · CAPEX shrinkage: propio vs. pares globales ────────────────────────────
def plot_capex_shrinkage(CH, outdir=None):
    cx = CH.get("capex") or {}
    det = cx.get("detalle")
    if not det or not det.get("aplicado"):
        return {"skip": "falta capex_shrinkage_detalle en m4_out.json (peer_capex_da no disponible)"}
    obs = det.get("peer_observaciones") or []
    if not obs:
        return {"skip": "capex_shrinkage_detalle sin observaciones de peers"}

    own_med, own_n = det["propio_mediana"], det["propio_n"]
    own_rstd = det["propio_std_robusto_mad"]
    peer_mean, peer_n, peer_std = det["peers_media"], det["peers_n"], det["peers_std"]
    w_own, w_peer = det["peso_propio"], det["peso_peers"]
    resultado = det["resultado"]
    da_pct = cx.get("da_pct_rev_median")

    fig, ax = scaffold(
        f"El CAPEX de ALUAR se combina con {peer_n} balances de 5 pares globales para superar N={own_n}",
        "Shrinkage por inversa de varianza (Fisher/Cochran) -- mismo principio que Vasicek/Bayes-Stein",
        "CAPEX / Revenue (%)", "yfinance (peers, cashflow/income statement en vivo) + canonical_financials.json (ALUAR)",
        f"Peso propio {w_own:.0%} (mediana, error robusto MAD) / peso peers {w_peer:.0%} (media, N={peer_n}).")

    peer_vals = [o["capex_pct_rev"] for o in obs]
    rng = np.random.default_rng(7)
    jitter = rng.uniform(-0.12, 0.12, size=len(peer_vals))
    ax.scatter(peer_vals, np.full(len(peer_vals), 0) + jitter, s=26, color=C["blue_lt"],
               alpha=0.65, zorder=2, label="Peers (obs. individuales)")

    label_peers = "Pares" + chr(10) + "(media)"
    label_propio = "ALUAR" + chr(10) + "(mediana propia)"
    label_result = "Resultado" + chr(10) + "(shrinkage)"
    rows = [
        (label_peers, peer_mean, peer_std / np.sqrt(peer_n), C["blue"], 1),
        (label_propio, own_med, own_rstd / np.sqrt(own_n), C["navy"], 2),
        (label_result, resultado, None, C["aluar"], 3),
    ]
    for label, val, err, col, y in rows:
        if err:
            ax.errorbar([val], [y], xerr=[[err], [err]], fmt="o", color=col, ms=11,
                        capsize=5, lw=LW["bold"], zorder=4)
        else:
            ax.scatter([val], [y], s=170, color=col, marker="D", zorder=5, edgecolor="white", linewidth=1.2)
        ax.text(val, y + 0.28, f"{val:.2%}", ha="center", va="bottom", fontsize=SZ["annot"],
                fontweight="bold", color=col)

    if da_pct is not None:
        ax.axvline(da_pct, color=C["risk"], lw=LW["medium"], ls="--", alpha=0.75, zorder=1)
        da_note = "D&A/Rev = " + format(da_pct, ".2%") + chr(10) + "(piso de perpetuidad," + chr(10) + "no aplica al horizonte explicito)"
        ax.text(da_pct, 3.55, da_note, ha="center", va="bottom", fontsize=SZ["annot"] - 0.5, color=C["risk"])

    ax.set_yticks([0, 1, 2, 3])
    ytick_labels = ["Peers" + chr(10) + "(individuales)", "Pares" + chr(10) + "(media)",
                    "ALUAR" + chr(10) + "(propio)", "Resultado" + chr(10) + "(usado en DCF)"]
    ax.set_yticklabels(ytick_labels, fontsize=SZ["tick"])
    ax.set_ylim(-0.6, 3.9)
    ax.grid(axis="y", visible=False)
    # Fix: _clean_y(ax, pct=True) reformatea el eje Y (categorico en este
    # grafico -- filas de grupo, no un valor numerico) como porcentaje,
    # pisando las etiquetas de categoria recien seteadas arriba. El
    # porcentaje va en el eje X (el valor CAPEX/Revenue), no en el Y.
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.tick_params(axis="y", length=0)
    ax.margins(x=0.02)
    ax.set_xlabel("CAPEX / Revenue", fontsize=SZ["axis"])
    return export(fig, "s20c_capex_shrinkage", outdir)

# ── Slide 21 · ROIC vs WACC ───────────────────────────────────────────────────
def plot_roic_vs_wacc(CH, outdir=None):
    d = CH["roic"]; wacc = CH["wacc_ref"]
    yrs = [y for y, v in zip(d["years"], d["values"]) if v is not None]
    roic = [v for v in d["values"] if v is not None]
    if not roic or wacc is None:
        return {"skip": "faltan ROIC histórico (m4) o WACC (m5)"}
    n_above = sum(1 for r in roic if r >= wacc)
    avg_roic = sum(roic) / len(roic)
    # Titulo dinamico: el ciclo del aluminio es volatil y el ROIC no supera al
    # WACC todos los anios (bug detectado: el titulo anterior afirmaba "todo
    # el ciclo" siendo falso para 4 de 6 anios historicos) -- se ajusta al
    # patron real observado, sin asumir de antemano el resultado.
    if n_above == len(roic):
        titulo = "ROIC por encima del WACC en todo el ciclo: foso económico positivo"
    elif avg_roic >= wacc:
        titulo = (f"El ROIC promedio del ciclo ({avg_roic:.1%}) supera al WACC pese a la "
                  f"volatilidad de {len(roic)-n_above} de {len(roic)} años por debajo del piso")
    else:
        titulo = (f"El ROIC supera al WACC sólo en {n_above} de {len(roic)} años: "
                  f"foso económico condicionado al ciclo del aluminio")
    fig, ax = scaffold(
        titulo,
        "Retorno sobre el capital invertido frente al costo de capital",
        "% anual", "ROIC histórico y WACC",
        "el spread ROIC−WACC mide la creación de valor.")
    x = np.arange(len(yrs))
    cols = [C["value"] if r >= wacc else C["risk"] for r in roic]
    bars = ax.bar(x, roic, color=cols, width=0.6, zorder=3)
    ax.axhline(wacc, color=C["navy"], lw=LW["bold"], ls="--", zorder=4)
    # Data label explicito sobre la linea (antes solo vivia en la leyenda) --
    # anclado a la derecha del ultimo bar para no colisionar con su etiqueta.
    ax.annotate(f"WACC {wacc:.1%}", xy=(x[-1] + 0.55, wacc), xytext=(2, 0),
                textcoords="offset points", ha="left", va="center",
                fontsize=SZ["annot"], fontweight="bold", color=C["navy"],
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=C["navy"], lw=0.6, alpha=0.95),
                zorder=6)
    # bbox blanco: anios cuyo ROIC queda pegado a la linea de WACC (ej. 2021,
    # 6.7% vs WACC 7.0%) quedaban con la etiqueta atravesada por la linea --
    # mismo patron de recuadro que el resto de graficos de este modulo.
    ax.bar_label(bars, fmt=lambda v: f"{v:.1%}", fontsize=SZ["annot"], padding=2, zorder=7,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.95))
    ax.set_xticks(x); ax.set_xticklabels(yrs, fontsize=SZ["tick"])
    ax.set_xlim(-0.6, len(yrs) - 1 + 1.15)
    ax.set_ylim(0, max(max(roic), wacc) * 1.20)
    ax.grid(axis="x", visible=False); _clean_y(ax, pct=True)
    return export(fig, "s21_roic_vs_wacc", outdir)


# ── Slide 23 · DCF Bridge EV → Equity ─────────────────────────────────────────
def plot_dcf_waterfall(CH, outdir=None):
    d = CH["dcf"]
    if any(d.get(k) is None for k in ("pv_fcffs", "pv_tv", "ev", "net_debt", "equity")):
        return {"skip": "faltan componentes DCF en m6_out.json"}
    labels = ["PV FCFF\nexplícitos", "PV Valor\nTerminal", "Enterprise\nValue",
              "(−) Deuda\nNeta", "Equity\nValue"]
    ev = d["ev"]
    fig, ax = scaffold(
        "Puente de valuación: de Enterprise Value a Equity Value",
        "Descomposición del DCF (Penman/Gordon) en USD MM",
        "USD MM", "DCF",
        "el puente a precio por acción aplica ÷acciones y ×CCL.")
    bars = [(0, d["pv_fcffs"], C["blue"]), (d["pv_fcffs"], d["pv_tv"], C["blue_lt"]),
            (0, ev, C["navy"]), (ev, -d["net_debt"], C["risk"]), (0, d["equity"], C["value"])]
    annots = [d["pv_fcffs"], d["pv_tv"], ev, d["net_debt"], d["equity"]]
    prefixes = ["", "", "", "(−) ", ""]
    for i, ((b, h, col), lbl, pref) in enumerate(zip(bars, annots, prefixes)):
        ax.bar(i, h, bottom=b, color=col, width=0.62, zorder=3)
        ax.text(i, max(b+h, b) + ev*0.015, f"{pref}{lbl:,.0f}", ha="center", va="bottom",
                fontsize=SZ["annot"], fontweight="bold",
                color=C["value"] if i == 4 else C["ink"])
    connector_pairs = [
        (0, d["pv_fcffs"], 1, d["pv_fcffs"]),
        (1, d["pv_fcffs"]+d["pv_tv"], 2, ev),
    ]
    for (xi, yi, xf, yf) in connector_pairs:
        ax.plot([xi+0.31, xf-0.31], [yi, yf], color=C["muted"], lw=LW["light"], ls="-", alpha=0.6)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=SZ["tick"])
    ax.set_ylim(0, ev * 1.25); ax.grid(axis="x", visible=False); _clean_y(ax, money=True)
    return export(fig, "s23_dcf_waterfall", outdir)


# ── Slide 22 · FCFF proyectado ────────────────────────────────────────────────
def plot_fcff_projection(CH, outdir=None):
    d = CH["fcff"]
    if not d["values"]:
        return {"skip": "falta fcff_proj en m6/m4"}
    fig, ax = scaffold(
        "El FCFF se expande tras superar el pico de CAPEX del Parque Eólico",
        "Free Cash Flow to Firm proyectado, método indirecto (Dumrauf cap. 9)",
        "USD MM", "proyecciones driver-based",
        "NOPAT + D&A − ΔWC − CAPEX.")
    x = np.arange(len(d["years"])); vals = d["values"]
    bars = ax.bar(x, vals, color=C["navy"], width=0.6, zorder=3)
    bars[-1].set_color(C["aluar"])
    ax.bar_label(bars, fmt=lambda v: f"{v:,.0f}", fontsize=SZ["annot"], fontweight="bold", padding=2)
    if len(vals) >= 2 and vals[0] and vals[-1] and vals[0] > 0:
        n = len(vals) - 1
        cagr = (vals[-1] / vals[0]) ** (1/n) - 1
        ax.annotate(f"CAGR {cagr:.1%}", xy=(x[-1], vals[-1]), xytext=(-10, 22),
                    textcoords="offset points", fontsize=SZ["annot"],
                    color=C["aluar"], fontweight="bold", ha="right",
                    arrowprops=dict(arrowstyle="-", color=C["aluar"], lw=LW["thin"]))
    ax.set_xticks(x); ax.set_xticklabels(d["years"], fontsize=SZ["tick"])
    ax.set_ylim(0, max(vals) * 1.22)
    ax.grid(axis="x", visible=False); _clean_y(ax, money=True)
    return export(fig, "s22_fcff_projection", outdir)


# ── Slide 18b · Cascada FCFF anual (hist + proy) ──────────────────────────────
def plot_fcff_bridge_annual(CH, outdir=None):
    d = CH["fcff_annual"]
    vals = [v for v in d["values"]]
    if all(v is None for v in vals):
        return {"skip": "faltan FCFF anuales (canonical + m6)"}
    # Fix (auditoria ronda 4, MEDIA): "v or 0" enmascararia un anio
    # proyectado sin dato como un FCFF=0 real -- mismo patron ya corregido en
    # el CH builder (cell 32) para ebitda['proj']. Hoy es inocuo (los 11
    # anios tienen dato real), pero si un anio de m6_out.json faltara a
    # futuro, no debe graficarse como una barra de $0 real.
    vals = [v if v is not None else 0 for v in vals]
    cols = [C["value"] if v >= 0 else C["risk"] for v in vals]
    # Fix (auditoria ronda 4, ALTA): "estados financieros de Aluar" atribuye
    # los 5 anios PROYECTADOS (2026E-2030E, driver-based de M6, no un dato
    # contable) a una fuente que solo corresponde a los 6 anios historicos.
    fig, ax = scaffold(
        "Generación de caja libre: del ciclo de inversión a la cosecha",
        "FCFF anual histórico y proyectado",
        "USD MM", "estados financieros de Aluar (histórico) + proyección driver-based M6 (2026E-2030E)",
        "valores negativos = años de CAPEX expansivo.")
    x = np.arange(len(d["years"]))
    bars = ax.bar(x, vals, color=cols, width=0.62, zorder=3)
    ax.axhline(0, color=C["ink"], lw=LW["thin"])
    for xi, v in zip(x, vals):
        offset = 10 if v >= 0 else -10
        va = "bottom" if v >= 0 else "top"
        ax.annotate(f"${v:,.0f}", (xi, v), textcoords="offset points", xytext=(0, offset),
                    ha="center", va=va, fontsize=SZ["annot"] - 1, fontweight="bold",
                    color=(C["value"] if v >= 0 else C["risk"]), zorder=6,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white",
                              ec=(C["value"] if v >= 0 else C["risk"]), lw=0.5, alpha=0.9))
    ax.set_xticks(x); ax.set_xticklabels(d["years"], fontsize=SZ["tick"], rotation=0)
    ymax = max(vals) * 1.28 if max(vals) > 0 else 0
    ymin = min(vals) * 1.28 if min(vals) < 0 else 0
    ax.set_ylim(ymin, ymax)
    ax.grid(axis="x", visible=False); _clean_y(ax, money=True)
    return export(fig, "s18b_fcff_bridge_annual", outdir)


# ── Slide 28/37 · Football field ──────────────────────────────────────────────
def plot_football_field(CH, outdir=None):
    target, mkt = CH["target"], CH["market"]
    mc = CH["mc"]; mult = CH.get("multiplos", {})
    if target is None or mkt is None:
        return {"skip": "faltan target (m6) o mercado (m6)"}
    rows = [{"label": "DCF determinístico", "lo": target*0.95, "hi": target*1.05}]
    if mc.get("p25") and mc.get("p75"):
        rows.append({"label": "Monte Carlo (P25–P75)", "lo": mc["p25"], "hi": mc["p75"]})
    ebitda_proj = (CH.get("ebitda") or {}).get("proj") or []
    ebitda_fwd  = ebitda_proj[0] if ebitda_proj else None
    net_debt_ff = (CH.get("dcf") or {}).get("net_debt") or 0
    shares_ff   = CONFIG["shares_mm"]
    # Fix (auditoria jul-09): "or 1522.0" hardcodeado -> CH["ccl"] ya tiene fallback
    # propio a m1.ccl (campo requerido, siempre vivo) desde el CH builder.
    ccl_ff      = CH.get("ccl")
    # Rango de peers calculado dinamicamente desde static_inputs['peers'] (excluye
    # la fila "ALUAR (implied)") -- el hardcode anterior (5.4x-9.0x, comentado
    # como "Constellium min, Chalco max") estaba mal etiquetado: el minimo real
    # del peer group es Rusal (5.1x), no Constellium (5.4x). Si el dataset de
    # peers cambia, este rango ahora se recalcula solo.
    _peers_static = (CH.get("static") or {}).get("peers") or {}
    _peers_ev = [v for n, v in zip(_peers_static.get("names", []), _peers_static.get("ev_ebitda", []))
                 if "ALUAR" not in n.upper()]
    if ebitda_fwd and ebitda_fwd > 0 and mult.get("ev_ebitda_fy25") and _peers_ev:
        peers_lo, peers_hi = min(_peers_ev), max(_peers_ev)
        px_lo = max(0, peers_lo * ebitda_fwd - net_debt_ff) / shares_ff * ccl_ff
        px_hi = max(0, peers_hi * ebitda_fwd - net_debt_ff) / shares_ff * ccl_ff
        rows.append({"label": f"Peers EV/EBITDA ({peers_lo:.1f}x–{peers_hi:.1f}x × FY26E)", "lo": px_lo, "hi": px_hi})
    # BUG detectado (auditoria jul-09): a diferencia del rango EV/EBITDA (arriba),
    # que se deriva dinamicamente de static_inputs["peers"], este rango P/E NO tiene
    # ningun dato de respaldo -- static_inputs.json no incluye multiplos P/E de
    # peers (solo EV/EBITDA). Es un supuesto sectorial generico (no espepecifico de
    # ALUAR ni de ningun peer real) -- documentado explicitamente, no se fabrica un
    # peer P/E dataset que no existe (Prohibicion de Alucinacion, CLAUDE.md).
    peer_pe_lo, peer_pe_hi = 8.0, 14.0  # _is_fallback: rango sectorial generico, sin peer P/E dataset propio
    if mult.get("p_e_fy25") and mult["p_e_fy25"] > 0:
        pe_mkt         = mult["p_e_fy25"]
        peer_pe_median = (peer_pe_lo + peer_pe_hi) / 2.0
        pe_threshold   = 5.0 * peer_pe_median
        if pe_mkt > pe_threshold:
            rows.append({"label": "P/E (no comparable)", "lo": None, "hi": None, "not_comparable": True})
        else:
            mkt_px   = CH.get("market") or 0
            eps_impl = mkt_px / pe_mkt if pe_mkt > 0 and mkt_px > 0 else None
            if eps_impl:
                rows.append({"label": f"Múltiplos P/E genérico ({peer_pe_lo:.0f}x–{peer_pe_hi:.0f}x × EPS, sin peers propios)",
                             "lo": peer_pe_lo * eps_impl, "hi": peer_pe_hi * eps_impl,
                             "not_comparable": False})
    # Fix (Directiva Refactorizacion 3.C): escenario de sensibilidad
    # "convergencia ROIC=WACC" del valor terminal (M6 detecto que el FCFF
    # terminal mecanico implica reinversion negativa/inconsistente con
    # perpetuidad) -- se expone como rango adicional en el football field,
    # NO reemplaza el DCF determinístico (target/dictamen primarios, fila de
    # arriba, siguen siendo el mecanico -- decision explícita de usuario).
    esc_conv = CH.get("escenario_convergencia")
    if esc_conv and esc_conv.get("target_ars"):
        _t_conv = esc_conv["target_ars"]
        rows.append({"label": "DCF Convergencia ROIC=WACC (sensibilidad TV)",
                     "lo": _t_conv * 0.95, "hi": _t_conv * 1.05})
    # Fix (auditoria jul-12, Hallazgo EX-01 cuantificado): escenario "1T26
    # sostenido" (EBITDA real anualizado x multiplo EV/EBITDA propio del
    # modelo, ver M6 [7b] / static_inputs.json['resultados_1q26'], fuente
    # Cohen 2-jun-2026) -- responde directamente a la objecion de que el
    # hallazgo del 1T26 quedaba solo como texto sin ningun numero/grafico.
    esc_1q26 = CH.get("escenario_1q26_sostenido")
    if esc_1q26 and esc_1q26.get("target_ars"):
        _t_1q26 = esc_1q26["target_ars"]
        rows.append({"label": "1T26 Anualizado × Múltiplo Propio (supuesto fuerte)",
                     "lo": _t_1q26 * 0.95, "hi": _t_1q26 * 1.05})
    # Titulo dinamico: la version anterior afirmaba fijo "tres metodologias...
    # convergen por encima del mercado" -- FALSO contra los rangos reales: Peers
    # EV/EBITDA cae ENTERAMENTE por debajo del mercado, y Monte Carlo (P25-P75)
    # atraviesa el precio de mercado (P25 tipicamente por debajo). Se clasifica
    # cada metodologia real (excluye la fila "not_comparable") segun su
    # posicion real frente al mercado.
    _valid_rows = [r for r in rows if not r.get("not_comparable")]
    _above_labels = [r["label"] for r in _valid_rows if r["lo"] > mkt]
    _n_above = len(_above_labels)
    _n_below = sum(1 for r in _valid_rows if r["hi"] < mkt)
    _n_valid = len(_valid_rows)
    if _n_valid and _n_above == _n_valid:
        _titulo_ff = "Todas las metodologías independientes convergen por encima del mercado"
    elif _n_above >= 1 and _n_below == 0:
        _titulo_ff = "El DCF central y su rango de sensibilidad se ubican por encima del mercado"
    elif _n_above >= 1:
        # Fix (auditoria jul-12): el template anterior asumia que la
        # metodologia "por encima" del mercado era siempre el DCF central --
        # con el nuevo escenario de sensibilidad 1T26 (Hallazgo EX-01
        # cuantificado, ver M6/escenario_1q26_sostenido) eso dejo de ser
        # cierto: puede ser OTRA metodologia la que supera al mercado
        # mientras el DCF central sigue por debajo. Se nombra explicitamente
        # cual, en vez de asumir que siempre es el DCF central.
        if _n_above == 1:
            _titulo_ff = f"El mercado supera al DCF central; solo '{_above_labels[0]}' converge por encima"
        else:
            _titulo_ff = (f"El DCF central queda por debajo del mercado; {_n_above} de {_n_valid} "
                           f"metodologías convergen por encima ({', '.join(_above_labels)})")
    else:
        _titulo_ff = "Comparación de metodologías de valuación frente al precio de mercado actual"
    _bench = (CH.get("static") or {}).get("sellside_benchmark") or {}
    _bench_note = (f" Allaria: único target externo público verificado -- 1 solo broker, "
                   f"no representa consenso de mercado." if _bench.get("target_ars_2026") else "")
    fig, ax = scaffold(
        _titulo_ff,
        "Rango de valor intrínseco por método vs. precio de mercado actual",
        "ARS / acción", "DCF, Monte Carlo y múltiplos",
        "P/E excluido por utilidades no normalizadas (ver nota en informe)." + _bench_note)
    y = np.arange(len(rows))[::-1]
    ax_xmax_hint = target * 1.5
    for yi, r in zip(y, rows):
        if r.get("not_comparable"):
            bar_w = ax_xmax_hint * 0.30
            bar_l = max(0, ax_xmax_hint * 0.05)
            # Fix (auditoria ronda 4, ALTA): "#CCCCCC"/"#888888" eran colores
            # crudos fuera de MASTER_STYLE -- C["muted"] (gris institucional,
            # ya usado para de-enfasis en todo el proyecto) + C["ink"] de
            # borde cubren la misma semantica ("excluido, no comparable")
            # sin salir de la paleta.
            ax.barh(yi, bar_w, left=bar_l, height=0.5,
                    color=C["muted"], hatch="//", edgecolor=C["ink"], zorder=3, alpha=0.7)
        else:
            ax.barh(yi, r["hi"] - r["lo"], left=r["lo"], height=0.5, color=C["blue_lt"], zorder=3)
    ax.axvline(mkt, color=C["risk"], lw=LW["bold"], label=f"Mercado {mkt:,.0f} — precio a batir")
    ax.axvline(target, color=C["aluar"], lw=LW["bold"], ls="--", label=f"Target base {target:,.0f} — DCF central")
    if _bench.get("target_ars_2026"):
        # Sin `label=` (no se agrega a la leyenda -- una 4ta entrada
        # empujaba la caja de leyenda hasta pisar la nota al pie, detectado
        # por render real). Se anota directo sobre el grafico, ARRIBA del
        # area de barras (axes fraction en Y, dato en X), lejos de leyenda
        # y nota.
        _bx = _bench["target_ars_2026"]
        ax.axvline(_bx, color=C["ink"], lw=LW["medium"], ls=":", zorder=2)
        ax.annotate(f"{_bench['broker']} ({_bench.get('rating', 'N/D')})\nARS {_bx:,.0f}",
                    xy=(_bx, 0.98), xycoords=("data", "axes fraction"),
                    fontsize=SZ["annot"], color=C["ink"], ha="left", va="top")
        ax_xmax_hint = max(ax_xmax_hint, _bx * 1.05)
    # Fix (auditoria jul-12, bug detectado por inspeccion visual del PNG
    # regenerado -- objecion de usuario "no veo cambios en graficos"): la
    # barra nueva del escenario 1T26 (ARS ~2.000+) quedaba dibujada FUERA del
    # rango visible del eje X (limite fijo en base a target*1.5/Allaria*1.05,
    # ambos ~600-1200), por lo que la fila aparecia con etiqueta pero SIN
    # barra visible -- mismo patron ya resuelto para Allaria, replicado aca.
    if esc_1q26 and esc_1q26.get("target_ars"):
        ax_xmax_hint = max(ax_xmax_hint, esc_1q26["target_ars"] * 1.05)
    ax.set_yticks(y); ax.set_yticklabels([r["label"] for r in rows], fontsize=SZ["tick"])
    ax.set_xlim(0, ax_xmax_hint * 1.15)
    _legend_outside(ax, ncol=1)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: _money_fmt(v)))
    ax.grid(axis="y", visible=False); _clean_y(ax)
    # Fix (Directiva 4.C -- Inferencia Narrativa Desconectada / slide 37):
    # se exponen n_above/n_valid/titulo en el dict de retorno (ademas de
    # png/svg/pdf) para que update_presentation_v10.py pueda leer el MISMO
    # conteo que ya calculo este grafico (via figures_meta.json) al armar el
    # texto "N de N metodologías..." de la slide 37 -- una sola fuente de
    # verdad para el conteo, en vez de reimplementar la logica de
    # clasificacion above/below una segunda vez en el generador de PPTX
    # (que podria divergir silenciosamente de lo que el grafico realmente
    # muestra, el mismo patron de bug ya cazado en otras rondas).
    return {**export(fig, "s37_football_field", outdir),
            "n_above": _n_above, "n_valid": _n_valid, "titulo": _titulo_ff}


# ── Slide 33 · Sensibilidad WACC × g (heatmap) ────────────────────────────────
def _scaffold_matrix(title, subtitle, unit, source, note_box_title, note_box_lines, note=""):
    """Lienzo para matrices/heatmaps: heatmap a la izquierda + cuadro tecnico
    explicativo por fuera del lienzo del heatmap (columna derecha) -- para
    que ninguna matriz quede huerfana sin interpretacion (contrato
    S22/S28/S29 de .claude/rules/20_financial_model_gate.md)."""
    apply_aluar_theme()
    fig = plt.figure(figsize=FIGSIZE)
    fig.text(MARG["left"], TITLE_Y, title, ha="left", va="top",
             fontsize=SZ["title"], fontweight="bold", color=C["navy"])
    fig.text(MARG["left"], SUB_Y, subtitle, ha="left", va="top",
             fontsize=SZ["subtitle"], color=C["muted"])
    if unit:
        fig.text(MARG["right"], SUB_Y, unit, ha="right", va="top",
                 fontsize=SZ["subtitle"], style="italic", color=C["muted"])
    foot = f"Fuente: {source}.  Elaboración propia - {_TODAY}."
    if note:
        foot += f"  Nota: {note}"
    fig.text(MARG["left"], SRC_Y, foot, ha="left", va="bottom",
             fontsize=SZ["source"], color=C["muted"])
    import matplotlib.gridspec as _gs_mtx
    from matplotlib.patches import Rectangle as _Rect_note
    spec = _gs_mtx.GridSpec(1, 2, figure=fig, left=MARG["left"], right=MARG["right"],
                            top=MARG["top"] - 0.03, bottom=MARG["bottom"] + 0.02,
                            width_ratios=[2.35, 1], wspace=0.30)
    ax = fig.add_subplot(spec[0, 0])
    ax_note = fig.add_subplot(spec[0, 1]); ax_note.axis("off")
    ax_note.text(0.08, 0.93, note_box_title, transform=ax_note.transAxes, ha="left", va="top",
                 fontsize=SZ["annot"] + 0.5, fontweight="bold", color=C["navy"])
    ax_note.text(0.08, 0.82, "\n".join(note_box_lines), transform=ax_note.transAxes,
                 ha="left", va="top", fontsize=SZ["annot"] - 1, color=C["ink"], linespacing=1.8)
    ax_note.add_patch(_Rect_note((0.02, 0.02), 0.96, 0.96, transform=ax_note.transAxes,
                       fill=False, edgecolor=C["muted"], lw=LW["light"], zorder=1))
    return fig, ax


def plot_sensitivity_wacc_g(CH, outdir=None):
    fcff = (CH["fcff"]["values"] or [])[:]; wacc = CH["wacc_ref"]
    nd = CH["dcf"]["net_debt"]
    if not fcff or wacc is None or nd is None:
        return {"skip": "faltan FCFF/WACC/deuda neta"}
    shares = CONFIG["shares_mm"]
    ccl    = CH.get("ccl")
    fcff_t = fcff[-1]
    # BUG detectado (auditoria jul-09, SHIPPEADO): g_base estaba hardcodeado en 0.020
    # -- coincidia con m5_out.json["g_terminal"] vigente (2.0%) por casualidad, pero
    # si el modelo cambiara su g terminal, esta grilla (y su celda "caso base")
    # quedarian silenciosamente desincronizadas del resto del deck. Ahora lee
    # CH["dcf"]["g_terminal"] (agregado al CH builder), unica fuente de verdad.
    g_base = CH["dcf"].get("g_terminal")
    if ccl is None or g_base is None:
        return {"skip": "falta CCL (m1/m6) o g_terminal (m5)"}
    n_sides = 3
    g_step  = 0.010
    w_step  = 0.010
    gs    = np.array([g_base + (k - n_sides) * g_step for k in range(2 * n_sides + 1)])
    waccs = np.array([wacc   + (k - n_sides) * w_step for k in range(2 * n_sides + 1)])
    i_base = n_sides
    j_base = n_sides
    grid = np.zeros((len(gs), len(waccs)))
    # Fix (auditoria ronda 4, ALTA): antes las celdas que activaban el guard
    # WACC>g+2% colapsaban silenciosamente a TV=0 (precio degenerado, ~1-2%
    # del valor de las celdas vecinas) sin ninguna senal visual -- a
    # diferencia de plot_football_field, que SI hachura/grisea su celda "no
    # comparable". Se marca cada celda invalida para excluirla del color-scale
    # y anotarla como "N/A" en vez de un numero enganosamente pequeño.
    invalid = np.zeros((len(gs), len(waccs)), dtype=bool)
    for i, gv in enumerate(gs):
        for j, wv in enumerate(waccs):
            pv = sum(fcff[k]/(1+wv)**(k+0.5) for k in range(len(fcff)))  # H-04: Mid-Year Convention
            # Guard con buffer +0.02, MISMO criterio ya aprobado en M7/M9
            # (wacc_sim > g_sim + 0.02) -- sin buffer, celdas con wacc-g~0.0001
            # explotaban a >500.000 ARS/accion (bug real detectado visualmente).
            if wv > gv + 0.02:
                tv = fcff_t*(1+gv)/(wv-gv)
            else:
                tv = 0
                invalid[i, j] = True
            ev = pv + tv/(1+wv)**(len(fcff)-0.5)
            grid[i, j] = (ev-nd)/shares*ccl
    box_lines = [
        f"Cada celda: precio objetivo (ARS)",
        f"si WACC/g varían ±{n_sides} pasos de",
        f"{w_step:.1%}/{g_step:.1%} respecto al caso base.",
        "",
        "Recuadro negro = caso base",
        f"(g={g_base:.1%}, WACC del modelo).",
        "",
        "Verde = mayor valor; rojo =",
        "menor valor (escala relativa",
        "al rango de esta grilla).",
        "",
        "Gris/N.A. = WACC≤g+2%: Gordon",
        "Growth no definido en esa celda.",
    ]
    # Titulo dinamico: el literal fijo anterior ("target robusto") es FALSO
    # contra la propia grilla -- el precio base (1.352) se mueve +32%/-21%
    # con solo +-1pp de WACC, comportamiento esperado de Gordon Growth
    # (TV ~ 1/(WACC-g)) pero lo opuesto de "robusto". Se mide el swing
    # real de los 4 vecinos inmediatos al caso base y se rotula en consecuencia.
    base_val = grid[i_base, j_base]
    _vecinos = []
    # Fix (auditoria ronda 4, ALTA): excluir vecinos invalidos (guard
    # WACC>g+2% disparado, TV colapsada a 0) del calculo de swing -- si no,
    # una celda degenerada inflaria artificialmente el swing reportado.
    if j_base + 1 < len(waccs) and not invalid[i_base, j_base + 1]: _vecinos.append(grid[i_base, j_base + 1])
    if j_base - 1 >= 0 and not invalid[i_base, j_base - 1]:         _vecinos.append(grid[i_base, j_base - 1])
    if i_base + 1 < len(gs) and not invalid[i_base + 1, j_base]:    _vecinos.append(grid[i_base + 1, j_base])
    if i_base - 1 >= 0 and not invalid[i_base - 1, j_base]:         _vecinos.append(grid[i_base - 1, j_base])
    max_swing = max((abs(v / base_val - 1) for v in _vecinos), default=0) if base_val else 0
    if max_swing < 0.15:
        titulo_sens = "El target es robusto a un rango razonable de WACC y crecimiento"
    elif max_swing < 0.35:
        titulo_sens = f"El target es moderadamente sensible al spread WACC−g (±{max_swing:.0%} ante ±1pp)"
    else:
        titulo_sens = f"El target es altamente sensible al spread WACC−g: ±1pp lo mueve hasta ±{max_swing:.0%}"
    fig, ax = _scaffold_matrix(
        titulo_sens,
        "Precio objetivo (ARS) según sensibilidad WACC × g terminal",
        "ARS / acción", "sensibilidad (Gordon Growth, Mid-Year Convention)",
        "Cómo leer esta matriz", box_lines,
        f"celda marcada = caso base (g = {g_base:.1%}, WACC = modelo).")
    # Fix (auditoria ronda 4, ALTA): las celdas invalidas (guard disparado) se
    # enmascaran (NaN) para el color-scale -- ya no comprimen el rango de
    # color de las celdas legitimas, y se pintan de gris (set_bad) en vez de
    # heredar un rojo/verde enganoso del colormap RdYlGn.
    grid_masked = np.where(invalid, np.nan, grid)
    cmap_sens = plt.get_cmap("RdYlGn").copy()
    cmap_sens.set_bad(color=C["muted"])
    im = ax.imshow(grid_masked, cmap=cmap_sens, aspect="auto")
    ax.set_xticks(range(len(waccs))); ax.set_xticklabels([f"{w:.1%}" for w in waccs], fontsize=SZ["tick"]-1)
    ax.set_yticks(range(len(gs))); ax.set_yticklabels([f"{g:.1%}" for g in gs], fontsize=SZ["tick"]-1)
    ax.set_xlabel("WACC", fontsize=SZ["axis"]); ax.set_ylabel("g terminal", fontsize=SZ["axis"])
    _grid_max_valid = np.nanmax(grid_masked)
    for i in range(len(gs)):
        for j in range(len(waccs)):
            if invalid[i, j]:
                ax.text(j, i, "N/A\n(WACC≤g+2%)", ha="center", va="center",
                        fontsize=6, color=C["ink"], style="italic")
                continue
            color_txt = "white" if abs(grid[i,j]) > _grid_max_valid*0.6 else C["ink"]
            fw = "bold" if (i == i_base and j == j_base) else "normal"
            ax.text(j, i, f"{grid[i,j]:,.0f}", ha="center", va="center",
                    fontsize=7, color=color_txt, fontweight=fw)
    from matplotlib.patches import Rectangle as _Rect
    ax.add_patch(_Rect((j_base - 0.47, i_base - 0.47), 0.94, 0.94,
                       fill=False, edgecolor=C["ink"], lw=LW["heavy"], zorder=5))
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="ARS / acción (gris = WACC≤g+2%, no definido)", shrink=0.85, pad=0.02)
    return export(fig, "s33_sensitivity_wacc_g", outdir)


# ── Slide 34 · Monte Carlo — distribución ─────────────────────────────────────
def plot_monte_carlo(CH, outdir=None):
    mc = CH["mc"]
    if mc.get("sample") is None or mc.get("p50") is None:
        return {"skip": "falta mc_results.npy o percentiles m7"}
    sims_all = np.asarray(mc["sample"])
    # Los escenarios con equity truncado a 0 (Merton 1974, call-option) son una
    # masa puntual de insolvencia, no parte de la distribucion continua de
    # precios -- mezclarlos en el mismo histograma genera un pico artificial
    # en el primer bin. Se excluyen del histograma y se reportan aparte,
    # calculado dinamicamente desde el propio vector de simulaciones (no
    # hardcodeado).
    pct_truncado = float(np.mean(sims_all <= 0)) if len(sims_all) else 0.0
    sims = sims_all[sims_all > 0]
    lo, hi = np.percentile(sims, [1, 99]); sims = sims[(sims >= lo) & (sims <= hi)]
    nota_trunc = (f"{pct_truncado:.1%} de simulaciones truncadas a 0 (insolvencia, equity "
                  f"como call option Merton 1974) — excluidas del histograma."
                  if pct_truncado > 0 else "")
    fig, ax = scaffold(
        "La distribución estocástica confirma una asimetría favorable al alza",
        "Precio objetivo simulado — OU + saltos de Merton, t-Student (df=5)",
        "ARS / acción", "Monte Carlo (20.000 sims, seed=42)",
        f"P10/P50/P90 sobre simulaciones válidas (WACC>g). {nota_trunc}")
    ax.hist(sims, bins=80, color=C["navy"], alpha=0.85, edgecolor="white", linewidth=0.2, zorder=3)
    for v, col, lab in [(mc["market"], C["risk"], f"Mercado {mc['market']:,.0f}"),
                        (mc["dcf_base"], C["aluar"], f"DCF base {mc['dcf_base']:,.0f}"),
                        (mc["p50"], C["value"], f"P50 {mc['p50']:,.0f}")]:
        if v is not None:
            ax.axvline(v, color=col, lw=LW["bold"], label=lab)
    for v in (mc.get("p10"), mc.get("p90")):
        if v is not None:
            ax.axvline(v, color=C["muted"], lw=LW["light"], ls=":")
    ax.set_ylabel("Frecuencia", fontsize=SZ["axis"])
    _legend_outside(ax, ncol=3)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: _money_fmt(v)))
    ax.grid(axis="x", visible=False); _clean_y(ax)
    return export(fig, "s34_monte_carlo", outdir)


# ── Slide 34b · Trayectorias estocásticas (re-simuladas desde params) ─────────
def plot_stochastic_paths(CH, outdir=None):
    ou, mj = CH.get("ou"), CH.get("merton"); mkt = CH["market"]
    if not ou or mkt is None:
        return {"skip": "faltan ou_params (m9) o precio mercado"}
    rng = np.random.default_rng(CONFIG["seed"]); T, npaths = 252, 60
    kappa, theta, sigma = ou["kappa"], ou["theta"], ou["sigma"]
    dt = 1/252; X = np.full((T+1, npaths), np.log(mkt))
    # Fix (auditoria ronda 4, MEDIA): mj (m9_out.json['merton_params'], real)
    # se leia pero nunca se usaba -- la simulacion era OU puro pese a que
    # CLAUDE.md exige mantener Merton Jump-Diffusion + OU como extension
    # conjunta. Se agrega el salto compuesto de Poisson (vectorizado sobre
    # las 60 sendas por paso) cuando mj esta disponible; si no, cae a OU
    # puro (mismo comportamiento de antes) y el titulo lo refleja.
    _tiene_jumps = bool(mj and all(k in mj for k in ("lambda_j", "mu_j", "sigma_j")))
    for t in range(1, T+1):
        dW = sigma*np.sqrt(dt)*rng.standard_normal(npaths)
        if _tiene_jumps:
            n_j = rng.poisson(lam=mj["lambda_j"]*dt, size=npaths)
            jump = np.where(n_j > 0, rng.normal(n_j*mj["mu_j"], mj["sigma_j"]*np.sqrt(np.maximum(n_j, 1))), 0.0)
        else:
            jump = 0.0
        X[t] = X[t-1] + kappa*(theta-X[t-1])*dt + dW + jump
    paths = np.exp(X)
    _subtitulo_proc = ("Trayectorias Ornstein-Uhlenbeck + saltos de Merton (1 año, 60 sendas), calibrado a ALUA.BA histórico"
                        if _tiene_jumps else
                        "Trayectorias Ornstein-Uhlenbeck (1 año, 60 sendas), calibrado a ALUA.BA histórico")
    fig, ax = scaffold(
        "El proceso OU calibrado revierte hacia el precio de equilibrio de largo plazo",
        _subtitulo_proc,
        "ARS / acción", "OU + Merton JD calibrados (m9_out.json)" if _tiene_jumps else "OU calibrado (Euler-Maruyama)",
        f"θ (media LP) ≈ {np.exp(theta):,.0f} ARS; vida media ≈ {ou.get('half_life_days','?')} d.")
    x = np.arange(T+1)
    for i in range(npaths):
        ax.plot(x, paths[:, i], color=C["blue_lt"], alpha=0.25, lw=LW["hairline"])
    ax.plot(x, paths.mean(axis=1), color=C["navy"], lw=LW["heavy"], label="Media simulada")
    ax.axhline(np.exp(theta), color=C["aluar"], lw=LW["regular"], ls=":",
               label="θ — nivel de reversión de largo plazo")
    ax.axhline(mkt, color=C["risk"], lw=LW["medium"], ls="--", label="Precio actual")
    ax.set_xlabel("Días", fontsize=SZ["axis"])
    _legend_outside(ax, ncol=3)
    ax.grid(axis="x", visible=False); _clean_y(ax, money=True)
    return export(fig, "s34b_stochastic_paths", outdir)


# ── Slide 36 · VaR / CVaR ─────────────────────────────────────────────────────
def plot_var_cvar(CH, outdir=None):
    mc = CH["mc"]; var = CH.get("var")
    if mc.get("sample") is None:
        return {"skip": "falta mc_results.npy"}
    sims = np.asarray(mc["sample"])
    # VaR/CVaR se calculan sobre el sample COMPLETO (incluye el piso de insolvencia
    # Merton) -- es correcto que VaR99~=0 refleje ese ~4% de escenarios de wipeout,
    # el propio pie de nota ya lo explica ("VaR99≈0 = floor Merton").
    p5, p1 = np.percentile(sims, [5, 1])
    cvar = sims[sims <= p5].mean() if (sims <= p5).any() else p5
    # BUG detectado (auditoria jul-09, SHIPPEADO): el histograma usaba el MISMO
    # trim [0.5,99.5] percentil sobre "sims" crudo -- como el pileup en $0 es 4.2%
    # (> 0.5%), el percentil 0.5 caia DENTRO del pileup (lo=0.0), asi que el trim no
    # excluia nada: el histograma mostraba un pico artificial de ~980 sims en $0,
    # mas alto que el pico real de la distribucion continua (confirmado visualmente
    # sobre el PNG regenerado). Mismo bug ya corregido en s34_monte_carlo (jul-05)
    # pero nunca propagado a este grafico -- se excluye el pileup SOLO del
    # histograma (las estadisticas de arriba usan el sample completo, correctas).
    pct_truncado_s36 = float((sims <= 0).mean())
    sims_hist = sims[sims > 0]
    lo, hi = np.percentile(sims_hist, [0.5, 99.5]); s = sims_hist[(sims_hist >= lo) & (sims_hist <= hi)]
    # Titulo dinamico: el VaR95 puede estar muy lejos del precio de mercado
    # (el piso de insolvencia de Merton trunca escenarios extremos a ~0) -- el
    # titulo anterior afirmaba de forma fija "el VaR esta proximo a la
    # cotizacion actual", falso cuando la caida implicita es severa (bug
    # detectado visualmente: VaR95=48 vs mercado=994 es -95%, no "acotado").
    mkt = mc.get("market")
    loss95 = (mkt - p5) / mkt if mkt else None
    if loss95 is None:
        titulo = "Distribución del precio objetivo con cortes de Value at Risk y Expected Shortfall"
    elif loss95 <= 0.15:
        titulo = "Riesgo de cola acotado: el VaR 95% está próximo a la cotización actual"
    elif loss95 <= 0.40:
        titulo = f"Riesgo de cola moderado: el VaR 95% implica una caída de {loss95:.0%} vs. mercado"
    else:
        titulo = (f"Cola de riesgo severa: el VaR 95% implica una caída de {loss95:.0%} vs. mercado, "
                  f"por el piso de insolvencia del modelo Merton")
    fig, ax = scaffold(
        titulo,
        "Distribución del precio objetivo con cortes de Value at Risk y Expected Shortfall",
        "ARS / acción", "VaR histórico/paramétrico + Monte Carlo",
        f"CVaR = pérdida media condicional más allá del VaR 95%. VaR99≈0 = floor Merton (resp. limitada). "
        f"Histograma excluye {pct_truncado_s36:.1%} truncado a $0.")
    ax.hist(s, bins=80, color=C["navy"], alpha=0.8, edgecolor="white", linewidth=0.2, zorder=3)
    ax.axvline(p5, color=C["aluar"], lw=LW["bold"], label=f"VaR 95% = {p5:,.0f}")
    ax.axvline(p1, color=C["risk"], lw=LW["bold"], label=f"VaR 99% = {p1:,.0f}")
    ax.axvline(cvar, color=C["risk"], lw=LW["regular"], ls="--", label=f"CVaR 95% = {cvar:,.0f}")
    if mc.get("market"):
        ax.axvline(mc["market"], color=C["ink"], lw=LW["medium"], ls=":", label=f"Mercado {mc['market']:,.0f}")
    ax.set_ylabel("Frecuencia", fontsize=SZ["axis"])
    _legend_outside(ax, ncol=2)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: _money_fmt(v)))
    ax.grid(axis="x", visible=False); _clean_y(ax)
    return export(fig, "s36_var_cvar", outdir)


# ── Slide 45 · Matriz de correlación ──────────────────────────────────────────
def plot_correlation_matrix(CH, outdir=None):
    corr = CH.get("corr")
    if not corr:
        return {"skip": "falta matriz de correlación (m11 o static)"}
    labels = corr.get("labels"); M = np.asarray(corr.get("matrix"))
    box_lines = [
        "Correlación de Pearson (ρ)",
        "entre retornos diarios,",
        "rango −1 a +1.",
        "",
        "ρ baja o negativa entre ALUAR",
        "y otros activos favorece la",
        "diversificación de cartera.",
        "",
        "Interpretar en frecuencia",
        "semanal/mensual: el cepo",
        "distorsiona la serie diaria.",
    ]
    fig, ax = _scaffold_matrix(
        "Correlaciones diarias distorsionadas por el cepo; el ciclo real es de baja frecuencia",
        "Matriz de correlación de retornos diarios entre activos y drivers",
        "ρ (−1 a 1)", "retornos diarios",
        "Cómo leer esta matriz", box_lines,
        "interpretar en frecuencia semanal/mensual.")
    n = len(labels)
    im = ax.imshow(M, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=SZ["tick"])
    ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=SZ["tick"])
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=SZ["annot"],
                    color="white" if abs(M[i, j]) > 0.6 else C["ink"])
    ax.grid(False); fig.colorbar(im, ax=ax)
    return export(fig, "s45_correlation_matrix", outdir)


# ── Slide 25 · Rendimiento acumulado (Base 100) ───────────────────────────────
def plot_cum_returns(CH, outdir=None):
    s = (CH.get("static") or {}).get("cum_returns")
    if not s:
        return {"skip": "falta static_inputs['cum_returns'] (series ALUA/Merval/TXAR Base100)"}
    dates = s["dates"]
    # Titulo dinamico: refleja el ranking REAL de retornos (no se asume de antemano
    # cual activo lidera -- el titulo original afirmaba que ALUAR superaba al
    # Merval cuando en realidad el Merval lidera en ARS nominal).
    _rets_finales = {"ALUA.BA": s.get("alua", [None])[-1], "S&P Merval": s.get("merv", [None])[-1],
                      "TXAR.BA": s.get("txar", [None])[-1]}
    _rets_finales = {k: v for k, v in _rets_finales.items() if v is not None}
    _lider = max(_rets_finales, key=_rets_finales.get) if _rets_finales else "el mercado"
    _titulo = (f"{_lider} lidera el retorno acumulado en ARS nominal — la dolarización de "
               f"ingresos de ALUAR actúa como escudo cambiario" if _lider != "ALUA.BA" else
               "ALUAR supera al Merval en ARS: la dolarización de ingresos actúa como escudo cambiario")
    fig, ax = scaffold(
        _titulo,
        "Retorno acumulado Base 100 en ARS nominal — comparación relativa válida (misma moneda)",
        "Índice Base 100 ARS", "yfinance (ALUA.BA, ^MERV, TXAR.BA)",
        "Base 100 ARS: permite comparar rendimientos relativos entre activos denominados en pesos.")
    x = np.arange(len(dates))
    series_specs = [("alua", C["aluar"], "ALUA.BA", "heavy"),
                     ("merv", C["navy"], "S&P Merval", "regular"),
                     ("txar", C["muted"], "TXAR.BA (Ternium)", "regular")]
    for key, col, lab, lw_key in series_specs:
        if key in s:
            ax.plot(x, s[key], color=col, lw=LW[lw_key], label=lab)
    ax.axhline(100, color=C["ink"], lw=LW["thin"], ls="--", alpha=0.5)
    # Escala log: la dispersion entre ALUA/MERV/TXAR distorsiona la lectura en
    # escala lineal cuando un activo se despega fuertemente de los otros dos.
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(mtick.ScalarFormatter())
    ax.yaxis.set_minor_formatter(mtick.NullFormatter())
    # Etiqueta de dato final para las 3 series (antes solo ALUA la tenia).
    # Offset vertical escalonado por orden de valor final para evitar que las
    # etiquetas de series con valores finales cercanos (ALUA/TXAR) se solapen.
    _finales = [(key, col, lab, s[key][-1]) for key, col, lab, lw_key in series_specs if key in s and s[key]]
    _finales.sort(key=lambda t: t[3])
    # bbox blanco: cuando 2 series terminan con valores finales cercanos
    # (ALUA +2260% / TXAR +1949%) sus trazos zigzaguean cerca del borde
    # derecho y la propia LINEA (no solo la otra etiqueta) atraviesa el
    # texto -- el escalonado vertical evita colision etiqueta/etiqueta pero
    # no etiqueta/linea. Mismo patron de recuadro que el resto del proyecto.
    for _rank, (key, col, lab, last_v) in enumerate(_finales):
        ax.annotate(f"{lab}: {last_v-100:+.0f}%", xy=(x[-1], last_v),
                    xytext=(-10, 6 + _rank*13), textcoords="offset points", ha="right",
                    fontsize=SZ["annot"]-1, fontweight="bold", color=col, zorder=6,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    step = max(1, len(dates)//6)
    xt = list(x[::step])
    if xt[-1] != x[-1]:
        xt.append(x[-1])  # asegura que el ultimo anio (2026) siempre tenga tick
    ax.set_xticks(xt); ax.set_xticklabels([dates[i] for i in xt], fontsize=SZ["tick"])
    _legend_outside(ax, ncol=3)
    ax.grid(axis="x", visible=False); _clean_y(ax)
    return export(fig, "s25_cum_returns", outdir)


# -- Slide 7 · Top productores mundiales (barras, reemplaza mapa Plotly generico) --
def plot_top_producers_global(CH, outdir=None):
    s = (CH.get("static") or {}).get("market_share_pies")
    if not s or not s.get("global"):
        return {"skip": "falta static_inputs['market_share_pies']['global']"}
    d = s["global"]
    names = list(d.keys()); vals = list(d.values())
    order = sorted(range(len(vals)), key=lambda i: vals[i])
    names = [names[i] for i in order]; vals = [vals[i] for i in order]
    cols = [C["risk"] if "china" in n.lower() else C["blue_lt"] for n in names]
    # Concentracion de paises nombrados calculada dinamicamente (excluye el
    # residual "Otros") -- antes el footer decia "~79%" como literal fijo;
    # ahora se recalcula del propio dict si static_inputs.json cambia.
    _named_share = sum(v for n, v in zip(names, vals) if n.lower() != "otros")
    _n_named = sum(1 for n in names if n.lower() != "otros")
    fig, ax = scaffold(
        "China domina la oferta global de aluminio primario",
        "Participacion en la produccion mundial de aluminio primario por pais/bloque",
        "% del total mundial", "IAI / World Aluminium (static_inputs)",
        f"oligopolio geografico: {_n_named} paises concentran ~{_named_share:.0%} de la oferta global.")
    y = np.arange(len(names))
    bars = ax.barh(y, vals, color=cols, height=0.62, zorder=3)
    ax.bar_label(bars, fmt=lambda v: f"{v:.1%}", fontsize=SZ["annot"], fontweight="bold", padding=3)
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=SZ["tick"])
    ax.set_xlim(0, max(vals) * 1.20)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.grid(axis="y", visible=False); _clean_y(ax)  # eje Y es categorico -- pct=True rompia los nombres
    return export(fig, "s07_top_producers_global", outdir)


# ── Slide 27 · Market share regional (barras) ─────────────────────────────────
def plot_market_share_regional(CH, outdir=None):
    s = (CH.get("static") or {}).get("market_share_regional")
    if not s:
        return {"skip": "falta static_inputs['market_share_regional']"}
    names, share = s["names"], s["share"]
    order = np.argsort(share)
    names = [names[i] for i in order]; share = [share[i] for i in order]
    cols = [C["aluar"] if "ALUAR" in n.upper() else C["blue_lt"] for n in names]
    # Texto corregido: este dataset (market_share_regional) es la composicion del
    # ABASTECIMIENTO DOMESTICO argentino (ALUAR vs importaciones Asia/EEUU), no la
    # produccion de Latinoamerica -- Brasil no es una variable aca (esa comparacion
    # vive en market_share_pies['regional'], grafico s27_market_share_pies). El
    # titulo/nota anteriores describian el grafico equivocado (bug de copy cruzado).
    fig, ax = scaffold(
        "ALUAR domina el abastecimiento doméstico, con competencia de importados",
        "Composición del abastecimiento de aluminio primario en el mercado argentino",
        "% del total doméstico", "static_inputs (IAI / World Aluminium)",
        "las importaciones de Asia y EE.UU. presionan el pricing power local.")
    y = np.arange(len(names))
    bars = ax.barh(y, share, color=cols, height=0.62, zorder=3)
    ax.bar_label(bars, fmt=lambda v: f"{v:.0%}", fontsize=SZ["annot"], fontweight="bold", padding=3)
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=SZ["tick"])
    ax.set_xlim(0, max(share) * 1.20)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.grid(axis="y", visible=False); _clean_y(ax)  # eje Y es categorico -- pct=True rompia los nombres
    return export(fig, "s27_market_share_regional", outdir)


# ── Slide 27 · Share global y regional (barras horizontales dobles) ──────────
def plot_market_share_pies(CH, outdir=None):
    s = (CH.get("static") or {}).get("market_share_pies")
    if not s:
        return {"skip": "falta static_inputs['market_share_pies']"}

    import matplotlib.gridspec as _gs
    # Lider regional calculado dinamicamente -- el titulo anterior afirmaba fijo
    # "ALUAR lidera en Latam", pero static_inputs['market_share_pies']['regional']
    # muestra a Brasil (Alumar+CBA) con 48% vs ALUAR 38% (Brasil lidera, no ALUAR).
    _reg = s.get("regional") or {}
    _reg_leader_raw = max(_reg, key=_reg.get) if _reg else None
    _reg_leader = (_reg_leader_raw.split("\n")[0] if _reg_leader_raw else "la región")
    _titulo_pies = ("China domina la oferta global; ALUAR lidera en Latam"
                    if _reg_leader_raw and "ALUAR" in _reg_leader_raw.upper()
                    else f"China domina la oferta global; {_reg_leader} lidera en Latam, ALUAR en 2do lugar")
    fig = plt.figure(figsize=FIGSIZE); fig.subplots_adjust(**MARG)
    fig.text(MARG["left"], TITLE_Y, _titulo_pies,
             fontsize=SZ["title"], fontweight="bold", color=C["navy"], va="top")
    fig.text(MARG["left"], SUB_Y,
             "Participación en producción de aluminio primario: global (izq.) y Latinoamérica (der.)",
             fontsize=SZ["subtitle"], color=C["muted"], va="top")
    fig.text(MARG["left"], SRC_Y, f"Fuente: IAI / World Aluminium (static_inputs). Elaboración propia - {_TODAY}.",
             fontsize=SZ["source"], color=C["muted"])

    spec = _gs.GridSpec(1, 2, figure=fig, left=MARG["left"], right=MARG["right"],
                        top=MARG["top"]-0.04, bottom=MARG["bottom"], wspace=0.35)

    for col_idx, (key, title) in enumerate([("global", "Producción Mundial"), ("regional", "Latam")]):
        ax = fig.add_subplot(spec[0, col_idx])
        d = s[key]
        names = list(d.keys()); vals = list(d.values())
        order = sorted(range(len(vals)), key=lambda i: vals[i], reverse=True)
        names = [names[i] for i in order]; vals = [vals[i] for i in order]
        cols = []
        for n in names:
            if "ALUAR" in n.upper():
                cols.append(C["aluar"])
            elif "CHINA" in n.upper() or "HONGQIAO" in n.upper() or "CHALCO" in n.upper():
                cols.append(C["risk"])
            else:
                cols.append(C["blue_lt"])
        y = np.arange(len(names))[::-1]
        bars = ax.barh(y, vals, color=cols, height=0.58, zorder=3)
        ax.bar_label(bars, fmt=lambda v: f"{v:.0%}", fontsize=SZ["annot"]-0.5, fontweight="bold", padding=2)
        ax.set_yticks(y); ax.set_yticklabels(names, fontsize=SZ["tick"]-0.5)
        ax.set_title(title, fontsize=SZ["subtitle"], fontweight="bold", color=C["navy"], pad=6)
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax.set_xlim(0, max(vals)*1.30)
        ax.spines[["top","right"]].set_visible(False)
        ax.tick_params(length=0); ax.grid(axis="y", visible=False)

    return export(fig, "s27_market_share_pies", outdir)


# ── Slide 28/29 · Múltiplos comparables (peers) ───────────────────────────────
def plot_peer_multiples(CH, outdir=None):
    s = (CH.get("static") or {}).get("peers")
    if not s:
        return {"skip": "falta static_inputs['peers'] (múltiplos comparables)"}
    names, evebitda = list(s["names"]), list(s["ev_ebitda"])
    multiplos = CH.get("multiplos") or {}
    mkt_cap   = multiplos.get("mkt_cap_usdmm")
    net_debt  = (CH.get("dcf") or {}).get("net_debt")
    ebitda_h  = (CH.get("ebitda") or {}).get("hist") or []
    ebitda_ltm = ebitda_h[-1] if ebitda_h else None
    if mkt_cap and net_debt and ebitda_ltm and ebitda_ltm > 0:
        mkt_ev = mkt_cap + net_debt
        aluar_mkt_multiple = round(mkt_ev / ebitda_ltm, 1)
    else:
        aluar_mkt_multiple = multiplos.get("ev_ebitda_fy25")  # fallback
    # Posicion de ALUAR vs el RANGO real de peers, calculada ANTES de sobre-
    # escribir su entrada -- el titulo anterior afirmaba fijo "cotiza al
    # multiplo del sector", pero el valor de mercado (~14x en esta corrida)
    # esta muy por encima del peer mas caro (Chalco 9.0x): ALUAR cotiza con
    # PREMIO, no "al multiplo". Titulo dinamico segun donde cae el numero real.
    _peer_idx = next((i for i, n in enumerate(names) if "ALUAR" in n.upper()), None)
    _peer_vals = [v for i, v in enumerate(evebitda) if i != _peer_idx]
    _peer_lo, _peer_hi = (min(_peer_vals), max(_peer_vals)) if _peer_vals else (None, None)
    for i, n in enumerate(names):
        if "ALUAR" in n.upper():
            evebitda[i] = aluar_mkt_multiple
            names[i] = "ALUAR\n(mercado)"
            break
    cols = [C["aluar"] if "ALUAR" in n.upper() else C["blue_lt"] for n in names]
    if aluar_mkt_multiple is not None and _peer_hi is not None and aluar_mkt_multiple > _peer_hi:
        _titulo_pm = (f"ALUAR cotiza con premio sobre el sector ({aluar_mkt_multiple:.1f}x vs. "
                      f"{_peer_hi:.1f}x del peer más caro): el upside exige ejecución, no re-rating")
        _nota_pm = "el mercado ya paga una prima frente al sector; el upside remanente depende del DCF."
    elif aluar_mkt_multiple is not None and _peer_lo is not None and aluar_mkt_multiple < _peer_lo:
        _titulo_pm = (f"ALUAR cotiza con descuento sobre el sector ({aluar_mkt_multiple:.1f}x vs. "
                      f"{_peer_lo:.1f}x del peer más barato): el upside proviene del DCF, no del re-rating")
        _nota_pm = "el descuento táctico explica el upside."
    else:
        _titulo_pm = f"ALUAR cotiza dentro del rango del sector ({aluar_mkt_multiple:.1f}x): el upside proviene del DCF"
        _nota_pm = "el múltiplo de mercado converge al del sector; el upside remanente proviene del DCF."
    # Fix (auditoria ronda 4, ALTA): "SEC / Damodaran / yfinance" no es la
    # fuente real del dataset de peers -- static_inputs.json['peers']['_fuente']
    # dice explicitamente "Bloomberg/Reuters consenso Q1-2026".
    fig, ax = scaffold(
        _titulo_pm,
        "EV/EBITDA LTM de mercado: (Cap. Bursátil + Deuda Neta) / EBITDA FY2025",
        "EV/EBITDA (x)", "Bloomberg/Reuters consenso Q1-2026 (static_inputs.json)",
        _nota_pm)
    y = np.arange(len(names))[::-1]
    bars = ax.barh(y, evebitda, color=cols, height=0.62, zorder=3)
    ax.bar_label(bars, fmt=lambda v: f"{v:.1f}x", fontsize=SZ["annot"], fontweight="bold", padding=3)
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=SZ["tick"])
    ax.set_xlim(0, max(evebitda) * 1.20)
    ax.grid(axis="y", visible=False); _clean_y(ax)
    return export(fig, "s28_peer_multiples", outdir)


# ── Slide 30 · Diagnóstico de retornos (distribución t-Student) ───────────────
def plot_returns_diagnostics(CH, outdir=None):
    s = (CH.get("static") or {}).get("alua_returns")
    if not s:
        return {"skip": "falta static_inputs['alua_returns'] (retornos diarios ALUA)"}
    r = np.asarray(s)
    df, loc, scale = _st.t.fit(r)
    x = np.linspace(np.percentile(r, 1), np.percentile(r, 99), 300)
    fig, ax = scaffold(
        "Los retornos exhiben colas pesadas: se rechaza la normalidad (t-Student)",
        "Distribución de retornos diarios de ALUA.BA vs ajuste t-Student y Normal",
        "retorno diario", "retornos diarios ALUA.BA",
        "df finito justifica el Monte Carlo con t-Student (test JB/KS en informe).")
    ax.hist(r, bins=80, density=True, color=C["navy"], alpha=0.45, label="Empírico")
    ax.plot(x, _st.t.pdf(x, df, loc, scale), color=C["aluar"], lw=LW["bold"], label=f"t-Student (df={df:.1f})")
    ax.plot(x, _st.norm.pdf(x, r.mean(), r.std()), color=C["muted"], lw=LW["regular"], ls="--", label="Normal")
    ax.set_xlabel("Retorno diario", fontsize=SZ["axis"])
    _legend_outside(ax, ncol=3)
    ax.grid(axis="x", visible=False); _clean_y(ax)
    return export(fig, "s30_returns_diagnostics", outdir)


# ── Slide 39 · Frontera eficiente / rol en cartera ────────────────────────────
def plot_efficient_frontier(CH, outdir=None):
    s = (CH.get("static") or {}).get("frontier")
    if not s:
        return {"skip": "falta static_inputs['frontier'] (vol/ret de cartera, vía M12)"}
    # Fix (auditoria ronda 5, ALTA -- data lineage): "ALUA/TXAR/GGAL/BMA"
    # estaba hardcodeado en la nota pese a que 2 lineas mas abajo el codigo
    # SI lee fa=CH.get("frontier_assets") dinamicamente para los marcadores
    # individuales -- si yfinance fallara para GGAL o BMA (o el dropna
    # recortara el sample), la nota seguiria prometiendo un universo de 4
    # activos que ya no seria el real. Se construye desde
    # frontier_assets["names"] (m12_out.json["assets"], los que
    # efectivamente sobrevivieron el fetch) y se avisa si algo quedo afuera.
    _fa_note = CH.get("frontier_assets") or {}
    _names_note = _fa_note.get("names") or []
    _nota_frontier = f"universo optimizado {'/'.join(_names_note)}; misma ventana que la nube de puntos."
    if _fa_note.get("excluidos"):
        _nota_frontier += f" Excluidos por falla de datos: {'/'.join(_fa_note['excluidos'])}."
    fig, ax = scaffold(
        "Incorporar ALUAR mejora el perfil riesgo-retorno de la cartera argentina",
        "Frontera eficiente media-varianza y posición individual de ALUA y TXAR",
        "retorno vs. volatilidad anual (%)", "media-varianza (M12)",
        _nota_frontier)
    sharpe_raw = np.array(s.get("sharpe") or [], dtype=float)
    sharpe_c = np.clip(np.nan_to_num(sharpe_raw, nan=0.0, posinf=5.0, neginf=-5.0), -5.0, 5.0)
    sc = ax.scatter(np.array(s["vol"])*100, np.array(s["ret"])*100, c=sharpe_c,
                    cmap="viridis", alpha=0.4, s=8)
    fig.colorbar(sc, ax=ax, label="Sharpe ratio", shrink=0.75, pad=0.02)
    # Puntos individuales: ALUA + TXAR (misma metodologia/ventana que la nube,
    # via m12_out.json["assets"]/mu_anual/vol_anual -- antes solo se graficaba
    # ALUA) + Merval como referencia externa (m8_out.json, ventana distinta,
    # aclarado en la nota de fuente para no mezclar metodologias sin avisar).
    # Solo se grafican activos con la MISMA metodologia/ventana que la nube
    # de puntos (m12_out.json, universo optimizado) -- se evaluo agregar
    # Merval (m8_out.json) como referencia externa, pero su ventana produce
    # un retorno anualizado (~130%) en una escala no comparable que aplasta
    # la nube de puntos real (bug detectado visualmente tras el primer
    # intento); se descarta para no mezclar metodologias sin avisar de forma
    # enganosa en el propio grafico.
    fa = CH.get("frontier_assets") or {}
    names = fa.get("names") or []; mu = fa.get("mu_anual") or []; vol = fa.get("vol_anual") or []
    asset_pts = []
    for nm, m, v in zip(names, mu, vol):
        if nm in ("ALUA", "TXAR") and m is not None and v is not None:
            asset_pts.append((nm, v * 100, m * 100))
    if not asset_pts and s.get("alua"):
        asset_pts.append(("ALUA", s["alua"]["vol"] * 100, s["alua"]["ret"] * 100))
    asset_pts.sort(key=lambda t: t[2])
    marker_map = {"ALUA": ("^", C["aluar"]), "TXAR": ("D", C["navy"])}
    for rank, (nm, vx, ry) in enumerate(asset_pts):
        mk, col = marker_map.get(nm, ("o", C["ink"]))
        ax.scatter(vx, ry, color=col, s=160, marker=mk, zorder=5, label=f"{nm} (individual)",
                   edgecolor="white", linewidth=0.8)
        ax.annotate(nm, xy=(vx, ry), xytext=(8, -10 + rank * 14), textcoords="offset points",
                    ha="left", fontsize=SZ["annot"] - 1, fontweight="bold", color=col)
    ms = s.get("max_sharpe") or {}
    if ms.get("vol_anual") and ms.get("ret_anual"):
        ax.scatter(ms["vol_anual"]*100, ms["ret_anual"]*100, color=C["value"],
                   s=220, marker="*", zorder=6, label=f'Máx. Sharpe ({ms["sharpe"]:.2f})')
    ax.set_xlabel("Volatilidad anual (%)", fontsize=SZ["axis"])
    ax.set_ylabel("Retorno anual (%)", fontsize=SZ["axis"])
    _legend_outside(ax, ncol=2)
    return export(fig, "s39_efficient_frontier", outdir)


# ── Backup · Descomposicion de varianza (CCR + Indice Unico) ─────────────────
def plot_variance_decomposition(CH, outdir=None):
    """Descompone el riesgo del portafolio (extension Alexander/Sharpe/Bailey,
    aprobada en CLAUDE.md): (izq.) Component Contribution to Risk -- cuanto
    aporta cada activo a la varianza TOTAL del portafolio Max Sharpe, vs. su
    peso nominal; (der.) Modelo de Indice Unico (Sharpe, 1963) -- cuanto de la
    varianza de cada activo es riesgo de mercado (beta^2 * Var(MERV)) vs.
    idiosincratico. Responde con un numero, no con una afirmacion cualitativa,
    por que el optimizador pondera fuerte a ALUA."""
    pr = CH.get("portfolio_risk") or {}
    assets = pr.get("assets"); w = pr.get("weights_max_sharpe"); ccr = pr.get("ccr_max_sharpe")
    sid = pr.get("single_index")
    if not (assets and w and ccr):
        return {"skip": "faltan risk_contribution_max_sharpe/weights en m12_out.json"}

    alua_w, alua_ccr = w.get("ALUA"), ccr.get("ALUA")
    if alua_w is not None and alua_ccr is not None and alua_w > 0:
        _rel = (alua_ccr - alua_w) / alua_w
        _rol = ("diversifica: aporta menos riesgo del que pesa" if _rel < -0.05
                else "concentra: aporta más riesgo del que pesa" if _rel > 0.05
                else "aporta riesgo proporcional a su peso")
    else:
        _rol = "rol no determinado"
    _titulo = f"ALUA {_rol} en el portafolio de Máx. Sharpe"

    fig = plt.figure(figsize=FIGSIZE); fig.subplots_adjust(**MARG)
    fig.text(MARG["left"], TITLE_Y, _titulo, fontsize=SZ["title"], fontweight="bold",
             color=C["navy"], va="top")
    fig.text(MARG["left"], SUB_Y,
             "Component Contribution to Risk (izq.) y Modelo de Índice Único vs. Merval (der.)",
             fontsize=SZ["subtitle"], color=C["muted"], va="top")
    fig.text(MARG["right"], SUB_Y, "% de la varianza total (M12)", ha="right", va="top",
             fontsize=SZ["subtitle"], style="italic", color=C["muted"])
    fig.text(MARG["left"], SRC_Y,
             "Fuente: Elaboración propia en base a estados financieros de Aluar e información de mercado. "
             f"Elaboración propia - {_TODAY}.  Nota: CCR (Alexander/Sharpe/Bailey) suma 100% de la varianza "
             "del portafolio Máx. Sharpe; Índice Único (Sharpe, 1963) usa Merval como proxy de mercado.",
             fontsize=SZ["source"], color=C["muted"])

    import matplotlib.gridspec as _gs
    spec = _gs.GridSpec(1, 2, figure=fig, left=MARG["left"], right=MARG["right"],
                        top=MARG["top"]-0.04, bottom=MARG["bottom"], wspace=0.55)

    # Panel izquierdo: peso vs. contribucion al riesgo (CCR), portafolio Max Sharpe
    ax1 = fig.add_subplot(spec[0, 0])
    names = list(assets)
    weights_v = np.array([w.get(a, 0.0) for a in names])
    ccr_v = np.array([ccr.get(a, 0.0) for a in names])
    order = np.argsort(ccr_v)
    names_o = [names[i] for i in order]; weights_o = weights_v[order]; ccr_o = ccr_v[order]
    y = np.arange(len(names_o)); h = 0.36
    cols_w = [C["aluar"] if n == "ALUA" else C["blue_lt"] for n in names_o]
    cols_c = [C["risk"] if n == "ALUA" else C["muted"] for n in names_o]
    b1 = ax1.barh(y + h/2, weights_o, height=h, color=cols_w, zorder=3, label="Peso en el portafolio")
    b2 = ax1.barh(y - h/2, ccr_o, height=h, color=cols_c, zorder=3, alpha=0.85,
                  hatch="//", edgecolor="white", label="Contribución al riesgo (CCR)")
    ax1.bar_label(b1, fmt=lambda v: f"{v:.0%}", fontsize=SZ["annot"]-1, padding=2)
    ax1.bar_label(b2, fmt=lambda v: f"{v:.0%}", fontsize=SZ["annot"]-1, padding=2)
    ax1.set_yticks(y); ax1.set_yticklabels(names_o, fontsize=SZ["tick"])
    ax1.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    _xmax = max(float(weights_o.max()), float(ccr_o.max())) if len(weights_o) else 1.0
    ax1.set_xlim(0, _xmax * 1.35)
    ax1.set_title("Peso vs. riesgo aportado — Máx. Sharpe", fontsize=SZ["subtitle"],
                  fontweight="bold", color=C["navy"], pad=6)
    ax1.spines[["top", "right"]].set_visible(False); ax1.tick_params(length=0)
    ax1.grid(axis="y", visible=False)
    _legend_outside(ax1, ncol=1, y=-0.16)

    # Panel derecho: sistematico vs. idiosincratico (Indice Unico, Sharpe 1963)
    ax2 = fig.add_subplot(spec[0, 1])
    if sid:
        names2 = [n for n in names if n in sid]
        r2 = np.array([sid[n]["r2"] for n in names2])
        order2 = np.argsort(r2)
        names2_o = [names2[i] for i in order2]; r2_o = r2[order2]
        y2 = np.arange(len(names2_o))
        ax2.barh(y2, r2_o, height=0.58, color=C["navy"], zorder=3, label="Sistemático (β² × Var. mercado)")
        ax2.barh(y2, 1 - r2_o, height=0.58, left=r2_o, color=C["blue_lt"], zorder=3,
                 label="Idiosincrático (específico)")
        for yi, n, r in zip(y2, names2_o, r2_o):
            ax2.annotate(f"β={sid[n]['beta_mkt']:.2f}  R²={r:.0%}", xy=(1.02, yi), xytext=(4, 0),
                         textcoords="offset points", va="center", fontsize=SZ["annot"]-1,
                         color=C["ink"], fontweight="bold" if n == "ALUA" else "normal")
        ax2.set_yticks(y2); ax2.set_yticklabels(names2_o, fontsize=SZ["tick"])
        ax2.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
        ax2.set_xlim(0, 1.32)
        ax2.set_title(f"Sistemático vs. idiosincrático (β vs. {pr.get('market_proxy') or 'MERV'})",
                      fontsize=SZ["subtitle"], fontweight="bold", color=C["navy"], pad=6)
        ax2.spines[["top", "right"]].set_visible(False); ax2.tick_params(length=0)
        ax2.grid(axis="y", visible=False)
        _legend_outside(ax2, ncol=1, y=-0.16)
    else:
        ax2.text(0.5, 0.5, "Índice Único no disponible\n(falta Merval en el universo optimizado)",
                 ha="center", va="center", fontsize=SZ["annot"], color=C["muted"])
        ax2.axis("off")

    return export(fig, "s_variance_decomposition", outdir)


# ── Slide 53 · Sensibilidad LME × Volumen ─────────────────────────────────────
def plot_sensitivity_lme_vol(CH, outdir=None):
    hist_ebitda  = CH["ebitda"]["hist"]  or []
    hist_margins = CH["ebitda_margin"]["values"] or []
    base = hist_ebitda[-1] if hist_ebitda else None
    if base is None or base <= 0:
        return {"skip": "falta EBITDA base (canonical FY2025)"}

    dol = 2.5  # fallback conservador (2x–3x típico para smelters con energía propia)
    if len(hist_ebitda) >= 5 and len(hist_margins) >= 5:
        e23 = hist_ebitda[-3]; m23 = hist_margins[-3]
        e24 = hist_ebitda[-2]; m24 = hist_margins[-2]
        if e23 and e24 and m23 and m24 and m23 > 0 and m24 > 0 and e23 > 0:
            r23 = e23 / m23
            r24 = e24 / m24
            dr = r24 - r23
            if dr > 0 and r23 > 0:
                pct_e = (e24 - e23) / e23
                pct_r = dr / r23
                dol = max(1.5, min(pct_e / pct_r, 5.0))

    # Nota (auditoria ronda 4, BAJA): 0.85 es un supuesto de mix exportador
    # (no viene de static_inputs.json ni canonical_financials.json, que no
    # tienen un campo de mix exportador dedicado) -- se declara explicitamente
    # en el pie de la matriz (elast_lme/elast_vol) para que no quede implicito.
    lme_rev_share = 0.85  # supuesto: fraccion de ingresos exportables sensibles al LME
    elast_lme = dol * lme_rev_share
    elast_vol  = dol * 1.00

    lme = np.linspace(-0.30, 0.30, 7)
    vol = np.linspace(-0.20, 0.20, 7)
    grid = np.maximum(0, np.array(
        [[base * (1 + elast_lme * dl + elast_vol * dv) for dl in lme] for dv in vol]))

    i_base = len(vol) // 2; j_base = len(lme) // 2

    box_lines = [
        "Cada celda: EBITDA (USD MM)",
        "ante shocks simultáneos de",
        "precio LME y volumen vendido.",
        "",
        f"DOL empírico: {dol:.2f}x (2023→24,",
        "apalancamiento operativo real).",
        "",
        "Recuadro negro = caso base",
        "(sin shock, EBITDA FY2025).",
        "",
        "Verde = mayor EBITDA; rojo =",
        "menor, vs. rango de la grilla.",
    ]
    fig, ax = _scaffold_matrix(
        "ALUAR tiene alto apalancamiento operativo: el LME amplifica el EBITDA",
        f"EBITDA (USD MM) ante shocks de LME y volumen — DOL empírico: {dol:.2f}x",
        "USD MM", "DOL empírico 2023→2024 (Dumrauf Cap. 9)",
        "Cómo leer esta matriz", box_lines,
        f"elast_LME={elast_lme:.2f}x; elast_Vol={elast_vol:.2f}x.")
    im = ax.imshow(grid, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(lme)))
    ax.set_xticklabels([f"{v:.0%}" for v in lme], fontsize=SZ["tick"]-1)
    ax.set_yticks(range(len(vol)))
    ax.set_yticklabels([f"{v:.0%}" for v in vol], fontsize=SZ["tick"]-1)
    ax.set_xlabel("Variación LME", fontsize=SZ["axis"])
    ax.set_ylabel("Variación Volumen", fontsize=SZ["axis"])
    for i in range(len(vol)):
        for j in range(len(lme)):
            fw = "bold" if (i == i_base and j == j_base) else "normal"
            ax.text(j, i, f"{grid[i,j]:,.0f}", ha="center", va="center",
                    fontsize=6.8, color=C["ink"], fontweight=fw)
    # Fix (auditoria ronda 4, ALTA): edgecolor="white" era inconsistente con
    # el recuadro de caso base "hermano" (s33_sensitivity_wacc_g, misma
    # funcion _scaffold_matrix(), usa negro) y ademas un color crudo fuera de
    # C[...]. Se estandariza a C["ink"] (negro institucional) en ambos, buen
    # contraste contra el colormap RdYlGn en cualquiera de sus extremos.
    from matplotlib.patches import Rectangle as _Rect2
    ax.add_patch(_Rect2((j_base-0.47, i_base-0.47), 0.94, 0.94,
                        fill=False, edgecolor=C["ink"], lw=LW["heavy"], zorder=5))
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="EBITDA USD MM", shrink=0.85, pad=0.02)
    return export(fig, "s53_sensitivity_lme_vol", outdir)


# ── Sensibilidad: Tornado, CDF Monte Carlo y Espacio de Fase Cholesky ────────
# Hallazgo de auditoría (script externo 01_ingesta_y_riesgo_aluar.ipynb,
# "v14.0 Golden Copy", jun-21 -- prototipo abandonado, output HTML, nunca
# conectado al pipeline PPTX vigente): tenía 3 TIPOS de gráfico que este
# registro no tenía (Tornado, CDF comparativa, espacio de fase Cholesky) --
# genuinamente valiosos como TIPO de visualización, pero ese script tenía
# bugs reales confirmados al ejecutarlo (precios de acción negativos por
# falta del piso de Merton 1974, un motor "base" con varianza casi nula,
# multiplicadores de tornado hardcodeados sin trazabilidad). Se portan los
# 3 TIPOS de gráfico, no el motor: recalculan el MISMO puente EV→Equity→
# Precio de M6 (tornado) o leen las MISMAS muestras ya simuladas y
# persistidas por M7 (CDF, espacio de fase) -- cero fórmulas nuevas, cero
# motor paralelo, paleta 100% MASTER_STYLE.

def plot_sensitivity_tornado(CH, outdir=None):
    """Tornado: recalcula el mismo puente EV→Equity→Precio de M6 bajo un
    shock relativo de ±10% en cada variable clave (WACC, g terminal, Dólar
    CCL, Lambda vía el canal Ke→WACC), ceteris paribus. Barras coloreadas
    por el SIGNO DEL RESULTADO (verde=sube el precio, rojo=baja) -- no por
    si el shock en sí es "bueno" o "malo" (ambiguo para el CCL: una
    devaluación no es claramente favorable en términos reales), consistente
    con el contrato de color institucional (verde=creación de valor,
    rojo=riesgo)."""
    dcf = CH.get("dcf") or {}
    wc  = CH.get("wacc") or {}
    fcff_years = (CH.get("fcff") or {}).get("values")
    ccl = CH.get("ccl")
    required = [dcf.get("net_debt"), dcf.get("fcff_terminal"), dcf.get("shares_mm"),
                dcf.get("g_terminal"), wc.get("wacc"), wc.get("rf"), wc.get("beta_l"),
                wc.get("erp"), wc.get("crp"), wc.get("lambda_ar"), wc.get("e_v_target"),
                wc.get("d_v_target"), wc.get("kd_after_tax"), ccl, fcff_years]
    if any(v is None for v in required):
        return {"skip": "faltan insumos para el tornado de sensibilidad (M5/M6)"}

    net_debt, fcff_t, shares = dcf["net_debt"], dcf["fcff_terminal"], dcf["shares_mm"]
    g_base, wacc_base = dcf["g_terminal"], wc["wacc"]
    n_years = len(fcff_years)

    def price_at(wacc, g, ccl_v=ccl):
        pv_fc = sum(fcff_years[i] / (1 + wacc) ** (i + 0.5) for i in range(n_years))
        tv    = fcff_t * (1 + g) / (wacc - g)
        pv_tv = tv / (1 + wacc) ** (n_years - 0.5)
        equity = (pv_fc + pv_tv) - net_debt
        return (equity / shares) * ccl_v

    target_base = price_at(wacc_base, g_base)
    if target_base == 0:
        return {"skip": "target_base=0, no se puede normalizar el tornado"}
    SHOCK = 0.10  # +/-10% relativo, mismo criterio que el resto del anexo de sensibilidad

    def d(new_price):
        return (new_price / target_base - 1) * 100

    def wacc_with_lambda(lam):
        ke_new = wc["rf"] + wc["beta_l"] * wc["erp"] + lam * wc["crp"]
        return wc["e_v_target"] * ke_new + wc["d_v_target"] * wc["kd_after_tax"]

    lam = wc["lambda_ar"]
    rows = [
        ("WACC",                 d(price_at(wacc_base * (1 - SHOCK), g_base)),
                                  d(price_at(wacc_base * (1 + SHOCK), g_base))),
        ("g terminal",           d(price_at(wacc_base, g_base * (1 - SHOCK))),
                                  d(price_at(wacc_base, g_base * (1 + SHOCK)))),
        ("Dólar CCL",            d(price_at(wacc_base, g_base, ccl * (1 - SHOCK))),
                                  d(price_at(wacc_base, g_base, ccl * (1 + SHOCK)))),
        ("Lambda (riesgo país)", d(price_at(wacc_with_lambda(lam * (1 - SHOCK)), g_base)),
                                  d(price_at(wacc_with_lambda(lam * (1 + SHOCK)), g_base))),
    ]
    rows.sort(key=lambda r: abs(r[1]) + abs(r[2]))
    labels  = [r[0] for r in rows]
    shock_m = [r[1] for r in rows]
    shock_p = [r[2] for r in rows]

    fig, ax = scaffold(
        "El Target Price es más sensible al Dólar CCL y al WACC que a Lambda",
        "Impacto en el precio objetivo (DCF) ante un shock de ±10% en cada variable, ceteris paribus",
        "% vs. Target Price base", "M5/M6 (recálculo del puente EV→Equity→Precio)",
        "barras coloreadas por el signo del resultado (verde=sube el precio, rojo=baja), no por si el shock es 'favorable'.")
    y = np.arange(len(labels))
    for yi, sm, sp in zip(y, shock_m, shock_p):
        ax.barh(yi, sm, color=(C["value"] if sm >= 0 else C["risk"]), height=0.5, zorder=3)
        ax.barh(yi, sp, color=(C["value"] if sp >= 0 else C["risk"]), height=0.5, zorder=3)
        for v in (sm, sp):
            off = 1.5 if v >= 0 else -1.5
            ax.text(v + off, yi, f"{v:+.1f}%", va="center",
                    ha=("left" if v >= 0 else "right"), fontsize=SZ["annot"],
                    fontweight="bold", color=(C["value"] if v >= 0 else C["risk"]))
    ax.axvline(0, color=C["ink"], lw=LW["thin"])
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=SZ["tick"])
    xmax = max(abs(v) for v in shock_m + shock_p) * 1.4
    ax.set_xlim(-xmax, xmax)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:+.0f}%"))
    ax.grid(axis="y", visible=False); _clean_y(ax)
    return export(fig, "s_sensibilidad_tornado", outdir)


def plot_mc_cdf(CH, outdir=None):
    """Función de Distribución Acumulada (CDF) empírica del Monte Carlo --
    más rigurosa que un histograma para leer la cola de riesgo (percentiles
    exactos, sin depender del ancho de bin). Misma muestra que alimenta
    s34_monte_carlo (mc_results.npy) -- ninguna re-simulación aparte."""
    sample = (CH.get("mc") or {}).get("sample")
    if not sample:
        return {"skip": "falta mc.sample (mc_results.npy)"}
    arr = np.sort(np.asarray(sample, dtype=float))
    n = len(arr)
    if n < 100:
        return {"skip": f"muestra insuficiente ({n})"}
    cdf = np.arange(1, n + 1) / n
    p10, p50, p90 = (float(np.percentile(arr, p)) for p in (10, 50, 90))
    mkt = CH.get("market")

    fig, ax = scaffold(
        "La cola izquierda de la CDF cuantifica el riesgo de insolvencia (piso Merton)",
        "Función de Distribución Acumulada empírica del Monte Carlo",
        "Probabilidad acumulada", "M7 (mc_results.npy)",
        "el escalón cerca de 0 es la masa puntual del piso de equity (Merton 1974, equity=max(0,EV-deuda)).")
    ax.plot(arr, cdf, color=C["navy"], lw=LW["heavy"], zorder=3)
    ax.axvline(p10, color=C["risk"],  lw=LW["medium"], ls="--", label=f"P10=${p10:,.0f}")
    ax.axvline(p50, color=C["aluar"], lw=LW["medium"], ls="--", label=f"P50=${p50:,.0f}")
    ax.axvline(p90, color=C["value"], lw=LW["medium"], ls="--", label=f"P90=${p90:,.0f}")
    if mkt:
        ax.axvline(mkt, color=C["ink"], lw=LW["medium"], ls=":", label=f"Mercado=${mkt:,.0f}")
    ax.legend(fontsize=SZ["legend"], frameon=False, loc="lower right")
    ax.set_ylim(0, 1)
    # Fix (revisión visual): el 0.5% superior de la cola derecha (fat-tail de
    # los saltos de Merton) estira el eje X hasta >10.000 dejando ~60% del
    # lienzo vacío -- se recorta la VISTA a p99.5 (la curva sigue calculada
    # sobre el 100% de la muestra, solo se acota el rango visible), mismo
    # criterio de recorte ya usado en el histograma de M7 (percentil 99).
    ax.set_xlim(0, float(np.percentile(arr, 99.5)))
    ax.set_xlabel("Precio objetivo ARS (por acción)", fontsize=SZ["axis"], color=C["ink"])
    _clean_y(ax, pct=True)
    return export(fig, "s_mc_cdf", outdir)


def plot_cholesky_phase_space(CH, outdir=None):
    """Espacio de fase de los shocks correlacionados (Cholesky) del Monte
    Carlo -- visualiza directamente la correlación WACC↔g asumida en M7,
    usando los MISMOS wacc_sim/g_sim ya simulados y persistidos por M7
    (mc_corr_sample.npy) -- sin una segunda simulación aparte que pueda
    desincronizarse del motor real."""
    corr = (CH.get("mc") or {}).get("corr_sample")
    if not corr or not corr.get("wacc") or not corr.get("g"):
        return {"skip": "falta mc.corr_sample (mc_corr_sample.npy, generado por M7)"}
    wacc_s = np.asarray(corr["wacc"]) * 100
    g_s    = np.asarray(corr["g"]) * 100
    if len(wacc_s) < 50:
        return {"skip": f"muestra insuficiente ({len(wacc_s)})"}
    corr_emp = float(np.corrcoef(wacc_s, g_s)[0, 1])

    fig, ax = scaffold(
        f"La correlación WACC↔g asumida (ρ={corr_emp:+.2f}) se refleja en la simulación",
        "Espacio de fase de los shocks correlacionados (Cholesky), muestra de simulaciones válidas",
        "", "M7 (mc_corr_sample.npy)",
        "cada punto es una simulación válida (post-guard WACC>g+2%); la recta visualiza la correlación asumida.")
    ax.scatter(wacc_s, g_s, s=8, color=C["blue_lt"], alpha=0.35, edgecolor="none", zorder=2)
    slope, intercept = np.polyfit(wacc_s, g_s, 1)
    x_line = np.linspace(wacc_s.min(), wacc_s.max(), 50)
    ax.plot(x_line, slope * x_line + intercept, color=C["risk"], lw=LW["bold"], zorder=3,
            label=f"Regresión (ρ={corr_emp:+.2f})")
    ax.set_xlabel("WACC simulado (%)", fontsize=SZ["axis"], color=C["ink"])
    ax.set_ylabel("g terminal simulado (%)", fontsize=SZ["axis"], color=C["ink"])
    ax.legend(fontsize=SZ["legend"], frameon=False, loc="upper right")
    _clean_y(ax)
    return export(fig, "s_cholesky_phase_space", outdir)


# Registro central de TODOS los gráficos (slide → función).
CHART_REGISTRY = {
    "s07_top_producers_global": plot_top_producers_global,
    "s09_lme_vs_dxy": plot_lme_vs_dxy, "s10_macro_argentina": plot_macro_argentina,
    "s11_embi_compression": plot_embi_compression, "s11_merval_pe": plot_merval_pe,
    "s15_energy_mix": plot_energy_mix, "s16_cost_curve": plot_cost_curve,
    "s17_ebitda_hist_proj": plot_ebitda_hist_proj, "s18_balance_structure": plot_balance_structure,
    "s18b_fcff_bridge_annual": plot_fcff_bridge_annual, "s20_wacc_decomposition": plot_wacc_decomposition,
    "s20b_ke_lambda_vs_conv": plot_ke_lambda_vs_conv,
    "s20c_capex_shrinkage": plot_capex_shrinkage, "s21_roic_vs_wacc": plot_roic_vs_wacc,
    "s22_fcff_projection": plot_fcff_projection, "s23_dcf_waterfall": plot_dcf_waterfall,
    "s25_ebitda_margin": plot_ebitda_margin, "s25_cum_returns": plot_cum_returns,
    "s27_market_share_regional": plot_market_share_regional, "s27_market_share_pies": plot_market_share_pies,
    "s28_peer_multiples": plot_peer_multiples, "s30_returns_diagnostics": plot_returns_diagnostics,
    "s31_beta_architecture": plot_beta_architecture, "s33_sensitivity_wacc_g": plot_sensitivity_wacc_g,
    "s34_monte_carlo": plot_monte_carlo, "s34b_stochastic_paths": plot_stochastic_paths,
    "s36_var_cvar": plot_var_cvar, "s37_football_field": plot_football_field,
    "s39_efficient_frontier": plot_efficient_frontier, "s45_correlation_matrix": plot_correlation_matrix,
    "s53_sensitivity_lme_vol": plot_sensitivity_lme_vol,
    "s_variance_decomposition": plot_variance_decomposition,
    "s_sensibilidad_tornado": plot_sensitivity_tornado,
    "s_mc_cdf": plot_mc_cdf,
    "s_cholesky_phase_space": plot_cholesky_phase_space,
}


## Exportación y pipeline maestro

In [19]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA · EXPORTACIÓN + PIPELINE MAESTRO                                     ║
# ║  Mecanismos listos. NADA se ejecuta salvo que Claude Code ponga los flags. ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Generación de TODOS los gráficos (PNG+SVG+PDF) ────────────────────────────
def export_all_figures(workdir=None):
    """Renderiza todos los gráficos del CHART_REGISTRY desde los outputs."""
    CH = build_chart_inputs(workdir)
    results = {}
    for name, fn in CHART_REGISTRY.items():
        try:
            results[name] = fn(CH, outdir=CONFIG["figdir"])
        except Exception as e:
            results[name] = {"error": str(e)}
    done = sum(1 for r in results.values() if "png" in r)
    skip = {k: v.get("skip") for k, v in results.items() if v.get("skip")}
    # Fix (auditoria ronda 4, ALTA): v.get("skip") es None para un error
    # REAL (queda como {"error": str(e)}) -- un crash autentico de un
    # grafico nunca entraba al reporte de "pendientes", pese a que el
    # string del error ya estaba disponible en el dict. Se reportan aparte.
    errors = {k: v.get("error") for k, v in results.items() if v.get("error")}
    print(f"[FIGURAS] generadas: {done}/{len(CHART_REGISTRY)}")
    if skip:
        print("[FIGURAS] pendientes de dato (completar static_inputs.json u outputs):")
        for k, v in skip.items():
            print(f"   - {k}: {v}")
    if errors:
        print("[FIGURAS] ERRORES reales (crash de la función, no falta de dato):")
        for k, v in errors.items():
            print(f"   ! {k}: {v}")
    # Fix (Directiva 4.C -- Inferencia Narrativa Desconectada): algunos
    # graficos (ej. s37_football_field) devuelven metadatos propios ademas
    # de las rutas png/svg/pdf (conteos, titulos data-driven ya calculados).
    # Se persiste 'results' a disco para que update_presentation_v10.py lea
    # esos metadatos como UNICA fuente de verdad al armar texto relacionado
    # en el PPTX, en vez de reimplementar la misma logica de clasificacion
    # una segunda vez (riesgo de divergencia silenciosa entre grafico y texto).
    _wd = workdir or CONFIG["workdir"]
    with open(os.path.join(_wd, "figures_meta.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2, default=str)
    return results


# ── Excel (openpyxl): supuestos + ratios + proyecciones + valuación ───────────
def export_excel(workdir=None, fname="ALUAR_modelo.xlsx"):
    """Construye el Excel con todas las hojas desde los outputs. Requiere openpyxl."""
    import openpyxl
    from openpyxl.utils.dataframe import dataframe_to_rows
    inp = Inputs.from_outputs(workdir)
    wb = openpyxl.Workbook(); wb.remove(wb.active)
    # Hoja de supuestos (desde CONFIG + m1/m5)
    ws = wb.create_sheet("0) Supuestos")
    for k, v in {**CONFIG, **(inp.get("m5", default={}))}.items():
        if isinstance(v, (int, float, str, bool)):
            ws.append([k, v])
    # Ratios por año (motor de ratios)
    can = inp.get("can", default={})
    if can:
        years = [int(k[2:]) for k in can if k.startswith("FY")]
        # Fix (auditoria jul-09): market_by_year=None dejaba TODA la categoria
        # "bursatiles" (7 ratios) y la mitad de "creacion_valor" (2 ratios) en
        # blanco para los 6 anios -- mkt_cap/EV/WACC actuales SI existen (m6/m5),
        # simplemente nunca se le pasaban al motor de ratios. Se wirea el
        # ULTIMO anio historico (misma base que usa m6 para sus propios
        # multiplos "de mercado") -- los anios anteriores quedan sin market
        # data real (no se fabrica un market cap historico que no existe en
        # el pipeline).
        m5d, m6d = inp.get("m5", default={}), inp.get("m6", default={})
        market_by_year = {}
        # mkt_cap/ev en USD MM (auditoria jul-09): compute_all_ratios() fue
        # corregido (celda 3) para que rdo_neto/pn/fco/ic/activo/deuda_fin
        # prefieran consistentemente "_usdmm" (moneda base del DCF, igual que
        # ebit/ebitda/ventas/capex/deuda_neta/nopat) -- mkt_cap/ev ya no
        # necesitan convertirse a ARS crudos para calzar unidades.
        if years and m6d.get("multiplos", {}).get("mkt_cap_usdmm") is not None:
            last_yr = max(years)
            # Fix (auditoria ronda 5, MEDIA -- excel sync): "dividendos" nunca
            # se pasaba al motor de ratios pese a que canonical_financials.json
            # SI trae dividendos_ars para los 6 anios -- bursatiles.dividend_yield
            # quedaba None todos los anios sin que faltara el dato real.
            # dividendos_ars se guarda NEGATIVO (convencion de flujo de caja,
            # mismo criterio que capex_ars) -- abs() para un yield positivo,
            # convertido a USD MM via ccl_cierre del ultimo anio historico.
            div_ars = can.get(f"FY{last_yr}", {}).get("dividendos_ars")
            ccl_last = can.get(f"FY{last_yr}", {}).get("ccl_cierre")
            div_usdmm = abs(div_ars) / (ccl_last * 1e6) if (div_ars is not None and ccl_last) else None
            market_by_year[str(last_yr)] = {
                "mkt_cap":    m6d["multiplos"]["mkt_cap_usdmm"],
                "ev":         m6d.get("ev_usdmm"),
                "wacc":       m5d.get("wacc"),
                "dividendos": div_usdmm,
            }
        rt = ratios_timeseries(can, years, market_by_year=market_by_year)
        ws2 = wb.create_sheet("1) Ratios")
        ws2.append(["Año", "Categoría", "Ratio", "Valor"])
        for yr, cats in rt.items():
            for cat, d in cats.items():
                for ratio, val in d.items():
                    ws2.append([yr, cat, ratio, val])
    # Proyecciones y valuación
    # Fix (auditoria ronda 4, ALTA): el filtro isinstance(v,(int,float,str,bool))
    # descartaba TODO dict/list -- m4["historical"]/["projections"]/
    # ["roic_historico"] (todos dicts) quedaban 100% excluidos, vaciando la
    # hoja "2) Proyecciones" de su contenido real. Se aplana un nivel de
    # dict/list en filas legibles en vez de descartarlos en silencio.
    def _append_flat(ws, key, value, prefix=""):
        label = f"{prefix}{key}" if prefix else key
        if isinstance(value, dict):
            for k2, v2 in value.items():
                _append_flat(ws, k2, v2, prefix=f"{label}.")
        elif isinstance(value, list):
            for i, v2 in enumerate(value):
                if isinstance(v2, (dict, list)):
                    _append_flat(ws, str(i), v2, prefix=f"{label}[]. ")
                else:
                    ws.append([f"{label}[{i}]", v2])
        elif isinstance(value, (int, float, str, bool)) or value is None:
            ws.append([label, value])

    for key, sheet in [("m4", "2) Proyecciones"), ("m6", "3) Valuación DCF"),
                       ("m7", "4) Monte Carlo")]:
        d = inp.get(key, default={})
        if d:
            ws3 = wb.create_sheet(sheet)
            for k, v in d.items():
                if k == "figures":
                    continue  # rutas de archivo PNG, no aportan al Excel
                _append_flat(ws3, k, v)
    path = os.path.join(workdir or CONFIG["workdir"], fname)
    wb.save(path); print(f"[EXCEL] {path}")
    return path


# ── PowerPoint (python-pptx): inserta cada figura en su slide ─────────────────
# AUDITORÍA (Fase 3): antes se fijaba width=9.2in para TODAS las imágenes sin
# mirar su aspect ratio real — cada PNG tenía un figsize distinto (12x6, 10x7,
# 12x5, ...), así que la altura resultante variaba slide a slide sin ningún
# criterio, y en el peor caso (aspect ratio muy vertical) el margen inferior
# se podía perder. Ahora se lee el tamaño real del PNG con Pillow, se calcula
# el rectángulo más grande que entra en el área útil del slide preservando el
# aspect ratio exacto de la imagen, y se centra horizontal y verticalmente.
# Slide en 16:9 (13.333x7.5in), estándar de deck institucional/IB.
# BUG detectado (auditoria jul-09, SEVERO): esta funcion escribia DIRECTO a
# "ALUAR_tesis.pptx" -- el MISMO nombre del entregable oficial que
# update_presentation_v10.py produce (via TEMPLATE + tags) un momento
# despues en la secuencia de run_pipeline.py. Confirmado en el log de la
# ultima corrida real: "[PPTX] ...ALUAR_tesis.pptx (30 slides, 16:9...)"
# se ejecuta en CADA pipeline, generando un deck naive (una imagen por
# slide, SIN template/tags/texto) que es pura computacion desperdiciada
# porque update_presentation_v10.py lo sobreescribe de inmediato despues.
# El riesgo real: si update_presentation_v10.py fallara A MITAD DE CAMINO
# (excepcion antes de prs.save(DST)), el archivo naive de ESTA funcion
# quedaria en disco como si fuera la tesis final -- .claude/rules/
# hard_write_gate.md exige que el pipeline nunca deje un binario a medio
# escribir haciendose pasar por el entregable. Se renombra el default (ya
# no puede colisionar aunque se llame sin argumentos) y se saca del
# auto-run de do_exports (ver run_all) -- sigue disponible para uso manual.
def export_pptx(workdir=None, fname="ALUAR_preview_raw_NO_USAR_COMO_ENTREGABLE.pptx"):
    """Arma un deck DE PREVIEW (una imagen por slide, sin template/tags) --
    NO es el entregable oficial (ese lo produce update_presentation_v10.py
    sobre el TEMPLATE con tags). Requiere python-pptx y Pillow."""
    from pptx import Presentation
    from pptx.util import Inches, Emu
    from PIL import Image

    figdir = CONFIG["figdir"]
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank = prs.slide_layouts[6]

    margin = Inches(0.5)
    max_w = prs.slide_width - 2 * margin
    max_h = prs.slide_height - 2 * margin

    n_placed = 0
    for name in CHART_REGISTRY:
        png = os.path.join(figdir, f"{name}.png")
        if not os.path.exists(png):
            continue
        with Image.open(png) as im:
            img_w_px, img_h_px = im.size
        aspect = img_w_px / img_h_px

        # Ajustar al área útil manteniendo el aspect ratio real del PNG.
        w, h = max_w, Emu(int(max_w / aspect))
        if h > max_h:
            h, w = max_h, Emu(int(max_h * aspect))

        left = int((prs.slide_width - w) / 2)
        top = int((prs.slide_height - h) / 2)

        slide = prs.slides.add_slide(blank)
        slide.shapes.add_picture(png, left, top, width=w, height=h)
        n_placed += 1

    path = os.path.join(workdir or CONFIG["workdir"], fname)
    prs.save(path)
    print(f"[PPTX] {path} ({n_placed} slides, 16:9, aspect ratio preservado)")
    return path


# ── Split a 13 módulos .py (cada módulo se exporta como archivo) ──────────────
def split_modules(workdir=None):
    """Mecanismo para emitir m1.py..m13.py desde las funciones run_* del notebook.
    Implementado como hook; Claude Code lo invoca si necesita los módulos sueltos."""
    print("[SPLIT] Disponible. Cada celda M1–M13 ya es un módulo lógico autocontenido.")
    return True


# ── ORQUESTACIÓN — orden de dependencias CORREGIDO ────────────────────────────
# Dependencias reales: M5(WACC) requiere M1; M4 requiere M1+M5; M6 requiere M1,M4,M5;
# M7 requiere M4,M5,M6; M8 indep.; M9 requiere M1,M5,M6; M10 requiere M1,M7;
# M11 indep.; M12 requiere M1,M8; M13 requiere M1,M4,M5,M6,M7.
# M3 retirado (auditoria jul-09): M3_run() era codigo muerto, ver celda 10.
PIPELINE_ORDER = ["M1", "M2", "M5", "M4", "M6", "M7",
                  "M8", "M9", "M10", "M11", "M12", "M13"]


def run_all(do_exports=None):
    """Ejecuta los 13 módulos en orden de dependencia y luego las exportaciones.
    Sólo corre si CONFIG['run_pipeline'] es True (lo activa Claude Code).

    AUDITORÍA (Fase 4): antes de exportar/graficar, corre
    validate_financial_arrays() (definida en module13_synthesis.py) sobre los
    m*_out.json y mc_results.npy ya persistidos. Si detecta NaN/Inf, campos
    requeridos ausentes o parámetros fuera de rango físico, levanta
    ValueError y el pipeline se detiene ANTES de generar un solo gráfico —
    nunca se grafica sobre datos corruptos."""
    if not CONFIG.get("run_pipeline"):
        print("[PIPELINE] run_pipeline=False → no se ejecuta. "
              "Claude Code debe poner CONFIG['run_pipeline']=True.")
        return
    runners = {"M1": M1_run, "M2": M2_run, "M4": M4_run, "M5": M5_run,
               "M6": M6_run, "M7": M7_run, "M8": M8_run, "M9": M9_run, "M10": M10_run,
               "M11": M11_run, "M12": M12_run, "M13": M13_run}
    for m in PIPELINE_ORDER:
        print(f"\n===== {m} =====")
        runners[m]()
    inject_lineage_metadata()
    do_exports = CONFIG.get("run_exports") if do_exports is None else do_exports
    if do_exports:
        import sys as _sys
        if CONFIG["workdir"] not in _sys.path:
            _sys.path.insert(0, CONFIG["workdir"])
        from module13_synthesis import validate_financial_arrays
        validate_financial_arrays(CONFIG["workdir"])  # levanta ValueError y frena acá si hay datos corruptos

        export_all_figures()
        # Fix (auditoria ronda 5, MEDIA -- excel sync): el print-only de
        # ronda 4 evitaba que un fallo de export_excel() (ej. archivo
        # bloqueado por tenerlo abierto, KeyError) tumbara el pipeline --
        # run_pipeline.py solo verifica existencia/mtime de ALUAR_tesis.pptx,
        # nunca de ALUAR_modelo.xlsx, asi que el proceso podia terminar con
        # exit 0 dejando el Excel silenciosamente desactualizado respecto a
        # los JSON recien generados. Se relanza para que el fallo sea
        # bloqueante (mismo criterio que validate_financial_arrays arriba).
        _excel_path = export_excel()
        if not os.path.exists(_excel_path):
            raise RuntimeError(
                f"export_excel() no levanto excepcion pero {_excel_path} no "
                f"existe en disco -- Excel quedaria desactualizado en silencio.")
        # export_pptx() NO se auto-corre (auditoria jul-09): ver docstring/nota
        # de la funcion -- update_presentation_v10.py (invocado por separado en
        # run_pipeline.py) es el UNICO productor autorizado de ALUAR_tesis.pptx.
        # Disponible para uso manual/preview si hace falta: export_pptx().


# No se auto-ejecuta. Claude Code: poner los flags y llamar run_all().
if CONFIG.get("run_pipeline"):
    run_all()



===== M1 =====
M1 v3 — INGESTION DATOS DE MERCADO (PARAMETROS AUDITADOS)


  rf = 4.5690% | ^TNX yfinance — US10Y último cierre


  CCL = 1,551.27 ARS/USD | GGAL.BA=8335.00×10 / GGAL=53.73 = 1551.27
  ERP = 4.1800% | Damodaran (NYU Stern), ERP implícito USA, julio-2026
  EMBI = 4.4100% | JPMorgan EMBI+ Argentina (proxy Bloomberg/Reuters) -- snapshot manual, actualizar periodicamente
  Lambda = 0.2 | Damodaran Lambda — CONFIG['lambda_ar']=0.2 (FALLBACK documentado, editar en celda CONFIG)


  Beta OLS = 0.8469 (SE=0.0564, n=2377) | OLS 5y daily ALUA.BA (USD via CCL) vs ^GSPC n=2377


  LME Aluminium = USD 3,304/Tn | ALI=F yfinance — LME Aluminium futures


...


  Kelly ALUA: f=0.3810 (38.1%), Half-Kelly=19.0%

[OK] m12_out.json + 2 figuras guardadas.

===== M13 =====
M13 SYNTHESIS — Dictamen Institucional + Graficos Completos



  ╔══════════════════════════════════════════╗
  ║  DICTAMEN: VENDER                          ║
  ║  Precio objetivo:    579 ARS              ║
  ║  Precio mercado:     976 ARS              ║
  ║  Prima DCF:       -40.7%                ║
  ║  P(upside MC):   19.9%                  ║
  ╚══════════════════════════════════════════╝

  [NOTA METODOLÓGICA DE MONTE CARLO]
  La mediana (P50) de Monte Carlo es menor al Target Price Caso Base debido a la asimetría de la Gordon Growth (1/(WACC-g)), que es una función no lineal hiperbólica. Además, el truncamiento a cero de escenarios insolventes (equity como call option - Merton 1974) absorbe las colas extremas de deuda neta alta, sesgando la mediana respecto al caso base.

[OK] m13_out.json + 11 figuras guardadas.


[AUDITORÍA DE DATOS] OK — 9 archivos validados sin NaN/Inf/outliers: ['m1_out.json', 'm2_out.json', 'm4_out.json', 'm5_out.json', 'm6_out.json', 'm7_out.json', 'm8_out.json', 'm13_out.json', 'mc_results.npy']


[FIGURAS] generadas: 34/34


[AVISO] Categoria 'liquidez' 100% NaN: activo_corriente/pasivo_corriente no existen en canonical_financials.json.
[EXCEL] C:\Users\fedea\Valuacion\ALUAR_modelo.xlsx


## Esquema de `static_inputs.json` (datos de Memoria / industria)

Algunos gráficos usan datos que NO provienen de APIs sino de la Memoria Anual o de
la industria (no se fabrican). Proveer este archivo en `CONFIG['workdir']` con claves:

- `ccl`: float
- `energy_mix`: {"Hidroeléctrica": .., "Eólico": .., "Térmica (Gas)": ..}
- `cost_curve`: {"names": [...], "cash_cost": [...]}
- `market_share_regional`: {"names": [...], "share": [...]}
- `market_share_pies`: {"global": {...}, "regional": {...}}
- `peers`: {"names": [...], "ev_ebitda": [...]}
- `macro_ar`: {"years": [...], "inflacion": [...], "pbi_growth": [...]}
- `embi_hist`: {"dates": [...], "values": [...]}
- `merval_pe`: {"years": [...], "values": [...]}
- `lme_dxy`: {"dates": [...], "lme": [...], "dxy": [...]}
- `cum_returns`: {"dates": [...], "alua": [...], "merv": [...], "txar": [...]}
- `alua_returns`: [retornos diarios...]
- `correlation`: {"labels": [...], "matrix": [[...]]}
- `frontier`: {"vol": [...], "ret": [...], "sharpe": [...], "alua": {"vol":.., "ret":..}}

Las series de mercado (lme_dxy, cum_returns, alua_returns, correlation, frontier)
pueden auto-generarse haciendo que M8/M9/M11/M12 las persistan en sus JSON; en ese
caso `build_chart_inputs` las toma de ahí y `static_inputs.json` es opcional.
